# Bomberland Rank #4 Agent - Training Notebook

Tổng quan agent:

- rule-based safety core để mô phỏng bom, blast, chain reaction và escape route;
- CNN-GRU neural policy để học hành vi theo trạng thái và phase trận;
- behavior cloning từ rule agents và elite leaderboard logs;
- PPO-style curriculum/self-play để tăng độ robust;
- PFSP league để chọn đối thủ train theo độ khó;
- beam-search planner để chọn action an toàn và có pressure trong runtime.


## 1. Official Engine Setup

Cell này chuẩn bị official Bomberland engine. Khi chạy trên Kaggle, notebook
ưu tiên dùng participant kit đã attach trong `/kaggle/input`; nếu không có,
nó fallback sang GitHub official repo.

Phần này chỉ phục vụ training/evaluation trong notebook. Submission cuối
vẫn chỉ cần `submission/agent.py` và `submission/policy.pt`.


In [ ]:
# Cell 1: Nạp official Bomberland engine và các baseline agents.
# Mục tiêu:
# - Dùng participant kit nếu notebook chạy trên Kaggle có attach input.
# - Fallback sang GitHub official repo nếu không tìm thấy engine local.
# - Đưa repo vào sys.path để import BomberEnv và các rule agents.
# Ghi chú:
# - Cell này là phần setup môi trường, không chứa logic agent.
# - Khi chỉ chạy submission/agent.py, cell này không cần thiết.

# Cell 1: Nạp official Bomberland engine và baseline agents.
import os
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

REPO_URL = "https://github.com/VLTisME/Bomberland-GDGoC-AI-Challenge.git"
REPO_DIR = Path("/kaggle/working/Bomberland-GDGoC-AI-Challenge")

if not (REPO_DIR / "engine" / "game.py").exists():
    bundled = None
    for game_py in Path("/kaggle/input").rglob("participant_kit/engine/game.py"):
        kit_root = game_py.parents[1]
        bundled = kit_root
        REPO_DIR.mkdir(parents=True, exist_ok=True)
        for item in kit_root.iterdir():
            target = REPO_DIR / item.name
            if item.is_dir():
                shutil.copytree(item, target, dirs_exist_ok=True)
            else:
                target.write_bytes(item.read_bytes())
        break
    for archive_path in Path("/kaggle/input").rglob("*.zip"):
        if bundled is not None:
            break
        try:
            with zipfile.ZipFile(archive_path) as archive:
                if "participant_kit/engine/game.py" in archive.namelist():
                    bundled = archive_path
                    REPO_DIR.mkdir(parents=True, exist_ok=True)
                    for member in archive.infolist():
                        if not member.filename.startswith("participant_kit/"):
                            continue
                        relative = Path(member.filename).relative_to("participant_kit")
                        if not relative.parts or member.is_dir():
                            continue
                        target = REPO_DIR / relative
                        target.parent.mkdir(parents=True, exist_ok=True)
                        target.write_bytes(archive.read(member))
                    break
        except (OSError, zipfile.BadZipFile, KeyError):
            continue
    if bundled is None:
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

if (REPO_DIR / ".git").exists():
    commit = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
else:
    commit = "bundled-official-source"
print("Repo:", REPO_DIR)
print("Commit:", commit)


## 2. Training Configuration

Cell này gom toàn bộ cấu hình train: seed, device, baseline agents,
curriculum stages và hyperparameters.

Cách tổ chức `Stage` và `TrainConfig` giúp đổi curriculum dễ hơn:

- giai đoạn đầu ưu tiên survival và box farming;
- giai đoạn giữa học đối đầu với rule agents mạnh hơn;
- giai đoạn cuối tăng self-play và pressure để tối ưu final rank.


In [ ]:
# Cell 2: Imports, seed cố định, DEVICE và curriculum cơ bản.
# Mục tiêu:
# - Cố định random seed để quá trình train/eval ít nhiễu hơn.
# - Tự chọn GPU nếu Kaggle có CUDA, nếu không sẽ chạy CPU.
# - Khai báo Stage và TrainConfig để gom các hyperparameters chính.
# - Định nghĩa các preset train: smoke, standard, long.

# Cell 2: Imports, reproducibility settings và curriculum config.
import copy
import csv
import json
import math
import random
import time
import zipfile
from collections import deque
from dataclasses import asdict, dataclass
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Categorical

from engine.game import BomberEnv
from agent import (
    BoxFarmerAgent,
    GeniusRuleAgent,
    RandomAgent,
    SimpleRuleAgent,
    SmarterRuleAgent,
    TacticalRuleAgent,
)

SEED = 202606
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.set_float32_matmul_precision("high")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Torch:", torch.__version__)
print("Device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

@dataclass(frozen=True)
class Stage:
    name: str
    updates: int
    policy_slots_min: int
    policy_slots_max: int
    baselines: tuple
    reward_profile: str
    league_probability: float = 0.0

@dataclass
class TrainConfig:
    preset: str
    seed: int
    num_envs: int
    episodes_per_update: int
    bc_teacher_steps: int
    bc_chunk_size: int
    bc_epochs: int
    max_steps: int = 500
    gamma: float = 0.995
    gae_lambda: float = 0.95
    learning_rate: float = 3e-4
    clip_coef: float = 0.20
    ent_coef: float = 0.015
    vf_coef: float = 0.50
    max_grad_norm: float = 0.50
    ppo_epochs: int = 4
    minibatch_size: int = 1024
    eval_matches: int = 100
    checkpoint_every: int = 5
    league_snapshot_every: int = 4
    league_max_size: int = 10

PRESET = "long"  # "smoke", "standard", or "long"

PRESETS = {
    "smoke": {
        "config": TrainConfig("smoke", SEED, 2, 2, 256, 128, 1, ppo_epochs=1, minibatch_size=128, eval_matches=8),
        "stages": [
            Stage("smoke", 1, 1, 1, ("random", "simple"), "survival"),
        ],
    },
    "standard": {
        "config": TrainConfig("standard", SEED, 12, 20, 15_000, 4096, 2, eval_matches=100),
        "stages": [
            Stage("survival", 8, 1, 1, ("random", "simple", "box_farmer"), "survival"),
            Stage("mixed_baselines", 18, 1, 1, ("simple", "box_farmer", "smarter", "genius"), "balanced"),
            Stage("strong_baselines", 28, 1, 2, ("smarter", "genius", "tactical"), "competitive", 0.15),
            Stage("shared_self_play", 70, 2, 3, ("genius", "tactical"), "final", 0.60),
        ],
    },
    "long": {
        "config": TrainConfig("long", SEED, 24, 32, 30_000, 4096, 3, eval_matches=200),
        "stages": [
            Stage("survival", 12, 1, 1, ("random", "simple", "box_farmer"), "survival"),
            Stage("mixed_baselines", 28, 1, 1, ("simple", "box_farmer", "smarter", "genius"), "balanced"),
            Stage("strong_baselines", 56, 1, 2, ("smarter", "genius", "tactical"), "competitive", 0.20),
            Stage("shared_self_play", 140, 2, 3, ("smarter", "genius", "tactical"), "final", 0.65),
        ],
    },
}

CFG = PRESETS[PRESET]["config"]
CURRICULUM = PRESETS[PRESET]["stages"]

ARTIFACT_DIR = Path("/kaggle/working/bomberland_artifacts")
CHECKPOINT_DIR = ARTIFACT_DIR / "checkpoints"
DEPLOY_DIR = ARTIFACT_DIR / "submission"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
DEPLOY_DIR.mkdir(parents=True, exist_ok=True)

BASELINE_CLASSES = {
    "random": RandomAgent,
    "simple": SimpleRuleAgent,
    "box_farmer": BoxFarmerAgent,
    "smarter": SmarterRuleAgent,
    "genius": GeniusRuleAgent,
    "tactical": TacticalRuleAgent,
}

print("Preset:", PRESET)
print("Curriculum:", [stage.name for stage in CURRICULUM])


## 3. Observation Encoding Và Safety Core

Notebook xây dựng:

- `danger_planes`: map nguy hiểm theo từng tick tương lai;
- `survivability_depth`: độ sâu sống sót sau mỗi action;
- `legal_action_mask`: mask action nguy hiểm trước khi policy chọn;
- `escape_exit_count_after_bomb`: luật đặt bom cần ít nhất 2 đường thoát;
- `encode_obs`: chuyển observation thành tensor cho neural policy.

Nhờ safety core, policy có thể aggressive hơn mà vẫn hạn chế rank-3 do
self-trap.


In [ ]:
# Cell 3: Feature engineering, bomb simulation và safe-action mask.
# Mục tiêu:
# - Mô phỏng blast, bomb timer, chain reaction và future danger planes.
# - Tính survivability depth cho từng action trong horizon ngắn.
# - Áp dụng luật đặt bom an toàn: cần ít nhất MIN_BOMB_EXITS = 2 lối thoát.
# - Encode observation thành map channels + scalar features cho neural policy.
# Đây là rule-based safety core giúp policy aggressive nhưng không tự kẹt bom.

# Cell 3: Feature engineering, bomb simulation và safe-action masks.
# IMPORTANT: engine coordinates are [row, col]. Action labels in the public guide are misleading.
MOVES = {
    0: (0, 0),
    1: (-1, 0),
    2: (1, 0),
    3: (0, -1),
    4: (0, 1),
}
HORIZON = 7
ADVERSARIAL_BOMB_DISTANCE = 4
MIN_BOMB_EXITS = 2
N_CHANNELS = 19
N_SCALARS = 8
HIDDEN_SIZE = 128

# Theo dõi bomb radius theo owner vì log/env chỉ lưu bomb timer; radius cần được suy ra từ bonus của người đặt bom.
def radius_lookup_from_env(env):
    return {(int(b.x), int(b.y), int(b.owner_id)): int(b.radius) for b in env.bombs}

def in_bounds(grid, row, col):
    return 0 <= row < grid.shape[0] and 0 <= col < grid.shape[1]

def passable(grid, row, col):
    return in_bounds(grid, row, col) and int(grid[row, col]) in (0, 3, 4)

def blast_tiles(grid, row, col, radius):
    tiles = {(int(row), int(col))}
    for drow, dcol in ((-1, 0), (1, 0), (0, -1), (0, 1)):
        for distance in range(1, int(radius) + 1):
            target_row = int(row) + drow * distance
            target_col = int(col) + dcol * distance
            if not in_bounds(grid, target_row, target_col):
                break
            cell = int(grid[target_row, target_col])
            if cell == 1:
                break
            tiles.add((target_row, target_col))
            if cell == 2:
                break
    return tiles

def resolved_bombs(obs, radius_lookup=None, extra_bomb=None, extra_bombs=None):
    players = obs["players"]
    radius_lookup = radius_lookup or {}
    entries = []
    for bomb in obs["bombs"]:
        row, col, timer, owner_id = [int(value) for value in bomb]
        fallback_radius = 1
        if 0 <= owner_id < len(players):
            fallback_radius = 1 + int(players[owner_id][4])
        radius = int(radius_lookup.get((row, col, owner_id), fallback_radius))
        entries.append((row, col, timer, owner_id, radius))
    if extra_bomb is not None:
        entries.append(tuple(int(value) for value in extra_bomb))
    for bomb in extra_bombs or ():
        entries.append(tuple(int(value) for value in bomb))
    return entries

# Xây future danger planes: mỗi plane là các ô nguy hiểm tại một tick tương lai, có xét chain reaction.
def danger_planes(obs, horizon=HORIZON, radius_lookup=None, extra_bomb=None, extra_bombs=None):
    grid = obs["map"]
    bombs = resolved_bombs(
        obs,
        radius_lookup=radius_lookup,
        extra_bomb=extra_bomb,
        extra_bombs=extra_bombs,
    )
    planes = np.zeros((horizon, grid.shape[0], grid.shape[1]), dtype=np.float32)
    if not bombs:
        return planes

    effective_timers = [max(1, int(bomb[2])) for bomb in bombs]
    blasts = [blast_tiles(grid, bomb[0], bomb[1], bomb[4]) for bomb in bombs]

    # Propagate chain reactions until reaching a fixed point.
    for _ in range(len(bombs)):
        changed = False
        for source_index, source_tiles in enumerate(blasts):
            for target_index, target in enumerate(bombs):
                if source_index == target_index:
                    continue
                if (target[0], target[1]) in source_tiles and effective_timers[target_index] > effective_timers[source_index]:
                    effective_timers[target_index] = effective_timers[source_index]
                    changed = True
        if not changed:
            break

    for timer, tiles in zip(effective_timers, blasts):
        if 1 <= timer <= horizon:
            for row, col in tiles:
                planes[timer - 1, row, col] = 1.0
    return planes

def potential_enemy_bombs(obs, player_id, max_distance=ADVERSARIAL_BOMB_DISTANCE):
    players = obs["players"]
    own_row, own_col = int(players[player_id][0]), int(players[player_id][1])
    occupied = {(int(bomb[0]), int(bomb[1])) for bomb in obs["bombs"]}
    threats = []
    for enemy_id, enemy in enumerate(players):
        if enemy_id == player_id or int(enemy[2]) != 1 or int(enemy[3]) <= 0:
            continue
        row, col = int(enemy[0]), int(enemy[1])
        if (row, col) in occupied:
            continue
        if abs(row - own_row) + abs(col - own_col) <= max_distance:
            threats.append((row, col, 6, enemy_id, 1 + int(enemy[4])))
    return threats

# Ước lượng agent sống được bao lâu sau một action đầu tiên trong horizon ngắn.
def survivability_depth_after_action(obs, player_id, first_action, radius_lookup=None, stress_bomb=None):
    grid = obs["map"]
    players = obs["players"]
    row, col, alive, _, radius_bonus = [int(value) for value in players[player_id]]
    if not alive:
        return HORIZON if int(first_action) == 0 else 0

    blocked = {(int(bomb[0]), int(bomb[1])) for bomb in obs["bombs"]}
    extra_bomb = None
    if int(first_action) == 5:
        # PLACE_BOMB consumes the current turn. The engine immediately
        # ticks the new bomb from 7 to 6.
        blocked.add((row, col))
        extra_bomb = (row, col, 6, player_id, 1 + radius_bonus)
        next_row, next_col = row, col
    else:
        drow, dcol = MOVES[int(first_action)]
        next_row, next_col = row + drow, col + dcol
        if int(first_action) != 0:
            if not passable(grid, next_row, next_col) or (next_row, next_col) in blocked:
                return 0

    extra_bombs = []
    if extra_bomb is not None:
        extra_bombs.append(extra_bomb)
    if stress_bomb is not None:
        blocked.add((int(stress_bomb[0]), int(stress_bomb[1])))
        extra_bombs.append(stress_bomb)
    future = danger_planes(
        obs,
        horizon=HORIZON,
        radius_lookup=radius_lookup,
        extra_bombs=extra_bombs,
    )
    if future[0, next_row, next_col] > 0.5:
        return 0

    positions = {(next_row, next_col)}
    for tick in range(1, HORIZON):
        next_positions = set()
        for current_row, current_col in positions:
            for action in range(5):
                drow, dcol = MOVES[action]
                next_row, next_col = current_row + drow, current_col + dcol
                if action != 0:
                    if not passable(grid, next_row, next_col):
                        continue
                    if (next_row, next_col) in blocked:
                        continue
                if future[tick, next_row, next_col] > 0.5:
                    continue
                next_positions.add((next_row, next_col))
        positions = next_positions
        if not positions:
            return tick
    return HORIZON

def adversarial_survivability_depth_after_action(obs, player_id, first_action, radius_lookup=None):
    depth = survivability_depth_after_action(obs, player_id, first_action, radius_lookup)
    if depth < HORIZON:
        return depth
    for stress_bomb in potential_enemy_bombs(obs, player_id):
        depth = min(
            depth,
            survivability_depth_after_action(
                obs,
                player_id,
                first_action,
                radius_lookup,
                stress_bomb=stress_bomb,
            ),
        )
    return depth

# Đếm số hướng thoát an toàn nếu đặt bom tại vị trí hiện tại; dùng cho luật two-exit.
def escape_exit_count_after_bomb(obs, player_id, radius_lookup=None):
    grid = obs["map"]
    row, col, alive, _, radius_bonus = [int(value) for value in obs["players"][player_id]]
    if not alive:
        return 0
    blocked = {(int(bomb[0]), int(bomb[1])) for bomb in obs["bombs"]}
    blocked.add((row, col))
    future = danger_planes(
        obs,
        horizon=HORIZON,
        radius_lookup=radius_lookup,
        extra_bomb=(row, col, 6, player_id, 1 + radius_bonus),
    )
    exits = 0
    for action in range(1, 5):
        drow, dcol = MOVES[action]
        next_row, next_col = row + drow, col + dcol
        if passable(grid, next_row, next_col) and (next_row, next_col) not in blocked:
            if future[0, next_row, next_col] < 0.5:
                exits += 1
    return exits

# Mask action không hợp lệ hoặc quá nguy hiểm trước khi đưa vào neural policy.
def legal_action_mask(obs, player_id, radius_lookup=None, strict_safety=True):
    grid = obs["map"]
    players = obs["players"]
    row, col, alive, bombs_left, _ = [int(value) for value in players[player_id]]
    mask = np.zeros(6, dtype=np.bool_)
    if not alive:
        mask[0] = True
        return mask

    bomb_positions = {(int(bomb[0]), int(bomb[1])) for bomb in obs["bombs"]}
    mask[0] = True
    for action in range(1, 5):
        drow, dcol = MOVES[action]
        next_row, next_col = row + drow, col + dcol
        if passable(grid, next_row, next_col) and (next_row, next_col) not in bomb_positions:
            mask[action] = True
    if bombs_left > 0 and (row, col) not in bomb_positions:
        mask[5] = True

    if not strict_safety:
        return mask

    depths = np.full(6, -1, dtype=np.int8)
    for action in np.flatnonzero(mask):
        depths[action] = survivability_depth_after_action(
            obs,
            player_id,
            int(action),
            radius_lookup=radius_lookup,
        )

    safe_mask = mask & (depths >= HORIZON)
    if safe_mask[5]:
        if escape_exit_count_after_bomb(obs, player_id, radius_lookup) < MIN_BOMB_EXITS:
            safe_mask[5] = False
            depths[5] = -1

    # Prefer actions that still keep a survival route if a nearby enemy
    # places a bomb on the next turn. Preserve the regular safe set only
    # when every action is vulnerable to the same pressure.
    if safe_mask.any() and potential_enemy_bombs(obs, player_id):
        robust_depths = np.full(6, -1, dtype=np.int8)
        for action in np.flatnonzero(safe_mask):
            robust_depths[action] = adversarial_survivability_depth_after_action(
                obs,
                player_id,
                int(action),
                radius_lookup,
            )
        robust_mask = safe_mask & (robust_depths >= HORIZON)
        if robust_mask.any():
            safe_mask = robust_mask
    if safe_mask.any():
        return safe_mask

    # If survival is impossible, keep only actions that delay death the
    # longest. Never add a new bomb while already in an unsolved trap.
    non_bomb_mask = mask.copy()
    non_bomb_mask[5] = False
    if non_bomb_mask.any():
        return non_bomb_mask & (depths == depths[non_bomb_mask].max())
    mask[5] = False
    if mask.any():
        return mask
    mask[0] = True
    return mask

# Encode observation thành tensor: static map, players, bombs, danger planes và scalar game context.
def encode_obs(obs, player_id, step_count=0, radius_lookup=None):
    grid = obs["map"]
    players = obs["players"]
    bombs = obs["bombs"]
    height, width = grid.shape

    channels = [(grid == tile).astype(np.float32) for tile in range(5)]

    own_plane = np.zeros((height, width), dtype=np.float32)
    own = players[player_id]
    if int(own[2]) == 1:
        own_plane[int(own[0]), int(own[1])] = 1.0
    channels.append(own_plane)

    enemy_ids = [index for index in range(len(players)) if index != player_id]
    for enemy_id in enemy_ids:
        enemy_plane = np.zeros((height, width), dtype=np.float32)
        enemy = players[enemy_id]
        if int(enemy[2]) == 1:
            enemy_plane[int(enemy[0]), int(enemy[1])] = 1.0
        channels.append(enemy_plane)

    bomb_plane = np.zeros((height, width), dtype=np.float32)
    timer_plane = np.zeros((height, width), dtype=np.float32)
    radius_plane = np.zeros((height, width), dtype=np.float32)
    for row, col, timer, owner_id, radius in resolved_bombs(obs, radius_lookup=radius_lookup):
        bomb_plane[row, col] = 1.0
        timer_plane[row, col] = max(timer_plane[row, col], float(timer) / 7.0)
        radius_plane[row, col] = max(radius_plane[row, col], float(radius) / 5.0)
    channels.extend((bomb_plane, timer_plane, radius_plane))
    future_danger = danger_planes(obs, horizon=HORIZON, radius_lookup=radius_lookup)
    channels.extend(future_danger)

    alive_enemies = sum(int(players[index][2]) for index in enemy_ids)
    own_row, own_col = int(own[0]), int(own[1])
    enemy_distances = [
        abs(int(players[index][0]) - own_row) + abs(int(players[index][1]) - own_col)
        for index in enemy_ids
        if int(players[index][2]) == 1
    ]
    nearest_enemy = min(enemy_distances) if enemy_distances else 24
    bomb_positions = {(int(bomb[0]), int(bomb[1])) for bomb in bombs}
    safe_moves = 0
    for action in range(5):
        drow, dcol = MOVES[action]
        next_row, next_col = own_row + drow, own_col + dcol
        if action != 0 and (not passable(grid, next_row, next_col) or (next_row, next_col) in bomb_positions):
            continue
        if future_danger[0, next_row, next_col] < 0.5:
            safe_moves += 1
    boxes_in_range = sum(
        int(grid[row, col]) == 2
        for row, col in blast_tiles(grid, own_row, own_col, 1 + int(own[4]))
    )
    scalars = np.asarray(
        [
            float(own[3]) / 5.0,
            float(own[4]) / 4.0,
            float(alive_enemies) / 3.0,
            float(player_id) / 3.0,
            min(float(step_count), 500.0) / 500.0,
            min(float(nearest_enemy), 24.0) / 24.0,
            float(safe_moves) / 5.0,
            min(float(boxes_in_range), 4.0) / 4.0,
        ],
        dtype=np.float32,
    )
    encoded = np.stack(channels, axis=0).astype(np.float32)
    assert encoded.shape == (N_CHANNELS, 13, 13), encoded.shape
    return encoded, scalars

PLANNER_DEPTH = HORIZON
PLANNER_WIDTH = 18

def strategic_objectives(grid):
    objectives = {}
    for row, col in np.argwhere((grid == 3) | (grid == 4)):
        objectives[(int(row), int(col))] = 1.25
    for box_row, box_col in np.argwhere(grid == 2):
        for drow, dcol in MOVES.values():
            row, col = int(box_row) + drow, int(box_col) + dcol
            if passable(grid, row, col):
                objectives[(row, col)] = max(objectives.get((row, col), 0.0), 0.60)
    return tuple((row, col, value) for (row, col), value in objectives.items())

def escape_region_size(grid, start_row, start_col, blocked, future, ticks=6):
    positions = {(int(start_row), int(start_col))}
    for tick in range(min(int(ticks), future.shape[0])):
        next_positions = set()
        for row, col in positions:
            for action in range(5):
                drow, dcol = MOVES[action]
                next_row, next_col = row + drow, col + dcol
                if action != 0 and (
                    not passable(grid, next_row, next_col)
                    or (next_row, next_col) in blocked
                ):
                    continue
                if future[tick, next_row, next_col] < 0.5:
                    next_positions.add((next_row, next_col))
        positions = next_positions
        if not positions:
            return 0
    return len(positions)

# Beam-search planner: dùng policy logits làm priors rồi chấm điểm các future positions.
def beam_plan_action(
    obs,
    player_id,
    radius_lookup=None,
    policy_logits=None,
    recent_positions=None,
    action_mask=None,
    step_count=0,
):
    grid = obs["map"]
    players = obs["players"]
    row, col, alive, _, radius_bonus = [int(value) for value in players[player_id]]
    if not alive:
        return 0

    mask = (
        legal_action_mask(obs, player_id, radius_lookup=radius_lookup, strict_safety=True)
        if action_mask is None
        else action_mask
    )
    logits = np.zeros(6, dtype=np.float32) if policy_logits is None else np.asarray(policy_logits, dtype=np.float32)
    valid_logits = logits[mask]
    priors = np.zeros(6, dtype=np.float32)
    priors[mask] = np.exp(valid_logits - np.max(valid_logits))
    priors /= max(float(priors.sum()), 1e-8)

    bomb_positions = {(int(bomb[0]), int(bomb[1])) for bomb in obs["bombs"]}
    enemy_records = [
        (
            index,
            int(players[index][0]),
            int(players[index][1]),
            int(players[index][3]),
            int(players[index][4]),
        )
        for index in range(len(players))
        if index != player_id and int(players[index][2]) == 1
    ]
    enemy_positions = [(enemy_row, enemy_col) for _, enemy_row, enemy_col, _, _ in enemy_records]
    objectives = strategic_objectives(grid)
    recent_sequence = tuple(recent_positions or ())
    recent_positions = set(recent_sequence)
    current_future = danger_planes(obs, horizon=HORIZON, radius_lookup=radius_lookup)
    current_threatened = bool(np.any(current_future[:, row, col] > 0.5))
    boxes_left = int(np.count_nonzero(grid == 2))
    endgame = int(step_count) >= 260 or len(enemy_records) <= 1 or boxes_left <= 4
    closing_phase = int(step_count) >= 400
    aggression = 1.0 + 0.35 * endgame + 0.25 * closing_phase
    baseline_regions = {
        enemy_id: escape_region_size(
            grid,
            enemy_row,
            enemy_col,
            bomb_positions,
            current_future,
        )
        for enemy_id, enemy_row, enemy_col, _, _ in enemy_records
    }

    def adversarial_exposure(target_row, target_col):
        return sum(
            bombs_left > 0
            and (target_row, target_col) in blast_tiles(grid, enemy_row, enemy_col, 1 + radius_bonus)
            for _, enemy_row, enemy_col, bombs_left, radius_bonus in enemy_records
        )

    def position_score(target_row, target_col, future, blocked, tick):
        mobility = 0
        for action in range(5):
            drow, dcol = MOVES[action]
            next_row, next_col = target_row + drow, target_col + dcol
            if action != 0 and (not passable(grid, next_row, next_col) or (next_row, next_col) in blocked):
                continue
            if future[min(tick, HORIZON - 1), next_row, next_col] < 0.5:
                mobility += 1
        safe_margin = sum(future[index, target_row, target_col] < 0.5 for index in range(tick, HORIZON))
        nearest_enemy = min(
            (abs(enemy_row - target_row) + abs(enemy_col - target_col) for enemy_row, enemy_col in enemy_positions),
            default=24,
        )
        item_bonus = 0.35 if int(grid[target_row, target_col]) in (3, 4) else 0.0
        objective_bonus = max(
            (value - 0.04 * (abs(objective_row - target_row) + abs(objective_col - target_col))
             for objective_row, objective_col, value in objectives),
            default=0.0,
        )
        corridor_penalty = 0.12 if mobility <= 1 else 0.0
        pressure_penalty = 0.12 * adversarial_exposure(target_row, target_col)
        objective_scale = 0.72 if endgame else 1.0
        distance_weight = 0.018 if endgame else 0.01
        return (
            0.08 * mobility
            + 0.04 * safe_margin
            + item_bonus
            + objective_scale * objective_bonus
            - distance_weight * nearest_enemy
            - corridor_penalty
            - pressure_penalty
        )

    beams = []
    for first_action in np.flatnonzero(mask):
        first_action = int(first_action)
        drow, dcol = MOVES.get(first_action, (0, 0))
        next_row, next_col = row + drow, col + dcol
        blocked = set(bomb_positions)
        extra_bomb = None
        tactical = 0.0
        if first_action == 0 and np.any(mask[1:5]):
            tactical += 0.10 if current_threatened or len(obs["bombs"]) else (-0.34 if endgame else -0.24)
        if first_action != 5 and (next_row, next_col) in recent_positions:
            tactical -= 0.12
        if first_action != 5 and len(recent_sequence) >= 2 and (next_row, next_col) == recent_sequence[-2]:
            tactical -= 0.30
        if first_action == 5:
            next_row, next_col = row, col
            blocked.add((row, col))
            radius = 1 + radius_bonus
            extra_bomb = (row, col, 6, player_id, radius)
            blast = blast_tiles(grid, row, col, radius)
            boxes_hit = sum(int(grid[target_row, target_col]) == 2 for target_row, target_col in blast)
            enemies_hit = sum((enemy_row, enemy_col) in blast for enemy_row, enemy_col in enemy_positions)
            pressure = 0.0
            for _, enemy_row, enemy_col, _, _ in enemy_records:
                escape_options = sum(
                    passable(grid, enemy_row + drow, enemy_col + dcol)
                    and (enemy_row + drow, enemy_col + dcol) not in blast
                    for drow, dcol in MOVES.values()
                )
                covered_options = sum(
                    (enemy_row + drow, enemy_col + dcol) in blast
                    for drow, dcol in MOVES.values()
                )
                if (enemy_row, enemy_col) in blast:
                    pressure += 0.45 + 0.22 * max(0, 2 - escape_options)
                elif abs(enemy_row - row) + abs(enemy_col - col) <= 4:
                    pressure += 0.06 * covered_options
            tactical += 1.00 * boxes_hit + aggression * (1.25 * enemies_hit + pressure)
            exits = escape_exit_count_after_bomb(obs, player_id, radius_lookup)
            if exits < MIN_BOMB_EXITS:
                tactical -= 4.0
            elif exits == MIN_BOMB_EXITS and len(obs["bombs"]) >= 2:
                tactical -= 0.30
            if boxes_hit == 0 and enemies_hit == 0:
                tactical -= 0.45
        future = danger_planes(obs, horizon=HORIZON, radius_lookup=radius_lookup, extra_bomb=extra_bomb)
        if first_action == 5:
            trap_gain = 0
            forced_traps = 0
            for enemy_id, enemy_row, enemy_col, _, _ in enemy_records:
                remaining = escape_region_size(
                    grid,
                    enemy_row,
                    enemy_col,
                    blocked,
                    future,
                )
                trap_gain += max(0, baseline_regions.get(enemy_id, remaining) - remaining)
                forced_traps += int(remaining == 0)
            tactical += aggression * (0.08 * min(trap_gain, 12) + 0.90 * forced_traps)
        score = 1.20 * float(priors[first_action]) + tactical
        score += position_score(next_row, next_col, future, blocked, 0)
        beams.append((score, next_row, next_col, first_action, blocked, future))

    for tick in range(1, PLANNER_DEPTH):
        expanded = []
        for score, current_row, current_col, first_action, blocked, future in beams:
            for action in range(5):
                drow, dcol = MOVES[action]
                next_row, next_col = current_row + drow, current_col + dcol
                if action != 0 and (not passable(grid, next_row, next_col) or (next_row, next_col) in blocked):
                    continue
                if future[min(tick, HORIZON - 1), next_row, next_col] > 0.5:
                    continue
                next_score = score + position_score(next_row, next_col, future, blocked, tick)
                expanded.append((next_score, next_row, next_col, first_action, blocked, future))
        if not expanded:
            break
        beams = sorted(expanded, key=lambda item: item[0], reverse=True)[:PLANNER_WIDTH]

    if beams:
        return int(max(beams, key=lambda item: item[0])[3])
    return int(np.flatnonzero(mask)[np.argmax(priors[mask])])

def safe_fallback(obs, player_id, radius_lookup=None):
    mask = legal_action_mask(obs, player_id, radius_lookup=radius_lookup, strict_safety=True)
    for action in (1, 2, 3, 4, 0, 5):
        if mask[action]:
            return int(action)
    return 0

# Sanity check against the official engine.
_env = BomberEnv(seed=0)
_obs = _env.reset(seed=0)
_features, _scalars = encode_obs(_obs, 0, step_count=0, radius_lookup=radius_lookup_from_env(_env))
print("Feature shape:", _features.shape, "Scalar shape:", _scalars.shape)
print("Initial legal mask:", legal_action_mask(_obs, 0, radius_lookup_from_env(_env)).tolist())


## 4. CNN-GRU Actor-Critic

Model dùng kiến trúc actor-critic nhỏ gọn:

- CNN xử lý board channels 13x13;
- scalar features mô tả phase trận, bomb count, radius, enemy distance;
- GRU giữ short-term memory cho nhịp đặt bom và chạy bom;
- actor sinh action logits;
- critic ước lượng value cho PPO.

Behavior cloning được dùng để warm-start policy trước khi PPO update.


In [ ]:
# Cell 4: CNN-GRU actor-critic và behavior cloning utilities.
# Mục tiêu:
# - CNN đọc board 13x13 nhiều channel.
# - GRU giữ short-term memory cho nhịp đặt bom, chạy bom và pressure late game.
# - Actor sinh logits cho 6 actions, critic ước lượng value cho PPO.
# - Behavior cloning dùng teacher actions để warm-start policy trước PPO.

# Cell 4: CNN-GRU actor-critic và behavior cloning utilities.
class ActorCritic(nn.Module):
    def __init__(self, channels=N_CHANNELS, scalars=N_SCALARS, actions=6):
        super().__init__()
        self.map_encoder = nn.Sequential(
            nn.Conv2d(channels, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Flatten(),
        )
        self.scalar_encoder = nn.Sequential(
            nn.Linear(scalars, 64),
            nn.ReLU(),
        )
        self.shared = nn.Sequential(
            nn.Linear(64 * 13 * 13 + 64, 256),
            nn.ReLU(),
        )
        self.gru = nn.GRUCell(256, HIDDEN_SIZE)
        self.policy_head = nn.Linear(HIDDEN_SIZE, actions)
        self.value_head = nn.Linear(HIDDEN_SIZE, 1)

    def forward(self, maps, scalars, action_masks, recurrent_hidden):
        map_features = self.map_encoder(maps)
        scalar_features = self.scalar_encoder(scalars)
        latent = self.shared(torch.cat((map_features, scalar_features), dim=1))
        next_hidden = self.gru(latent, recurrent_hidden)
        logits = self.policy_head(next_hidden)
        logits = logits.masked_fill(~action_masks, -1.0e9)
        values = self.value_head(next_hidden).squeeze(-1)
        return logits, values, next_hidden

def tensor_batch(array, dtype=None):
    tensor = torch.as_tensor(array, dtype=dtype)
    return tensor.to(DEVICE, non_blocking=True)

def train_bc_chunk(model, optimizer, records, epochs, batch_size):
    if not records:
        return float("nan")
    maps = np.stack([record[0] for record in records])
    scalars = np.stack([record[1] for record in records])
    masks = np.stack([record[2] for record in records])
    actions = np.asarray([record[3] for record in records], dtype=np.int64)
    hidden = np.zeros((len(records), HIDDEN_SIZE), dtype=np.float32)
    losses = []
    model.train()
    for _ in range(epochs):
        order = np.random.permutation(len(records))
        for start in range(0, len(order), batch_size):
            indices = order[start : start + batch_size]
            logits, _, _ = model(
                tensor_batch(maps[indices], torch.float32),
                tensor_batch(scalars[indices], torch.float32),
                tensor_batch(masks[indices], torch.bool),
                tensor_batch(hidden[indices], torch.float32),
            )
            target = tensor_batch(actions[indices], torch.long)
            loss = nn.functional.cross_entropy(logits, target)
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), CFG.max_grad_norm)
            optimizer.step()
            losses.append(float(loss.detach().cpu()))
    return float(np.mean(losses))

def behavior_clone_from_tactical(model, optimizer, teacher_steps):
    if teacher_steps <= 0:
        return
    print(f"Behavior cloning from TacticalRuleAgent for {teacher_steps:,} teacher decisions")
    records = []
    collected = 0
    episode = 0
    while collected < teacher_steps:
        env = BomberEnv(seed=CFG.seed + episode, max_steps=CFG.max_steps)
        obs = env.reset(seed=CFG.seed + episode)
        teachers = [TacticalRuleAgent(index) for index in range(4)]
        done = False
        while not done and collected < teacher_steps:
            radius_lookup = radius_lookup_from_env(env)
            actions = []
            for player_id, teacher in enumerate(teachers):
                if env.players[player_id].alive:
                    action = int(teacher.act(obs))
                    features, scalars = encode_obs(obs, player_id, env.current_step, radius_lookup)
                    mask = legal_action_mask(obs, player_id, radius_lookup, strict_safety=False)
                    if not mask[action]:
                        action = safe_fallback(obs, player_id, radius_lookup)
                    records.append((features, scalars, mask, action))
                    collected += 1
                else:
                    action = 0
                actions.append(action)
            obs, terminated, truncated = env.step(actions)
            done = terminated or truncated
            if len(records) >= CFG.bc_chunk_size:
                loss = train_bc_chunk(model, optimizer, records, CFG.bc_epochs, CFG.minibatch_size)
                print(f"  BC decisions={collected:,}/{teacher_steps:,} loss={loss:.4f}")
                records.clear()
        episode += 1

    if records:
        loss = train_bc_chunk(model, optimizer, records, CFG.bc_epochs, CFG.minibatch_size)
        print(f"  BC decisions={collected:,}/{teacher_steps:,} loss={loss:.4f}")
    print("Behavior cloning complete")


## 5. PPO, Curriculum Learning Và Self-Play

Cell này triển khai rollout collection và PPO update.

Reward shaping không chỉ thưởng win/loss, mà còn hướng policy theo các
hành vi có ích:

- farm box/item an toàn;
- đặt bom có target;
- thoát blast line sau khi đặt bom;
- tạo trap pressure;
- giảm wait/passive behavior;
- phạt death, draw và mutual destruction.

Curriculum learning giúp policy học từ dễ đến khó thay vì nhảy thẳng vào
self-play mạnh.


In [ ]:
# Cell 5: PPO rollout, shaped reward, curriculum và self-play.
# Mục tiêu:
# - Chạy nhiều arena song song để thu thập trajectories.
# - Dùng shaped reward cho box/item, safe bomb, trap pressure, survival và final rank.
# - Tính GAE/advantage rồi update actor-critic bằng PPO clipped objective.
# - Shared-policy self-play giúp policy học tương tác nhiều agent cùng lúc.

# Cell 5: PPO rollout, curriculum learning và self-play.
REWARD_PROFILES = {
    "survival": {
        "kill": 0.10,
        "box": 0.025,
        "item": 0.100,
        "bomb_target": 0.020,
        "wasted_bomb": -0.020,
        "low_exit_bomb": -0.120,
        "self_trap_bomb": -0.300,
        "wait": -0.002,
        "death": -1.00,
        "danger": -0.14,
        "potential": 0.12,
        "sole_win": 1.0,
        "draw": -1.0,
        "mutual_destruction": -1.0,
        "ranks": (4.0, 1.0, -0.5, -2.5),
    },
    "balanced": {
        "kill": 0.18,
        "box": 0.035,
        "item": 0.120,
        "bomb_target": 0.025,
        "wasted_bomb": -0.020,
        "low_exit_bomb": -0.140,
        "self_trap_bomb": -0.350,
        "wait": -0.002,
        "death": -0.90,
        "danger": -0.12,
        "potential": 0.10,
        "sole_win": 1.5,
        "draw": -1.5,
        "mutual_destruction": -1.5,
        "ranks": (5.0, 1.5, -0.5, -3.0),
    },
    "competitive": {
        "kill": 0.25,
        "box": 0.025,
        "item": 0.080,
        "bomb_target": 0.018,
        "wasted_bomb": -0.025,
        "low_exit_bomb": -0.180,
        "self_trap_bomb": -0.450,
        "wait": -0.001,
        "death": -0.85,
        "danger": -0.11,
        "potential": 0.08,
        "sole_win": 2.0,
        "draw": -2.0,
        "mutual_destruction": -2.0,
        "ranks": (6.0, 2.0, -0.5, -3.5),
    },
    "final": {
        "kill": 0.28,
        "box": 0.015,
        "item": 0.050,
        "bomb_target": 0.012,
        "wasted_bomb": -0.020,
        "low_exit_bomb": -0.220,
        "self_trap_bomb": -0.600,
        "wait": -0.001,
        "death": -0.80,
        "danger": -0.10,
        "potential": 0.05,
        "sole_win": 3.0,
        "draw": -3.0,
        "mutual_destruction": -3.0,
        "ranks": (8.0, 2.5, -0.5, -4.0),
    },
}

def state_potential(obs, player_id, radius_lookup=None):
    players = obs["players"]
    if int(players[player_id][2]) != 1:
        return -1.0
    row, col = int(players[player_id][0]), int(players[player_id][1])
    mask = legal_action_mask(obs, player_id, radius_lookup=radius_lookup, strict_safety=False)
    immediate = danger_planes(obs, horizon=1, radius_lookup=radius_lookup)[0]
    safe_moves = 0
    for action in range(5):
        if not mask[action]:
            continue
        drow, dcol = MOVES[action]
        if immediate[row + drow, col + dcol] < 0.5:
            safe_moves += 1
    return float(safe_moves) / 5.0 - (0.75 if immediate[row, col] > 0.5 else 0.0)

# Reward shaping cân bằng survival, farming, bombing pressure và final rank.
def shaped_reward(before_obs, after_obs, before_stats, after_stats, player_id, profile_name, before_radius, after_radius):
    profile = REWARD_PROFILES[profile_name]
    reward = 0.0
    reward += profile["kill"] * (after_stats["kills"] - before_stats["kills"])
    reward += profile["box"] * (after_stats["boxes"] - before_stats["boxes"])
    reward += profile["item"] * (after_stats["items"] - before_stats["items"])

    was_alive = int(before_obs["players"][player_id][2]) == 1
    is_alive = int(after_obs["players"][player_id][2]) == 1
    if was_alive and not is_alive:
        reward += profile["death"]
    if is_alive:
        row, col = int(after_obs["players"][player_id][0]), int(after_obs["players"][player_id][1])
        upcoming = np.flatnonzero(
            danger_planes(after_obs, horizon=HORIZON, radius_lookup=after_radius)[:, row, col] > 0.5
        )
        if len(upcoming):
            first_tick = int(upcoming[0]) + 1
            reward += profile["danger"] * (1.0 + 0.20 * (HORIZON - first_tick))

    before_phi = state_potential(before_obs, player_id, before_radius)
    after_phi = state_potential(after_obs, player_id, after_radius)
    reward += profile["potential"] * (CFG.gamma * after_phi - before_phi)

    bomb_delta = after_stats["bombs"] - before_stats["bombs"]
    before_row, before_col = int(before_obs["players"][player_id][0]), int(before_obs["players"][player_id][1])
    after_row, after_col = int(after_obs["players"][player_id][0]), int(after_obs["players"][player_id][1])
    if bomb_delta > 0:
        radius = 1 + int(before_obs["players"][player_id][4])
        blast = blast_tiles(before_obs["map"], before_row, before_col, radius)
        boxes_hit = sum(int(before_obs["map"][row, col]) == 2 for row, col in blast)
        enemies_hit = sum(
            index != player_id
            and int(before_obs["players"][index][2]) == 1
            and (int(before_obs["players"][index][0]), int(before_obs["players"][index][1])) in blast
            for index in range(len(before_obs["players"]))
        )
        targets = boxes_hit + 2 * enemies_hit
        if targets > 0:
            reward += profile["bomb_target"] * min(float(targets), 4.0) * bomb_delta
        else:
            reward += profile["wasted_bomb"] * bomb_delta
        exits = escape_exit_count_after_bomb(before_obs, player_id, before_radius)
        if exits < MIN_BOMB_EXITS:
            reward += profile["low_exit_bomb"] * float(MIN_BOMB_EXITS - exits) * bomb_delta
        if survivability_depth_after_action(before_obs, player_id, 5, before_radius) < HORIZON:
            reward += profile["self_trap_bomb"] * bomb_delta
    elif was_alive and is_alive and (before_row, before_col) == (after_row, after_col):
        upcoming = danger_planes(before_obs, horizon=HORIZON, radius_lookup=before_radius)
        tactical_wait = len(before_obs["bombs"]) > 0 or np.any(upcoming[:, before_row, before_col] > 0.5)
        reward += (0.25 if tactical_wait else 1.0) * profile["wait"]
    return float(reward)

def ranks_from_state(players, death_order):
    order = [list(group) for group in death_order]
    survivors = [index for index, player in enumerate(players) if player.alive]
    if survivors:
        def stats(index):
            player_stats = players[index].stats
            return (
                player_stats["kills"],
                player_stats["boxes"],
                player_stats["items"],
                player_stats["bombs"],
            )
        survivors.sort(key=stats, reverse=True)
        groups = []
        current = [survivors[0]]
        current_stats = stats(survivors[0])
        for index in survivors[1:]:
            if stats(index) == current_stats:
                current.append(index)
            else:
                groups.append(current)
                current = [index]
                current_stats = stats(index)
        groups.append(current)
        order.extend(reversed(groups))

    ranks = [0] * len(players)
    for rank, group in enumerate(reversed(order)):
        for index in group:
            ranks[index] = rank
    return ranks

def make_league_snapshot(model):
    snapshot = copy.deepcopy(model).to(DEVICE).eval()
    for parameter in snapshot.parameters():
        parameter.requires_grad_(False)
    return snapshot

# Arena bọc BomberEnv để thu trajectory cho nhiều policy slots và baseline opponents.
class Arena:
    def __init__(self, arena_index, stage, rng, league):
        self.arena_index = arena_index
        self.reset(stage, rng, league)

    def reset(self, stage, rng, league):
        self.stage = stage
        seed = rng.randrange(1_000_000_000)
        self.env = BomberEnv(seed=seed, max_steps=CFG.max_steps)
        self.obs = self.env.reset(seed=seed)
        count = rng.randint(stage.policy_slots_min, stage.policy_slots_max)
        self.policy_ids = set(rng.sample(range(4), count))
        self.opponents = {}
        for player_id in range(4):
            if player_id in self.policy_ids:
                continue
            if league and rng.random() < stage.league_probability:
                self.opponents[player_id] = {
                    "kind": "snapshot",
                    "snapshot_index": rng.randrange(len(league)),
                }
            else:
                self.opponents[player_id] = {
                    "kind": "baseline",
                    "agent": BASELINE_CLASSES[rng.choice(stage.baselines)](player_id),
                }
        self.trajectories = {player_id: [] for player_id in self.policy_ids}
        self.hidden_by_player = {
            player_id: np.zeros(HIDDEN_SIZE, dtype=np.float32) for player_id in range(4)
        }
        self.alive_mask = [True] * 4
        self.death_order = []

    def radius_lookup(self):
        return radius_lookup_from_env(self.env)

    def step(self, staged_transitions, snapshot_actions):
        before_obs = self.obs
        before_radius = self.radius_lookup()
        before_stats = [dict(player.stats) for player in self.env.players]
        actions = []
        for player_id, player in enumerate(self.env.players):
            if not player.alive:
                action = 0
            elif player_id in self.policy_ids:
                action = int(staged_transitions[player_id]["action"])
            elif self.opponents[player_id]["kind"] == "snapshot":
                action = int(snapshot_actions[player_id])
            else:
                action = int(self.opponents[player_id]["agent"].act(before_obs))
            actions.append(action)

        before_step = int(self.env.current_step)
        self.obs, terminated, truncated = self.env.step(actions)
        done = bool(terminated or truncated)
        after_radius = self.radius_lookup()

        deaths = []
        for player_id, player in enumerate(self.env.players):
            if self.alive_mask[player_id] and not player.alive:
                self.alive_mask[player_id] = False
                deaths.append(player_id)
        if deaths:
            self.death_order.append(deaths)

        for player_id, transition in staged_transitions.items():
            transition["reward"] = shaped_reward(
                before_obs,
                self.obs,
                before_stats[player_id],
                dict(self.env.players[player_id].stats),
                player_id,
                self.stage.reward_profile,
                before_radius,
                after_radius,
                step=before_step,
            )
            transition["done"] = bool(done or not self.env.players[player_id].alive)
            self.trajectories[player_id].append(transition)

        if done:
            ranks = ranks_from_state(self.env.players, self.death_order)
            profile = REWARD_PROFILES[self.stage.reward_profile]
            rank_rewards = profile["ranks"]
            best_rank = min(ranks)
            winners = {index for index, rank in enumerate(ranks) if rank == best_rank}
            final_death_group = set(self.death_order[-1]) if self.death_order else set()
            for player_id, trajectory in self.trajectories.items():
                if trajectory:
                    trajectory[-1]["reward"] += float(rank_rewards[min(ranks[player_id], 3)])
                    if player_id in winners:
                        if len(winners) == 1:
                            trajectory[-1]["reward"] += float(profile["sole_win"])
                        else:
                            trajectory[-1]["reward"] += float(profile["draw"])
                    if player_id in final_death_group and len(final_death_group) > 1:
                        trajectory[-1]["reward"] += float(profile["mutual_destruction"])
                    trajectory[-1]["done"] = True
            return True, ranks
        return False, None

def finish_trajectory(trajectory):
    gae = 0.0
    next_value = 0.0
    for transition in reversed(trajectory):
        nonterminal = 0.0 if transition["done"] else 1.0
        delta = transition["reward"] + CFG.gamma * next_value * nonterminal - transition["value"]
        gae = delta + CFG.gamma * CFG.gae_lambda * nonterminal * gae
        transition["advantage"] = float(gae)
        transition["return"] = float(gae + transition["value"])
        next_value = transition["value"]
    return trajectory

@torch.no_grad()
def sample_policy_requests(model, requests):
    maps = np.stack([request["map"] for request in requests])
    scalars = np.stack([request["scalars"] for request in requests])
    masks = np.stack([request["mask"] for request in requests])
    hidden = np.stack([request["hidden"] for request in requests])
    logits, values, next_hidden = model(
        tensor_batch(maps, torch.float32),
        tensor_batch(scalars, torch.float32),
        tensor_batch(masks, torch.bool),
        tensor_batch(hidden, torch.float32),
    )
    distribution = Categorical(logits=logits)
    actions = distribution.sample()
    log_probs = distribution.log_prob(actions)
    return (
        actions.cpu().numpy(),
        log_probs.cpu().numpy(),
        values.cpu().numpy(),
        next_hidden.cpu().numpy(),
    )

# Thu PPO batch từ nhiều arenas song song; mỗi transition giữ obs, action, log_prob, value và reward.
def collect_ppo_batch(model, stage, rng, league):
    model.eval()
    arenas = [Arena(index, stage, rng, league) for index in range(CFG.num_envs)]
    records = []
    episode_metrics = []

    while len(episode_metrics) < CFG.episodes_per_update:
        requests = []
        snapshot_groups = {}
        for arena in arenas:
            radius_lookup = arena.radius_lookup()
            for player_id in sorted(arena.policy_ids):
                if not arena.env.players[player_id].alive:
                    continue
                features, scalars = encode_obs(arena.obs, player_id, arena.env.current_step, radius_lookup)
                mask = legal_action_mask(arena.obs, player_id, radius_lookup, strict_safety=True)
                requests.append(
                    {
                        "arena": arena,
                        "player_id": player_id,
                        "map": features,
                        "scalars": scalars,
                        "mask": mask,
                        "hidden": arena.hidden_by_player[player_id].copy(),
                    }
                )
            for player_id, opponent in arena.opponents.items():
                if opponent["kind"] != "snapshot" or not arena.env.players[player_id].alive:
                    continue
                features, scalars = encode_obs(arena.obs, player_id, arena.env.current_step, radius_lookup)
                mask = legal_action_mask(arena.obs, player_id, radius_lookup, strict_safety=True)
                snapshot_groups.setdefault(opponent["snapshot_index"], []).append(
                    {
                        "arena": arena,
                        "player_id": player_id,
                        "map": features,
                        "scalars": scalars,
                        "mask": mask,
                        "hidden": arena.hidden_by_player[player_id].copy(),
                    }
                )

        per_arena = {id(arena): {} for arena in arenas}
        if requests:
            actions, log_probs, values, next_hidden = sample_policy_requests(model, requests)
            for request, action, log_prob, value, hidden in zip(requests, actions, log_probs, values, next_hidden):
                request["arena"].hidden_by_player[request["player_id"]] = hidden.copy()
                transition = {
                    "map": request["map"],
                    "scalars": request["scalars"],
                    "mask": request["mask"],
                    "hidden": request["hidden"],
                    "action": int(action),
                    "log_prob": float(log_prob),
                    "value": float(value),
                }
                per_arena[id(request["arena"])][request["player_id"]] = transition

        snapshot_actions = {id(arena): {} for arena in arenas}
        for snapshot_index, group in snapshot_groups.items():
            actions, _, _, next_hidden = sample_policy_requests(league[snapshot_index], group)
            for request, action, hidden in zip(group, actions, next_hidden):
                request["arena"].hidden_by_player[request["player_id"]] = hidden.copy()
                snapshot_actions[id(request["arena"])][request["player_id"]] = int(action)

        for arena in arenas:
            finished, ranks = arena.step(per_arena[id(arena)], snapshot_actions[id(arena)])
            if not finished:
                continue
            for trajectory in arena.trajectories.values():
                if trajectory:
                    records.extend(finish_trajectory(trajectory))
            policy_ranks = [ranks[player_id] for player_id in arena.policy_ids]
            episode_metrics.append(
                {
                    "mean_policy_rank": float(np.mean(policy_ranks)),
                    "best_policy_rank": int(min(policy_ranks)),
                    "steps": int(arena.env.current_step),
                }
            )
            arena.reset(stage, rng, league)
    return records, episode_metrics

# PPO update: clipped policy loss + value loss + entropy bonus, có gradient clipping.
def ppo_update(model, optimizer, records):
    maps = np.stack([record["map"] for record in records])
    scalars = np.stack([record["scalars"] for record in records])
    masks = np.stack([record["mask"] for record in records])
    hidden = np.stack([record["hidden"] for record in records])
    actions = np.asarray([record["action"] for record in records], dtype=np.int64)
    old_log_probs = np.asarray([record["log_prob"] for record in records], dtype=np.float32)
    advantages = np.asarray([record["advantage"] for record in records], dtype=np.float32)
    returns = np.asarray([record["return"] for record in records], dtype=np.float32)
    advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)

    metrics = {"policy_loss": [], "value_loss": [], "entropy": [], "approx_kl": []}
    model.train()
    for _ in range(CFG.ppo_epochs):
        order = np.random.permutation(len(records))
        for start in range(0, len(order), CFG.minibatch_size):
            indices = order[start : start + CFG.minibatch_size]
            logits, values, _ = model(
                tensor_batch(maps[indices], torch.float32),
                tensor_batch(scalars[indices], torch.float32),
                tensor_batch(masks[indices], torch.bool),
                tensor_batch(hidden[indices], torch.float32),
            )
            distribution = Categorical(logits=logits)
            new_log_probs = distribution.log_prob(tensor_batch(actions[indices], torch.long))
            entropy = distribution.entropy().mean()
            old_lp = tensor_batch(old_log_probs[indices], torch.float32)
            batch_advantages = tensor_batch(advantages[indices], torch.float32)
            batch_returns = tensor_batch(returns[indices], torch.float32)
            log_ratio = new_log_probs - old_lp
            ratio = log_ratio.exp()
            unclipped = ratio * batch_advantages
            clipped = torch.clamp(ratio, 1.0 - CFG.clip_coef, 1.0 + CFG.clip_coef) * batch_advantages
            policy_loss = -torch.min(unclipped, clipped).mean()
            value_loss = 0.5 * (values - batch_returns).pow(2).mean()
            loss = policy_loss + CFG.vf_coef * value_loss - CFG.ent_coef * entropy

            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), CFG.max_grad_norm)
            optimizer.step()

            metrics["policy_loss"].append(float(policy_loss.detach().cpu()))
            metrics["value_loss"].append(float(value_loss.detach().cpu()))
            metrics["entropy"].append(float(entropy.detach().cpu()))
            metrics["approx_kl"].append(float(((ratio - 1.0) - log_ratio).mean().detach().cpu()))
    return {key: float(np.mean(values)) for key, values in metrics.items()}

def save_checkpoint(model, optimizer, global_update, stage_name):
    path = CHECKPOINT_DIR / f"ppo_{global_update:04d}_{stage_name}.pth"
    torch.save(
        {
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "global_update": global_update,
            "stage_name": stage_name,
            "config": asdict(CFG),
            "channels": N_CHANNELS,
            "scalars": N_SCALARS,
        },
        path,
    )
    return path

def train_curriculum(model, optimizer):
    rng = random.Random(CFG.seed)
    history = []
    global_update = 0
    league = [make_league_snapshot(model)]
    for stage in CURRICULUM:
        print(f"\n=== Stage: {stage.name} ({stage.updates} updates) ===")
        for stage_update in range(1, stage.updates + 1):
            started = time.perf_counter()
            records, episodes = collect_ppo_batch(model, stage, rng, league)
            update_metrics = ppo_update(model, optimizer, records)
            global_update += 1
            if global_update % CFG.league_snapshot_every == 0:
                league.append(make_league_snapshot(model))
                if len(league) > CFG.league_max_size:
                    league.pop(0)
            row = {
                "global_update": global_update,
                "stage": stage.name,
                "stage_update": stage_update,
                "transitions": len(records),
                "mean_rank": float(np.mean([episode["mean_policy_rank"] for episode in episodes])),
                "best_rank_rate": float(np.mean([episode["best_policy_rank"] == 0 for episode in episodes])),
                "mean_steps": float(np.mean([episode["steps"] for episode in episodes])),
                "seconds": time.perf_counter() - started,
                **update_metrics,
            }
            history.append(row)
            print(
                f"[{global_update:04d}] {stage.name:<18} "
                f"transitions={row['transitions']:>6} "
                f"rank={row['mean_rank']:.3f} "
                f"best={row['best_rank_rate']:.1%} "
                f"entropy={row['entropy']:.3f} "
                f"league={len(league)} "
                f"time={row['seconds']:.1f}s"
            )
            if global_update % CFG.checkpoint_every == 0:
                checkpoint = save_checkpoint(model, optimizer, global_update, stage.name)
                print("  checkpoint:", checkpoint)
        save_checkpoint(model, optimizer, global_update, stage.name)

    history_path = ARTIFACT_DIR / "training_history.json"
    history_path.write_text(json.dumps(history, indent=2), encoding="utf-8")
    return history


## 6. Evaluation Và Submission Export

Cell này đánh giá policy theo behavior giống deploy thật: neural policy
sinh logits, planner chọn action cuối cùng.

Export tạo:

- `agent.py`: inference logic tự chứa;
- `policy.pt`: TorchScript model;
- `submission.zip`: file nộp chính thức.


In [ ]:
# Cell 6: Evaluation và export helper cho deploy agent.
# Mục tiêu:
# - Đánh giá policy bằng deploy-time greedy policy + planner.
# - Sinh agent.py tự chứa toàn bộ inference logic.
# - Export TorchScript policy.pt để official evaluator load nhanh trên CPU.
# - Định nghĩa hàm export_submission(model). Cell này chưa chọn checkpoint cuối;
#   Cell 10 mới gọi hàm này để đóng gói submission.zip chính thức.

# Cell 6: Evaluation loop và deploy/export helper.
class BombTracker:
    def __init__(self):
        self.radius_by_bomb = {}

    def update(self, obs):
        current = set()
        players = obs["players"]
        for bomb in obs["bombs"]:
            row, col, _, owner_id = [int(value) for value in bomb]
            key = (row, col, owner_id)
            current.add(key)
            if key not in self.radius_by_bomb:
                self.radius_by_bomb[key] = 1 + int(players[owner_id][4])
        self.radius_by_bomb = {
            key: radius for key, radius in self.radius_by_bomb.items() if key in current
        }
        return self.radius_by_bomb

# Agent dùng trong evaluation: chọn action greedy qua model + planner giống deploy-time behavior.
class GreedyPolicyAgent:
    def __init__(self, player_id, model, device):
        self.player_id = int(player_id)
        self.model = model
        self.device = device
        self.tracker = BombTracker()
        self.step_count = 0
        self.hidden = torch.zeros(1, HIDDEN_SIZE, dtype=torch.float32, device=device)
        self.recent_positions = deque(maxlen=6)
        self.action_counts = np.zeros(6, dtype=np.int64)
        self.bomb_exit_counts = []

    @torch.no_grad()
    def act(self, obs):
        self.step_count += 1
        radius_lookup = self.tracker.update(obs)
        features, scalars = encode_obs(obs, self.player_id, self.step_count, radius_lookup)
        mask = legal_action_mask(obs, self.player_id, radius_lookup, strict_safety=True)
        logits, _, self.hidden = self.model(
            torch.as_tensor(features[None], dtype=torch.float32, device=self.device),
            torch.as_tensor(scalars[None], dtype=torch.float32, device=self.device),
            torch.as_tensor(mask[None], dtype=torch.bool, device=self.device),
            self.hidden,
        )
        self.hidden = self.hidden.detach()
        action = beam_plan_action(
            obs,
            self.player_id,
            radius_lookup=radius_lookup,
            policy_logits=logits[0].detach().cpu().numpy(),
            recent_positions=self.recent_positions,
            action_mask=mask,
            step_count=self.step_count,
        )
        self.recent_positions.append(
            (int(obs["players"][self.player_id][0]), int(obs["players"][self.player_id][1]))
        )
        self.action_counts[action] += 1
        if action == 5:
            self.bomb_exit_counts.append(
                escape_exit_count_after_bomb(obs, self.player_id, radius_lookup)
            )
        return action

def evaluate_policy(model, matches=None, seed=900_000):
    matches = matches or CFG.eval_matches
    model.eval()
    baseline_names = ("smarter", "genius", "tactical")
    ranks = []
    wins = 0
    draws = 0
    steps = []
    action_counts = np.zeros(6, dtype=np.int64)
    bomb_exit_counts = []
    rng = random.Random(seed)
    for match_index in range(matches):
        seat = match_index % 4
        env = BomberEnv(seed=seed + match_index, max_steps=CFG.max_steps)
        obs = env.reset(seed=seed + match_index)
        agents = []
        for player_id in range(4):
            if player_id == seat:
                agents.append(GreedyPolicyAgent(player_id, model, DEVICE))
            else:
                agents.append(BASELINE_CLASSES[rng.choice(baseline_names)](player_id))
        alive_mask = [True] * 4
        death_order = []
        done = False
        while not done:
            actions = [
                int(agent.act(obs)) if env.players[player_id].alive else 0
                for player_id, agent in enumerate(agents)
            ]
            obs, terminated, truncated = env.step(actions)
            done = bool(terminated or truncated)
            deaths = []
            for player_id, player in enumerate(env.players):
                if alive_mask[player_id] and not player.alive:
                    alive_mask[player_id] = False
                    deaths.append(player_id)
            if deaths:
                death_order.append(deaths)
        final_ranks = ranks_from_state(env.players, death_order)
        action_counts += agents[seat].action_counts
        bomb_exit_counts.extend(agents[seat].bomb_exit_counts)
        ranks.append(final_ranks[seat])
        best_rank = min(final_ranks)
        winners = {index for index, rank in enumerate(final_ranks) if rank == best_rank}
        wins += int(seat in winners and len(winners) == 1)
        draws += int(seat in winners and len(winners) > 1)
        steps.append(env.current_step)
    result = {
        "matches": matches,
        "average_rank": float(np.mean(ranks)),
        "rank_0_rate": float(sum(rank == 0 for rank in ranks) / matches),
        "sole_win_rate": float(wins / matches),
        "draw_rate": float(draws / matches),
        "mean_steps": float(np.mean(steps)),
        "rank_histogram": {str(rank): int(ranks.count(rank)) for rank in range(4)},
        "action_histogram": {str(action): int(action_counts[action]) for action in range(6)},
        "wait_rate": float(action_counts[0] / max(int(action_counts.sum()), 1)),
        "bomb_rate": float(action_counts[5] / max(int(action_counts.sum()), 1)),
        "low_exit_bomb_rate": float(
            sum(exit_count < MIN_BOMB_EXITS for exit_count in bomb_exit_counts)
            / max(len(bomb_exit_counts), 1)
        ),
        "mean_bomb_exits": float(np.mean(bomb_exit_counts)) if bomb_exit_counts else None,
    }
    print(json.dumps(result, indent=2))
    return result

# Sinh source code agent.py tự chứa inference helpers để submission không phụ thuộc notebook.
def deployment_agent_source():
    # This intentionally duplicates the compact inference helpers so submission.zip is self-contained.
    return r'''
from pathlib import Path
from collections import deque
import numpy as np
import torch

MOVES = {0:(0,0),1:(-1,0),2:(1,0),3:(0,-1),4:(0,1)}
HORIZON = 7
ADVERSARIAL_BOMB_DISTANCE = 4
MIN_BOMB_EXITS = 2
HIDDEN_SIZE = 128
N_CHANNELS = 19
N_SCALARS = 8
PLANNER_DEPTH = HORIZON
PLANNER_WIDTH = 18

def inside(grid, r, c):
    return 0 <= r < grid.shape[0] and 0 <= c < grid.shape[1]

def passable(grid, r, c):
    return inside(grid, r, c) and int(grid[r, c]) in (0, 3, 4)

def blast(grid, r, c, radius):
    tiles = {(int(r), int(c))}
    for dr, dc in ((-1,0),(1,0),(0,-1),(0,1)):
        for d in range(1, int(radius) + 1):
            nr, nc = int(r) + dr*d, int(c) + dc*d
            if not inside(grid, nr, nc):
                break
            cell = int(grid[nr, nc])
            if cell == 1:
                break
            tiles.add((nr, nc))
            if cell == 2:
                break
    return tiles

class BombTracker:
    def __init__(self):
        self.radius = {}
    def update(self, obs):
        current = set()
        players = obs["players"]
        for b in obs["bombs"]:
            r, c, _, owner = [int(v) for v in b]
            key = (r, c, owner)
            current.add(key)
            if key not in self.radius:
                self.radius[key] = 1 + int(players[owner][4])
        self.radius = {key: value for key, value in self.radius.items() if key in current}
        return self.radius

def bombs(obs, radius_lookup, extra=None, extras=None):
    players = obs["players"]
    result = []
    for b in obs["bombs"]:
        r, c, timer, owner = [int(v) for v in b]
        fallback = 1 + int(players[owner][4])
        result.append((r, c, timer, owner, int(radius_lookup.get((r,c,owner), fallback))))
    if extra is not None:
        result.append(tuple(int(v) for v in extra))
    for b in extras or ():
        result.append(tuple(int(v) for v in b))
    return result

def danger(obs, horizon=HORIZON, radius_lookup=None, extra=None, extras=None):
    grid = obs["map"]
    entries = bombs(obs, radius_lookup or {}, extra, extras)
    planes = np.zeros((horizon, 13, 13), dtype=np.float32)
    if not entries:
        return planes
    timers = [max(1, int(entry[2])) for entry in entries]
    blasts = [blast(grid, entry[0], entry[1], entry[4]) for entry in entries]
    for _ in range(len(entries)):
        changed = False
        for i, source in enumerate(blasts):
            for j, target in enumerate(entries):
                if i != j and (target[0], target[1]) in source and timers[j] > timers[i]:
                    timers[j] = timers[i]
                    changed = True
        if not changed:
            break
    for timer, tiles in zip(timers, blasts):
        if 1 <= timer <= horizon:
            for r, c in tiles:
                planes[timer - 1, r, c] = 1.0
    return planes

def enemy_bomb_threats(obs, player_id, max_distance=ADVERSARIAL_BOMB_DISTANCE):
    players = obs["players"]
    own_row, own_col = int(players[player_id][0]), int(players[player_id][1])
    occupied = {(int(b[0]), int(b[1])) for b in obs["bombs"]}
    result = []
    for enemy_id, enemy in enumerate(players):
        if enemy_id == player_id or int(enemy[2]) != 1 or int(enemy[3]) <= 0:
            continue
        row, col = int(enemy[0]), int(enemy[1])
        if (row, col) in occupied:
            continue
        if abs(row - own_row) + abs(col - own_col) <= max_distance:
            result.append((row, col, 6, enemy_id, 1 + int(enemy[4])))
    return result

def survival_depth(obs, player_id, first_action, radius_lookup, stress=None):
    grid = obs["map"]
    row, col, alive, _, bonus = [int(v) for v in obs["players"][player_id]]
    if not alive:
        return HORIZON if int(first_action) == 0 else 0
    blocked = {(int(b[0]), int(b[1])) for b in obs["bombs"]}
    extra = None
    if int(first_action) == 5:
        blocked.add((row, col))
        extra = (row, col, 6, player_id, 1 + bonus)
        nr, nc = row, col
    else:
        dr, dc = MOVES[int(first_action)]
        nr, nc = row + dr, col + dc
        if int(first_action) != 0 and (not passable(grid, nr, nc) or (nr, nc) in blocked):
            return 0
    extras = []
    if extra is not None:
        extras.append(extra)
    if stress is not None:
        blocked.add((int(stress[0]), int(stress[1])))
        extras.append(stress)
    future = danger(obs, HORIZON, radius_lookup, extras=extras)
    if future[0, nr, nc] > 0.5:
        return 0
    positions = {(nr, nc)}
    for tick in range(1, HORIZON):
        following = set()
        for r, c in positions:
            for action in range(5):
                dr, dc = MOVES[action]
                nr, nc = r + dr, c + dc
                if action != 0 and (not passable(grid, nr, nc) or (nr, nc) in blocked):
                    continue
                if future[tick, nr, nc] < 0.5:
                    following.add((nr, nc))
        positions = following
        if not positions:
            return tick
    return HORIZON

def robust_survival_depth(obs, player_id, first_action, radius_lookup):
    depth = survival_depth(obs, player_id, first_action, radius_lookup)
    if depth < HORIZON:
        return depth
    for stress in enemy_bomb_threats(obs, player_id):
        depth = min(depth, survival_depth(obs, player_id, first_action, radius_lookup, stress))
    return depth

def bomb_exit_count(obs, player_id, radius_lookup):
    grid = obs["map"]
    row, col, alive, _, bonus = [int(v) for v in obs["players"][player_id]]
    if not alive:
        return 0
    blocked = {(int(b[0]), int(b[1])) for b in obs["bombs"]}
    blocked.add((row, col))
    future = danger(obs, HORIZON, radius_lookup, (row, col, 6, player_id, 1 + bonus))
    exits = 0
    for action in range(1, 5):
        dr, dc = MOVES[action]
        nr, nc = row + dr, col + dc
        if passable(grid, nr, nc) and (nr, nc) not in blocked and future[0, nr, nc] < 0.5:
            exits += 1
    return exits

def mask_actions(obs, player_id, radius_lookup):
    grid = obs["map"]
    row, col, alive, left, _ = [int(v) for v in obs["players"][player_id]]
    result = np.zeros(6, dtype=np.bool_)
    if not alive:
        result[0] = True
        return result
    blocked = {(int(b[0]), int(b[1])) for b in obs["bombs"]}
    result[0] = True
    for action in range(1, 5):
        dr, dc = MOVES[action]
        nr, nc = row + dr, col + dc
        result[action] = passable(grid, nr, nc) and (nr, nc) not in blocked
    result[5] = left > 0 and (row, col) not in blocked
    depths = np.full(6, -1, dtype=np.int8)
    for action in np.flatnonzero(result):
        depths[action] = survival_depth(obs, player_id, int(action), radius_lookup)
    safe = result & (depths >= HORIZON)
    if safe[5]:
        if bomb_exit_count(obs, player_id, radius_lookup) < MIN_BOMB_EXITS:
            safe[5] = False
            depths[5] = -1
    if safe.any() and enemy_bomb_threats(obs, player_id):
        robust = np.full(6, -1, dtype=np.int8)
        for action in np.flatnonzero(safe):
            robust[action] = robust_survival_depth(obs, player_id, int(action), radius_lookup)
        robust_safe = safe & (robust >= HORIZON)
        if robust_safe.any():
            safe = robust_safe
    if safe.any():
        return safe
    non_bomb = result.copy()
    non_bomb[5] = False
    if non_bomb.any():
        return non_bomb & (depths == depths[non_bomb].max())
    result[5] = False
    if result.any():
        return result
    result[0] = True
    return result

def encode(obs, player_id, step_count, radius_lookup):
    grid, players = obs["map"], obs["players"]
    channels = [(grid == tile).astype(np.float32) for tile in range(5)]
    own = np.zeros((13, 13), dtype=np.float32)
    if int(players[player_id][2]) == 1:
        own[int(players[player_id][0]), int(players[player_id][1])] = 1.0
    channels.append(own)
    enemies = [index for index in range(4) if index != player_id]
    for enemy_id in enemies:
        plane = np.zeros((13, 13), dtype=np.float32)
        if int(players[enemy_id][2]) == 1:
            plane[int(players[enemy_id][0]), int(players[enemy_id][1])] = 1.0
        channels.append(plane)
    bomb_plane = np.zeros((13, 13), dtype=np.float32)
    timer_plane = np.zeros((13, 13), dtype=np.float32)
    radius_plane = np.zeros((13, 13), dtype=np.float32)
    for r, c, timer, _, radius in bombs(obs, radius_lookup):
        bomb_plane[r, c] = 1.0
        timer_plane[r, c] = max(timer_plane[r, c], float(timer) / 7.0)
        radius_plane[r, c] = max(radius_plane[r, c], float(radius) / 5.0)
    channels.extend((bomb_plane, timer_plane, radius_plane))
    future = danger(obs, HORIZON, radius_lookup)
    channels.extend(future)
    own_row, own_col = int(players[player_id][0]), int(players[player_id][1])
    enemy_distances = [
        abs(int(players[index][0]) - own_row) + abs(int(players[index][1]) - own_col)
        for index in enemies if int(players[index][2]) == 1
    ]
    nearest_enemy = min(enemy_distances) if enemy_distances else 24
    blocked = {(int(b[0]), int(b[1])) for b in obs["bombs"]}
    safe_moves = 0
    for action in range(5):
        dr, dc = MOVES[action]
        nr, nc = own_row + dr, own_col + dc
        if action != 0 and (not passable(grid, nr, nc) or (nr, nc) in blocked):
            continue
        if future[0, nr, nc] < 0.5:
            safe_moves += 1
    boxes_in_range = sum(
        int(grid[r, c]) == 2 for r, c in blast(grid, own_row, own_col, 1 + int(players[player_id][4]))
    )
    scalar = np.asarray([
        float(players[player_id][3]) / 5.0,
        float(players[player_id][4]) / 4.0,
        float(sum(int(players[index][2]) for index in enemies)) / 3.0,
        float(player_id) / 3.0,
        min(float(step_count), 500.0) / 500.0,
        min(float(nearest_enemy), 24.0) / 24.0,
        float(safe_moves) / 5.0,
        min(float(boxes_in_range), 4.0) / 4.0,
    ], dtype=np.float32)
    return np.stack(channels).astype(np.float32), scalar

def objectives(grid):
    result = {}
    for r, c in np.argwhere((grid == 3) | (grid == 4)):
        result[(int(r), int(c))] = 1.25
    for br, bc in np.argwhere(grid == 2):
        for dr, dc in MOVES.values():
            r, c = int(br) + dr, int(bc) + dc
            if passable(grid, r, c):
                result[(r, c)] = max(result.get((r, c), 0.0), 0.60)
    return tuple((r, c, value) for (r, c), value in result.items())

def escape_region_size(grid, start_row, start_col, blocked, future, ticks=6):
    positions = {(int(start_row), int(start_col))}
    for tick in range(min(int(ticks), future.shape[0])):
        next_positions = set()
        for row, col in positions:
            for action in range(5):
                drow, dcol = MOVES[action]
                next_row, next_col = row + drow, col + dcol
                if action != 0 and (
                    not passable(grid, next_row, next_col)
                    or (next_row, next_col) in blocked
                ):
                    continue
                if future[tick, next_row, next_col] < 0.5:
                    next_positions.add((next_row, next_col))
        positions = next_positions
        if not positions:
            return 0
    return len(positions)

def plan_action(obs, player_id, radius_lookup, policy_logits, recent_positions, action_mask=None, step_count=0):
    grid, players = obs["map"], obs["players"]
    row, col, alive, _, bonus = [int(v) for v in players[player_id]]
    if not alive:
        return 0
    mask = mask_actions(obs, player_id, radius_lookup) if action_mask is None else action_mask
    logits = np.asarray(policy_logits, dtype=np.float32)
    priors = np.zeros(6, dtype=np.float32)
    priors[mask] = np.exp(logits[mask] - np.max(logits[mask]))
    priors /= max(float(priors.sum()), 1e-8)
    bomb_positions = {(int(b[0]), int(b[1])) for b in obs["bombs"]}
    enemy_records = [
        (
            index,
            int(players[index][0]),
            int(players[index][1]),
            int(players[index][3]),
            int(players[index][4]),
        )
        for index in range(len(players)) if index != player_id and int(players[index][2]) == 1
    ]
    enemies = [(er, ec) for _, er, ec, _, _ in enemy_records]
    targets = objectives(grid)
    recent_sequence = tuple(recent_positions)
    recent_positions = set(recent_sequence)
    current_future = danger(obs, HORIZON, radius_lookup)
    threatened = bool(np.any(current_future[:, row, col] > 0.5))
    boxes_left = int(np.count_nonzero(grid == 2))
    endgame = int(step_count) >= 260 or len(enemy_records) <= 1 or boxes_left <= 4
    closing_phase = int(step_count) >= 400
    aggression = 1.0 + 0.35 * endgame + 0.25 * closing_phase
    baseline_regions = {
        enemy_id: escape_region_size(grid, er, ec, bomb_positions, current_future)
        for enemy_id, er, ec, _, _ in enemy_records
    }

    def adversarial_exposure(r, c):
        return sum(
            bombs_left > 0 and (r, c) in blast(grid, er, ec, 1 + radius)
            for _, er, ec, bombs_left, radius in enemy_records
        )

    def position_score(r, c, future, blocked, tick):
        mobility = 0
        for action in range(5):
            dr, dc = MOVES[action]
            nr, nc = r + dr, c + dc
            if action != 0 and (not passable(grid, nr, nc) or (nr, nc) in blocked):
                continue
            if future[min(tick, HORIZON - 1), nr, nc] < 0.5:
                mobility += 1
        margin = sum(future[index, r, c] < 0.5 for index in range(tick, HORIZON))
        distance = min((abs(er - r) + abs(ec - c) for er, ec in enemies), default=24)
        item = 0.35 if int(grid[r, c]) in (3, 4) else 0.0
        objective = max(
            (value - 0.04 * (abs(target_row - r) + abs(target_col - c))
             for target_row, target_col, value in targets),
            default=0.0,
        )
        corridor = 0.12 if mobility <= 1 else 0.0
        pressure = 0.12 * adversarial_exposure(r, c)
        objective_scale = 0.72 if endgame else 1.0
        distance_weight = 0.018 if endgame else 0.01
        return (
            0.08 * mobility
            + 0.04 * margin
            + item
            + objective_scale * objective
            - distance_weight * distance
            - corridor
            - pressure
        )

    beams = []
    for first in np.flatnonzero(mask):
        first = int(first)
        dr, dc = MOVES.get(first, (0, 0))
        nr, nc = row + dr, col + dc
        blocked, extra, tactical = set(bomb_positions), None, 0.0
        if first == 0 and np.any(mask[1:5]):
            tactical += 0.10 if threatened or len(obs["bombs"]) else (-0.34 if endgame else -0.24)
        if first != 5 and (nr, nc) in recent_positions:
            tactical -= 0.12
        if first != 5 and len(recent_sequence) >= 2 and (nr, nc) == recent_sequence[-2]:
            tactical -= 0.30
        if first == 5:
            nr, nc = row, col
            blocked.add((row, col))
            extra = (row, col, 6, player_id, 1 + bonus)
            tiles = blast(grid, row, col, 1 + bonus)
            box_hits = sum(int(grid[r, c]) == 2 for r, c in tiles)
            enemy_hits = sum((er, ec) in tiles for er, ec in enemies)
            pressure = 0.0
            for _, er, ec, _, _ in enemy_records:
                escapes = sum(
                    passable(grid, er + dr, ec + dc) and (er + dr, ec + dc) not in tiles
                    for dr, dc in MOVES.values()
                )
                covered = sum((er + dr, ec + dc) in tiles for dr, dc in MOVES.values())
                if (er, ec) in tiles:
                    pressure += 0.45 + 0.22 * max(0, 2 - escapes)
                elif abs(er - row) + abs(ec - col) <= 4:
                    pressure += 0.06 * covered
            tactical += 1.00 * box_hits + aggression * (1.25 * enemy_hits + pressure)
            exits = bomb_exit_count(obs, player_id, radius_lookup)
            if exits < MIN_BOMB_EXITS:
                tactical -= 4.0
            elif exits == MIN_BOMB_EXITS and len(obs["bombs"]) >= 2:
                tactical -= 0.30
            if box_hits == 0 and enemy_hits == 0:
                tactical -= 0.45
        future = danger(obs, HORIZON, radius_lookup, extra)
        if first == 5:
            trap_gain = 0
            forced_traps = 0
            for enemy_id, er, ec, _, _ in enemy_records:
                remaining = escape_region_size(grid, er, ec, blocked, future)
                trap_gain += max(0, baseline_regions.get(enemy_id, remaining) - remaining)
                forced_traps += int(remaining == 0)
            tactical += aggression * (0.08 * min(trap_gain, 12) + 0.90 * forced_traps)
        score = 1.20 * float(priors[first]) + tactical + position_score(nr, nc, future, blocked, 0)
        beams.append((score, nr, nc, first, blocked, future))

    for tick in range(1, PLANNER_DEPTH):
        expanded = []
        for score, r, c, first, blocked, future in beams:
            for action in range(5):
                dr, dc = MOVES[action]
                nr, nc = r + dr, c + dc
                if action != 0 and (not passable(grid, nr, nc) or (nr, nc) in blocked):
                    continue
                if future[min(tick, HORIZON - 1), nr, nc] > 0.5:
                    continue
                expanded.append((score + position_score(nr, nc, future, blocked, tick), nr, nc, first, blocked, future))
        if not expanded:
            break
        beams = sorted(expanded, key=lambda item: item[0], reverse=True)[:PLANNER_WIDTH]
    if beams:
        return int(max(beams, key=lambda item: item[0])[3])
    return int(np.flatnonzero(mask)[np.argmax(priors[mask])])

# Lớp Agent chính của submission; evaluator chỉ cần khởi tạo class này và gọi act(obs).
class Agent:
    def __init__(self, agent_id: int):
        torch.set_num_threads(1)
        self.agent_id = int(agent_id)
        self.step_count = 0
        self.tracker = BombTracker()
        self.model = torch.jit.load(str(Path(__file__).parent / "policy.pt"), map_location="cpu")
        self.model.eval()
        self.hidden = torch.zeros(1, HIDDEN_SIZE, dtype=torch.float32)
        self.recent_positions = deque(maxlen=6)
        # Move TorchScript cold-start work into the 20-second startup
        # budget. The evaluator gives the first act() call only 100ms.
        with torch.no_grad():
            warm_maps = torch.zeros(1, N_CHANNELS, 13, 13, dtype=torch.float32)
            warm_scalars = torch.zeros(1, N_SCALARS, dtype=torch.float32)
            warm_masks = torch.ones(1, 6, dtype=torch.bool)
            warm_hidden = torch.zeros(1, HIDDEN_SIZE, dtype=torch.float32)
            self.model(warm_maps, warm_scalars, warm_masks, warm_hidden)

    def act(self, obs: dict) -> int:
        try:
            self.step_count += 1
            radius_lookup = self.tracker.update(obs)
            features, scalars = encode(obs, self.agent_id, self.step_count, radius_lookup)
            masks = mask_actions(obs, self.agent_id, radius_lookup)
            with torch.no_grad():
                logits, _, self.hidden = self.model(
                    torch.from_numpy(features).unsqueeze(0),
                    torch.from_numpy(scalars).unsqueeze(0),
                    torch.from_numpy(masks).unsqueeze(0),
                    self.hidden,
                )
            action = plan_action(
                obs,
                self.agent_id,
                radius_lookup,
                logits[0].numpy(),
                self.recent_positions,
                masks,
                step_count=self.step_count,
            )
            self.recent_positions.append(
                (int(obs["players"][self.agent_id][0]), int(obs["players"][self.agent_id][1]))
            )
            return action
        except Exception:
            return 0
'''.strip() + "\n"

# Helper export: nhận model đã chọn, rồi ghi TorchScript policy.pt + agent.py vào submission.zip.
def export_submission(model):
    DEPLOY_DIR.mkdir(parents=True, exist_ok=True)
    model_cpu = copy.deepcopy(model).cpu().eval()
    example = (
        torch.zeros(1, N_CHANNELS, 13, 13, dtype=torch.float32),
        torch.zeros(1, N_SCALARS, dtype=torch.float32),
        torch.ones(1, 6, dtype=torch.bool),
        torch.zeros(1, HIDDEN_SIZE, dtype=torch.float32),
    )
    scripted = torch.jit.trace(model_cpu, example)
    policy_path = DEPLOY_DIR / "policy.pt"
    scripted.save(str(policy_path))
    agent_path = DEPLOY_DIR / "agent.py"
    agent_path.write_text(deployment_agent_source(), encoding="utf-8")

    zip_path = ARTIFACT_DIR / "submission.zip"
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        archive.write(agent_path, arcname="agent.py")
        archive.write(policy_path, arcname="policy.pt")
    print("Exported:", zip_path)
    print("ZIP contents:", zipfile.ZipFile(zip_path).namelist())
    return zip_path


## 7. Runtime Planner Và Top-Meta Overrides

Planner là lớp quyết định cuối cùng tại runtime. Neural logits chỉ đóng
vai trò priors; planner vẫn kiểm tra danger, mobility, trap potential và
objective score trước khi chọn action.

Đây là lý do agent có thể vừa farm/snowball vừa late-game pressure mà
không phụ thuộc hoàn toàn vào neural policy.


In [ ]:
# Cell 7: Planner nâng cấp và top-meta overrides.
# Mục tiêu:
# - Tăng độ thực chiến của deploy planner so với rule planner ban đầu.
# - Tải/chọn elite logs từ leaderboard nếu chạy trong notebook train.
# - Huấn luyện weighted behavior cloning từ top agents.
# - Chọn checkpoint bằng score ưu tiên avg rank thấp, rank0 cao, rank3 thấp.

# Cell 7: Runtime planner nâng cấp và top-meta overrides.
# This cell intentionally overrides configuration, rewards, top-log imitation,
# deploy-time planning, and checkpoint selection from the base top-44 notebook.
import dataclasses
import re
import shutil
import urllib.request

V2_STARTED_AT = time.perf_counter()
WALL_CLOCK_LIMIT_SECONDS = 10.25 * 3600
TOP_K = 8
TOP_MAX_RANK = 12
TOP_LOGS_PER_AGENT = 12
TOP_LOG_BC_MAX_SAMPLES = 220_000
TOP_LOG_BC_EPOCHS = 2
TOP_LOG_BC_BATCH = 1024
TOP_LOG_BC_DIR = ARTIFACT_DIR / "top_logs"
SHEET_ID = "****"
LOG_GID = "****"

# Faster and more targeted than the original 236-update run. The old notebook
# already reaches top 44; V2 spends more budget on elite imitation and late
# conversion, not on replaying weak-baseline survival.
CFG = TrainConfig(
    "top44_v2",
    SEED,
    num_envs=24,
    episodes_per_update=32,
    bc_teacher_steps=12_000,
    bc_chunk_size=4096,
    bc_epochs=2,
    max_steps=500,
    gamma=0.995,
    gae_lambda=0.95,
    learning_rate=2.2e-4,
    clip_coef=0.18,
    ent_coef=0.010,
    vf_coef=0.50,
    max_grad_norm=0.50,
    ppo_epochs=3,
    minibatch_size=1024,
    eval_matches=96,
    checkpoint_every=4,
    league_snapshot_every=3,
    league_max_size=12,
)

CURRICULUM = [
    Stage("opening_topfit", 20, 1, 1, ("box_farmer", "smarter", "genius"), "opening", 0.10),
    Stage("post_bomb_control", 22, 1, 1, ("smarter", "genius", "tactical"), "control", 0.25),
    Stage("strong_top_league", 58, 1, 1, ("smarter", "genius", "tactical"), "competitive", 0.55),
    Stage("late_conversion", 50, 1, 1, ("genius", "tactical"), "final", 0.75),
]

REWARD_PROFILES = {
    "opening": {
        "kill": 0.22, "box": 0.075, "item": 0.130,
        "bomb_target": 0.060, "wasted_bomb": -0.070,
        "low_exit_bomb": -0.220, "self_trap_bomb": -0.620,
        "wait": -0.010, "death": -1.10, "danger": -0.13,
        "potential": 0.045, "sole_win": 5.0, "draw": -5.0,
        "mutual_destruction": -4.0, "ranks": (7.0, 2.0, -1.0, -5.0),
    },
    "control": {
        "kill": 0.42, "box": 0.035, "item": 0.080,
        "bomb_target": 0.045, "wasted_bomb": -0.055,
        "low_exit_bomb": -0.260, "self_trap_bomb": -0.700,
        "wait": -0.006, "death": -1.05, "danger": -0.12,
        "potential": 0.035, "sole_win": 8.0, "draw": -8.0,
        "mutual_destruction": -5.0, "ranks": (9.0, 2.5, -1.5, -6.5),
    },
    "competitive": {
        "kill": 0.50, "box": 0.026, "item": 0.065,
        "bomb_target": 0.035, "wasted_bomb": -0.045,
        "low_exit_bomb": -0.280, "self_trap_bomb": -0.780,
        "wait": -0.004, "death": -1.00, "danger": -0.115,
        "potential": 0.025, "sole_win": 10.0, "draw": -10.0,
        "mutual_destruction": -6.0, "ranks": (11.0, 3.0, -2.0, -7.5),
    },
    "final": {
        "kill": 0.62, "box": 0.020, "item": 0.055,
        "bomb_target": 0.030, "wasted_bomb": -0.050,
        "low_exit_bomb": -0.320, "self_trap_bomb": -0.900,
        "wait": -0.007, "death": -1.15, "danger": -0.125,
        "potential": 0.015, "sole_win": 12.0, "draw": -11.5,
        "mutual_destruction": -7.0, "ranks": (12.0, 3.0, -2.0, -8.0),
    },
}

def _drive_file_id(url):
    match = re.search(r"/d/([^/]+)", str(url))
    return match.group(1) if match else None

def _download(url, path, retries=3):
    path.parent.mkdir(parents=True, exist_ok=True)
    for attempt in range(retries):
        try:
            urllib.request.urlretrieve(url, path)
            if path.stat().st_size > 1000:
                return
        except Exception:
            if attempt + 1 == retries:
                raise
            time.sleep(1.5 * (attempt + 1))
    raise RuntimeError(f"download failed: {url}")

def fetch_elite_logs(log_dir=TOP_LOG_BC_DIR, top_k=TOP_K, logs_per_agent=TOP_LOGS_PER_AGENT):
    import pandas as pd
    log_dir = Path(log_dir)
    json_dir = log_dir / "json"
    json_dir.mkdir(parents=True, exist_ok=True)
    lb_url = "****"
    logs_url = "****"
    lb = pd.read_csv(lb_url)
    logs = pd.read_csv(logs_url)
    lb.to_csv(log_dir / "leaderboard.csv", index=False)
    logs.to_csv(log_dir / "logs.csv", index=False)
    top = lb[lb["Is Active"].astype(int).eq(1) & (lb["Rank"].astype(int) <= TOP_MAX_RANK)].head(top_k)
    if len(top) < min(6, top_k):
        top = lb[lb["Is Active"].astype(int).eq(1)].head(top_k)
    rows = []
    match_cache = {}
    for _, team in top.iterrows():
        sid = str(team["Submission ID"])
        matches = logs[logs["Submission IDs"].astype(str).str.contains(sid, regex=False)].head(logs_per_agent)
        for sample, (_, row) in enumerate(matches.iterrows(), start=1):
            fid = _drive_file_id(row["JSON Drive URL"])
            if not fid:
                continue
            safe_team = re.sub(r"[^A-Za-z0-9_.-]+", "_", str(team["Team"]))
            path = json_dir / f"top{int(team['Rank']):02d}_{safe_team}_{sample}_{row['Match ID']}.json"
            match_id = str(row["Match ID"])
            if not path.exists():
                if match_id in match_cache and Path(match_cache[match_id]).exists():
                    shutil.copy2(match_cache[match_id], path)
                else:
                    _download("****", path)
                    match_cache[match_id] = str(path)
            rows.append({
                "rank": int(team["Rank"]), "team": str(team["Team"]),
                "submission_id": sid, "match_id": row["Match ID"], "path": str(path),
            })
            print("elite log", path.name, path.stat().st_size)
    selected = pd.DataFrame(rows)
    selected.to_csv(log_dir / "selected_top_logs.csv", index=False)
    if selected["submission_id"].nunique() < min(6, top_k):
        raise RuntimeError("Too few elite agents downloaded for top-log BC")
    return log_dir

def obs_from_log_frame(frame):
    return {
        "map": np.asarray(frame["map"], dtype=np.int16),
        "players": np.asarray(frame["players"], dtype=np.int16),
        "bombs": np.asarray(frame["bombs"], dtype=np.int16).reshape(-1, 4),
    }

def _phase_id(step):
    if step <= 80:
        return 0
    if step <= 220:
        return 1
    if step <= 380:
        return 2
    return 3

def extract_elite_bc_records(log_dir=TOP_LOG_BC_DIR, max_samples=TOP_LOG_BC_MAX_SAMPLES):
    import pandas as pd
    selected = pd.read_csv(Path(log_dir) / "selected_top_logs.csv")
    selected_ids = set(selected["submission_id"].astype(str))
    paths = [Path(p) for p in selected["path"].astype(str)]
    records = []
    weights = []
    rng = random.Random(SEED + 91)
    for path in paths:
        if not path.exists():
            continue
        data = json.loads(path.read_text(encoding="utf-8"))
        team_ids = [str(x) for x in data["team_ids"]]
        slots = [i for i, sid in enumerate(team_ids) if sid in selected_ids]
        hist = data.get("history", [])
        for step in range(1, len(hist)):
            actions = hist[step].get("actions")
            if actions is None:
                continue
            prev = obs_from_log_frame(hist[step - 1])
            for slot in slots:
                if int(prev["players"][slot][2]) != 1:
                    continue
                action = int(actions[slot])
                if not 0 <= action <= 5:
                    continue
                features, scalars = encode_obs(prev, slot, step, radius_lookup={})
                mask = legal_action_mask(prev, slot, radius_lookup={}, strict_safety=False)
                if not mask[action]:
                    continue
                phase = _phase_id(step)
                # Correct the exact gaps we measured: more opening farm, less STOP,
                # more accurate mid/late pressure, and stronger post-bomb movement.
                weight = 1.0
                if phase == 0:
                    weight *= 1.35
                    if action == 0:
                        weight *= 0.45
                    if action == 5:
                        weight *= 1.25
                elif phase >= 2:
                    if action == 0:
                        weight *= 0.55
                    if action == 5:
                        weight *= 1.45
                    if action in (1, 2, 3, 4):
                        weight *= 1.18
                records.append((features, scalars, mask, action))
                weights.append(weight)
    if len(records) > max_samples:
        indices = rng.sample(range(len(records)), max_samples)
        records = [records[i] for i in indices]
        weights = [weights[i] for i in indices]
    print("elite BC samples:", len(records), "mean_weight:", float(np.mean(weights)) if weights else None)
    return records, np.asarray(weights, dtype=np.float32)

def train_bc_weighted(model, optimizer, records, weights, epochs, batch_size):
    if not records:
        return float("nan")
    maps = np.stack([r[0] for r in records])
    scalars = np.stack([r[1] for r in records])
    masks = np.stack([r[2] for r in records])
    actions = np.asarray([r[3] for r in records], dtype=np.int64)
    hidden = np.zeros((len(records), HIDDEN_SIZE), dtype=np.float32)
    weights = np.asarray(weights, dtype=np.float32)
    weights = weights / max(float(weights.mean()), 1e-6)
    losses = []
    model.train()
    for epoch in range(epochs):
        order = np.random.permutation(len(records))
        for start in range(0, len(order), batch_size):
            idx = order[start:start + batch_size]
            logits, _, _ = model(
                tensor_batch(maps[idx], torch.float32),
                tensor_batch(scalars[idx], torch.float32),
                tensor_batch(masks[idx], torch.bool),
                tensor_batch(hidden[idx], torch.float32),
            )
            target = tensor_batch(actions[idx], torch.long)
            per = nn.functional.cross_entropy(logits, target, reduction="none")
            loss = (per * tensor_batch(weights[idx], torch.float32)).mean()
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), CFG.max_grad_norm)
            optimizer.step()
            losses.append(float(loss.detach().cpu()))
        print(f"  elite BC epoch={epoch + 1} loss={np.mean(losses):.4f}")
    return float(np.mean(losses))

def train_curriculum(model, optimizer):
    rng = random.Random(CFG.seed)
    history = []
    global_update = 0
    league = [make_league_snapshot(model)]
    for stage in CURRICULUM:
        print(f"\n=== Stage: {stage.name} ({stage.updates} updates) ===")
        for stage_update in range(1, stage.updates + 1):
            if time.perf_counter() - V2_STARTED_AT > WALL_CLOCK_LIMIT_SECONDS:
                print("wall-clock cutoff reached; saving and leaving time for evaluation/export")
                save_checkpoint(model, optimizer, global_update, "wall_clock_cutoff")
                (ARTIFACT_DIR / "training_history.json").write_text(json.dumps(history, indent=2), encoding="utf-8")
                return history
            started = time.perf_counter()
            records, episodes = collect_ppo_batch(model, stage, rng, league)
            update_metrics = ppo_update(model, optimizer, records)
            global_update += 1
            if global_update % CFG.league_snapshot_every == 0:
                league.append(make_league_snapshot(model))
                if len(league) > CFG.league_max_size:
                    league.pop(0)
            row = {
                "global_update": global_update, "stage": stage.name,
                "stage_update": stage_update, "transitions": len(records),
                "mean_rank": float(np.mean([e["mean_policy_rank"] for e in episodes])),
                "best_rank_rate": float(np.mean([e["best_policy_rank"] == 0 for e in episodes])),
                "mean_steps": float(np.mean([e["steps"] for e in episodes])),
                "seconds": time.perf_counter() - started, **update_metrics,
            }
            history.append(row)
            print(
                f"[{global_update:04d}] {stage.name:<18} transitions={row['transitions']:>6} "
                f"rank={row['mean_rank']:.3f} best={row['best_rank_rate']:.1%} "
                f"entropy={row['entropy']:.3f} league={len(league)} time={row['seconds']:.1f}s"
            )
            if global_update % CFG.checkpoint_every == 0:
                print("  checkpoint:", save_checkpoint(model, optimizer, global_update, stage.name))
        save_checkpoint(model, optimizer, global_update, stage.name)
    (ARTIFACT_DIR / "training_history.json").write_text(json.dumps(history, indent=2), encoding="utf-8")
    return history

_DEPLOY_AGENT_SOURCE_V2 = "from pathlib import Path\nfrom collections import deque\nimport numpy as np\nimport torch\n\nMOVES = {0:(0,0),1:(-1,0),2:(1,0),3:(0,-1),4:(0,1)}\nHORIZON = 7\nADVERSARIAL_BOMB_DISTANCE = 4\nMIN_BOMB_EXITS = 2\nHIDDEN_SIZE = 128\nN_CHANNELS = 19\nN_SCALARS = 8\nPLANNER_DEPTH = HORIZON\nPLANNER_WIDTH = 18\n\ndef inside(grid, r, c):\n    return 0 <= r < grid.shape[0] and 0 <= c < grid.shape[1]\n\ndef passable(grid, r, c):\n    return inside(grid, r, c) and int(grid[r, c]) in (0, 3, 4)\n\ndef blast(grid, r, c, radius):\n    tiles = {(int(r), int(c))}\n    for dr, dc in ((-1,0),(1,0),(0,-1),(0,1)):\n        for d in range(1, int(radius) + 1):\n            nr, nc = int(r) + dr*d, int(c) + dc*d\n            if not inside(grid, nr, nc):\n                break\n            cell = int(grid[nr, nc])\n            if cell == 1:\n                break\n            tiles.add((nr, nc))\n            if cell == 2:\n                break\n    return tiles\n\nclass BombTracker:\n    def __init__(self):\n        self.radius = {}\n    def update(self, obs):\n        current = set()\n        players = obs[\"players\"]\n        for b in obs[\"bombs\"]:\n            r, c, _, owner = [int(v) for v in b]\n            key = (r, c, owner)\n            current.add(key)\n            if key not in self.radius:\n                self.radius[key] = 1 + int(players[owner][4])\n        self.radius = {key: value for key, value in self.radius.items() if key in current}\n        return self.radius\n\ndef bombs(obs, radius_lookup, extra=None, extras=None):\n    players = obs[\"players\"]\n    result = []\n    for b in obs[\"bombs\"]:\n        r, c, timer, owner = [int(v) for v in b]\n        fallback = 1 + int(players[owner][4])\n        result.append((r, c, timer, owner, int(radius_lookup.get((r,c,owner), fallback))))\n    if extra is not None:\n        result.append(tuple(int(v) for v in extra))\n    for b in extras or ():\n        result.append(tuple(int(v) for v in b))\n    return result\n\ndef danger(obs, horizon=HORIZON, radius_lookup=None, extra=None, extras=None):\n    grid = obs[\"map\"]\n    entries = bombs(obs, radius_lookup or {}, extra, extras)\n    planes = np.zeros((horizon, 13, 13), dtype=np.float32)\n    if not entries:\n        return planes\n    timers = [max(1, int(entry[2])) for entry in entries]\n    blasts = [blast(grid, entry[0], entry[1], entry[4]) for entry in entries]\n    for _ in range(len(entries)):\n        changed = False\n        for i, source in enumerate(blasts):\n            for j, target in enumerate(entries):\n                if i != j and (target[0], target[1]) in source and timers[j] > timers[i]:\n                    timers[j] = timers[i]\n                    changed = True\n        if not changed:\n            break\n    for timer, tiles in zip(timers, blasts):\n        if 1 <= timer <= horizon:\n            for r, c in tiles:\n                planes[timer - 1, r, c] = 1.0\n    return planes\n\ndef enemy_bomb_threats(obs, player_id, max_distance=ADVERSARIAL_BOMB_DISTANCE):\n    players = obs[\"players\"]\n    own_row, own_col = int(players[player_id][0]), int(players[player_id][1])\n    occupied = {(int(b[0]), int(b[1])) for b in obs[\"bombs\"]}\n    result = []\n    for enemy_id, enemy in enumerate(players):\n        if enemy_id == player_id or int(enemy[2]) != 1 or int(enemy[3]) <= 0:\n            continue\n        row, col = int(enemy[0]), int(enemy[1])\n        if (row, col) in occupied:\n            continue\n        if abs(row - own_row) + abs(col - own_col) <= max_distance:\n            result.append((row, col, 6, enemy_id, 1 + int(enemy[4])))\n    return result\n\ndef survival_depth(obs, player_id, first_action, radius_lookup, stress=None):\n    grid = obs[\"map\"]\n    row, col, alive, _, bonus = [int(v) for v in obs[\"players\"][player_id]]\n    if not alive:\n        return HORIZON if int(first_action) == 0 else 0\n    blocked = {(int(b[0]), int(b[1])) for b in obs[\"bombs\"]}\n    extra = None\n    if int(first_action) == 5:\n        blocked.add((row, col))\n        extra = (row, col, 6, player_id, 1 + bonus)\n        nr, nc = row, col\n    else:\n        dr, dc = MOVES[int(first_action)]\n        nr, nc = row + dr, col + dc\n        if int(first_action) != 0 and (not passable(grid, nr, nc) or (nr, nc) in blocked):\n            return 0\n    extras = []\n    if extra is not None:\n        extras.append(extra)\n    if stress is not None:\n        blocked.add((int(stress[0]), int(stress[1])))\n        extras.append(stress)\n    future = danger(obs, HORIZON, radius_lookup, extras=extras)\n    if future[0, nr, nc] > 0.5:\n        return 0\n    positions = {(nr, nc)}\n    for tick in range(1, HORIZON):\n        following = set()\n        for r, c in positions:\n            for action in range(5):\n                dr, dc = MOVES[action]\n                nr, nc = r + dr, c + dc\n                if action != 0 and (not passable(grid, nr, nc) or (nr, nc) in blocked):\n                    continue\n                if future[tick, nr, nc] < 0.5:\n                    following.add((nr, nc))\n        positions = following\n        if not positions:\n            return tick\n    return HORIZON\n\ndef robust_survival_depth(obs, player_id, first_action, radius_lookup):\n    depth = survival_depth(obs, player_id, first_action, radius_lookup)\n    if depth < HORIZON:\n        return depth\n    for stress in enemy_bomb_threats(obs, player_id):\n        depth = min(depth, survival_depth(obs, player_id, first_action, radius_lookup, stress))\n    return depth\n\ndef bomb_exit_count(obs, player_id, radius_lookup):\n    grid = obs[\"map\"]\n    row, col, alive, _, bonus = [int(v) for v in obs[\"players\"][player_id]]\n    if not alive:\n        return 0\n    blocked = {(int(b[0]), int(b[1])) for b in obs[\"bombs\"]}\n    blocked.add((row, col))\n    future = danger(obs, HORIZON, radius_lookup, (row, col, 6, player_id, 1 + bonus))\n    exits = 0\n    for action in range(1, 5):\n        dr, dc = MOVES[action]\n        nr, nc = row + dr, col + dc\n        if passable(grid, nr, nc) and (nr, nc) not in blocked and future[0, nr, nc] < 0.5:\n            exits += 1\n    return exits\n\ndef mask_actions(obs, player_id, radius_lookup):\n    grid = obs[\"map\"]\n    row, col, alive, left, _ = [int(v) for v in obs[\"players\"][player_id]]\n    result = np.zeros(6, dtype=np.bool_)\n    if not alive:\n        result[0] = True\n        return result\n    blocked = {(int(b[0]), int(b[1])) for b in obs[\"bombs\"]}\n    result[0] = True\n    for action in range(1, 5):\n        dr, dc = MOVES[action]\n        nr, nc = row + dr, col + dc\n        result[action] = passable(grid, nr, nc) and (nr, nc) not in blocked\n    result[5] = left > 0 and (row, col) not in blocked\n    depths = np.full(6, -1, dtype=np.int8)\n    for action in np.flatnonzero(result):\n        depths[action] = survival_depth(obs, player_id, int(action), radius_lookup)\n    safe = result & (depths >= HORIZON)\n    if safe[5]:\n        if bomb_exit_count(obs, player_id, radius_lookup) < MIN_BOMB_EXITS:\n            safe[5] = False\n            depths[5] = -1\n    if safe.any() and enemy_bomb_threats(obs, player_id):\n        robust = np.full(6, -1, dtype=np.int8)\n        for action in np.flatnonzero(safe):\n            robust[action] = robust_survival_depth(obs, player_id, int(action), radius_lookup)\n        robust_safe = safe & (robust >= HORIZON)\n        if robust_safe.any():\n            safe = robust_safe\n    if safe.any():\n        return safe\n    non_bomb = result.copy()\n    non_bomb[5] = False\n    if non_bomb.any():\n        return non_bomb & (depths == depths[non_bomb].max())\n    result[5] = False\n    if result.any():\n        return result\n    result[0] = True\n    return result\n\ndef encode(obs, player_id, step_count, radius_lookup):\n    grid, players = obs[\"map\"], obs[\"players\"]\n    channels = [(grid == tile).astype(np.float32) for tile in range(5)]\n    own = np.zeros((13, 13), dtype=np.float32)\n    if int(players[player_id][2]) == 1:\n        own[int(players[player_id][0]), int(players[player_id][1])] = 1.0\n    channels.append(own)\n    enemies = [index for index in range(4) if index != player_id]\n    for enemy_id in enemies:\n        plane = np.zeros((13, 13), dtype=np.float32)\n        if int(players[enemy_id][2]) == 1:\n            plane[int(players[enemy_id][0]), int(players[enemy_id][1])] = 1.0\n        channels.append(plane)\n    bomb_plane = np.zeros((13, 13), dtype=np.float32)\n    timer_plane = np.zeros((13, 13), dtype=np.float32)\n    radius_plane = np.zeros((13, 13), dtype=np.float32)\n    for r, c, timer, _, radius in bombs(obs, radius_lookup):\n        bomb_plane[r, c] = 1.0\n        timer_plane[r, c] = max(timer_plane[r, c], float(timer) / 7.0)\n        radius_plane[r, c] = max(radius_plane[r, c], float(radius) / 5.0)\n    channels.extend((bomb_plane, timer_plane, radius_plane))\n    future = danger(obs, HORIZON, radius_lookup)\n    channels.extend(future)\n    own_row, own_col = int(players[player_id][0]), int(players[player_id][1])\n    enemy_distances = [\n        abs(int(players[index][0]) - own_row) + abs(int(players[index][1]) - own_col)\n        for index in enemies if int(players[index][2]) == 1\n    ]\n    nearest_enemy = min(enemy_distances) if enemy_distances else 24\n    blocked = {(int(b[0]), int(b[1])) for b in obs[\"bombs\"]}\n    safe_moves = 0\n    for action in range(5):\n        dr, dc = MOVES[action]\n        nr, nc = own_row + dr, own_col + dc\n        if action != 0 and (not passable(grid, nr, nc) or (nr, nc) in blocked):\n            continue\n        if future[0, nr, nc] < 0.5:\n            safe_moves += 1\n    boxes_in_range = sum(\n        int(grid[r, c]) == 2 for r, c in blast(grid, own_row, own_col, 1 + int(players[player_id][4]))\n    )\n    scalar = np.asarray([\n        float(players[player_id][3]) / 5.0,\n        float(players[player_id][4]) / 4.0,\n        float(sum(int(players[index][2]) for index in enemies)) / 3.0,\n        float(player_id) / 3.0,\n        min(float(step_count), 500.0) / 500.0,\n        min(float(nearest_enemy), 24.0) / 24.0,\n        float(safe_moves) / 5.0,\n        min(float(boxes_in_range), 4.0) / 4.0,\n    ], dtype=np.float32)\n    return np.stack(channels).astype(np.float32), scalar\n\ndef objectives(grid):\n    result = {}\n    for r, c in np.argwhere((grid == 3) | (grid == 4)):\n        result[(int(r), int(c))] = 1.25\n    for br, bc in np.argwhere(grid == 2):\n        for dr, dc in MOVES.values():\n            r, c = int(br) + dr, int(bc) + dc\n            if passable(grid, r, c):\n                result[(r, c)] = max(result.get((r, c), 0.0), 0.60)\n    return tuple((r, c, value) for (r, c), value in result.items())\n\ndef escape_region_size(grid, start_row, start_col, blocked, future, ticks=6):\n    positions = {(int(start_row), int(start_col))}\n    for tick in range(min(int(ticks), future.shape[0])):\n        next_positions = set()\n        for row, col in positions:\n            for action in range(5):\n                drow, dcol = MOVES[action]\n                next_row, next_col = row + drow, col + dcol\n                if action != 0 and (\n                    not passable(grid, next_row, next_col)\n                    or (next_row, next_col) in blocked\n                ):\n                    continue\n                if future[tick, next_row, next_col] < 0.5:\n                    next_positions.add((next_row, next_col))\n        positions = next_positions\n        if not positions:\n            return 0\n    return len(positions)\n\ndef plan_action(obs, player_id, radius_lookup, policy_logits, recent_positions, action_mask=None, step_count=0):\n    grid, players = obs[\"map\"], obs[\"players\"]\n    row, col, alive, _, bonus = [int(v) for v in players[player_id]]\n    if not alive:\n        return 0\n    mask = mask_actions(obs, player_id, radius_lookup) if action_mask is None else action_mask\n    logits = np.asarray(policy_logits, dtype=np.float32)\n    priors = np.zeros(6, dtype=np.float32)\n    priors[mask] = np.exp(logits[mask] - np.max(logits[mask]))\n    priors /= max(float(priors.sum()), 1e-8)\n    bomb_positions = {(int(b[0]), int(b[1])) for b in obs[\"bombs\"]}\n    enemy_records = [\n        (\n            index,\n            int(players[index][0]),\n            int(players[index][1]),\n            int(players[index][3]),\n            int(players[index][4]),\n        )\n        for index in range(len(players)) if index != player_id and int(players[index][2]) == 1\n    ]\n    enemies = [(er, ec) for _, er, ec, _, _ in enemy_records]\n    targets = objectives(grid)\n    recent_sequence = tuple(recent_positions)\n    recent_positions = set(recent_sequence)\n    current_future = danger(obs, HORIZON, radius_lookup)\n    threatened = bool(np.any(current_future[:, row, col] > 0.5))\n    boxes_left = int(np.count_nonzero(grid == 2))\n    opening = int(step_count) <= 80\n    endgame = int(step_count) >= 260 or len(enemy_records) <= 1 or boxes_left <= 4\n    closing_phase = int(step_count) >= 400\n    aggression = 1.0 + 0.45 * endgame + 0.35 * closing_phase\n    own_recent_bombs = [\n        (int(b[0]), int(b[1]), int(b[2])) for b in obs['bombs']\n        if int(b[3]) == player_id and int(b[2]) >= 4\n    ]\n    baseline_regions = {\n        enemy_id: escape_region_size(grid, er, ec, bomb_positions, current_future)\n        for enemy_id, er, ec, _, _ in enemy_records\n    }\n\n    def adversarial_exposure(r, c):\n        return sum(\n            bombs_left > 0 and (r, c) in blast(grid, er, ec, 1 + radius)\n            for _, er, ec, bombs_left, radius in enemy_records\n        )\n\n    def position_score(r, c, future, blocked, tick):\n        mobility = 0\n        for action in range(5):\n            dr, dc = MOVES[action]\n            nr, nc = r + dr, c + dc\n            if action != 0 and (not passable(grid, nr, nc) or (nr, nc) in blocked):\n                continue\n            if future[min(tick, HORIZON - 1), nr, nc] < 0.5:\n                mobility += 1\n        margin = sum(future[index, r, c] < 0.5 for index in range(tick, HORIZON))\n        distance = min((abs(er - r) + abs(ec - c) for er, ec in enemies), default=24)\n        item = 0.35 if int(grid[r, c]) in (3, 4) else 0.0\n        objective = max(\n            (value - 0.04 * (abs(target_row - r) + abs(target_col - c))\n             for target_row, target_col, value in targets),\n            default=0.0,\n        )\n        corridor = 0.12 if mobility <= 1 else 0.0\n        pressure = 0.12 * adversarial_exposure(r, c)\n        objective_scale = 1.35 if opening else (0.72 if endgame else 1.0)\n        distance_weight = 0.018 if endgame else 0.01\n        return (\n            0.08 * mobility\n            + 0.04 * margin\n            + item\n            + objective_scale * objective\n            - distance_weight * distance\n            - corridor\n            - pressure\n        )\n\n    beams = []\n    for first in np.flatnonzero(mask):\n        first = int(first)\n        dr, dc = MOVES.get(first, (0, 0))\n        nr, nc = row + dr, col + dc\n        blocked, extra, tactical = set(bomb_positions), None, 0.0\n        if first == 0 and np.any(mask[1:5]):\n            near_own_bomb = any(abs(row - br) + abs(col - bc) <= 2 for br, bc, _ in own_recent_bombs)\n            if near_own_bomb:\n                tactical -= 1.25\n            elif threatened or len(obs['bombs']):\n                tactical += 0.05\n            else:\n                tactical -= 0.60 if opening else (0.50 if endgame else 0.30)\n        if first != 5 and (nr, nc) in recent_positions:\n            tactical -= 0.12\n        if first != 5 and len(recent_sequence) >= 2 and (nr, nc) == recent_sequence[-2]:\n            tactical -= 0.30\n        if first in (1, 2, 3, 4) and own_recent_bombs:\n            br, bc, _ = min(own_recent_bombs, key=lambda b: abs(row - b[0]) + abs(col - b[1]))\n            before_d = abs(row - br) + abs(col - bc)\n            after_d = abs(nr - br) + abs(nc - bc)\n            if after_d > before_d:\n                tactical += 0.70\n            elif after_d < before_d:\n                tactical -= 0.55\n            if row == br or col == bc:\n                tactical += 0.25 if (nr != br and nc != bc) else -0.20\n        if first == 5:\n            nr, nc = row, col\n            blocked.add((row, col))\n            extra = (row, col, 6, player_id, 1 + bonus)\n            tiles = blast(grid, row, col, 1 + bonus)\n            box_hits = sum(int(grid[r, c]) == 2 for r, c in tiles)\n            enemy_hits = sum((er, ec) in tiles for er, ec in enemies)\n            pressure = 0.0\n            for _, er, ec, _, _ in enemy_records:\n                escapes = sum(\n                    passable(grid, er + dr, ec + dc) and (er + dr, ec + dc) not in tiles\n                    for dr, dc in MOVES.values()\n                )\n                covered = sum((er + dr, ec + dc) in tiles for dr, dc in MOVES.values())\n                if (er, ec) in tiles:\n                    pressure += 0.45 + 0.22 * max(0, 2 - escapes)\n                elif abs(er - row) + abs(ec - col) <= 4:\n                    pressure += 0.06 * covered\n            tactical += (1.65 if opening else 1.00) * box_hits + aggression * (1.50 * enemy_hits + 1.18 * pressure)\n            exits = bomb_exit_count(obs, player_id, radius_lookup)\n            if exits < MIN_BOMB_EXITS:\n                tactical -= 4.0\n            elif exits == MIN_BOMB_EXITS and len(obs[\"bombs\"]) >= 2:\n                tactical -= 0.30\n            if box_hits == 0 and enemy_hits == 0:\n                tactical -= 1.05 if opening else 0.58\n            elif opening and box_hits > 0:\n                tactical += 0.55\n        future = danger(obs, HORIZON, radius_lookup, extra)\n        if first == 5:\n            trap_gain = 0\n            forced_traps = 0\n            for enemy_id, er, ec, _, _ in enemy_records:\n                remaining = escape_region_size(grid, er, ec, blocked, future)\n                trap_gain += max(0, baseline_regions.get(enemy_id, remaining) - remaining)\n                forced_traps += int(remaining == 0)\n            tactical += aggression * (0.11 * min(trap_gain, 14) + 1.10 * forced_traps)\n        score = 1.20 * float(priors[first]) + tactical + position_score(nr, nc, future, blocked, 0)\n        beams.append((score, nr, nc, first, blocked, future))\n\n    for tick in range(1, PLANNER_DEPTH):\n        expanded = []\n        for score, r, c, first, blocked, future in beams:\n            for action in range(5):\n                dr, dc = MOVES[action]\n                nr, nc = r + dr, c + dc\n                if action != 0 and (not passable(grid, nr, nc) or (nr, nc) in blocked):\n                    continue\n                if future[min(tick, HORIZON - 1), nr, nc] > 0.5:\n                    continue\n                expanded.append((score + position_score(nr, nc, future, blocked, tick), nr, nc, first, blocked, future))\n        if not expanded:\n            break\n        beams = sorted(expanded, key=lambda item: item[0], reverse=True)[:PLANNER_WIDTH]\n    if beams:\n        return int(max(beams, key=lambda item: item[0])[3])\n    return int(np.flatnonzero(mask)[np.argmax(priors[mask])])\n\nclass Agent:\n    def __init__(self, agent_id: int):\n        torch.set_num_threads(1)\n        self.agent_id = int(agent_id)\n        self.step_count = 0\n        self.tracker = BombTracker()\n        self.model = torch.jit.load(str(Path(__file__).parent / \"policy.pt\"), map_location=\"cpu\")\n        self.model.eval()\n        self.hidden = torch.zeros(1, HIDDEN_SIZE, dtype=torch.float32)\n        self.recent_positions = deque(maxlen=6)\n        # Move TorchScript cold-start work into the 20-second startup\n        # budget. The evaluator gives the first act() call only 100ms.\n        with torch.no_grad():\n            warm_maps = torch.zeros(1, N_CHANNELS, 13, 13, dtype=torch.float32)\n            warm_scalars = torch.zeros(1, N_SCALARS, dtype=torch.float32)\n            warm_masks = torch.ones(1, 6, dtype=torch.bool)\n            warm_hidden = torch.zeros(1, HIDDEN_SIZE, dtype=torch.float32)\n            self.model(warm_maps, warm_scalars, warm_masks, warm_hidden)\n\n    def act(self, obs: dict) -> int:\n        try:\n            self.step_count += 1\n            radius_lookup = self.tracker.update(obs)\n            features, scalars = encode(obs, self.agent_id, self.step_count, radius_lookup)\n            masks = mask_actions(obs, self.agent_id, radius_lookup)\n            with torch.no_grad():\n                logits, _, self.hidden = self.model(\n                    torch.from_numpy(features).unsqueeze(0),\n                    torch.from_numpy(scalars).unsqueeze(0),\n                    torch.from_numpy(masks).unsqueeze(0),\n                    self.hidden,\n                )\n            action = plan_action(\n                obs,\n                self.agent_id,\n                radius_lookup,\n                logits[0].numpy(),\n                self.recent_positions,\n                masks,\n                step_count=self.step_count,\n            )\n            self.recent_positions.append(\n                (int(obs[\"players\"][self.agent_id][0]), int(obs[\"players\"][self.agent_id][1]))\n            )\n            return action\n        except Exception:\n            return 0\n"

_deploy_ns = {}
exec(_DEPLOY_AGENT_SOURCE_V2, _deploy_ns)
PLANNER_ROLLOUT_PROB = 0.85

@torch.no_grad()
def sample_policy_requests(model, requests):
    maps = np.stack([request["map"] for request in requests])
    scalars = np.stack([request["scalars"] for request in requests])
    masks = np.stack([request["mask"] for request in requests])
    hidden = np.stack([request["hidden"] for request in requests])
    logits, values, next_hidden = model(
        tensor_batch(maps, torch.float32),
        tensor_batch(scalars, torch.float32),
        tensor_batch(masks, torch.bool),
        tensor_batch(hidden, torch.float32),
    )
    distribution = Categorical(logits=logits)
    sampled = distribution.sample().detach().cpu().numpy()
    logits_np = logits.detach().cpu().numpy()
    actions = []
    for index, request in enumerate(requests):
        action = int(sampled[index])
        arena = request.get("arena")
        player_id = int(request.get("player_id", 0))
        if arena is not None and random.random() < PLANNER_ROLLOUT_PROB:
            if not hasattr(arena, "recent_positions_by_player") or arena.env.current_step <= 1:
                arena.recent_positions_by_player = {i: deque(maxlen=6) for i in range(4)}
            recent = arena.recent_positions_by_player[player_id]
            action = int(_deploy_ns["plan_action"](
                arena.obs,
                player_id,
                arena.radius_lookup(),
                logits_np[index],
                recent,
                masks[index],
                step_count=arena.env.current_step,
            ))
            player = arena.obs["players"][player_id]
            if int(player[2]) == 1:
                recent.append((int(player[0]), int(player[1])))
        actions.append(action)
    action_tensor = tensor_batch(np.asarray(actions, dtype=np.int64), torch.long)
    log_probs = distribution.log_prob(action_tensor)
    return (
        np.asarray(actions, dtype=np.int64),
        log_probs.detach().cpu().numpy(),
        values.detach().cpu().numpy(),
        next_hidden.detach().cpu().numpy(),
    )

# Agent dùng trong evaluation: chọn action greedy qua model + planner giống deploy-time behavior.
class GreedyPolicyAgent:
    def __init__(self, player_id, model, device):
        self.player_id = int(player_id)
        self.model = model
        self.device = device
        self.tracker = _deploy_ns["BombTracker"]()
        self.step_count = 0
        self.hidden = torch.zeros(1, HIDDEN_SIZE, dtype=torch.float32, device=device)
        self.recent_positions = deque(maxlen=6)
        self.action_counts = np.zeros(6, dtype=np.int64)
        self.bomb_exit_counts = []

    @torch.no_grad()
    def act(self, obs):
        self.step_count += 1
        radius_lookup = self.tracker.update(obs)
        features, scalars = _deploy_ns["encode"](obs, self.player_id, self.step_count, radius_lookup)
        mask = _deploy_ns["mask_actions"](obs, self.player_id, radius_lookup)
        logits, _, self.hidden = self.model(
            torch.as_tensor(features[None], dtype=torch.float32, device=self.device),
            torch.as_tensor(scalars[None], dtype=torch.float32, device=self.device),
            torch.as_tensor(mask[None], dtype=torch.bool, device=self.device),
            self.hidden,
        )
        self.hidden = self.hidden.detach()
        action = _deploy_ns["plan_action"](
            obs, self.player_id, radius_lookup, logits[0].detach().cpu().numpy(),
            self.recent_positions, mask, step_count=self.step_count,
        )
        self.recent_positions.append((int(obs["players"][self.player_id][0]), int(obs["players"][self.player_id][1])))
        self.action_counts[action] += 1
        if action == 5:
            self.bomb_exit_counts.append(_deploy_ns["bomb_exit_count"](obs, self.player_id, radius_lookup))
        return int(action)

# Sinh source code agent.py tự chứa inference helpers để submission không phụ thuộc notebook.
def deployment_agent_source():
    return _DEPLOY_AGENT_SOURCE_V2

def candidate_score(metrics):
    return (
        140.0 * metrics["sole_win_rate"]
        + 35.0 * metrics["rank_0_rate"]
        - 90.0 * metrics["average_rank"]
        + 0.025 * metrics["mean_steps"]
        - 50.0 * metrics["draw_rate"]
        - 18.0 * max(0.0, metrics["wait_rate"] - 0.08)
    )

def select_best_checkpoint(model, optimizer, matches=48):
    candidates = []
    ckpts = sorted(CHECKPOINT_DIR.glob("ppo_*.pth"))[-10:]
    for path in ckpts:
        trial = ActorCritic().to(DEVICE)
        state = torch.load(path, map_location=DEVICE)
        trial.load_state_dict(state["model_state_dict"])
        metrics = evaluate_policy(trial, matches=matches, seed=910_000 + len(candidates) * 2000)
        metrics["checkpoint"] = str(path)
        metrics["candidate_score"] = candidate_score(metrics)
        candidates.append((metrics["candidate_score"], path, metrics))
    current_metrics = evaluate_policy(model, matches=matches, seed=970_000)
    current_metrics["checkpoint"] = "current"
    current_metrics["candidate_score"] = candidate_score(current_metrics)
    candidates.append((current_metrics["candidate_score"], None, current_metrics))
    candidates.sort(key=lambda x: x[0], reverse=True)
    best_score, best_path, best_metrics = candidates[0]
    if best_path is not None:
        state = torch.load(best_path, map_location=DEVICE)
        model.load_state_dict(state["model_state_dict"])
        try:
            optimizer.load_state_dict(state["optimizer_state_dict"])
        except Exception:
            pass
    report = {"selected": best_metrics, "candidates": [m for _, _, m in candidates]}
    (ARTIFACT_DIR / "checkpoint_selection.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
    print("selected checkpoint:", json.dumps(best_metrics, indent=2))
    return best_metrics

print("V2 config ready:", CFG)
print("V2 curriculum:", [stage.name for stage in CURRICULUM])


## 8. Elite-Log Imitation Và PFSP League

Sau khi policy đủ ổn định, notebook học thêm từ top leaderboard agents.
Recurrent sequence imitation giúp học rhythm theo phase trận:

- early game: phá box, lấy item, không đặt bom vô nghĩa;
- mid game: kiểm soát corridor và chạy sau khi đặt bom;
- late game: đặt bom tạo pressure nhưng vẫn giữ survival.

**PFSP (Prioritized Fictitious Self-Play)** trong notebook:

- lưu các frozen snapshots/elite clones vào league;
- đánh giá độ khó của từng opponent;
- sample opponent theo priority thay vì uniform random;
- ưu tiên các đối thủ mà policy còn yếu hoặc thắng chưa ổn định;
- tránh overfit vào một nhóm opponent duy nhất.

Nói ngắn gọn: PFSP giúp training tập trung vào những matchup có giá trị
học cao nhất.


In [ ]:
# Cell 8: Recurrent imitation, planner distillation và PFSP league.
# Mục tiêu:
# - Học sequence behavior từ elite logs thay vì học từng frame độc lập.
# - Distill planner decisions vào policy để neural model gần hơn với action an toàn.
# - Tạo league gồm policy snapshots và elite clones.
# - Dùng PFSP (Prioritized Fictitious Self-Play) để sample đối thủ theo độ khó.
# PFSP trong notebook này:
# - Lưu các snapshot/clone vào pool.
# - Theo dõi strength/loss profile của từng opponent.
# - Ưu tiên sample đối thủ vừa đủ khó, tránh train quá nhiều với đối thủ quá yếu
#   hoặc quá mạnh, nhờ đó policy ổn định hơn.

# Cell 8: Recurrent imitation, planner distillation và PFSP league.
import glob
import os
from collections import Counter, defaultdict

V4_STARTED_AT = time.perf_counter()
V4_WALL_CLOCK_LIMIT_SECONDS = 9.25 * 3600
TOP_K = 8
TOP_MAX_RANK = 12
TOP_LOGS_PER_AGENT = 14
TOP_LOG_DIR = ARTIFACT_DIR / "v4_top_logs"
SEQUENCE_LENGTH = 32
SEQUENCE_STRIDE = 16
SEQUENCE_BURN_IN = 8
SEQUENCE_BC_EPOCHS_RESUME = 4
SEQUENCE_BC_EPOCHS_FRESH = 6
SEQUENCE_BATCH_SIZE = 8
ELITE_CLONE_COUNT = 8
ELITE_CLONE_EPOCHS = 2
MAX_SEQUENCES = 5000
MAX_ELITE_REPLAY_FRAMES = 100_000
PLANNER_LOGIT_BONUS = 2.75
PLANNER_QUERY_PROB = 0.30
PLANNER_DISTILL_COEF = 0.045
ELITE_BC_COEF = 0.020
ELITE_BC_BATCH = 256
TARGET_KL = 0.012
VALIDATE_EVERY = 8
VALIDATION_MATCHES = 32
FINAL_EVAL_MATCHES = 96
V4_RNG = random.Random(SEED + 404)

# The previous deploy planner occasionally measured slightly above 100 ms.
# Use the same smaller search both during training and in submission.
PLANNER_DEPTH = 6
PLANNER_WIDTH = 12
_DEPLOY_AGENT_SOURCE_V4 = deployment_agent_source().replace(
    "PLANNER_DEPTH = HORIZON\nPLANNER_WIDTH = 18",
    "PLANNER_DEPTH = 6\nPLANNER_WIDTH = 12",
)


# Sinh source code agent.py tự chứa inference helpers để submission không phụ thuộc notebook.
def deployment_agent_source():
    return _DEPLOY_AGENT_SOURCE_V4

CFG = TrainConfig(
    "meta_top_sequence_pfsp_v4",
    SEED,
    num_envs=20,
    episodes_per_update=24,
    bc_teacher_steps=0,
    bc_chunk_size=4096,
    bc_epochs=1,
    max_steps=500,
    gamma=0.995,
    gae_lambda=0.95,
    learning_rate=4.0e-5,
    clip_coef=0.12,
    ent_coef=0.008,
    vf_coef=0.55,
    max_grad_norm=0.50,
    ppo_epochs=2,
    minibatch_size=1024,
    eval_matches=FINAL_EVAL_MATCHES,
    checkpoint_every=4,
    league_snapshot_every=4,
    league_max_size=8,
)

# Exactly one trainable current policy is used in every arena. The remaining
# three seats are independent opponents, avoiding shared-policy cooperation.
CURRICULUM = [
    Stage("sequence_stabilize", 8, 1, 1, ("box_farmer", "smarter", "genius"), "opening_v4", 0.35),
    Stage("mixed_hard", 18, 1, 1, ("smarter", "genius", "tactical"), "balanced_v4", 0.55),
    Stage("elite_pfsp", 26, 1, 1, ("genius", "tactical"), "competitive_v4", 0.75),
    Stage("late_conversion", 28, 1, 1, ("genius", "tactical"), "late_v4", 0.86),
    Stage("robust_league", 16, 1, 1, ("genius", "tactical"), "robust_v4", 0.92),
]
STAGE_TARGET_RANK = {
    "sequence_stabilize": 0.82,
    "mixed_hard": 0.72,
    "elite_pfsp": 0.66,
    "late_conversion": 0.62,
    "robust_league": 0.60,
}
STAGE_EXTRA_CAP = {
    "sequence_stabilize": 2,
    "mixed_hard": 4,
    "elite_pfsp": 6,
    "late_conversion": 6,
    "robust_league": 6,
}

# Terminal rank dominates. Dense terms are deliberately small so the policy
# cannot obtain a high training return by farming while repeatedly finishing
# third or fourth.
REWARD_PROFILES = {
    "opening_v4": {
        "kill": 0.24, "box": 0.050, "item": 0.080,
        "bomb_target": 0.035, "bomb_enemy": 0.050, "trap_gain": 0.006,
        "wasted_bomb": -0.035, "low_exit_bomb": -0.55, "self_trap_bomb": -1.20,
        "wait": -0.010, "death": -0.30, "danger": -0.050,
        "potential": 0.012, "post_bomb_move": 0.008, "leave_blast_line": 0.012,
        "sole_win": 1.50, "draw": -1.00, "mutual_destruction": -1.50,
        "ranks": (3.0, 1.0, -1.0, -3.0),
    },
    "balanced_v4": {
        "kill": 0.32, "box": 0.030, "item": 0.070,
        "bomb_target": 0.030, "bomb_enemy": 0.070, "trap_gain": 0.008,
        "wasted_bomb": -0.040, "low_exit_bomb": -0.60, "self_trap_bomb": -1.30,
        "wait": -0.012, "death": -0.35, "danger": -0.055,
        "potential": 0.010, "post_bomb_move": 0.010, "leave_blast_line": 0.014,
        "sole_win": 1.60, "draw": -1.10, "mutual_destruction": -1.70,
        "ranks": (3.2, 1.0, -1.1, -3.2),
    },
    "competitive_v4": {
        "kill": 0.42, "box": 0.018, "item": 0.060,
        "bomb_target": 0.026, "bomb_enemy": 0.090, "trap_gain": 0.010,
        "wasted_bomb": -0.045, "low_exit_bomb": -0.65, "self_trap_bomb": -1.40,
        "wait": -0.014, "death": -0.40, "danger": -0.060,
        "potential": 0.008, "post_bomb_move": 0.012, "leave_blast_line": 0.016,
        "sole_win": 1.75, "draw": -1.20, "mutual_destruction": -1.90,
        "ranks": (3.4, 1.1, -1.2, -3.4),
    },
    "late_v4": {
        "kill": 0.55, "box": 0.010, "item": 0.050,
        "bomb_target": 0.022, "bomb_enemy": 0.110, "trap_gain": 0.013,
        "wasted_bomb": -0.025, "low_exit_bomb": -0.70, "self_trap_bomb": -1.50,
        "wait": -0.018, "death": -0.45, "danger": -0.065,
        "potential": 0.006, "post_bomb_move": 0.014, "leave_blast_line": 0.020,
        "sole_win": 2.00, "draw": -1.40, "mutual_destruction": -2.10,
        "ranks": (3.8, 1.2, -1.3, -3.8),
    },
    "robust_v4": {
        "kill": 0.50, "box": 0.012, "item": 0.055,
        "bomb_target": 0.024, "bomb_enemy": 0.105, "trap_gain": 0.012,
        "wasted_bomb": -0.030, "low_exit_bomb": -0.75, "self_trap_bomb": -1.60,
        "wait": -0.017, "death": -0.50, "danger": -0.070,
        "potential": 0.007, "post_bomb_move": 0.014, "leave_blast_line": 0.020,
        "sole_win": 2.00, "draw": -1.50, "mutual_destruction": -2.20,
        "ranks": (3.8, 1.2, -1.4, -4.0),
    },
}


# Reward shaping cân bằng survival, farming, bombing pressure và final rank.
def shaped_reward(
    before_obs,
    after_obs,
    before_stats,
    after_stats,
    player_id,
    profile_name,
    before_radius,
    after_radius,
    step=0,
):
    profile = REWARD_PROFILES[profile_name]
    step = int(step)
    reward = 0.0
    kill_gain = after_stats["kills"] - before_stats["kills"]
    box_gain = after_stats["boxes"] - before_stats["boxes"]
    item_gain = after_stats["items"] - before_stats["items"]
    reward += profile["kill"] * kill_gain
    reward += profile["box"] * (1.35 if step <= 90 else 0.35) * box_gain
    reward += profile["item"] * (1.20 if step <= 220 else 0.90) * item_gain

    was_alive = int(before_obs["players"][player_id][2]) == 1
    is_alive = int(after_obs["players"][player_id][2]) == 1
    if was_alive and not is_alive:
        reward += profile["death"]
    if is_alive:
        row = int(after_obs["players"][player_id][0])
        col = int(after_obs["players"][player_id][1])
        future = danger_planes(after_obs, horizon=HORIZON, radius_lookup=after_radius)
        danger_ticks = np.flatnonzero(future[:, row, col] > 0.5)
        if len(danger_ticks):
            reward += profile["danger"] * (1.0 + 0.15 * (HORIZON - int(danger_ticks[0])))

    before_phi = state_potential(before_obs, player_id, before_radius)
    after_phi = state_potential(after_obs, player_id, after_radius)
    reward += profile["potential"] * (CFG.gamma * after_phi - before_phi)

    bomb_delta = after_stats["bombs"] - before_stats["bombs"]
    before_row = int(before_obs["players"][player_id][0])
    before_col = int(before_obs["players"][player_id][1])
    after_row = int(after_obs["players"][player_id][0])
    after_col = int(after_obs["players"][player_id][1])
    if bomb_delta > 0:
        radius = 1 + int(before_obs["players"][player_id][4])
        blast = blast_tiles(before_obs["map"], before_row, before_col, radius)
        boxes_hit = sum(int(before_obs["map"][row, col]) == 2 for row, col in blast)
        enemies_hit = sum(
            index != player_id
            and int(before_obs["players"][index][2]) == 1
            and (int(before_obs["players"][index][0]), int(before_obs["players"][index][1])) in blast
            for index in range(len(before_obs["players"]))
        )
        if boxes_hit + enemies_hit:
            reward += profile["bomb_target"] * min(4, boxes_hit + 2 * enemies_hit)
        elif step >= 400:
            # Bombs are the fourth official step-500 tie-break. Keep this tiny:
            # survival and higher tie-break fields must still dominate.
            reward += 0.004
        else:
            reward += profile["wasted_bomb"]
        reward += profile["bomb_enemy"] * enemies_hit
        exits = escape_exit_count_after_bomb(before_obs, player_id, before_radius)
        if exits < MIN_BOMB_EXITS:
            reward += profile["low_exit_bomb"] * (MIN_BOMB_EXITS - exits)
        if survivability_depth_after_action(before_obs, player_id, 5, before_radius) < HORIZON:
            reward += profile["self_trap_bomb"]
    elif was_alive and is_alive:
        moved = (before_row, before_col) != (after_row, after_col)
        if moved:
            own_blast_before = any(
                int(bomb[3]) == player_id
                and (int(bomb[0]) == before_row or int(bomb[1]) == before_col)
                for bomb in before_obs["bombs"]
            )
            own_blast_after = any(
                int(bomb[3]) == player_id
                and (int(bomb[0]) == after_row or int(bomb[1]) == after_col)
                for bomb in before_obs["bombs"]
            )
            if own_blast_before and not own_blast_after:
                reward += profile["leave_blast_line"]
            if any(int(bomb[3]) == player_id for bomb in before_obs["bombs"]):
                reward += profile["post_bomb_move"]
        else:
            safe_mask = legal_action_mask(before_obs, player_id, before_radius, strict_safety=True)
            reward += profile["wait"] * (1.0 if np.any(safe_mask[1:5]) else 0.10)
    return float(reward)


# Tìm checkpoint/policy resume trong /kaggle/input để train tiếp từ bản mạnh trước đó.
def _resume_candidates():
    root = Path("/kaggle/input")
    if not root.exists():
        return []
    checkpoints = []
    policies = []
    archives = []
    for path in root.rglob("*"):
        if not path.is_file():
            continue
        lower = path.name.lower()
        if lower.endswith(".pth") and ("ppo_" in lower or "checkpoint" in lower or "resume" in lower):
            match = re.search(r"(\d{3,5})", lower)
            update = int(match.group(1)) if match else -1
            checkpoints.append((update, path))
        elif lower == "policy.pt":
            policies.append(path)
        elif lower.endswith(".zip") and (
            "submission" in lower or "continuation" in lower or "bundle" in lower
        ):
            archives.append(path)
    checkpoints.sort(key=lambda item: (item[0], str(item[1])), reverse=True)
    return [("checkpoint", path) for _, path in checkpoints] + [
        ("torchscript", path) for path in sorted(policies)
    ] + [("archive", path) for path in sorted(archives)]


# Load resume policy; nếu không có checkpoint hợp lệ thì dùng warm start/fresh fallback.
def load_resume_or_fresh(model):
    errors = []
    for source_type, source_path in _resume_candidates():
        try:
            actual_path = source_path
            if source_type == "archive":
                extract_dir = ARTIFACT_DIR / "resume_submission_v4"
                extract_dir.mkdir(parents=True, exist_ok=True)
                with zipfile.ZipFile(source_path) as archive:
                    archive.extractall(extract_dir)
                archive_checkpoints = sorted(extract_dir.rglob("*.pth"))
                archive_policies = sorted(extract_dir.rglob("policy.pt"))
                if archive_checkpoints:
                    actual_path = archive_checkpoints[-1]
                    source_type = "checkpoint"
                elif archive_policies:
                    actual_path = archive_policies[0]
                    source_type = "torchscript"
                else:
                    raise FileNotFoundError("archive contains neither a checkpoint nor policy.pt")
            if source_type == "checkpoint":
                state = torch.load(actual_path, map_location=DEVICE)
                state_dict = state.get("model_state_dict", state)
                model.load_state_dict(state_dict, strict=True)
                update = int(state.get("global_update", 0)) if isinstance(state, dict) else 0
            else:
                scripted = torch.jit.load(str(actual_path), map_location="cpu")
                model.load_state_dict(scripted.state_dict(), strict=True)
                update = 42
            checksum = float(sum(p.detach().float().abs().sum().cpu() for p in model.parameters()))
            print("Resume source:", source_type, actual_path)
            print(f"Loaded resume policy: update={update} checksum={checksum:.3f}")
            return update, str(actual_path), False
        except Exception as exc:
            errors.append(f"{source_path}: {type(exc).__name__}: {exc}")
    if errors:
        print("Ignored incompatible resume candidates:")
        for error in errors[:8]:
            print(" ", error)
    print("Resume source: none; using fresh tactical warm start")
    return 0, None, True


ROTATION_ACTION_MAPS = (
    np.asarray([0, 1, 2, 3, 4, 5], dtype=np.int64),
    np.asarray([0, 3, 4, 2, 1, 5], dtype=np.int64),
    np.asarray([0, 2, 1, 4, 3, 5], dtype=np.int64),
    np.asarray([0, 4, 3, 1, 2, 5], dtype=np.int64),
)


def rotate_sequence(sequence, turns):
    turns = int(turns) % 4
    if turns == 0:
        return (
            sequence["maps"],
            sequence["scalars"],
            sequence["masks"],
            sequence["actions"],
            sequence["weights"],
        )
    mapping = ROTATION_ACTION_MAPS[turns]
    maps = np.rot90(sequence["maps"], k=turns, axes=(-2, -1)).copy()
    actions = mapping[sequence["actions"]]
    masks = np.zeros_like(sequence["masks"])
    for old_action, new_action in enumerate(mapping):
        masks[:, new_action] = sequence["masks"][:, old_action]
    return maps, sequence["scalars"], masks, actions, sequence["weights"]


# Trích recurrent sequences từ elite logs, gán weight theo rank, phase và chất lượng action.
def extract_elite_sequences(log_dir=TOP_LOG_DIR, max_sequences=MAX_SEQUENCES):
    import pandas as pd

    selected = pd.read_csv(Path(log_dir) / "selected_top_logs.csv")
    sequences = []
    seen = set()
    action_histogram = Counter()
    invalid_actions = 0
    for _, row in selected.iterrows():
        path = Path(row["path"])
        sid = str(row["submission_id"])
        match_id = str(row["match_id"])
        key = (match_id, sid)
        if key in seen or not path.exists():
            continue
        seen.add(key)
        data = json.loads(path.read_text(encoding="utf-8"))
        team_ids = [str(value) for value in data["team_ids"]]
        if sid not in team_ids:
            continue
        slot = team_ids.index(sid)
        rank = int(data["ranks"][slot])
        leaderboard_rank = int(row["rank"])
        team_name = str(row["team"])
        survival = int(data["survival_steps"][slot])
        outcome_weight = (1.80, 1.25, 0.65, 0.22)[min(rank, 3)]
        meta_weight = 1.50 if leaderboard_rank <= 3 else (1.25 if leaderboard_rank <= 5 else 1.05)
        if "****" in team_name:
            meta_weight *= 1.15

        tracker = BombTracker()
        trajectory = []
        history = data.get("history", [])
        for step in range(1, len(history)):
            previous = history[step - 1]
            current = history[step]
            obs = obs_from_log_frame(previous)
            radius_lookup = tracker.update(obs)
            actions = current.get("actions")
            alive = bool(previous["alive"][slot])
            if actions is None or not alive:
                trajectory = []
                continue
            action = int(actions[slot])
            if not 0 <= action <= 5:
                trajectory = []
                continue
            features, scalars = encode_obs(obs, slot, step, radius_lookup=radius_lookup)
            mask = legal_action_mask(obs, slot, radius_lookup=radius_lookup, strict_safety=True)
            if not bool(mask[action]):
                invalid_actions += 1
                trajectory = []
                continue

            phase_weight = 1.0
            if step <= 90:
                phase_weight = 1.15
            elif step >= 300:
                phase_weight = 1.35
            if action == 5 and step >= 220:
                phase_weight *= 1.25
            if action == 0 and np.any(mask[1:5]):
                phase_weight *= 0.30
            if rank >= 3 and step >= max(1, survival - 12):
                phase_weight *= 0.25
            weight = outcome_weight * meta_weight * phase_weight
            trajectory.append((features, scalars, mask, action, weight))
            action_histogram[action] += 1

            if len(trajectory) >= SEQUENCE_LENGTH:
                start = len(trajectory) - SEQUENCE_LENGTH
                if start % SEQUENCE_STRIDE == 0:
                    chunk = trajectory[-SEQUENCE_LENGTH:]
                    sequences.append(
                        {
                            "maps": np.stack([item[0] for item in chunk]),
                            "scalars": np.stack([item[1] for item in chunk]),
                            "masks": np.stack([item[2] for item in chunk]),
                            "actions": np.asarray([item[3] for item in chunk], dtype=np.int64),
                            "weights": np.asarray([item[4] for item in chunk], dtype=np.float32),
                            "sid": sid,
                            "team": team_name,
                            "leaderboard_rank": leaderboard_rank,
                            "match_id": match_id,
                        }
                    )

    if len(sequences) > max_sequences:
        rng = np.random.default_rng(SEED + 405)
        quality = np.asarray(
            [
                float(sequence["weights"].mean())
                * (1.0 + 0.10 * np.count_nonzero(sequence["actions"] == 5))
                for sequence in sequences
            ],
            dtype=np.float64,
        )
        quality /= quality.sum()
        keep = rng.choice(len(sequences), size=max_sequences, replace=False, p=quality)
        sequences = [sequences[int(index)] for index in keep]
    print(
        "V4 sequence dataset:",
        f"sequences={len(sequences)}",
        f"frames={len(sequences) * SEQUENCE_LENGTH}",
        f"matches={len(seen)}",
        f"invalid_or_unsafe={invalid_actions}",
        f"actions={dict(sorted(action_histogram.items()))}",
    )
    if len(sequences) < 200:
        raise RuntimeError("Too few valid recurrent elite sequences were extracted")
    return sequences


# Recurrent behavior cloning: học chuỗi action để giữ rhythm đặt bom/chạy bom theo phase trận.
def train_sequence_bc(model, sequences, epochs, learning_rate, label):
    if not sequences:
        return []
    optimizer = optim.Adam(model.parameters(), lr=learning_rate, eps=1e-5)
    losses = []
    model.train()
    for epoch in range(int(epochs)):
        order = np.random.permutation(len(sequences))
        epoch_losses = []
        for start in range(0, len(order), SEQUENCE_BATCH_SIZE):
            selected = [sequences[int(index)] for index in order[start : start + SEQUENCE_BATCH_SIZE]]
            if not selected:
                continue
            rotated = [rotate_sequence(sequence, V4_RNG.randrange(4)) for sequence in selected]
            maps = np.stack([item[0] for item in rotated])
            scalars = np.stack([item[1] for item in rotated])
            masks = np.stack([item[2] for item in rotated])
            actions = np.stack([item[3] for item in rotated])
            weights = np.stack([item[4] for item in rotated])
            hidden = torch.zeros(len(selected), HIDDEN_SIZE, dtype=torch.float32, device=DEVICE)
            total_loss = torch.zeros((), dtype=torch.float32, device=DEVICE)
            total_weight = torch.zeros((), dtype=torch.float32, device=DEVICE)
            for tick in range(SEQUENCE_LENGTH):
                logits, _, hidden = model(
                    tensor_batch(maps[:, tick], torch.float32),
                    tensor_batch(scalars[:, tick], torch.float32),
                    tensor_batch(masks[:, tick], torch.bool),
                    hidden,
                )
                if tick < SEQUENCE_BURN_IN:
                    continue
                per_item = nn.functional.cross_entropy(
                    logits,
                    tensor_batch(actions[:, tick], torch.long),
                    reduction="none",
                )
                batch_weights = tensor_batch(weights[:, tick], torch.float32)
                total_loss = total_loss + (per_item * batch_weights).sum()
                total_weight = total_weight + batch_weights.sum()
            loss = total_loss / total_weight.clamp_min(1.0)
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), CFG.max_grad_norm)
            optimizer.step()
            epoch_losses.append(float(loss.detach().cpu()))
        mean_loss = float(np.mean(epoch_losses))
        losses.append(mean_loss)
        print(f"  sequence BC {label} epoch={epoch + 1}/{epochs} loss={mean_loss:.4f}")
    return losses


ELITE_REPLAY = None


@torch.no_grad()
# Cache elite frames để thêm regularization imitation trong PPO update.
def cache_elite_replay(model, sequences, max_frames=MAX_ELITE_REPLAY_FRAMES):
    global ELITE_REPLAY
    chosen = list(sequences)
    V4_RNG.shuffle(chosen)
    max_sequences = max(1, int(max_frames) // SEQUENCE_LENGTH)
    chosen = chosen[:max_sequences]
    maps_out = []
    scalars_out = []
    masks_out = []
    actions_out = []
    weights_out = []
    hidden_out = []
    model.eval()
    batch_size = 24
    for start in range(0, len(chosen), batch_size):
        batch = chosen[start : start + batch_size]
        maps = np.stack([sequence["maps"] for sequence in batch])
        scalars = np.stack([sequence["scalars"] for sequence in batch])
        masks = np.stack([sequence["masks"] for sequence in batch])
        hidden = torch.zeros(len(batch), HIDDEN_SIZE, dtype=torch.float32, device=DEVICE)
        for tick in range(SEQUENCE_LENGTH):
            if tick >= SEQUENCE_BURN_IN:
                maps_out.append(maps[:, tick])
                scalars_out.append(scalars[:, tick])
                masks_out.append(masks[:, tick])
                actions_out.append(np.stack([sequence["actions"][tick] for sequence in batch]))
                weights_out.append(np.stack([sequence["weights"][tick] for sequence in batch]))
                hidden_out.append(hidden.detach().cpu().numpy())
            _, _, hidden = model(
                tensor_batch(maps[:, tick], torch.float32),
                tensor_batch(scalars[:, tick], torch.float32),
                tensor_batch(masks[:, tick], torch.bool),
                hidden,
            )
    ELITE_REPLAY = {
        "maps": np.concatenate(maps_out, axis=0),
        "scalars": np.concatenate(scalars_out, axis=0),
        "masks": np.concatenate(masks_out, axis=0),
        "actions": np.concatenate(actions_out, axis=0).astype(np.int64),
        "weights": np.concatenate(weights_out, axis=0).astype(np.float32),
        "hidden": np.concatenate(hidden_out, axis=0).astype(np.float32),
    }
    ELITE_REPLAY["weights"] /= max(float(ELITE_REPLAY["weights"].mean()), 1e-6)
    print("Elite recurrent replay cached:", len(ELITE_REPLAY["actions"]))


# PFSPPool lưu opponent snapshots/clones và sampling weights cho Prioritized Fictitious Self-Play.
class PFSPPool:
    def __init__(self, snapshot_limit=8):
        self.entries = {}
        self.next_id = 0
        self.snapshot_limit = int(snapshot_limit)

    def add_model(self, model, label, kind):
        entry_id = self.next_id
        self.next_id += 1
        frozen = copy.deepcopy(model).to(DEVICE).eval()
        for parameter in frozen.parameters():
            parameter.requires_grad_(False)
        self.entries[entry_id] = {
            "id": entry_id,
            "model": frozen,
            "label": str(label),
            "kind": str(kind),
            "games": 0,
            "score": 0.50,
        }
        if kind == "snapshot":
            snapshots = [
                entry for entry in self.entries.values() if entry["kind"] == "snapshot"
            ]
            while len(snapshots) > self.snapshot_limit:
                victim = min(snapshots, key=lambda entry: entry["id"])
                del self.entries[victim["id"]]
                snapshots = [
                    entry for entry in self.entries.values() if entry["kind"] == "snapshot"
                ]
        return entry_id

    def sample(self, rng, prefer_elite=True):
        entries = list(self.entries.values())
        if not entries:
            return None
        if prefer_elite:
            preferred = [entry for entry in entries if entry["kind"] == "elite"]
        else:
            preferred = [entry for entry in entries if entry["kind"] == "snapshot"]
        candidates = preferred or entries
        weights = []
        for entry in candidates:
            win_rate = float(entry["score"])
            uncertainty = 1.0 / np.sqrt(1.0 + float(entry["games"]))
            pfsp = (1.0 - abs(2.0 * win_rate - 1.0)) ** 2
            weights.append(0.12 + pfsp + 0.35 * uncertainty)
        return rng.choices(candidates, weights=weights, k=1)[0]

    def update(self, entry_id, result):
        if entry_id not in self.entries:
            return
        entry = self.entries[entry_id]
        entry["games"] += 1
        alpha = 0.10 if entry["games"] >= 10 else 1.0 / entry["games"]
        entry["score"] = (1.0 - alpha) * entry["score"] + alpha * float(result)

    def summary(self):
        return [
            {
                "id": entry["id"],
                "label": entry["label"],
                "kind": entry["kind"],
                "games": entry["games"],
                "current_pairwise_score": round(float(entry["score"]), 4),
            }
            for entry in sorted(self.entries.values(), key=lambda item: item["id"])
        ]


PFSP_POOL = PFSPPool(snapshot_limit=CFG.league_max_size)


class V4Arena:
    def __init__(self, arena_index, stage, rng, pool):
        self.arena_index = arena_index
        self.reset(stage, rng, pool)

    def reset(self, stage, rng, pool):
        self.stage = stage
        self.pool = pool
        seed = rng.randrange(1_000_000_000)
        self.env = BomberEnv(seed=seed, max_steps=CFG.max_steps)
        self.obs = self.env.reset(seed=seed)
        current_player = rng.randrange(4)
        self.policy_ids = {current_player}
        self.current_player = current_player
        self.opponents = {}
        model_count = 0
        for player_id in range(4):
            if player_id == current_player:
                continue
            use_model = bool(pool.entries) and rng.random() < stage.league_probability
            if use_model:
                prefer_elite = rng.random() < (0.65 if stage.league_probability >= 0.70 else 0.50)
                entry = pool.sample(rng, prefer_elite=prefer_elite)
                if entry is not None:
                    self.opponents[player_id] = {"kind": "model", "entry_id": entry["id"]}
                    model_count += 1
                    continue
            self.opponents[player_id] = {
                "kind": "baseline",
                "agent": BASELINE_CLASSES[rng.choice(stage.baselines)](player_id),
            }
        # Hard stages always contain at least one learned opponent.
        if stage.league_probability >= 0.70 and model_count == 0 and pool.entries:
            replace_id = next(iter(self.opponents))
            entry = pool.sample(rng, prefer_elite=True)
            self.opponents[replace_id] = {"kind": "model", "entry_id": entry["id"]}
        self.trajectories = {current_player: []}
        self.hidden_by_player = {
            player_id: np.zeros(HIDDEN_SIZE, dtype=np.float32) for player_id in range(4)
        }
        self.recent_by_player = {player_id: deque(maxlen=6) for player_id in range(4)}
        self.alive_mask = [True] * 4
        self.death_order = []

    def radius_lookup(self):
        return radius_lookup_from_env(self.env)

    def step(self, staged_transitions, model_actions):
        before_obs = self.obs
        before_radius = self.radius_lookup()
        before_stats = [dict(player.stats) for player in self.env.players]
        actions = []
        for player_id, player in enumerate(self.env.players):
            if not player.alive:
                action = 0
            elif player_id == self.current_player:
                action = int(staged_transitions[player_id]["action"])
            elif self.opponents[player_id]["kind"] == "model":
                action = int(model_actions[player_id])
            else:
                action = int(self.opponents[player_id]["agent"].act(before_obs))
            actions.append(action)
            if player.alive:
                self.recent_by_player[player_id].append((int(player.x), int(player.y)))

        before_step = int(self.env.current_step)
        self.obs, terminated, truncated = self.env.step(actions)
        done = bool(terminated or truncated)
        after_radius = self.radius_lookup()

        deaths = []
        for player_id, player in enumerate(self.env.players):
            if self.alive_mask[player_id] and not player.alive:
                self.alive_mask[player_id] = False
                deaths.append(player_id)
        if deaths:
            self.death_order.append(deaths)

        transition = staged_transitions.get(self.current_player)
        if transition is not None:
            transition["reward"] = shaped_reward(
                before_obs,
                self.obs,
                before_stats[self.current_player],
                dict(self.env.players[self.current_player].stats),
                self.current_player,
                self.stage.reward_profile,
                before_radius,
                after_radius,
                step=before_step,
            )
            transition["done"] = bool(done or not self.env.players[self.current_player].alive)
            self.trajectories[self.current_player].append(transition)

        if not done:
            return False, None

        ranks = ranks_from_state(self.env.players, self.death_order)
        profile = REWARD_PROFILES[self.stage.reward_profile]
        trajectory = self.trajectories[self.current_player]
        best_rank = min(ranks)
        winners = {index for index, rank in enumerate(ranks) if rank == best_rank}
        final_death_group = set(self.death_order[-1]) if self.death_order else set()
        if trajectory:
            own_rank = min(int(ranks[self.current_player]), 3)
            trajectory[-1]["reward"] += float(profile["ranks"][own_rank])
            if self.current_player in winners:
                trajectory[-1]["reward"] += (
                    float(profile["sole_win"]) if len(winners) == 1 else float(profile["draw"])
                )
            if self.current_player in final_death_group and len(final_death_group) > 1:
                trajectory[-1]["reward"] += float(profile["mutual_destruction"])
            trajectory[-1]["done"] = True

        for player_id, opponent in self.opponents.items():
            if opponent["kind"] != "model":
                continue
            if ranks[self.current_player] < ranks[player_id]:
                result = 1.0
            elif ranks[self.current_player] == ranks[player_id]:
                result = 0.5
            else:
                result = 0.0
            self.pool.update(opponent["entry_id"], result)
        return True, ranks


def _planner_query(request):
    arena = request["arena"]
    player_id = request["player_id"]
    obs = arena.obs
    step = int(arena.env.current_step)
    player = obs["players"][player_id]
    row, col = int(player[0]), int(player[1])
    enemies = [
        abs(int(enemy[0]) - row) + abs(int(enemy[1]) - col)
        for index, enemy in enumerate(obs["players"])
        if index != player_id and int(enemy[2]) == 1
    ]
    nearby_enemy = bool(enemies and min(enemies) <= 5)
    threatened = bool(
        np.any(danger_planes(obs, HORIZON, request["radius_lookup"])[:, row, col] > 0.5)
    )
    blast = blast_tiles(
        obs["map"],
        row,
        col,
        1 + int(player[4]),
    )
    bomb_has_target = bool(request["mask"][5]) and (
        nearby_enemy
        or any(int(obs["map"][target_row, target_col]) == 2 for target_row, target_col in blast)
        or step >= 320
    )
    tactical = threatened or nearby_enemy or step >= 260 or bomb_has_target
    probability = PLANNER_QUERY_PROB + (0.18 if step >= 300 else 0.0)
    return tactical or V4_RNG.random() < probability


@torch.no_grad()
def sample_current_requests(model, requests):
    maps = np.stack([request["map"] for request in requests])
    scalars = np.stack([request["scalars"] for request in requests])
    masks = np.stack([request["mask"] for request in requests])
    hidden = np.stack([request["hidden"] for request in requests])
    logits, values, next_hidden = model(
        tensor_batch(maps, torch.float32),
        tensor_batch(scalars, torch.float32),
        tensor_batch(masks, torch.bool),
        tensor_batch(hidden, torch.float32),
    )
    logits_np = logits.detach().cpu().numpy()
    planner_bias = np.zeros((len(requests), 6), dtype=np.float32)
    teacher_actions = np.full(len(requests), -1, dtype=np.int64)
    for index, request in enumerate(requests):
        if not _planner_query(request):
            continue
        action = beam_plan_action(
            request["arena"].obs,
            request["player_id"],
            radius_lookup=request["radius_lookup"],
            policy_logits=logits_np[index],
            recent_positions=request["arena"].recent_by_player[request["player_id"]],
            action_mask=request["mask"],
            step_count=request["arena"].env.current_step,
        )
        planner_bias[index, int(action)] = PLANNER_LOGIT_BONUS
        teacher_actions[index] = int(action)
    distribution = Categorical(logits=logits + tensor_batch(planner_bias, torch.float32))
    actions = distribution.sample()
    return (
        actions.cpu().numpy(),
        distribution.log_prob(actions).cpu().numpy(),
        values.cpu().numpy(),
        next_hidden.cpu().numpy(),
        planner_bias,
        teacher_actions,
    )


@torch.no_grad()
# Sample actions từ frozen opponent models trong league.
def sample_model_opponents(pool, groups):
    results = {}
    for entry_id, requests in groups.items():
        entry = pool.entries.get(entry_id)
        if entry is None or not requests:
            continue
        model = entry["model"]
        maps = np.stack([request["map"] for request in requests])
        scalars = np.stack([request["scalars"] for request in requests])
        masks = np.stack([request["mask"] for request in requests])
        hidden = np.stack([request["hidden"] for request in requests])
        logits, _, next_hidden = model(
            tensor_batch(maps, torch.float32),
            tensor_batch(scalars, torch.float32),
            tensor_batch(masks, torch.bool),
            tensor_batch(hidden, torch.float32),
        )
        distribution = Categorical(logits=logits)
        actions = distribution.sample().cpu().numpy()
        for request, action, next_state in zip(requests, actions, next_hidden.cpu().numpy()):
            request["arena"].hidden_by_player[request["player_id"]] = next_state.copy()
            results[(id(request["arena"]), request["player_id"])] = int(action)
    return results


# PPO rollout phiên bản PFSP: trộn current policy, snapshots, elite clones và planner teacher.
def collect_ppo_batch_v4(model, stage, rng, pool):
    model.eval()
    arenas = [V4Arena(index, stage, rng, pool) for index in range(CFG.num_envs)]
    records = []
    episode_metrics = []
    while len(episode_metrics) < CFG.episodes_per_update:
        requests = []
        opponent_groups = defaultdict(list)
        for arena in arenas:
            radius_lookup = arena.radius_lookup()
            player_id = arena.current_player
            if arena.env.players[player_id].alive:
                features, scalars = encode_obs(
                    arena.obs, player_id, arena.env.current_step, radius_lookup
                )
                mask = legal_action_mask(
                    arena.obs, player_id, radius_lookup, strict_safety=True
                )
                requests.append(
                    {
                        "arena": arena,
                        "player_id": player_id,
                        "map": features,
                        "scalars": scalars,
                        "mask": mask,
                        "hidden": arena.hidden_by_player[player_id].copy(),
                        "radius_lookup": radius_lookup,
                    }
                )
            for opponent_id, opponent in arena.opponents.items():
                if opponent["kind"] != "model" or not arena.env.players[opponent_id].alive:
                    continue
                features, scalars = encode_obs(
                    arena.obs, opponent_id, arena.env.current_step, radius_lookup
                )
                mask = legal_action_mask(
                    arena.obs, opponent_id, radius_lookup, strict_safety=True
                )
                opponent_groups[opponent["entry_id"]].append(
                    {
                        "arena": arena,
                        "player_id": opponent_id,
                        "map": features,
                        "scalars": scalars,
                        "mask": mask,
                        "hidden": arena.hidden_by_player[opponent_id].copy(),
                    }
                )

        staged = {id(arena): {} for arena in arenas}
        if requests:
            outputs = sample_current_requests(model, requests)
            actions, log_probs, values, next_hidden, biases, teachers = outputs
            for request, action, log_prob, value, hidden, bias, teacher in zip(
                requests, actions, log_probs, values, next_hidden, biases, teachers
            ):
                arena = request["arena"]
                player_id = request["player_id"]
                arena.hidden_by_player[player_id] = hidden.copy()
                staged[id(arena)][player_id] = {
                    "map": request["map"],
                    "scalars": request["scalars"],
                    "mask": request["mask"],
                    "hidden": request["hidden"],
                    "action": int(action),
                    "log_prob": float(log_prob),
                    "value": float(value),
                    "planner_bias": bias,
                    "teacher_action": int(teacher),
                }
        model_actions_flat = sample_model_opponents(pool, opponent_groups)
        for arena in arenas:
            model_actions = {
                player_id: model_actions_flat[(id(arena), player_id)]
                for player_id, opponent in arena.opponents.items()
                if opponent["kind"] == "model" and (id(arena), player_id) in model_actions_flat
            }
            finished, ranks = arena.step(staged[id(arena)], model_actions)
            if not finished:
                continue
            trajectory = arena.trajectories[arena.current_player]
            if trajectory:
                records.extend(finish_trajectory(trajectory))
            own_rank = int(ranks[arena.current_player])
            episode_metrics.append(
                {
                    "mean_policy_rank": float(own_rank),
                    "best_policy_rank": own_rank,
                    "rank3": int(own_rank >= 3),
                    "steps": int(arena.env.current_step),
                }
            )
            arena.reset(stage, rng, pool)
    return records, episode_metrics


# PPO update có thêm planner distillation và elite behavior cloning regularization.
def ppo_update_v4(model, optimizer, records):
    maps = np.stack([record["map"] for record in records])
    scalars = np.stack([record["scalars"] for record in records])
    masks = np.stack([record["mask"] for record in records])
    hidden = np.stack([record["hidden"] for record in records])
    actions = np.asarray([record["action"] for record in records], dtype=np.int64)
    old_log_probs = np.asarray([record["log_prob"] for record in records], dtype=np.float32)
    old_values = np.asarray([record["value"] for record in records], dtype=np.float32)
    advantages = np.asarray([record["advantage"] for record in records], dtype=np.float32)
    returns = np.asarray([record["return"] for record in records], dtype=np.float32)
    planner_bias = np.stack([record["planner_bias"] for record in records])
    teacher_actions = np.asarray(
        [record["teacher_action"] for record in records], dtype=np.int64
    )
    advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)
    metrics = defaultdict(list)
    model.train()
    stop_for_kl = False
    for _ in range(CFG.ppo_epochs):
        order = np.random.permutation(len(records))
        for start in range(0, len(order), CFG.minibatch_size):
            indices = order[start : start + CFG.minibatch_size]
            raw_logits, values, _ = model(
                tensor_batch(maps[indices], torch.float32),
                tensor_batch(scalars[indices], torch.float32),
                tensor_batch(masks[indices], torch.bool),
                tensor_batch(hidden[indices], torch.float32),
            )
            biased_logits = raw_logits + tensor_batch(planner_bias[indices], torch.float32)
            distribution = Categorical(logits=biased_logits)
            new_log_probs = distribution.log_prob(tensor_batch(actions[indices], torch.long))
            old_lp = tensor_batch(old_log_probs[indices], torch.float32)
            log_ratio = new_log_probs - old_lp
            ratio = log_ratio.exp()
            batch_advantages = tensor_batch(advantages[indices], torch.float32)
            unclipped = ratio * batch_advantages
            clipped = torch.clamp(
                ratio, 1.0 - CFG.clip_coef, 1.0 + CFG.clip_coef
            ) * batch_advantages
            policy_loss = -torch.min(unclipped, clipped).mean()

            batch_returns = tensor_batch(returns[indices], torch.float32)
            batch_old_values = tensor_batch(old_values[indices], torch.float32)
            value_clipped = batch_old_values + torch.clamp(
                values - batch_old_values, -CFG.clip_coef, CFG.clip_coef
            )
            value_loss = 0.5 * torch.maximum(
                (values - batch_returns).pow(2),
                (value_clipped - batch_returns).pow(2),
            ).mean()
            entropy = distribution.entropy().mean()

            teacher_mask = teacher_actions[indices] >= 0
            planner_loss = torch.zeros((), dtype=torch.float32, device=DEVICE)
            if np.any(teacher_mask):
                planner_loss = nn.functional.cross_entropy(
                    raw_logits[tensor_batch(teacher_mask, torch.bool)],
                    tensor_batch(teacher_actions[indices][teacher_mask], torch.long),
                )

            bc_loss = torch.zeros((), dtype=torch.float32, device=DEVICE)
            if ELITE_REPLAY is not None:
                replay_indices = np.random.randint(
                    0, len(ELITE_REPLAY["actions"]), size=ELITE_BC_BATCH
                )
                bc_logits, _, _ = model(
                    tensor_batch(ELITE_REPLAY["maps"][replay_indices], torch.float32),
                    tensor_batch(ELITE_REPLAY["scalars"][replay_indices], torch.float32),
                    tensor_batch(ELITE_REPLAY["masks"][replay_indices], torch.bool),
                    tensor_batch(ELITE_REPLAY["hidden"][replay_indices], torch.float32),
                )
                per_item = nn.functional.cross_entropy(
                    bc_logits,
                    tensor_batch(ELITE_REPLAY["actions"][replay_indices], torch.long),
                    reduction="none",
                )
                bc_loss = (
                    per_item
                    * tensor_batch(ELITE_REPLAY["weights"][replay_indices], torch.float32)
                ).mean()

            loss = (
                policy_loss
                + CFG.vf_coef * value_loss
                - CFG.ent_coef * entropy
                + PLANNER_DISTILL_COEF * planner_loss
                + ELITE_BC_COEF * bc_loss
            )
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), CFG.max_grad_norm)
            optimizer.step()

            approx_kl = float(((ratio - 1.0) - log_ratio).mean().detach().cpu())
            metrics["policy_loss"].append(float(policy_loss.detach().cpu()))
            metrics["value_loss"].append(float(value_loss.detach().cpu()))
            metrics["entropy"].append(float(entropy.detach().cpu()))
            metrics["approx_kl"].append(approx_kl)
            metrics["planner_loss"].append(float(planner_loss.detach().cpu()))
            metrics["bc_loss"].append(float(bc_loss.detach().cpu()))
            if approx_kl > TARGET_KL:
                stop_for_kl = True
                break
        if stop_for_kl:
            break
    result = {key: float(np.mean(values)) for key, values in metrics.items()}
    result["kl_early_stop"] = bool(stop_for_kl)
    return result


class V4PolicyOpponent:
    def __init__(self, player_id, model):
        self.player_id = int(player_id)
        self.model = model
        self.hidden = torch.zeros(1, HIDDEN_SIZE, dtype=torch.float32, device=DEVICE)
        self.tracker = BombTracker()

    @torch.no_grad()
    def act(self, obs, step_count):
        radius_lookup = self.tracker.update(obs)
        features, scalars = encode_obs(obs, self.player_id, step_count, radius_lookup)
        mask = legal_action_mask(obs, self.player_id, radius_lookup, strict_safety=True)
        logits, _, self.hidden = self.model(
            tensor_batch(features[None], torch.float32),
            tensor_batch(scalars[None], torch.float32),
            tensor_batch(mask[None], torch.bool),
            self.hidden,
        )
        self.hidden = self.hidden.detach()
        return int(torch.argmax(logits[0]).item())


class V4DeployEvalAgent:
    def __init__(self, player_id, model):
        self.player_id = int(player_id)
        self.model = model
        self.hidden = torch.zeros(1, HIDDEN_SIZE, dtype=torch.float32, device=DEVICE)
        self.tracker = BombTracker()
        self.recent = deque(maxlen=6)
        self.action_counts = np.zeros(6, dtype=np.int64)

    @torch.no_grad()
    def act(self, obs, step_count):
        radius_lookup = self.tracker.update(obs)
        features, scalars = encode_obs(obs, self.player_id, step_count, radius_lookup)
        mask = legal_action_mask(obs, self.player_id, radius_lookup, strict_safety=True)
        logits, _, self.hidden = self.model(
            tensor_batch(features[None], torch.float32),
            tensor_batch(scalars[None], torch.float32),
            tensor_batch(mask[None], torch.bool),
            self.hidden,
        )
        self.hidden = self.hidden.detach()
        action = beam_plan_action(
            obs,
            self.player_id,
            radius_lookup=radius_lookup,
            policy_logits=logits[0].detach().cpu().numpy(),
            recent_positions=self.recent,
            action_mask=mask,
            step_count=step_count,
        )
        self.recent.append(
            (int(obs["players"][self.player_id][0]), int(obs["players"][self.player_id][1]))
        )
        self.action_counts[action] += 1
        return int(action)


# Hard evaluation với pool đối thủ mạnh hơn baseline thường, dùng để chọn checkpoint.
def evaluate_hard_policy(model, pool, matches=VALIDATION_MATCHES, seed=2_000_000):
    rng = random.Random(seed)
    ranks = []
    wins = 0
    draws = 0
    rank3 = 0
    steps = []
    action_counts = np.zeros(6, dtype=np.int64)
    model_entries = list(pool.entries.values())
    for match_index in range(int(matches)):
        seat = match_index % 4
        env = BomberEnv(seed=seed + match_index, max_steps=CFG.max_steps)
        obs = env.reset(seed=seed + match_index)
        current = V4DeployEvalAgent(seat, model)
        agents = []
        for player_id in range(4):
            if player_id == seat:
                agents.append(current)
                continue
            use_model = bool(model_entries) and ((player_id + match_index) % 3 != 0)
            if use_model:
                entry = rng.choice(model_entries)
                agents.append(V4PolicyOpponent(player_id, entry["model"]))
            else:
                baseline = TacticalRuleAgent if (player_id + match_index) % 2 else GeniusRuleAgent
                agents.append(baseline(player_id))
        alive_mask = [True] * 4
        death_order = []
        done = False
        while not done:
            actions = []
            for player_id, agent in enumerate(agents):
                if not env.players[player_id].alive:
                    actions.append(0)
                elif player_id == seat or isinstance(agent, V4PolicyOpponent):
                    actions.append(int(agent.act(obs, env.current_step)))
                else:
                    actions.append(int(agent.act(obs)))
            obs, terminated, truncated = env.step(actions)
            done = bool(terminated or truncated)
            deaths = []
            for player_id, player in enumerate(env.players):
                if alive_mask[player_id] and not player.alive:
                    alive_mask[player_id] = False
                    deaths.append(player_id)
            if deaths:
                death_order.append(deaths)
        match_ranks = ranks_from_state(env.players, death_order)
        own_rank = int(match_ranks[seat])
        ranks.append(own_rank)
        rank3 += int(own_rank >= 3)
        best_rank = min(match_ranks)
        winners = [index for index, rank in enumerate(match_ranks) if rank == best_rank]
        wins += int(seat in winners and len(winners) == 1)
        draws += int(seat in winners and len(winners) > 1)
        steps.append(int(env.current_step))
        action_counts += current.action_counts
    total_actions = max(int(action_counts.sum()), 1)
    result = {
        "matches": int(matches),
        "average_rank": float(np.mean(ranks)),
        "rank_0_rate": float(np.mean(np.asarray(ranks) == 0)),
        "rank_3_rate": float(rank3 / matches),
        "sole_win_rate": float(wins / matches),
        "draw_rate": float(draws / matches),
        "mean_steps": float(np.mean(steps)),
        "rank_histogram": {str(rank): int(np.count_nonzero(np.asarray(ranks) == rank)) for rank in range(4)},
        "action_histogram": {str(action): int(action_counts[action]) for action in range(6)},
        "wait_rate": float(action_counts[0] / total_actions),
        "bomb_rate": float(action_counts[5] / total_actions),
    }
    print(json.dumps(result, indent=2))
    return result


def hard_validation_score(metrics):
    return (
        -100.0 * metrics["average_rank"]
        + 24.0 * metrics["sole_win_rate"]
        - 28.0 * metrics["rank_3_rate"]
        - 12.0 * metrics["draw_rate"]
    )


def build_elite_clones(base_model, sequences, pool):
    grouped = defaultdict(list)
    for sequence in sequences:
        grouped[sequence["sid"]].append(sequence)
    selected = sorted(grouped.items(), key=lambda item: len(item[1]), reverse=True)[:ELITE_CLONE_COUNT]
    for index, (sid, own_sequences) in enumerate(selected):
        if len(own_sequences) < 12:
            continue
        clone = copy.deepcopy(base_model).to(DEVICE)
        train_sequence_bc(
            clone,
            own_sequences,
            epochs=ELITE_CLONE_EPOCHS,
            learning_rate=3.0e-5,
            label=f"elite_{index + 1}",
        )
        team = own_sequences[0]["team"]
        pool.add_model(clone, f"elite:{team}:{sid[:8]}", "elite")
        del clone
    print("Elite PFSP entries:", len([e for e in pool.entries.values() if e["kind"] == "elite"]))


# Curriculum PFSP: train theo stages, validate định kỳ, lưu checkpoint tốt nhất.
def train_curriculum_v4(model, optimizer, pool, start_update=0):
    rng = random.Random(CFG.seed + 406)
    history = []
    global_update = int(start_update)
    pool.add_model(model, f"resume_{start_update}", "snapshot")
    initial_metrics = evaluate_hard_policy(
        model, pool, matches=VALIDATION_MATCHES, seed=2_100_000
    )
    best_score = hard_validation_score(initial_metrics)
    best_path = save_checkpoint(model, optimizer, global_update, "v4_initial")
    bad_checks = 0
    print(f"Initial hard score={best_score:.3f} avg_rank={initial_metrics['average_rank']:.3f}")

    for stage in CURRICULUM:
        target_rank = STAGE_TARGET_RANK[stage.name]
        max_stage_updates = stage.updates + STAGE_EXTRA_CAP[stage.name]
        stage_rows = []
        stage_update = 0
        print(
            f"\n=== V4 Stage: {stage.name} "
            f"(base={stage.updates}, max={max_stage_updates}, target_rank<={target_rank:.2f}) ==="
        )
        while stage_update < max_stage_updates:
            if stage_update >= stage.updates and len(stage_rows) >= 4:
                rolling_rank = float(np.mean([item["mean_rank"] for item in stage_rows[-4:]]))
                if rolling_rank <= target_rank:
                    print(
                        f"  curriculum gate passed: rolling_rank={rolling_rank:.3f} "
                        f"target={target_rank:.3f}"
                    )
                    break
                print(
                    f"  extending stage: rolling_rank={rolling_rank:.3f} "
                    f"target={target_rank:.3f}"
                )
            stage_update += 1
            if time.perf_counter() - V4_STARTED_AT > V4_WALL_CLOCK_LIMIT_SECONDS:
                print("V4 wall-clock cutoff reached; preserving evaluation/export time")
                save_checkpoint(model, optimizer, global_update, "v4_wall_clock_cutoff")
                if best_path is not None:
                    state = torch.load(best_path, map_location=DEVICE)
                    model.load_state_dict(state["model_state_dict"])
                (ARTIFACT_DIR / "training_history_v4.json").write_text(
                    json.dumps(history, indent=2), encoding="utf-8"
                )
                return history, best_path

            started = time.perf_counter()
            records, episodes = collect_ppo_batch_v4(model, stage, rng, pool)
            update_metrics = ppo_update_v4(model, optimizer, records)
            global_update += 1
            if global_update % CFG.league_snapshot_every == 0:
                pool.add_model(model, f"snapshot_{global_update}", "snapshot")
            row = {
                "global_update": global_update,
                "stage": stage.name,
                "stage_update": stage_update,
                "transitions": len(records),
                "mean_rank": float(np.mean([episode["mean_policy_rank"] for episode in episodes])),
                "rank0_rate": float(np.mean([episode["best_policy_rank"] == 0 for episode in episodes])),
                "rank3_rate": float(np.mean([episode["rank3"] for episode in episodes])),
                "mean_steps": float(np.mean([episode["steps"] for episode in episodes])),
                "seconds": time.perf_counter() - started,
                **update_metrics,
            }
            history.append(row)
            stage_rows.append(row)
            print(
                f"[{global_update:04d}] {stage.name:<19} transitions={row['transitions']:>6} "
                f"rank={row['mean_rank']:.3f} r0={row['rank0_rate']:.1%} r3={row['rank3_rate']:.1%} "
                f"entropy={row['entropy']:.3f} kl={row['approx_kl']:.4f} "
                f"planner={row['planner_loss']:.3f} bc={row['bc_loss']:.3f} "
                f"time={row['seconds']:.1f}s"
            )
            if global_update % CFG.checkpoint_every == 0:
                checkpoint = save_checkpoint(model, optimizer, global_update, stage.name)
                print("  checkpoint:", checkpoint)

            if global_update % VALIDATE_EVERY == 0:
                metrics = evaluate_hard_policy(
                    model,
                    pool,
                    matches=VALIDATION_MATCHES,
                    seed=2_100_000 + (global_update % 3) * 10_000,
                )
                score = hard_validation_score(metrics)
                print(
                    f"  hard score={score:.3f} best={best_score:.3f} "
                    f"avg_rank={metrics['average_rank']:.3f} rank3={metrics['rank_3_rate']:.1%}"
                )
                if score > best_score:
                    best_score = score
                    best_path = save_checkpoint(model, optimizer, global_update, "v4_best")
                    bad_checks = 0
                    print("  new V4 best:", best_path)
                elif score < best_score - 25.0:
                    bad_checks += 1
                    print("  hard regression warning:", bad_checks)
                    if bad_checks >= 3 and best_path is not None:
                        state = torch.load(best_path, map_location=DEVICE)
                        model.load_state_dict(state["model_state_dict"])
                        optimizer = optim.Adam(model.parameters(), lr=CFG.learning_rate, eps=1e-5)
                        pool.add_model(model, f"rollback_{global_update}", "snapshot")
                        bad_checks = 0
                        print("  rolled back to:", best_path)
                else:
                    bad_checks = 0
                (ARTIFACT_DIR / "pfsp_pool_v4.json").write_text(
                    json.dumps(pool.summary(), indent=2), encoding="utf-8"
                )

    (ARTIFACT_DIR / "training_history_v4.json").write_text(
        json.dumps(history, indent=2), encoding="utf-8"
    )
    if best_path is not None:
        state = torch.load(best_path, map_location=DEVICE)
        model.load_state_dict(state["model_state_dict"])
    return history, best_path


def select_best_checkpoint_v4(model, pool, matches=48):
    preferred = (
        sorted(CHECKPOINT_DIR.glob("*v4_resume_raw.pth"))
        + sorted(CHECKPOINT_DIR.glob("*v4_after_sequence_bc.pth"))
        + sorted(CHECKPOINT_DIR.glob("*v4_best.pth"))
    )
    recent = sorted(CHECKPOINT_DIR.glob("ppo_*.pth"))[-5:]
    paths = []
    for path in preferred + recent:
        if path not in paths:
            paths.append(path)
    candidates = []
    for path in paths:
        trial = ActorCritic().to(DEVICE)
        state = torch.load(path, map_location=DEVICE)
        trial.load_state_dict(state["model_state_dict"])
        metrics = evaluate_hard_policy(trial, pool, matches=matches, seed=2_600_000)
        metrics["checkpoint"] = str(path)
        metrics["candidate_score"] = hard_validation_score(metrics)
        candidates.append((metrics["candidate_score"], path, metrics))
        del trial
    current_metrics = evaluate_hard_policy(model, pool, matches=matches, seed=2_600_000)
    current_metrics["checkpoint"] = "current"
    current_metrics["candidate_score"] = hard_validation_score(current_metrics)
    candidates.append((current_metrics["candidate_score"], None, current_metrics))
    candidates.sort(key=lambda item: item[0], reverse=True)
    _, best_path, best_metrics = candidates[0]
    if best_path is not None:
        state = torch.load(best_path, map_location=DEVICE)
        model.load_state_dict(state["model_state_dict"])
    report = {"selected": best_metrics, "candidates": [item[2] for item in candidates]}
    (ARTIFACT_DIR / "checkpoint_selection_v4.json").write_text(
        json.dumps(report, indent=2), encoding="utf-8"
    )
    print("V4 selected checkpoint:", json.dumps(best_metrics, indent=2))
    return best_metrics, best_path


print("V4 config:", CFG)
print("V4 curriculum:", [stage.name for stage in CURRICULUM])


# ---------------------------------------------------------------------------
# V11: fresh 0042-style seed, then 0056-style recurrent elite imitation.
# This intentionally follows the two notebooks that produced 0042 and 0056.

V11_STARTED_AT = time.perf_counter()
V4_STARTED_AT = V11_STARTED_AT
V4_WALL_CLOCK_LIMIT_SECONDS = 9.35 * 3600
TOP_LOG_DIR = ARTIFACT_DIR / "v11_elite_teachers_logs"
TOP_LOG_BC_DIR = TOP_LOG_DIR
TOP_LOG_BC_MAX_SAMPLES = 110_000
TOP_LOG_BC_EPOCHS = 2
TOP_LOG_BC_BATCH = 1024
SEQUENCE_LENGTH = 32
SEQUENCE_STRIDE = 16
SEQUENCE_BURN_IN = 8
SEQUENCE_BC_EPOCHS_FRESH = 5
SEQUENCE_BATCH_SIZE = 8
ELITE_CLONE_COUNT = 6
ELITE_CLONE_EPOCHS = 2
MAX_SEQUENCES = 6500
MAX_ELITE_REPLAY_FRAMES = 110_000
VALIDATE_EVERY = 6
VALIDATION_MATCHES = 32
FINAL_EVAL_MATCHES = 96
TARGET_KL = 0.010
PLANNER_QUERY_PROB = 0.28
PLANNER_DISTILL_COEF = 0.040
ELITE_BC_COEF = 0.030
ELITE_BC_BATCH = 256

ELITE_TEACHERS_MANIFEST = [{'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 2, 'match_id': "****", 'drive_id': "****", 'seed': 486100, 'direct_teacher': 0, 'direct_reference_agent': 1}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 2, 'match_id': "****", 'drive_id': "****", 'seed': 772274, 'direct_teacher': 0, 'direct_reference_agent': 1}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 2, 'match_id': "****", 'drive_id': "****", 'seed': 968535, 'direct_teacher': 0, 'direct_reference_agent': 1}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 2, 'match_id': "****", 'drive_id': "****", 'seed': 616551, 'direct_teacher': 1, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 2, 'match_id': "****", 'drive_id': "****", 'seed': 462448, 'direct_teacher': 1, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 2, 'match_id': "****", 'drive_id': "****", 'seed': 313250, 'direct_teacher': 1, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 2, 'match_id': "****", 'drive_id': "****", 'seed': 331061, 'direct_teacher': 1, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 2, 'match_id': "****", 'drive_id': "****", 'seed': 889777, 'direct_teacher': 0, 'direct_reference_agent': 1}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 2, 'match_id': "****", 'drive_id': "****", 'seed': 852184, 'direct_teacher': 0, 'direct_reference_agent': 1}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 2, 'match_id': "****", 'drive_id': "****", 'seed': 545520, 'direct_teacher': 1, 'direct_reference_agent': 1}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 2, 'match_id': "****", 'drive_id': "****", 'seed': 710696, 'direct_teacher': 0, 'direct_reference_agent': 1}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 2, 'match_id': "****", 'drive_id': "****", 'seed': 973440, 'direct_teacher': 0, 'direct_reference_agent': 1}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 2, 'match_id': "****", 'drive_id': "****", 'seed': 45696, 'direct_teacher': 1, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 2, 'match_id': "****", 'drive_id': "****", 'seed': 865367, 'direct_teacher': 0, 'direct_reference_agent': 1}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 2, 'match_id': "****", 'drive_id': "****", 'seed': 112029, 'direct_teacher': 1, 'direct_reference_agent': 1}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 2, 'match_id': "****", 'drive_id': "****", 'seed': 759944, 'direct_teacher': 1, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 2, 'match_id': "****", 'drive_id': "****", 'seed': 799019, 'direct_teacher': 0, 'direct_reference_agent': 1}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 2, 'match_id': "****", 'drive_id': "****", 'seed': 748742, 'direct_teacher': 0, 'direct_reference_agent': 1}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 2, 'match_id': "****", 'drive_id': "****", 'seed': 228023, 'direct_teacher': 1, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 2, 'match_id': "****", 'drive_id': "****", 'seed': 721340, 'direct_teacher': 0, 'direct_reference_agent': 1}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 2, 'match_id': "****", 'drive_id': "****", 'seed': 798434, 'direct_teacher': 1, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 2, 'match_id': "****", 'drive_id': "****", 'seed': 850927, 'direct_teacher': 0, 'direct_reference_agent': 1}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 2, 'match_id': "****", 'drive_id': "****", 'seed': 938305, 'direct_teacher': 0, 'direct_reference_agent': 1}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 2, 'match_id': "****", 'drive_id': "****", 'seed': 815788, 'direct_teacher': 0, 'direct_reference_agent': 1}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 2, 'match_id': "****", 'drive_id': "****", 'seed': 623923, 'direct_teacher': 0, 'direct_reference_agent': 1}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 2, 'match_id': "****", 'drive_id': "****", 'seed': 414907, 'direct_teacher': 0, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 2, 'match_id': "****", 'drive_id': "****", 'seed': 985706, 'direct_teacher': 0, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 2, 'match_id': "****", 'drive_id': "****", 'seed': 118998, 'direct_teacher': 0, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 2, 'match_id': "****", 'drive_id': "****", 'seed': 747043, 'direct_teacher': 0, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 2, 'match_id': "****", 'drive_id': "****", 'seed': 719143, 'direct_teacher': 0, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 2, 'match_id': "****", 'drive_id': "****", 'seed': 552083, 'direct_teacher': 0, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 2, 'match_id': "****", 'drive_id': "****", 'seed': 11801, 'direct_teacher': 0, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 2, 'match_id': "****", 'drive_id': "****", 'seed': 874089, 'direct_teacher': 0, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 2, 'match_id': "****", 'drive_id': "****", 'seed': 350747, 'direct_teacher': 0, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 2, 'match_id': "****", 'drive_id': "****", 'seed': 405951, 'direct_teacher': 0, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 2, 'match_id': "****", 'drive_id': "****", 'seed': 762328, 'direct_teacher': 0, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 2, 'match_id': "****", 'drive_id': "****", 'seed': 557021, 'direct_teacher': 0, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 2, 'match_id': "****", 'drive_id': "****", 'seed': 848800, 'direct_teacher': 0, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 2, 'match_id': "****", 'drive_id': "****", 'seed': 303075, 'direct_teacher': 0, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 2, 'match_id': "****", 'drive_id': "****", 'seed': 155448, 'direct_teacher': 0, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 2, 'match_id': "****", 'drive_id': "****", 'seed': 918035, 'direct_teacher': 0, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 2, 'match_id': "****", 'drive_id': "****", 'seed': 732534, 'direct_teacher': 0, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 2, 'match_id': "****", 'drive_id': "****", 'seed': 493758, 'direct_teacher': 0, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 2, 'match_id': "****", 'drive_id': "****", 'seed': 766656, 'direct_teacher': 0, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 2, 'match_id': "****", 'drive_id': "****", 'seed': 237971, 'direct_teacher': 0, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 2, 'match_id': "****", 'drive_id': "****", 'seed': 253396, 'direct_teacher': 0, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 2, 'match_id': "****", 'drive_id': "****", 'seed': 258591, 'direct_teacher': 0, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 2, 'match_id': "****", 'drive_id': "****", 'seed': 352185, 'direct_teacher': 0, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 2, 'match_id': "****", 'drive_id': "****", 'seed': 109611, 'direct_teacher': 0, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 2, 'match_id': "****", 'drive_id': "****", 'seed': 856032, 'direct_teacher': 0, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 3, 'match_id': "****", 'drive_id': "****", 'seed': 147049, 'direct_teacher': 0, 'direct_reference_agent': 1}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 3, 'match_id': "****", 'drive_id': "****", 'seed': 138067, 'direct_teacher': 0, 'direct_reference_agent': 1}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 3, 'match_id': "****", 'drive_id': "****", 'seed': 490056, 'direct_teacher': 0, 'direct_reference_agent': 1}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 3, 'match_id': "****", 'drive_id': "****", 'seed': 582337, 'direct_teacher': 1, 'direct_reference_agent': 1}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 3, 'match_id': "****", 'drive_id': "****", 'seed': 618851, 'direct_teacher': 0, 'direct_reference_agent': 1}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 3, 'match_id': "****", 'drive_id': "****", 'seed': 284753, 'direct_teacher': 0, 'direct_reference_agent': 1}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 3, 'match_id': "****", 'drive_id': "****", 'seed': 106067, 'direct_teacher': 1, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 3, 'match_id': "****", 'drive_id': "****", 'seed': 798231, 'direct_teacher': 1, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 3, 'match_id': "****", 'drive_id': "****", 'seed': 295720, 'direct_teacher': 1, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 3, 'match_id': "****", 'drive_id': "****", 'seed': 119162, 'direct_teacher': 0, 'direct_reference_agent': 1}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 3, 'match_id': "****", 'drive_id': "****", 'seed': 592151, 'direct_teacher': 0, 'direct_reference_agent': 1}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 3, 'match_id': "****", 'drive_id': "****", 'seed': 467832, 'direct_teacher': 1, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 3, 'match_id': "****", 'drive_id': "****", 'seed': 382084, 'direct_teacher': 1, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 3, 'match_id': "****", 'drive_id': "****", 'seed': 396909, 'direct_teacher': 0, 'direct_reference_agent': 1}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 3, 'match_id': "****", 'drive_id': "****", 'seed': 545520, 'direct_teacher': 1, 'direct_reference_agent': 1}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 3, 'match_id': "****", 'drive_id': "****", 'seed': 124928, 'direct_teacher': 1, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 3, 'match_id': "****", 'drive_id': "****", 'seed': 108332, 'direct_teacher': 1, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 3, 'match_id': "****", 'drive_id': "****", 'seed': 597932, 'direct_teacher': 0, 'direct_reference_agent': 1}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 3, 'match_id': "****", 'drive_id': "****", 'seed': 221269, 'direct_teacher': 1, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 3, 'match_id': "****", 'drive_id': "****", 'seed': 29738, 'direct_teacher': 0, 'direct_reference_agent': 1}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 3, 'match_id': "****", 'drive_id': "****", 'seed': 529684, 'direct_teacher': 0, 'direct_reference_agent': 1}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 3, 'match_id': "****", 'drive_id': "****", 'seed': 152090, 'direct_teacher': 0, 'direct_reference_agent': 1}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 3, 'match_id': "****", 'drive_id': "****", 'seed': 237960, 'direct_teacher': 0, 'direct_reference_agent': 1}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 3, 'match_id': "****", 'drive_id': "****", 'seed': 877647, 'direct_teacher': 0, 'direct_reference_agent': 1}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 3, 'match_id': "****", 'drive_id': "****", 'seed': 525100, 'direct_teacher': 0, 'direct_reference_agent': 1}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 3, 'match_id': "****", 'drive_id': "****", 'seed': 89505, 'direct_teacher': 0, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 3, 'match_id': "****", 'drive_id': "****", 'seed': 615208, 'direct_teacher': 0, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 3, 'match_id': "****", 'drive_id': "****", 'seed': 388421, 'direct_teacher': 0, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 3, 'match_id': "****", 'drive_id': "****", 'seed': 98366, 'direct_teacher': 0, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 3, 'match_id': "****", 'drive_id': "****", 'seed': 246080, 'direct_teacher': 0, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 3, 'match_id': "****", 'drive_id': "****", 'seed': 539691, 'direct_teacher': 0, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 3, 'match_id': "****", 'drive_id': "****", 'seed': 838474, 'direct_teacher': 0, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 3, 'match_id': "****", 'drive_id': "****", 'seed': 383510, 'direct_teacher': 0, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 3, 'match_id': "****", 'drive_id': "****", 'seed': 889023, 'direct_teacher': 0, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 3, 'match_id': "****", 'drive_id': "****", 'seed': 193542, 'direct_teacher': 0, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 3, 'match_id': "****", 'drive_id': "****", 'seed': 610672, 'direct_teacher': 0, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 3, 'match_id': "****", 'drive_id': "****", 'seed': 149435, 'direct_teacher': 0, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 3, 'match_id': "****", 'drive_id': "****", 'seed': 430454, 'direct_teacher': 0, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 3, 'match_id': "****", 'drive_id': "****", 'seed': 356420, 'direct_teacher': 0, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 3, 'match_id': "****", 'drive_id': "****", 'seed': 268365, 'direct_teacher': 0, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 3, 'match_id': "****", 'drive_id': "****", 'seed': 493554, 'direct_teacher': 0, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 3, 'match_id': "****", 'drive_id': "****", 'seed': 963326, 'direct_teacher': 0, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 3, 'match_id': "****", 'drive_id': "****", 'seed': 940429, 'direct_teacher': 0, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 3, 'match_id': "****", 'drive_id': "****", 'seed': 704298, 'direct_teacher': 0, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 3, 'match_id': "****", 'drive_id': "****", 'seed': 740553, 'direct_teacher': 0, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 3, 'match_id': "****", 'drive_id': "****", 'seed': 240420, 'direct_teacher': 0, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 3, 'match_id': "****", 'drive_id': "****", 'seed': 999124, 'direct_teacher': 0, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 3, 'match_id': "****", 'drive_id': "****", 'seed': 486135, 'direct_teacher': 0, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 3, 'match_id': "****", 'drive_id': "****", 'seed': 399022, 'direct_teacher': 0, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 3, 'match_id': "****", 'drive_id': "****", 'seed': 92165, 'direct_teacher': 0, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 5, 'match_id': "****", 'drive_id': "****", 'seed': 90139, 'direct_teacher': 0, 'direct_reference_agent': 1}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 5, 'match_id': "****", 'drive_id': "****", 'seed': 505755, 'direct_teacher': 0, 'direct_reference_agent': 1}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 5, 'match_id': "****", 'drive_id': "****", 'seed': 428393, 'direct_teacher': 0, 'direct_reference_agent': 1}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 5, 'match_id': "****", 'drive_id': "****", 'seed': 559537, 'direct_teacher': 0, 'direct_reference_agent': 1}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 5, 'match_id': "****", 'drive_id': "****", 'seed': 582337, 'direct_teacher': 1, 'direct_reference_agent': 1}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 5, 'match_id': "****", 'drive_id': "****", 'seed': 668313, 'direct_teacher': 0, 'direct_reference_agent': 1}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 5, 'match_id': "****", 'drive_id': "****", 'seed': 350022, 'direct_teacher': 0, 'direct_reference_agent': 1}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 5, 'match_id': "****", 'drive_id': "****", 'seed': 909076, 'direct_teacher': 1, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 5, 'match_id': "****", 'drive_id': "****", 'seed': 106067, 'direct_teacher': 1, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 5, 'match_id': "****", 'drive_id': "****", 'seed': 924375, 'direct_teacher': 0, 'direct_reference_agent': 1}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 5, 'match_id': "****", 'drive_id': "****", 'seed': 784259, 'direct_teacher': 0, 'direct_reference_agent': 1}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 5, 'match_id': "****", 'drive_id': "****", 'seed': 943191, 'direct_teacher': 0, 'direct_reference_agent': 1}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 5, 'match_id': "****", 'drive_id': "****", 'seed': 313250, 'direct_teacher': 1, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 5, 'match_id': "****", 'drive_id': "****", 'seed': 910328, 'direct_teacher': 1, 'direct_reference_agent': 1}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 5, 'match_id': "****", 'drive_id': "****", 'seed': 552990, 'direct_teacher': 0, 'direct_reference_agent': 1}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 5, 'match_id': "****", 'drive_id': "****", 'seed': 333934, 'direct_teacher': 0, 'direct_reference_agent': 1}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 5, 'match_id': "****", 'drive_id': "****", 'seed': 382084, 'direct_teacher': 1, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 5, 'match_id': "****", 'drive_id': "****", 'seed': 579603, 'direct_teacher': 0, 'direct_reference_agent': 1}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 5, 'match_id': "****", 'drive_id': "****", 'seed': 840464, 'direct_teacher': 1, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 5, 'match_id': "****", 'drive_id': "****", 'seed': 504215, 'direct_teacher': 0, 'direct_reference_agent': 1}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 5, 'match_id': "****", 'drive_id': "****", 'seed': 267869, 'direct_teacher': 1, 'direct_reference_agent': 1}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 5, 'match_id': "****", 'drive_id': "****", 'seed': 805657, 'direct_teacher': 0, 'direct_reference_agent': 1}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 5, 'match_id': "****", 'drive_id': "****", 'seed': 127959, 'direct_teacher': 1, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 5, 'match_id': "****", 'drive_id': "****", 'seed': 917012, 'direct_teacher': 0, 'direct_reference_agent': 1}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 5, 'match_id': "****", 'drive_id': "****", 'seed': 457559, 'direct_teacher': 0, 'direct_reference_agent': 1}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 5, 'match_id': "****", 'drive_id': "****", 'seed': 561055, 'direct_teacher': 0, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 5, 'match_id': "****", 'drive_id': "****", 'seed': 461304, 'direct_teacher': 0, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 5, 'match_id': "****", 'drive_id': "****", 'seed': 108235, 'direct_teacher': 0, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 5, 'match_id': "****", 'drive_id': "****", 'seed': 252797, 'direct_teacher': 0, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 5, 'match_id': "****", 'drive_id': "****", 'seed': 44466, 'direct_teacher': 0, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 5, 'match_id': "****", 'drive_id': "****", 'seed': 257098, 'direct_teacher': 0, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 5, 'match_id': "****", 'drive_id': "****", 'seed': 744519, 'direct_teacher': 0, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 5, 'match_id': "****", 'drive_id': "****", 'seed': 929265, 'direct_teacher': 0, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 5, 'match_id': "****", 'drive_id': "****", 'seed': 460481, 'direct_teacher': 0, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 5, 'match_id': "****", 'drive_id': "****", 'seed': 116583, 'direct_teacher': 0, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 5, 'match_id': "****", 'drive_id': "****", 'seed': 700099, 'direct_teacher': 0, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 5, 'match_id': "****", 'drive_id': "****", 'seed': 293437, 'direct_teacher': 0, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 5, 'match_id': "****", 'drive_id': "****", 'seed': 796980, 'direct_teacher': 0, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 5, 'match_id': "****", 'drive_id': "****", 'seed': 958042, 'direct_teacher': 0, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 5, 'match_id': "****", 'drive_id': "****", 'seed': 521723, 'direct_teacher': 0, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 5, 'match_id': "****", 'drive_id': "****", 'seed': 684201, 'direct_teacher': 0, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 5, 'match_id': "****", 'drive_id': "****", 'seed': 727126, 'direct_teacher': 0, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 5, 'match_id': "****", 'drive_id': "****", 'seed': 278499, 'direct_teacher': 0, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 5, 'match_id': "****", 'drive_id': "****", 'seed': 294421, 'direct_teacher': 0, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 5, 'match_id': "****", 'drive_id': "****", 'seed': 807376, 'direct_teacher': 0, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 5, 'match_id': "****", 'drive_id': "****", 'seed': 734514, 'direct_teacher': 0, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 5, 'match_id': "****", 'drive_id': "****", 'seed': 364200, 'direct_teacher': 0, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 5, 'match_id': "****", 'drive_id': "****", 'seed': 878954, 'direct_teacher': 0, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 5, 'match_id': "****", 'drive_id': "****", 'seed': 406248, 'direct_teacher': 0, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 5, 'match_id': "****", 'drive_id': "****", 'seed': 269829, 'direct_teacher': 0, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 6, 'match_id': "****", 'drive_id': "****", 'seed': 928089, 'direct_teacher': 0, 'direct_reference_agent': 1}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 6, 'match_id': "****", 'drive_id': "****", 'seed': 689854, 'direct_teacher': 0, 'direct_reference_agent': 1}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 6, 'match_id': "****", 'drive_id': "****", 'seed': 909076, 'direct_teacher': 1, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 6, 'match_id': "****", 'drive_id': "****", 'seed': 798231, 'direct_teacher': 1, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 6, 'match_id': "****", 'drive_id': "****", 'seed': 295720, 'direct_teacher': 1, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 6, 'match_id': "****", 'drive_id': "****", 'seed': 281318, 'direct_teacher': 0, 'direct_reference_agent': 1}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 6, 'match_id': "****", 'drive_id': "****", 'seed': 616551, 'direct_teacher': 1, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 6, 'match_id': "****", 'drive_id': "****", 'seed': 462448, 'direct_teacher': 1, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 6, 'match_id': "****", 'drive_id': "****", 'seed': 228292, 'direct_teacher': 0, 'direct_reference_agent': 1}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 6, 'match_id': "****", 'drive_id': "****", 'seed': 313250, 'direct_teacher': 1, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 6, 'match_id': "****", 'drive_id': "****", 'seed': 910328, 'direct_teacher': 1, 'direct_reference_agent': 1}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 6, 'match_id': "****", 'drive_id': "****", 'seed': 331061, 'direct_teacher': 1, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 6, 'match_id': "****", 'drive_id': "****", 'seed': 467832, 'direct_teacher': 1, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 6, 'match_id': "****", 'drive_id': "****", 'seed': 509375, 'direct_teacher': 0, 'direct_reference_agent': 1}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 6, 'match_id': "****", 'drive_id': "****", 'seed': 230986, 'direct_teacher': 0, 'direct_reference_agent': 1}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 6, 'match_id': "****", 'drive_id': "****", 'seed': 124928, 'direct_teacher': 1, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 6, 'match_id': "****", 'drive_id': "****", 'seed': 108332, 'direct_teacher': 1, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 6, 'match_id': "****", 'drive_id': "****", 'seed': 921523, 'direct_teacher': 1, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 6, 'match_id': "****", 'drive_id': "****", 'seed': 45696, 'direct_teacher': 1, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 6, 'match_id': "****", 'drive_id': "****", 'seed': 127959, 'direct_teacher': 1, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 6, 'match_id': "****", 'drive_id': "****", 'seed': 112029, 'direct_teacher': 1, 'direct_reference_agent': 1}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 6, 'match_id': "****", 'drive_id': "****", 'seed': 574175, 'direct_teacher': 0, 'direct_reference_agent': 1}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 6, 'match_id': "****", 'drive_id': "****", 'seed': 135766, 'direct_teacher': 0, 'direct_reference_agent': 1}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 6, 'match_id': "****", 'drive_id': "****", 'seed': 759944, 'direct_teacher': 1, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 6, 'match_id': "****", 'drive_id': "****", 'seed': 47666, 'direct_teacher': 0, 'direct_reference_agent': 1}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 6, 'match_id': "****", 'drive_id': "****", 'seed': 513759, 'direct_teacher': 0, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 6, 'match_id': "****", 'drive_id': "****", 'seed': 491330, 'direct_teacher': 0, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 6, 'match_id': "****", 'drive_id': "****", 'seed': 196021, 'direct_teacher': 0, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 6, 'match_id': "****", 'drive_id': "****", 'seed': 107998, 'direct_teacher': 0, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 6, 'match_id': "****", 'drive_id': "****", 'seed': 390737, 'direct_teacher': 0, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 6, 'match_id': "****", 'drive_id': "****", 'seed': 49880, 'direct_teacher': 0, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 6, 'match_id': "****", 'drive_id': "****", 'seed': 424037, 'direct_teacher': 0, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 6, 'match_id': "****", 'drive_id': "****", 'seed': 219568, 'direct_teacher': 0, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 6, 'match_id': "****", 'drive_id': "****", 'seed': 95254, 'direct_teacher': 0, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 6, 'match_id': "****", 'drive_id': "****", 'seed': 63197, 'direct_teacher': 0, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 6, 'match_id': "****", 'drive_id': "****", 'seed': 478999, 'direct_teacher': 0, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 6, 'match_id': "****", 'drive_id': "****", 'seed': 691856, 'direct_teacher': 0, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 6, 'match_id': "****", 'drive_id': "****", 'seed': 992523, 'direct_teacher': 0, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 6, 'match_id': "****", 'drive_id': "****", 'seed': 310981, 'direct_teacher': 0, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 6, 'match_id': "****", 'drive_id': "****", 'seed': 147929, 'direct_teacher': 0, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 6, 'match_id': "****", 'drive_id': "****", 'seed': 532494, 'direct_teacher': 0, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 6, 'match_id': "****", 'drive_id': "****", 'seed': 974509, 'direct_teacher': 0, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 6, 'match_id': "****", 'drive_id': "****", 'seed': 678069, 'direct_teacher': 0, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 6, 'match_id': "****", 'drive_id': "****", 'seed': 398179, 'direct_teacher': 0, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 6, 'match_id': "****", 'drive_id': "****", 'seed': 700659, 'direct_teacher': 0, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 6, 'match_id': "****", 'drive_id': "****", 'seed': 209050, 'direct_teacher': 0, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 6, 'match_id': "****", 'drive_id': "****", 'seed': 573084, 'direct_teacher': 0, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 6, 'match_id': "****", 'drive_id': "****", 'seed': 353918, 'direct_teacher': 0, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 6, 'match_id': "****", 'drive_id': "****", 'seed': 226633, 'direct_teacher': 0, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 6, 'match_id': "****", 'drive_id': "****", 'seed': 2735, 'direct_teacher': 0, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 11, 'match_id': "****", 'drive_id': "****", 'seed': 884804, 'direct_teacher': 0, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 11, 'match_id': "****", 'drive_id': "****", 'seed': 826917, 'direct_teacher': 0, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 11, 'match_id': "****", 'drive_id': "****", 'seed': 310838, 'direct_teacher': 0, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 11, 'match_id': "****", 'drive_id': "****", 'seed': 38012, 'direct_teacher': 0, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 11, 'match_id': "****", 'drive_id': "****", 'seed': 630247, 'direct_teacher': 0, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 11, 'match_id': "****", 'drive_id': "****", 'seed': 51669, 'direct_teacher': 0, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 11, 'match_id': "****", 'drive_id': "****", 'seed': 75176, 'direct_teacher': 0, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 11, 'match_id': "****", 'drive_id': "****", 'seed': 278977, 'direct_teacher': 0, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 11, 'match_id': "****", 'drive_id': "****", 'seed': 746747, 'direct_teacher': 0, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 11, 'match_id': "****", 'drive_id': "****", 'seed': 403165, 'direct_teacher': 0, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 11, 'match_id': "****", 'drive_id': "****", 'seed': 95706, 'direct_teacher': 0, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 11, 'match_id': "****", 'drive_id': "****", 'seed': 938207, 'direct_teacher': 0, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 11, 'match_id': "****", 'drive_id': "****", 'seed': 405429, 'direct_teacher': 0, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 11, 'match_id': "****", 'drive_id': "****", 'seed': 321305, 'direct_teacher': 0, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 11, 'match_id': "****", 'drive_id': "****", 'seed': 370494, 'direct_teacher': 0, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 11, 'match_id': "****", 'drive_id': "****", 'seed': 757750, 'direct_teacher': 0, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 11, 'match_id': "****", 'drive_id': "****", 'seed': 991716, 'direct_teacher': 0, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 11, 'match_id': "****", 'drive_id': "****", 'seed': 400741, 'direct_teacher': 0, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 11, 'match_id': "****", 'drive_id': "****", 'seed': 77915, 'direct_teacher': 0, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 11, 'match_id': "****", 'drive_id': "****", 'seed': 400385, 'direct_teacher': 0, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 11, 'match_id': "****", 'drive_id': "****", 'seed': 562639, 'direct_teacher': 0, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 11, 'match_id': "****", 'drive_id': "****", 'seed': 844202, 'direct_teacher': 0, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 11, 'match_id': "****", 'drive_id': "****", 'seed': 583833, 'direct_teacher': 0, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 11, 'match_id': "****", 'drive_id': "****", 'seed': 840464, 'direct_teacher': 1, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 11, 'match_id': "****", 'drive_id': "****", 'seed': 330410, 'direct_teacher': 0, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 11, 'match_id': "****", 'drive_id': "****", 'seed': 238023, 'direct_teacher': 0, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 11, 'match_id': "****", 'drive_id': "****", 'seed': 221269, 'direct_teacher': 1, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 11, 'match_id': "****", 'drive_id': "****", 'seed': 294200, 'direct_teacher': 0, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 11, 'match_id': "****", 'drive_id': "****", 'seed': 921523, 'direct_teacher': 1, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 11, 'match_id': "****", 'drive_id': "****", 'seed': 322891, 'direct_teacher': 0, 'direct_reference_agent': 1}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 11, 'match_id': "****", 'drive_id': "****", 'seed': 626618, 'direct_teacher': 0, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 11, 'match_id': "****", 'drive_id': "****", 'seed': 9002, 'direct_teacher': 0, 'direct_reference_agent': 1}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 11, 'match_id': "****", 'drive_id': "****", 'seed': 267869, 'direct_teacher': 1, 'direct_reference_agent': 1}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 11, 'match_id': "****", 'drive_id': "****", 'seed': 366805, 'direct_teacher': 0, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 11, 'match_id': "****", 'drive_id': "****", 'seed': 568601, 'direct_teacher': 0, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 11, 'match_id': "****", 'drive_id': "****", 'seed': 863896, 'direct_teacher': 0, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 11, 'match_id': "****", 'drive_id': "****", 'seed': 566129, 'direct_teacher': 0, 'direct_reference_agent': 0}, {'agent_key': "****", 'team': "****", 'submission_id': "****", 'rank': 11, 'match_id': "****", 'drive_id': "****", 'seed': 279173, 'direct_teacher': 0, 'direct_reference_agent': 0}]

SEED_CFG = TrainConfig(
    "v11_seed_0042_like",
    SEED,
    num_envs=24,
    episodes_per_update=32,
    bc_teacher_steps=12_000,
    bc_chunk_size=4096,
    bc_epochs=2,
    max_steps=500,
    gamma=0.995,
    gae_lambda=0.95,
    learning_rate=2.0e-4,
    clip_coef=0.18,
    ent_coef=0.011,
    vf_coef=0.50,
    max_grad_norm=0.50,
    ppo_epochs=3,
    minibatch_size=1024,
    eval_matches=96,
    checkpoint_every=4,
    league_snapshot_every=3,
    league_max_size=12,
)
SEED_CURRICULUM = [
    Stage("v11_opening_topfit", 20, 1, 1, ("box_farmer", "smarter", "genius"), "v11_seed_opening", 0.10),
    Stage("v11_post_bomb_control", 22, 1, 1, ("smarter", "genius", "tactical"), "v11_seed_control", 0.25),
]

PFSP_CFG = TrainConfig(
    "v11_hk_sequence_pfsp",
    SEED,
    num_envs=20,
    episodes_per_update=24,
    bc_teacher_steps=0,
    bc_chunk_size=4096,
    bc_epochs=1,
    max_steps=500,
    gamma=0.995,
    gae_lambda=0.95,
    learning_rate=3.5e-5,
    clip_coef=0.12,
    ent_coef=0.010,
    vf_coef=0.55,
    max_grad_norm=0.50,
    ppo_epochs=2,
    minibatch_size=1024,
    eval_matches=FINAL_EVAL_MATCHES,
    checkpoint_every=4,
    league_snapshot_every=4,
    league_max_size=8,
)
PFSP_CURRICULUM = [
    Stage("sequence_stabilize", 8, 1, 1, ("box_farmer", "smarter", "genius"), "opening_v4", 0.45),
    Stage("mixed_hard", 12, 1, 1, ("smarter", "genius", "tactical"), "balanced_v4", 0.62),
    Stage("elite_pfsp", 16, 1, 1, ("genius", "tactical"), "competitive_v4", 0.78),
    Stage("late_conversion", 12, 1, 1, ("genius", "tactical"), "late_v4", 0.84),
    Stage("robust_league", 8, 1, 1, ("genius", "tactical"), "robust_v4", 0.88),
]
STAGE_TARGET_RANK.update({
    "sequence_stabilize": 0.58,
    "mixed_hard": 0.56,
    "elite_pfsp": 0.54,
    "late_conversion": 0.54,
    "robust_league": 0.52,
})
STAGE_EXTRA_CAP.update({
    "sequence_stabilize": 2,
    "mixed_hard": 3,
    "elite_pfsp": 4,
    "late_conversion": 4,
    "robust_league": 3,
})

REWARD_PROFILES.update({
    "v11_seed_opening": {
        "kill": 0.22, "box": 0.110, "item": 0.155,
        "bomb_target": 0.080, "bomb_enemy": 0.045, "trap_gain": 0.004,
        "wasted_bomb": -0.080,
        "low_exit_bomb": -0.320, "self_trap_bomb": -0.820,
        "wait": -0.010, "death": -1.10, "danger": -0.125,
        "potential": 0.055, "post_bomb_move": 0.010,
        "leave_blast_line": 0.014,
        "sole_win": 6.0, "draw": -6.0,
        "mutual_destruction": -4.5, "ranks": (7.5, 2.0, -1.0, -5.5),
    },
    "v11_seed_control": {
        "kill": 0.50, "box": 0.040, "item": 0.085,
        "bomb_target": 0.055, "bomb_enemy": 0.060, "trap_gain": 0.006,
        "wasted_bomb": -0.065,
        "low_exit_bomb": -0.360, "self_trap_bomb": -0.920,
        "wait": -0.004, "death": -1.08, "danger": -0.120,
        "potential": 0.030, "post_bomb_move": 0.012,
        "leave_blast_line": 0.016,
        "sole_win": 9.0, "draw": -9.0,
        "mutual_destruction": -5.8, "ranks": (9.8, 2.5, -1.5, -7.2),
    },
})


def fetch_elite_teachers_logs_v11(log_dir=TOP_LOG_DIR):
    import pandas as pd

    log_dir = Path(log_dir)
    json_dir = log_dir / "json"
    json_dir.mkdir(parents=True, exist_ok=True)
    input_payloads = {}
    for path in Path("/kaggle/input").rglob("*.json"):
        try:
            raw = path.read_bytes()
            payload = json.loads(raw.decode("utf-8"))
        except Exception:
            continue
        if "history" in payload and "team_ids" in payload:
            input_payloads.setdefault(str(payload.get("match_id", path.stem)), raw)
    for archive_path in Path("/kaggle/input").rglob("*.zip"):
        try:
            with zipfile.ZipFile(archive_path) as archive:
                for name in archive.namelist():
                    if not name.lower().endswith(".json"):
                        continue
                    raw = archive.read(name)
                    try:
                        payload = json.loads(raw)
                    except Exception:
                        continue
                    if "history" in payload and "team_ids" in payload:
                        input_payloads.setdefault(str(payload.get("match_id", Path(name).stem)), raw)
        except (OSError, zipfile.BadZipFile, KeyError, ValueError):
            continue

    rows = []
    failures = []
    for item in ELITE_TEACHERS_MANIFEST:
        match_id = str(item["match_id"])
        out_path = json_dir / f"{item['agent_key']}_{match_id}.json"
        try:
            if not out_path.exists():
                if match_id not in input_payloads:
                    raise FileNotFoundError(match_id)
                out_path.write_bytes(input_payloads[match_id])
            payload = json.loads(out_path.read_text(encoding="utf-8"))
            team_ids = [str(value) for value in payload["team_ids"]]
            if item["submission_id"] not in team_ids:
                raise ValueError("target submission missing")
            # Rank is deliberately adjusted for training weights: **** is the main
            # teacher bank, **** is a secondary style prior.
            train_rank = int(item["rank"])
            if item["team"] == "****_TNTT":
                train_rank = 2 if train_rank <= 3 else 4
            elif item["team"] == "****":
                train_rank = 11
            rows.append({
                "rank": train_rank,
                "actual_rank": int(item["rank"]),
                "team": item["team"],
                "agent_key": item["agent_key"],
                "submission_id": item["submission_id"],
                "match_id": match_id,
                "path": str(out_path),
                "seed": int(item["seed"]),
                "direct_teacher": int(item.get("direct_teacher", 0)),
                "direct_reference_agent": int(item.get("direct_reference_agent", 0)),
            })
        except Exception as exc:
            failures.append(f"{match_id}: {type(exc).__name__}: {exc}")
    selected = pd.DataFrame(rows)
    selected.to_csv(log_dir / "selected_top_logs.csv", index=False)
    counts = selected.groupby("agent_key").size().to_dict() if not selected.empty else {}
    expected = {
        "elite_rank_2": 50,
        "elite_rank_3": 50,
        "elite_rank_5": 50,
        "elite_rank_6": 50,
        "elite_primary": 38,
    }
    missing = {key: (int(counts.get(key, 0)), value) for key, value in expected.items() if int(counts.get(key, 0)) < value}
    if missing:
        raise RuntimeError(f"Missing V11 elite logs: {missing}; attach ELITE_TEACHERS_SEQUENCE_PFSP_V11_LOGS.zip")
    print("V11 elite logs:", counts, "failures=", len(failures))
    if failures:
        print("First warnings:", failures[:5])
    return log_dir


def _v11_blast_context(obs, slot, radius_lookup):
    row = int(obs["players"][slot][0])
    col = int(obs["players"][slot][1])
    radius = 1 + int(obs["players"][slot][4])
    tiles = blast_tiles(obs["map"], row, col, radius)
    covered = set()
    for bomb_row, bomb_col, _, _, bomb_radius in resolved_bombs(obs, radius_lookup=radius_lookup):
        covered.update(blast_tiles(obs["map"], bomb_row, bomb_col, bomb_radius))
    boxes = {(r, c) for r, c in tiles if int(obs["map"][r][c]) == 2}
    fresh = boxes - covered
    enemies = {
        (int(player[0]), int(player[1]))
        for index, player in enumerate(obs["players"])
        if index != slot and int(player[2]) == 1
    }
    chain = any((int(bomb[0]), int(bomb[1])) in tiles for bomb in obs["bombs"])
    return len(boxes), len(fresh), len(enemies & tiles), int(chain)


def extract_hk_flat_bc_records_v11(log_dir=TOP_LOG_DIR, max_samples=TOP_LOG_BC_MAX_SAMPLES):
    import pandas as pd

    selected = pd.read_csv(Path(log_dir) / "selected_top_logs.csv")
    records = []
    weights = []
    for _, row in selected.iterrows():
        data = json.loads(Path(row["path"]).read_text(encoding="utf-8"))
        sid = str(row["submission_id"])
        team = str(row["team"])
        agent_key = str(row["agent_key"])
        team_ids = [str(value) for value in data["team_ids"]]
        if sid not in team_ids:
            continue
        slot = team_ids.index(sid)
        rank = int(data["ranks"][slot])
        survival = int(data["survival_steps"][slot])
        outcome = (2.25, 1.20, 0.50, 0.16)[min(rank, 3)]
        team_weight = 1.65 if team == "****_TNTT" else 0.75
        if agent_key in {"elite_rank_2", "elite_rank_3"}:
            team_weight *= 1.12
        if int(row.get("direct_reference_agent", 0)):
            team_weight *= 1.15
        hist = data.get("history", [])
        previous_action = None
        tracker = BombTracker()
        for step in range(1, len(hist)):
            actions = hist[step].get("actions")
            if actions is None:
                continue
            prev = obs_from_log_frame(hist[step - 1])
            radius_lookup = tracker.update(prev)
            if int(prev["players"][slot][2]) != 1:
                continue
            action = int(actions[slot])
            if not 0 <= action <= 5:
                continue
            mask = legal_action_mask(prev, slot, radius_lookup=radius_lookup, strict_safety=False)
            if not bool(mask[action]):
                continue
            phase = _phase_id(step)
            weight = float(outcome * team_weight)
            if action == 5:
                boxes, fresh, enemies, chain = _v11_blast_context(prev, slot, radius_lookup)
                exits = escape_exit_count_after_bomb(prev, slot, radius_lookup)
                if exits < MIN_BOMB_EXITS:
                    continue
                if phase == 0:
                    weight *= (1.95 + 0.25 * min(fresh, 3)) if fresh else 0.22
                elif phase >= 2:
                    weight *= 1.75 if enemies else (1.15 if chain else 0.40)
                else:
                    weight *= 1.55 if (enemies or chain) else (0.65 if fresh else 0.30)
            elif action == 0:
                weight *= 0.70 if len(prev["bombs"]) else 0.38
            elif action in (1, 2, 3, 4):
                weight *= 1.12 if phase >= 1 else 1.00
            if previous_action == 5 and action in (1, 2, 3, 4):
                weight *= 1.85
            if (previous_action, action) in {(1, 2), (2, 1), (3, 4), (4, 3)}:
                weight *= 0.78
            if rank > 0 and step >= survival - 10:
                continue
            features, scalars = encode_obs(prev, slot, step, radius_lookup=radius_lookup)
            records.append((features, scalars, mask, action))
            weights.append(weight)
            previous_action = action
    if len(records) > max_samples:
        probs = np.asarray(weights, dtype=np.float64)
        probs /= probs.sum()
        indices = np.random.default_rng(SEED + 1111).choice(len(records), size=max_samples, replace=False, p=probs)
        records = [records[int(i)] for i in indices]
        weights = [weights[int(i)] for i in indices]
    print("V11 **** flat BC samples:", len(records), "mean_weight:", float(np.mean(weights)) if weights else None)
    return records, np.asarray(weights, dtype=np.float32)


_BASE_EXTRACT_ELITE_SEQUENCES = extract_elite_sequences


# Trích recurrent sequences từ elite logs, gán weight theo rank, phase và chất lượng action.
def extract_elite_sequences(log_dir=TOP_LOG_DIR, max_sequences=MAX_SEQUENCES):
    sequences = _BASE_EXTRACT_ELITE_SEQUENCES(log_dir=log_dir, max_sequences=max_sequences)
    hk = sum(1 for seq in sequences if "****_TNTT" in str(seq.get("team", "")))
    elite_primary = sum(1 for seq in sequences if "****" in str(seq.get("team", "")))
    print("V11 sequence mix:", {"****_TNTT": hk, "****": elite_primary, "total": len(sequences)})
    return sequences


def hard_validation_score_v11(metrics):
    return (
        19.0 * metrics["sole_win_rate"]
        + 7.0 * metrics["rank_0_rate"]
        - 13.0 * metrics["average_rank"]
        - 22.0 * metrics["rank_3_rate"]
        - 18.0 * metrics["draw_rate"]
        + 0.002 * metrics["mean_steps"]
        - 3.5 * max(0.0, metrics.get("bomb_rate", 0.0) - 0.22)
        - 3.0 * max(0.0, 0.105 - metrics.get("bomb_rate", 0.0))
    )


def select_best_checkpoint_v11(model, pool, matches=64):
    preferred = (
        sorted(CHECKPOINT_DIR.glob("*v11_seed_0042_like.pth"))
        + sorted(CHECKPOINT_DIR.glob("*v11_after_hk_sequence_bc.pth"))
        + sorted(CHECKPOINT_DIR.glob("*v4_best.pth"))
        + sorted(CHECKPOINT_DIR.glob("*v4_initial.pth"))
    )
    recent = sorted(CHECKPOINT_DIR.glob("ppo_*.pth"))[-8:]
    paths = []
    for path in preferred + recent:
        if path not in paths and "after_tactical_bc" not in path.name:
            paths.append(path)
    candidates = []
    for path in paths:
        trial = ActorCritic().to(DEVICE)
        state = torch.load(path, map_location=DEVICE)
        trial.load_state_dict(state["model_state_dict"])
        metrics = evaluate_hard_policy(trial, pool, matches=matches, seed=3_100_000)
        metrics["checkpoint"] = str(path)
        metrics["candidate_score"] = hard_validation_score_v11(metrics)
        if "after_hk_sequence_bc" in path.name:
            metrics["candidate_score"] += 0.35
        candidates.append((metrics["candidate_score"], path, metrics))
        del trial
        torch.cuda.empty_cache()
    current_metrics = evaluate_hard_policy(model, pool, matches=matches, seed=3_100_000)
    current_metrics["checkpoint"] = "current"
    current_metrics["candidate_score"] = hard_validation_score_v11(current_metrics)
    candidates.append((current_metrics["candidate_score"], None, current_metrics))
    candidates.sort(key=lambda item: item[0], reverse=True)
    _, best_path, best_metrics = candidates[0]
    if best_path is not None:
        state = torch.load(best_path, map_location=DEVICE)
        model.load_state_dict(state["model_state_dict"])
    report = {
        "selected": best_metrics,
        "candidates": [item[2] for item in candidates],
        "strategy": "fresh 0042 seed + ****-heavy recurrent Sequence BC + short PFSP",
    }
    (ARTIFACT_DIR / "checkpoint_selection_v11.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
    print("V11 selected checkpoint:", json.dumps(best_metrics, indent=2))
    return best_metrics, best_path


def configure_seed_phase():
    global CFG, CURRICULUM
    CFG = SEED_CFG
    CURRICULUM = SEED_CURRICULUM
    print("V11 seed config:", CFG)
    print("V11 seed curriculum:", [stage.name for stage in CURRICULUM])


def configure_pfsp_phase():
    global CFG, CURRICULUM, PFSP_POOL
    CFG = PFSP_CFG
    CURRICULUM = PFSP_CURRICULUM
    PFSP_POOL = PFSPPool(snapshot_limit=CFG.league_max_size)
    print("V11 PFSP config:", CFG)
    print("V11 PFSP curriculum:", [stage.name for stage in CURRICULUM])
    return PFSP_POOL


print("V11 teacher trajectories:", len(ELITE_TEACHERS_MANIFEST))
print("V11 teacher teams:", dict(Counter(item["team"] for item in ELITE_TEACHERS_MANIFEST)))


# ---------------------------------------------------------------------------
# V12: conservative continuation from V11 using rank-drop/top8 logs.

V12_STARTED_AT = time.perf_counter()
V4_STARTED_AT = V12_STARTED_AT
V12_CPU_MODE = DEVICE.type != "cuda"
V4_WALL_CLOCK_LIMIT_SECONDS = (2.25 if V12_CPU_MODE else 6.25) * 3600
V12_LOG_DIR = ARTIFACT_DIR / "v12_rankdrop_top8_logs"
TOP_LOG_DIR = V12_LOG_DIR

V12_TARGET_IDS = ['****', '****', '****', '****']
V12_TOP8_IDS = ['****', '****', '****', '****', '****', '****', '****', '****']
V12_LEADERBOARD_INFO = {'****': {'rank': 1, 'team': "****", 'mu': 115.2322, 'sigma': 0.734, 'score': 113.0303, 'games': 569, 'win_rate': 0.6942, 'avg_rank': 0.4868}, '****': {'rank': 2, 'team': "****", 'mu': 114.8587, 'sigma': 0.7358, 'score': 112.6512, 'games': 327, 'win_rate': 0.6483, 'avg_rank': 0.5657}, '****': {'rank': 3, 'team': "****", 'mu': 114.5271, 'sigma': 0.7738, 'score': 112.2058, 'games': 119, 'win_rate': 0.605, 'avg_rank': 0.5378}, '****': {'rank': 4, 'team': "****", 'mu': 114.3208, 'sigma': 0.7263, 'score': 112.1419, 'games': 267, 'win_rate': 0.6442, 'avg_rank': 0.5056}, '****': {'rank': 5, 'team': "****", 'mu': 114.1899, 'sigma': 0.7212, 'score': 112.0264, 'games': 1118, 'win_rate': 0.5957, 'avg_rank': 0.5769}, '****': {'rank': 6, 'team': "****", 'mu': 114.5147, 'sigma': 0.8312, 'score': 112.021, 'games': 74, 'win_rate': 0.5946, 'avg_rank': 0.527}, '****': {'rank': 7, 'team': "****", 'mu': 114.4577, 'sigma': 0.8245, 'score': 111.9841, 'games': 75, 'win_rate': 0.6, 'avg_rank': 0.56}, '****': {'rank': 8, 'team': "****", 'mu': 114.0649, 'sigma': 0.7185, 'score': 111.9094, 'games': 727, 'win_rate': 0.608, 'avg_rank': 0.5516}, '****': {'rank': 9, 'team': "****", 'mu': 114.046, 'sigma': 0.7217, 'score': 111.8808, 'games': 422, 'win_rate': 0.5829, 'avg_rank': 0.5806}, '****': {'rank': 10, 'team': "****", 'mu': 114.0224, 'sigma': 0.7141, 'score': 111.8802, 'games': 265, 'win_rate': 0.5698, 'avg_rank': 0.6943}, '****': {'rank': 11, 'team': "****", 'mu': 114.0322, 'sigma': 0.7181, 'score': 111.8778, 'games': 382, 'win_rate': 0.5838, 'avg_rank': 0.5995}, '****': {'rank': 12, 'team': "****", 'mu': 114.0443, 'sigma': 0.7226, 'score': 111.8766, 'games': 175, 'win_rate': 0.5943, 'avg_rank': 0.6229}, '****': {'rank': 13, 'team': "****", 'mu': 114.0175, 'sigma': 0.7147, 'score': 111.8735, 'games': 398, 'win_rate': 0.5427, 'avg_rank': 0.5779}, '****': {'rank': 14, 'team': "****", 'mu': 114.0731, 'sigma': 0.7333, 'score': 111.8733, 'games': 1051, 'win_rate': 0.6099, 'avg_rank': 0.5347}, '****': {'rank': 15, 'team': "****", 'mu': 114.0506, 'sigma': 0.7259, 'score': 111.8728, 'games': 472, 'win_rate': 0.5593, 'avg_rank': 0.6102}, '****': {'rank': 16, 'team': "****", 'mu': 114.0601, 'sigma': 0.7327, 'score': 111.8619, 'games': 562, 'win_rate': 0.6957, 'avg_rank': 0.4858}, '****': {'rank': 17, 'team': "****", 'mu': 114.0235, 'sigma': 0.7218, 'score': 111.8582, 'games': 723, 'win_rate': 0.592, 'avg_rank': 0.5864}, '****': {'rank': 18, 'team': "****", 'mu': 114.0207, 'sigma': 0.7229, 'score': 111.8521, 'games': 334, 'win_rate': 0.5629, 'avg_rank': 0.6317}, '****': {'rank': 19, 'team': "****", 'mu': 114.0087, 'sigma': 0.7192, 'score': 111.8512, 'games': 346, 'win_rate': 0.5751, 'avg_rank': 0.5983}, '****': {'rank': 20, 'team': "****", 'mu': 114.0689, 'sigma': 0.7404, 'score': 111.8476, 'games': 136, 'win_rate': 0.5956, 'avg_rank': 0.5588}, '****': {'rank': 21, 'team': "****", 'mu': 113.9941, 'sigma': 0.717, 'score': 111.843, 'games': 243, 'win_rate': 0.6008, 'avg_rank': 0.6008}, '****': {'rank': 22, 'team': "****", 'mu': 113.9958, 'sigma': 0.7178, 'score': 111.8424, 'games': 468, 'win_rate': 0.5705, 'avg_rank': 0.5491}, '****': {'rank': 23, 'team': "****", 'mu': 113.9784, 'sigma': 0.7132, 'score': 111.8389, 'games': 178, 'win_rate': 0.5506, 'avg_rank': 0.6124}, '****': {'rank': 24, 'team': "****", 'mu': 114.0226, 'sigma': 0.7283, 'score': 111.8376, 'games': 380, 'win_rate': 0.5737, 'avg_rank': 0.5289}, '****': {'rank': 25, 'team': "****", 'mu': 114.0034, 'sigma': 0.7232, 'score': 111.8337, 'games': 336, 'win_rate': 0.5655, 'avg_rank': 0.6071}, '****': {'rank': 26, 'team': "****", 'mu': 113.9537, 'sigma': 0.7076, 'score': 111.8309, 'games': 215, 'win_rate': 0.5163, 'avg_rank': 0.6698}, '****': {'rank': 27, 'team': "****", 'mu': 114.0145, 'sigma': 0.7284, 'score': 111.8293, 'games': 616, 'win_rate': 0.6721, 'avg_rank': 0.4968}, '****': {'rank': 28, 'team': "****", 'mu': 113.997, 'sigma': 0.7226, 'score': 111.8292, 'games': 218, 'win_rate': 0.5734, 'avg_rank': 0.5872}, '****': {'rank': 29, 'team': "****", 'mu': 113.9768, 'sigma': 0.7161, 'score': 111.8285, 'games': 483, 'win_rate': 0.5859, 'avg_rank': 0.5942}, '****': {'rank': 30, 'team': "****", 'mu': 114.0204, 'sigma': 0.733, 'score': 111.8213, 'games': 137, 'win_rate': 0.5766, 'avg_rank': 0.6277}, '****': {'rank': 31, 'team': "****", 'mu': 113.9768, 'sigma': 0.7195, 'score': 111.8182, 'games': 174, 'win_rate': 0.477, 'avg_rank': 0.6839}, '****': {'rank': 32, 'team': "****", 'mu': 113.9953, 'sigma': 0.7262, 'score': 111.8166, 'games': 750, 'win_rate': 0.592, 'avg_rank': 0.5347}, '****': {'rank': 33, 'team': "****", 'mu': 113.9939, 'sigma': 0.7263, 'score': 111.815, 'games': 308, 'win_rate': 0.5747, 'avg_rank': 0.5747}, '****': {'rank': 34, 'team': "****", 'mu': 113.9509, 'sigma': 0.7128, 'score': 111.8127, 'games': 516, 'win_rate': 0.564, 'avg_rank': 0.5698}, '****': {'rank': 35, 'team': "****", 'mu': 113.9795, 'sigma': 0.7224, 'score': 111.8123, 'games': 478, 'win_rate': 0.569, 'avg_rank': 0.6444}, '****': {'rank': 36, 'team': "****", 'mu': 114.0161, 'sigma': 0.7349, 'score': 111.8114, 'games': 143, 'win_rate': 0.5315, 'avg_rank': 0.5734}, '****': {'rank': 37, 'team': "****", 'mu': 114.0231, 'sigma': 0.7376, 'score': 111.8105, 'games': 644, 'win_rate': 0.5978, 'avg_rank': 0.587}, '****': {'rank': 38, 'team': "****", 'mu': 113.9523, 'sigma': 0.7154, 'score': 111.806, 'games': 228, 'win_rate': 0.6009, 'avg_rank': 0.6184}, '****': {'rank': 39, 'team': "****", 'mu': 113.9546, 'sigma': 0.7167, 'score': 111.8045, 'games': 203, 'win_rate': 0.5468, 'avg_rank': 0.5911}, '****': {'rank': 40, 'team': "****", 'mu': 113.9576, 'sigma': 0.718, 'score': 111.8036, 'games': 490, 'win_rate': 0.5633, 'avg_rank': 0.6}, '****': {'rank': 41, 'team': "****", 'mu': 113.9896, 'sigma': 0.7291, 'score': 111.8023, 'games': 112, 'win_rate': 0.4286, 'avg_rank': 0.75}, '****': {'rank': 42, 'team': "****", 'mu': 113.955, 'sigma': 0.7184, 'score': 111.8, 'games': 497, 'win_rate': 0.5895, 'avg_rank': 0.5614}, '****': {'rank': 43, 'team': "****", 'mu': 113.985, 'sigma': 0.7285, 'score': 111.7995, 'games': 164, 'win_rate': 0.5366, 'avg_rank': 0.7134}, '****': {'rank': 44, 'team': "****", 'mu': 113.9674, 'sigma': 0.7227, 'score': 111.7994, 'games': 572, 'win_rate': 0.5559, 'avg_rank': 0.5944}, '****': {'rank': 45, 'team': "****", 'mu': 113.9442, 'sigma': 0.7151, 'score': 111.7988, 'games': 694, 'win_rate': 0.5951, 'avg_rank': 0.5216}, '****': {'rank': 46, 'team': "****", 'mu': 113.9506, 'sigma': 0.7177, 'score': 111.7974, 'games': 655, 'win_rate': 0.6031, 'avg_rank': 0.5771}, '****': {'rank': 47, 'team': "****", 'mu': 113.9488, 'sigma': 0.7178, 'score': 111.7953, 'games': 251, 'win_rate': 0.5339, 'avg_rank': 0.6056}, '****': {'rank': 48, 'team': "****", 'mu': 113.9766, 'sigma': 0.7272, 'score': 111.7952, 'games': 407, 'win_rate': 0.6462, 'avg_rank': 0.5356}, '****': {'rank': 49, 'team': "****", 'mu': 113.9419, 'sigma': 0.7166, 'score': 111.792, 'games': 208, 'win_rate': 0.5721, 'avg_rank': 0.6346}, '****': {'rank': 50, 'team': "****", 'mu': 113.9806, 'sigma': 0.7302, 'score': 111.7901, 'games': 186, 'win_rate': 0.6129, 'avg_rank': 0.5269}, '****': {'rank': 51, 'team': "****", 'mu': 113.9392, 'sigma': 0.7178, 'score': 111.7859, 'games': 362, 'win_rate': 0.6354, 'avg_rank': 0.547}, '****': {'rank': 52, 'team': "****", 'mu': 114.0177, 'sigma': 0.7445, 'score': 111.7842, 'games': 123, 'win_rate': 0.5203, 'avg_rank': 0.626}, '****': {'rank': 53, 'team': "****", 'mu': 113.9906, 'sigma': 0.7359, 'score': 111.7829, 'games': 137, 'win_rate': 0.5693, 'avg_rank': 0.5839}, '****': {'rank': 54, 'team': "****", 'mu': 113.9195, 'sigma': 0.7123, 'score': 111.7826, 'games': 228, 'win_rate': 0.5526, 'avg_rank': 0.6184}, '****': {'rank': 55, 'team': "****", 'mu': 114.042, 'sigma': 0.7543, 'score': 111.7789, 'games': 121, 'win_rate': 0.595, 'avg_rank': 0.6777}, '****': {'rank': 56, 'team': "****", 'mu': 113.9183, 'sigma': 0.7155, 'score': 111.7719, 'games': 534, 'win_rate': 0.6592, 'avg_rank': 0.5112}, '****': {'rank': 57, 'team': "****", 'mu': 113.8418, 'sigma': 0.6905, 'score': 111.7702, 'games': 210, 'win_rate': 0.3714, 'avg_rank': 0.919}, '****': {'rank': 58, 'team': "****", 'mu': 113.958, 'sigma': 0.733, 'score': 111.7591, 'games': 186, 'win_rate': 0.5645, 'avg_rank': 0.5968}, '****': {'rank': 59, 'team': "****", 'mu': 113.9257, 'sigma': 0.7226, 'score': 111.758, 'games': 426, 'win_rate': 0.6338, 'avg_rank': 0.5962}, '****': {'rank': 60, 'team': "****", 'mu': 113.898, 'sigma': 0.7151, 'score': 111.7526, 'games': 187, 'win_rate': 0.5187, 'avg_rank': 0.7005}, '****': {'rank': 61, 'team': "****", 'mu': 114.0487, 'sigma': 0.7655, 'score': 111.7523, 'games': 97, 'win_rate': 0.5258, 'avg_rank': 0.6289}, '****': {'rank': 62, 'team': "****", 'mu': 113.9251, 'sigma': 0.7258, 'score': 111.7478, 'games': 655, 'win_rate': 0.6061, 'avg_rank': 0.542}, '****': {'rank': 63, 'team': "****", 'mu': 113.903, 'sigma': 0.7187, 'score': 111.7467, 'games': 373, 'win_rate': 0.5094, 'avg_rank': 0.6434}, '****': {'rank': 64, 'team': "****", 'mu': 113.8833, 'sigma': 0.7124, 'score': 111.7459, 'games': 458, 'win_rate': 0.5786, 'avg_rank': 0.6157}, '****': {'rank': 65, 'team': "****", 'mu': 113.8915, 'sigma': 0.7155, 'score': 111.7451, 'games': 557, 'win_rate': 0.5673, 'avg_rank': 0.6535}, '****': {'rank': 66, 'team': "****", 'mu': 113.8612, 'sigma': 0.7056, 'score': 111.7445, 'games': 610, 'win_rate': 0.5639, 'avg_rank': 0.6033}, '****': {'rank': 67, 'team': "****", 'mu': 113.9125, 'sigma': 0.723, 'score': 111.7436, 'games': 749, 'win_rate': 0.6168, 'avg_rank': 0.534}, '****': {'rank': 68, 'team': "****", 'mu': 113.9173, 'sigma': 0.7259, 'score': 111.7395, 'games': 406, 'win_rate': 0.6084, 'avg_rank': 0.5887}, '****': {'rank': 69, 'team': "****", 'mu': 113.8885, 'sigma': 0.7182, 'score': 111.734, 'games': 424, 'win_rate': 0.5566, 'avg_rank': 0.6203}, '****': {'rank': 70, 'team': "****", 'mu': 113.8525, 'sigma': 0.7066, 'score': 111.7329, 'games': 180, 'win_rate': 0.4278, 'avg_rank': 0.8167}, '****': {'rank': 71, 'team': "****", 'mu': 113.8557, 'sigma': 0.7127, 'score': 111.7175, 'games': 300, 'win_rate': 0.6267, 'avg_rank': 0.6133}, '****': {'rank': 72, 'team': "****", 'mu': 113.8423, 'sigma': 0.7087, 'score': 111.7163, 'games': 316, 'win_rate': 0.4241, 'avg_rank': 0.7468}, '****': {'rank': 73, 'team': "****", 'mu': 113.8918, 'sigma': 0.7261, 'score': 111.7134, 'games': 133, 'win_rate': 0.5714, 'avg_rank': 0.6241}, '****': {'rank': 74, 'team': "****", 'mu': 113.8568, 'sigma': 0.7163, 'score': 111.7078, 'games': 479, 'win_rate': 0.5699, 'avg_rank': 0.6367}, '****': {'rank': 75, 'team': "****", 'mu': 113.9381, 'sigma': 0.7443, 'score': 111.7052, 'games': 133, 'win_rate': 0.594, 'avg_rank': 0.5789}, '****': {'rank': 76, 'team': "****", 'mu': 113.83, 'sigma': 0.709, 'score': 111.7028, 'games': 202, 'win_rate': 0.4901, 'avg_rank': 0.7426}, '****': {'rank': 77, 'team': "****", 'mu': 113.8419, 'sigma': 0.7147, 'score': 111.698, 'games': 222, 'win_rate': 0.464, 'avg_rank': 0.6982}, '****': {'rank': 78, 'team': "****", 'mu': 113.8826, 'sigma': 0.7282, 'score': 111.6979, 'games': 246, 'win_rate': 0.5772, 'avg_rank': 0.6179}, '****': {'rank': 79, 'team': "****", 'mu': 113.8995, 'sigma': 0.7341, 'score': 111.6971, 'games': 102, 'win_rate': 0.4706, 'avg_rank': 0.902}, '****': {'rank': 80, 'team': "****", 'mu': 113.7986, 'sigma': 0.7026, 'score': 111.6908, 'games': 331, 'win_rate': 0.5801, 'avg_rank': 0.6375}, '****': {'rank': 81, 'team': "****", 'mu': 113.8476, 'sigma': 0.7204, 'score': 111.6863, 'games': 305, 'win_rate': 0.5377, 'avg_rank': 0.6754}, '****': {'rank': 82, 'team': "****", 'mu': 113.8481, 'sigma': 0.7208, 'score': 111.6856, 'games': 295, 'win_rate': 0.5627, 'avg_rank': 0.5898}, '****': {'rank': 83, 'team': "****", 'mu': 113.8379, 'sigma': 0.7186, 'score': 111.6821, 'games': 434, 'win_rate': 0.5714, 'avg_rank': 0.629}, '****': {'rank': 84, 'team': "****", 'mu': 113.8188, 'sigma': 0.7129, 'score': 111.6801, 'games': 505, 'win_rate': 0.5683, 'avg_rank': 0.604}, '****': {'rank': 85, 'team': "****", 'mu': 113.8849, 'sigma': 0.7359, 'score': 111.6773, 'games': 156, 'win_rate': 0.5897, 'avg_rank': 0.5833}, '****': {'rank': 86, 'team': "****", 'mu': 113.7481, 'sigma': 0.6917, 'score': 111.6731, 'games': 435, 'win_rate': 0.4529, 'avg_rank': 0.7747}, '****': {'rank': 87, 'team': "****", 'mu': 113.8222, 'sigma': 0.7176, 'score': 111.6693, 'games': 392, 'win_rate': 0.5128, 'avg_rank': 0.6224}, '****': {'rank': 88, 'team': "****", 'mu': 113.806, 'sigma': 0.7124, 'score': 111.6689, 'games': 597, 'win_rate': 0.5796, 'avg_rank': 0.5595}, '****': {'rank': 89, 'team': "****", 'mu': 113.8172, 'sigma': 0.7182, 'score': 111.6627, 'games': 358, 'win_rate': 0.6341, 'avg_rank': 0.6006}, '****': {'rank': 90, 'team': "****", 'mu': 113.9643, 'sigma': 0.7717, 'score': 111.6493, 'games': 99, 'win_rate': 0.5556, 'avg_rank': 0.5859}, '****': {'rank': 91, 'team': "****", 'mu': 113.8378, 'sigma': 0.7316, 'score': 111.643, 'games': 147, 'win_rate': 0.5918, 'avg_rank': 0.7143}, '****': {'rank': 92, 'team': "****", 'mu': 113.7951, 'sigma': 0.7182, 'score': 111.6406, 'games': 403, 'win_rate': 0.5583, 'avg_rank': 0.5782}, '****': {'rank': 93, 'team': "****", 'mu': 113.8152, 'sigma': 0.726, 'score': 111.6372, 'games': 154, 'win_rate': 0.5909, 'avg_rank': 0.5974}, '****': {'rank': 94, 'team': "****", 'mu': 113.796, 'sigma': 0.7239, 'score': 111.6244, 'games': 260, 'win_rate': 0.5962, 'avg_rank': 0.5538}, '****': {'rank': 95, 'team': "****", 'mu': 113.7591, 'sigma': 0.7171, 'score': 111.608, 'games': 585, 'win_rate': 0.5282, 'avg_rank': 0.6786}, '****': {'rank': 96, 'team': "****", 'mu': 114.1962, 'sigma': 0.8634, 'score': 111.606, 'games': 62, 'win_rate': 0.6129, 'avg_rank': 0.5645}, '****': {'rank': 97, 'team': "****", 'mu': 113.8089, 'sigma': 0.7346, 'score': 111.6052, 'games': 504, 'win_rate': 0.5913, 'avg_rank': 0.5337}, '****': {'rank': 98, 'team': "****", 'mu': 113.7741, 'sigma': 0.7241, 'score': 111.6018, 'games': 361, 'win_rate': 0.6454, 'avg_rank': 0.5263}, '****': {'rank': 99, 'team': "****", 'mu': 113.9832, 'sigma': 0.7955, 'score': 111.5968, 'games': 82, 'win_rate': 0.5488, 'avg_rank': 0.622}, '****': {'rank': 100, 'team': "****", 'mu': 113.8916, 'sigma': 0.7663, 'score': 111.5927, 'games': 106, 'win_rate': 0.566, 'avg_rank': 0.6698}, '****': {'rank': 101, 'team': "****", 'mu': 113.751, 'sigma': 0.7195, 'score': 111.5924, 'games': 307, 'win_rate': 0.5896, 'avg_rank': 0.6156}, '****': {'rank': 102, 'team': "****", 'mu': 113.7431, 'sigma': 0.718, 'score': 111.5893, 'games': 198, 'win_rate': 0.6263, 'avg_rank': 0.5657}, '****': {'rank': 103, 'team': "****", 'mu': 113.6924, 'sigma': 0.7033, 'score': 111.5826, 'games': 170, 'win_rate': 0.5118, 'avg_rank': 0.9235}, '****': {'rank': 104, 'team': "****", 'mu': 113.7367, 'sigma': 0.7186, 'score': 111.5809, 'games': 258, 'win_rate': 0.5581, 'avg_rank': 0.5775}, '****': {'rank': 105, 'team': "****", 'mu': 113.7279, 'sigma': 0.7161, 'score': 111.5795, 'games': 677, 'win_rate': 0.5968, 'avg_rank': 0.5657}, '****': {'rank': 106, 'team': "****", 'mu': 113.739, 'sigma': 0.7211, 'score': 111.5758, 'games': 619, 'win_rate': 0.588, 'avg_rank': 0.6236}, '****': {'rank': 107, 'team': "****", 'mu': 113.7483, 'sigma': 0.7245, 'score': 111.5747, 'games': 167, 'win_rate': 0.503, 'avg_rank': 0.6347}, '****': {'rank': 108, 'team': "****", 'mu': 113.7253, 'sigma': 0.7178, 'score': 111.572, 'games': 329, 'win_rate': 0.6322, 'avg_rank': 0.5562}, '****': {'rank': 109, 'team': "****", 'mu': 113.7377, 'sigma': 0.7243, 'score': 111.5648, 'games': 512, 'win_rate': 0.6641, 'avg_rank': 0.4473}, '****': {'rank': 110, 'team': "****", 'mu': 113.6938, 'sigma': 0.7101, 'score': 111.5635, 'games': 531, 'win_rate': 0.548, 'avg_rank': 0.5895}, '****': {'rank': 111, 'team': "****", 'mu': 113.715, 'sigma': 0.7176, 'score': 111.5621, 'games': 112, 'win_rate': 0.375, 'avg_rank': 1.0982}, '****': {'rank': 112, 'team': "****", 'mu': 114.12, 'sigma': 0.853, 'score': 111.5612, 'games': 56, 'win_rate': 0.5, 'avg_rank': 0.8929}, '****': {'rank': 113, 'team': "****", 'mu': 113.6816, 'sigma': 0.7069, 'score': 111.561, 'games': 222, 'win_rate': 0.491, 'avg_rank': 0.7207}, '****': {'rank': 114, 'team': "****", 'mu': 113.7507, 'sigma': 0.7304, 'score': 111.5595, 'games': 202, 'win_rate': 0.6337, 'avg_rank': 0.5792}, '****': {'rank': 115, 'team': "****", 'mu': 113.692, 'sigma': 0.7131, 'score': 111.5528, 'games': 541, 'win_rate': 0.5564, 'avg_rank': 0.6414}, '****': {'rank': 116, 'team': "****", 'mu': 113.9347, 'sigma': 0.7959, 'score': 111.547, 'games': 83, 'win_rate': 0.5422, 'avg_rank': 0.6145}, '****': {'rank': 117, 'team': "****", 'mu': 113.7909, 'sigma': 0.7538, 'score': 111.5293, 'games': 84, 'win_rate': 0.4643, 'avg_rank': 0.75}, '****': {'rank': 118, 'team': "****", 'mu': 113.6776, 'sigma': 0.7163, 'score': 111.5288, 'games': 286, 'win_rate': 0.5804, 'avg_rank': 0.6049}, '****': {'rank': 119, 'team': "****", 'mu': 113.6663, 'sigma': 0.7129, 'score': 111.5277, 'games': 698, 'win_rate': 0.6003, 'avg_rank': 0.5745}, '****': {'rank': 120, 'team': "****", 'mu': 113.6665, 'sigma': 0.7133, 'score': 111.5267, 'games': 151, 'win_rate': 0.4636, 'avg_rank': 0.7682}, '****': {'rank': 121, 'team': "****", 'mu': 113.6402, 'sigma': 0.7053, 'score': 111.5244, 'games': 208, 'win_rate': 0.3894, 'avg_rank': 0.8269}, '****': {'rank': 122, 'team': "****", 'mu': 113.9598, 'sigma': 0.8119, 'score': 111.5239, 'games': 83, 'win_rate': 0.6265, 'avg_rank': 0.494}, '****': {'rank': 123, 'team': "****", 'mu': 113.6661, 'sigma': 0.7181, 'score': 111.5117, 'games': 400, 'win_rate': 0.56, 'avg_rank': 0.6}, '****': {'rank': 124, 'team': "****", 'mu': 113.685, 'sigma': 0.7251, 'score': 111.5095, 'games': 440, 'win_rate': 0.5773, 'avg_rank': 0.5523}, '****': {'rank': 125, 'team': "****", 'mu': 113.7565, 'sigma': 0.7491, 'score': 111.5092, 'games': 127, 'win_rate': 0.5433, 'avg_rank': 0.5827}, '****': {'rank': 126, 'team': "****", 'mu': 113.6134, 'sigma': 0.7017, 'score': 111.5083, 'games': 135, 'win_rate': 0.3037, 'avg_rank': 1.1185}, '****': {'rank': 127, 'team': "****", 'mu': 113.6542, 'sigma': 0.7163, 'score': 111.5052, 'games': 429, 'win_rate': 0.5921, 'avg_rank': 0.5688}, '****': {'rank': 128, 'team': "****", 'mu': 113.5864, 'sigma': 0.6965, 'score': 111.4969, 'games': 233, 'win_rate': 0.382, 'avg_rank': 0.8712}, '****': {'rank': 129, 'team': "****", 'mu': 113.6233, 'sigma': 0.7104, 'score': 111.4923, 'games': 125, 'win_rate': 0.352, 'avg_rank': 1.04}, '****': {'rank': 130, 'team': "****", 'mu': 113.6379, 'sigma': 0.726, 'score': 111.4598, 'games': 563, 'win_rate': 0.5915, 'avg_rank': 0.5453}, '****': {'rank': 131, 'team': "****", 'mu': 113.553, 'sigma': 0.6995, 'score': 111.4546, 'games': 242, 'win_rate': 0.4008, 'avg_rank': 0.8512}, '****': {'rank': 132, 'team': "****", 'mu': 113.5476, 'sigma': 0.699, 'score': 111.4506, 'games': 137, 'win_rate': 0.3504, 'avg_rank': 0.9781}, '****': {'rank': 133, 'team': "****", 'mu': 113.7548, 'sigma': 0.7687, 'score': 111.4487, 'games': 105, 'win_rate': 0.6095, 'avg_rank': 0.5524}, '****': {'rank': 134, 'team': "****", 'mu': 113.5568, 'sigma': 0.7065, 'score': 111.4374, 'games': 433, 'win_rate': 0.4942, 'avg_rank': 0.7552}, '****': {'rank': 135, 'team': "****", 'mu': 113.565, 'sigma': 0.716, 'score': 111.4169, 'games': 239, 'win_rate': 0.5565, 'avg_rank': 0.6151}, '****': {'rank': 136, 'team': "****", 'mu': 113.571, 'sigma': 0.7259, 'score': 111.3932, 'games': 195, 'win_rate': 0.559, 'avg_rank': 0.6256}, '****': {'rank': 137, 'team': "****", 'mu': 115.0922, 'sigma': 1.2362, 'score': 111.3834, 'games': 22, 'win_rate': 0.6364, 'avg_rank': 0.6364}, '****': {'rank': 138, 'team': "****", 'mu': 113.5228, 'sigma': 0.7175, 'score': 111.3704, 'games': 636, 'win_rate': 0.4607, 'avg_rank': 0.7531}, '****': {'rank': 139, 'team': "****", 'mu': 113.625, 'sigma': 0.7537, 'score': 111.3639, 'games': 101, 'win_rate': 0.4851, 'avg_rank': 0.6436}, '****': {'rank': 140, 'team': "****", 'mu': 113.5041, 'sigma': 0.7175, 'score': 111.3514, 'games': 270, 'win_rate': 0.5667, 'avg_rank': 0.6259}, '****': {'rank': 141, 'team': "****", 'mu': 113.4867, 'sigma': 0.7154, 'score': 111.3405, 'games': 272, 'win_rate': 0.5294, 'avg_rank': 0.6213}, '****': {'rank': 142, 'team': "****", 'mu': 113.6577, 'sigma': 0.7771, 'score': 111.3263, 'games': 95, 'win_rate': 0.6, 'avg_rank': 0.5579}, '****': {'rank': 143, 'team': "****", 'mu': 113.4525, 'sigma': 0.7105, 'score': 111.3211, 'games': 474, 'win_rate': 0.5886, 'avg_rank': 0.5612}, '****': {'rank': 144, 'team': "****", 'mu': 113.4624, 'sigma': 0.718, 'score': 111.3085, 'games': 224, 'win_rate': 0.5536, 'avg_rank': 0.6205}, '****': {'rank': 145, 'team': "****", 'mu': 113.5904, 'sigma': 0.7693, 'score': 111.2825, 'games': 97, 'win_rate': 0.5773, 'avg_rank': 0.5876}, '****': {'rank': 146, 'team': "****", 'mu': 113.4247, 'sigma': 0.7143, 'score': 111.2819, 'games': 389, 'win_rate': 0.5347, 'avg_rank': 0.6941}, '****': {'rank': 147, 'team': "****", 'mu': 114.1518, 'sigma': 0.9597, 'score': 111.2728, 'games': 41, 'win_rate': 0.4878, 'avg_rank': 0.7561}, '****': {'rank': 148, 'team': "****", 'mu': 113.751, 'sigma': 0.8286, 'score': 111.2652, 'games': 66, 'win_rate': 0.5, 'avg_rank': 0.7273}, '****': {'rank': 149, 'team': "****", 'mu': 113.4119, 'sigma': 0.7171, 'score': 111.2606, 'games': 256, 'win_rate': 0.5781, 'avg_rank': 0.6445}, '****': {'rank': 150, 'team': "****", 'mu': 113.3903, 'sigma': 0.7135, 'score': 111.2499, 'games': 940, 'win_rate': 0.4628, 'avg_rank': 0.7032}, '****': {'rank': 151, 'team': "****", 'mu': 113.3605, 'sigma': 0.7045, 'score': 111.2471, 'games': 399, 'win_rate': 0.4762, 'avg_rank': 0.7694}, '****': {'rank': 152, 'team': "****", 'mu': 113.3723, 'sigma': 0.7107, 'score': 111.2402, 'games': 918, 'win_rate': 0.4869, 'avg_rank': 0.7081}, '****': {'rank': 153, 'team': "****", 'mu': 113.3826, 'sigma': 0.7151, 'score': 111.2374, 'games': 225, 'win_rate': 0.5333, 'avg_rank': 0.6222}, '****': {'rank': 154, 'team': "****", 'mu': 113.3637, 'sigma': 0.7113, 'score': 111.2297, 'games': 211, 'win_rate': 0.4787, 'avg_rank': 0.7678}, '****': {'rank': 155, 'team': "****", 'mu': 113.345, 'sigma': 0.7055, 'score': 111.2285, 'games': 308, 'win_rate': 0.4221, 'avg_rank': 0.7987}, '****': {'rank': 156, 'team': "****", 'mu': 113.6223, 'sigma': 0.7995, 'score': 111.224, 'games': 79, 'win_rate': 0.481, 'avg_rank': 0.6835}, '****': {'rank': 157, 'team': "****", 'mu': 113.7651, 'sigma': 0.8517, 'score': 111.2102, 'games': 60, 'win_rate': 0.5, 'avg_rank': 0.6333}, '****': {'rank': 158, 'team': "****", 'mu': 113.3042, 'sigma': 0.6984, 'score': 111.2091, 'games': 386, 'win_rate': 0.5233, 'avg_rank': 0.7098}, '****': {'rank': 159, 'team': "****", 'mu': 113.3261, 'sigma': 0.7069, 'score': 111.2055, 'games': 254, 'win_rate': 0.5118, 'avg_rank': 0.7638}, '****': {'rank': 160, 'team': "****", 'mu': 113.3476, 'sigma': 0.7147, 'score': 111.2037, 'games': 583, 'win_rate': 0.5712, 'avg_rank': 0.6038}, '****': {'rank': 161, 'team': "****", 'mu': 113.3327, 'sigma': 0.7137, 'score': 111.1916, 'games': 1000, 'win_rate': 0.602, 'avg_rank': 0.576}, '****': {'rank': 162, 'team': "****", 'mu': 113.3385, 'sigma': 0.7164, 'score': 111.1892, 'games': 212, 'win_rate': 0.5236, 'avg_rank': 0.8396}, '****': {'rank': 163, 'team': "****", 'mu': 113.313, 'sigma': 0.711, 'score': 111.1799, 'games': 612, 'win_rate': 0.5605, 'avg_rank': 0.6144}, '****': {'rank': 164, 'team': "****", 'mu': 113.2782, 'sigma': 0.7037, 'score': 111.1672, 'games': 861, 'win_rate': 0.4251, 'avg_rank': 0.8966}, '****': {'rank': 165, 'team': "****", 'mu': 113.2817, 'sigma': 0.707, 'score': 111.1608, 'games': 243, 'win_rate': 0.4856, 'avg_rank': 0.8025}, '****': {'rank': 166, 'team': "****", 'mu': 113.277, 'sigma': 0.7077, 'score': 111.1538, 'games': 407, 'win_rate': 0.5504, 'avg_rank': 0.6658}, '****': {'rank': 167, 'team': "****", 'mu': 113.4464, 'sigma': 0.766, 'score': 111.1484, 'games': 96, 'win_rate': 0.4896, 'avg_rank': 0.6771}, '****': {'rank': 168, 'team': "****", 'mu': 113.2288, 'sigma': 0.7002, 'score': 111.1283, 'games': 1248, 'win_rate': 0.4736, 'avg_rank': 0.6851}, '****': {'rank': 169, 'team': "****", 'mu': 113.3354, 'sigma': 0.7382, 'score': 111.1207, 'games': 117, 'win_rate': 0.4359, 'avg_rank': 0.7607}, '****': {'rank': 170, 'team': "****", 'mu': 113.2401, 'sigma': 0.7124, 'score': 111.1029, 'games': 279, 'win_rate': 0.5054, 'avg_rank': 0.7455}, '****': {'rank': 171, 'team': "****", 'mu': 114.6581, 'sigma': 1.1859, 'score': 111.1004, 'games': 29, 'win_rate': 0.7241, 'avg_rank': 0.3793}, '****': {'rank': 172, 'team': "****", 'mu': 113.2094, 'sigma': 0.7032, 'score': 111.0997, 'games': 221, 'win_rate': 0.5023, 'avg_rank': 0.7692}, '****': {'rank': 173, 'team': "****", 'mu': 113.1801, 'sigma': 0.6939, 'score': 111.0985, 'games': 145, 'win_rate': 0.2069, 'avg_rank': 1.1517}, '****': {'rank': 174, 'team': "****", 'mu': 113.2081, 'sigma': 0.7045, 'score': 111.0947, 'games': 305, 'win_rate': 0.5377, 'avg_rank': 0.7607}, '****': {'rank': 175, 'team': "****", 'mu': 113.4794, 'sigma': 0.7961, 'score': 111.0912, 'games': 70, 'win_rate': 0.4429, 'avg_rank': 0.9}, '****': {'rank': 176, 'team': "****", 'mu': 113.1898, 'sigma': 0.7023, 'score': 111.083, 'games': 170, 'win_rate': 0.4235, 'avg_rank': 0.9647}, '****': {'rank': 177, 'team': "****", 'mu': 113.2712, 'sigma': 0.7302, 'score': 111.0805, 'games': 160, 'win_rate': 0.4875, 'avg_rank': 0.7562}, '****': {'rank': 178, 'team': "****", 'mu': 113.2038, 'sigma': 0.7121, 'score': 111.0675, 'games': 320, 'win_rate': 0.575, 'avg_rank': 0.7094}, '****': {'rank': 179, 'team': "****", 'mu': 113.1732, 'sigma': 0.7151, 'score': 111.0279, 'games': 227, 'win_rate': 0.5551, 'avg_rank': 0.6872}, '****': {'rank': 180, 'team': "****", 'mu': 113.535, 'sigma': 0.8363, 'score': 111.0261, 'games': 68, 'win_rate': 0.5441, 'avg_rank': 0.6618}, '****': {'rank': 181, 'team': "****", 'mu': 115.0372, 'sigma': 1.3379, 'score': 111.0234, 'games': 17, 'win_rate': 0.4118, 'avg_rank': 0.7059}, '****': {'rank': 182, 'team': "****", 'mu': 113.1128, 'sigma': 0.6981, 'score': 111.0187, 'games': 178, 'win_rate': 0.3371, 'avg_rank': 1.1685}, '****': {'rank': 183, 'team': "****", 'mu': 113.1472, 'sigma': 0.7112, 'score': 111.0137, 'games': 155, 'win_rate': 0.4774, 'avg_rank': 0.7871}, '****': {'rank': 184, 'team': "****", 'mu': 114.9298, 'sigma': 1.3062, 'score': 111.0112, 'games': 18, 'win_rate': 0.3333, 'avg_rank': 0.8889}, '****': {'rank': 185, 'team': "****", 'mu': 113.1614, 'sigma': 0.7187, 'score': 111.0054, 'games': 431, 'win_rate': 0.5197, 'avg_rank': 0.6914}, '****': {'rank': 186, 'team': "****", 'mu': 113.1179, 'sigma': 0.7129, 'score': 110.9793, 'games': 540, 'win_rate': 0.537, 'avg_rank': 0.613}, '****': {'rank': 187, 'team': "****", 'mu': 113.6747, 'sigma': 0.8986, 'score': 110.979, 'games': 49, 'win_rate': 0.4082, 'avg_rank': 0.9592}, '****': {'rank': 188, 'team': "****", 'mu': 113.7131, 'sigma': 0.9122, 'score': 110.9766, 'games': 49, 'win_rate': 0.5306, 'avg_rank': 0.8571}, '****': {'rank': 189, 'team': "****", 'mu': 113.3953, 'sigma': 0.8196, 'score': 110.9365, 'games': 69, 'win_rate': 0.4783, 'avg_rank': 0.6232}, '****': {'rank': 190, 'team': "****", 'mu': 113.3468, 'sigma': 0.8148, 'score': 110.9024, 'games': 72, 'win_rate': 0.5417, 'avg_rank': 0.625}, '****': {'rank': 191, 'team': "****", 'mu': 113.3137, 'sigma': 0.8107, 'score': 110.8817, 'games': 73, 'win_rate': 0.5068, 'avg_rank': 0.6712}, '****': {'rank': 192, 'team': "****", 'mu': 113.5053, 'sigma': 0.8824, 'score': 110.8581, 'games': 54, 'win_rate': 0.537, 'avg_rank': 0.7222}, '****': {'rank': 193, 'team': "****", 'mu': 113.8674, 'sigma': 1.0038, 'score': 110.856, 'games': 33, 'win_rate': 0.2424, 'avg_rank': 1.1515}, '****': {'rank': 194, 'team': "****", 'mu': 113.0855, 'sigma': 0.7541, 'score': 110.8233, 'games': 94, 'win_rate': 0.3404, 'avg_rank': 0.9574}, '****': {'rank': 195, 'team': "****", 'mu': 112.9023, 'sigma': 0.7042, 'score': 110.7896, 'games': 427, 'win_rate': 0.4824, 'avg_rank': 0.7143}, '****': {'rank': 196, 'team': "****", 'mu': 112.8515, 'sigma': 0.7087, 'score': 110.7254, 'games': 197, 'win_rate': 0.4721, 'avg_rank': 0.7005}, '****': {'rank': 197, 'team': "****", 'mu': 113.1768, 'sigma': 0.8182, 'score': 110.7223, 'games': 61, 'win_rate': 0.3443, 'avg_rank': 1.0164}, '****': {'rank': 198, 'team': "****", 'mu': 112.859, 'sigma': 0.7196, 'score': 110.7002, 'games': 131, 'win_rate': 0.4885, 'avg_rank': 0.7939}, '****': {'rank': 199, 'team': "****", 'mu': 114.7091, 'sigma': 1.3397, 'score': 110.6901, 'games': 19, 'win_rate': 0.5789, 'avg_rank': 0.7895}, '****': {'rank': 200, 'team': "****", 'mu': 112.9663, 'sigma': 0.7603, 'score': 110.6855, 'games': 77, 'win_rate': 0.2987, 'avg_rank': 1.026}, '****': {'rank': 201, 'team': "****", 'mu': 113.9408, 'sigma': 1.0947, 'score': 110.6566, 'games': 27, 'win_rate': 0.1481, 'avg_rank': 1.2593}, '****': {'rank': 202, 'team': "****", 'mu': 112.9796, 'sigma': 0.776, 'score': 110.6516, 'games': 85, 'win_rate': 0.4588, 'avg_rank': 0.7059}, '****': {'rank': 203, 'team': "****", 'mu': 112.7418, 'sigma': 0.6969, 'score': 110.6511, 'games': 920, 'win_rate': 0.4674, 'avg_rank': 0.8272}, '****': {'rank': 204, 'team': "****", 'mu': 112.7053, 'sigma': 0.703, 'score': 110.5964, 'games': 393, 'win_rate': 0.4784, 'avg_rank': 0.7074}, '****': {'rank': 205, 'team': "****", 'mu': 112.699, 'sigma': 0.7062, 'score': 110.5803, 'games': 639, 'win_rate': 0.4773, 'avg_rank': 0.6808}, '****': {'rank': 206, 'team': "****", 'mu': 112.6795, 'sigma': 0.7007, 'score': 110.5774, 'games': 528, 'win_rate': 0.4659, 'avg_rank': 0.7973}, '****': {'rank': 207, 'team': "****", 'mu': 112.6374, 'sigma': 0.6885, 'score': 110.5719, 'games': 732, 'win_rate': 0.3607, 'avg_rank': 0.8566}, '****': {'rank': 208, 'team': "****", 'mu': 112.7008, 'sigma': 0.7098, 'score': 110.5713, 'games': 329, 'win_rate': 0.4559, 'avg_rank': 0.7295}, '****': {'rank': 209, 'team': "****", 'mu': 112.7042, 'sigma': 0.7132, 'score': 110.5646, 'games': 369, 'win_rate': 0.4336, 'avg_rank': 0.7913}, '****': {'rank': 210, 'team': "****", 'mu': 112.7019, 'sigma': 0.7128, 'score': 110.5634, 'games': 807, 'win_rate': 0.5105, 'avg_rank': 0.6568}, '****': {'rank': 211, 'team': "****", 'mu': 112.6518, 'sigma': 0.7033, 'score': 110.5417, 'games': 805, 'win_rate': 0.5528, 'avg_rank': 0.718}, '****': {'rank': 212, 'team': "****", 'mu': 112.6272, 'sigma': 0.696, 'score': 110.5391, 'games': 472, 'win_rate': 0.4153, 'avg_rank': 0.803}, '****': {'rank': 213, 'team': "****", 'mu': 112.6802, 'sigma': 0.7143, 'score': 110.5373, 'games': 987, 'win_rate': 0.5106, 'avg_rank': 0.6839}, '****': {'rank': 214, 'team': "****", 'mu': 113.0288, 'sigma': 0.8322, 'score': 110.5322, 'games': 67, 'win_rate': 0.5224, 'avg_rank': 0.7463}, '****': {'rank': 215, 'team': "****", 'mu': 112.793, 'sigma': 0.7558, 'score': 110.5255, 'games': 86, 'win_rate': 0.3372, 'avg_rank': 0.9767}, '****': {'rank': 216, 'team': "****", 'mu': 112.6574, 'sigma': 0.711, 'score': 110.5245, 'games': 222, 'win_rate': 0.455, 'avg_rank': 0.7838}, '****': {'rank': 217, 'team': "****", 'mu': 112.6572, 'sigma': 0.7119, 'score': 110.5214, 'games': 201, 'win_rate': 0.4229, 'avg_rank': 0.7313}, '****': {'rank': 218, 'team': "****", 'mu': 112.6418, 'sigma': 0.7105, 'score': 110.5102, 'games': 638, 'win_rate': 0.4859, 'avg_rank': 0.721}, '****': {'rank': 219, 'team': "****", 'mu': 112.6276, 'sigma': 0.7074, 'score': 110.5053, 'games': 237, 'win_rate': 0.4726, 'avg_rank': 0.8143}, '****': {'rank': 220, 'team': "****", 'mu': 112.6053, 'sigma': 0.701, 'score': 110.5022, 'games': 1008, 'win_rate': 0.4742, 'avg_rank': 0.7312}, '****': {'rank': 221, 'team': "****", 'mu': 112.6196, 'sigma': 0.7062, 'score': 110.5011, 'games': 973, 'win_rate': 0.4779, 'avg_rank': 0.7318}, '****': {'rank': 222, 'team': "****", 'mu': 112.7297, 'sigma': 0.7508, 'score': 110.4773, 'games': 107, 'win_rate': 0.5047, 'avg_rank': 0.7103}, '****': {'rank': 223, 'team': "****", 'mu': 112.5844, 'sigma': 0.7044, 'score': 110.4711, 'games': 499, 'win_rate': 0.487, 'avg_rank': 0.8036}, '****': {'rank': 224, 'team': "****", 'mu': 112.563, 'sigma': 0.6991, 'score': 110.4658, 'games': 447, 'win_rate': 0.4877, 'avg_rank': 0.8121}, '****': {'rank': 225, 'team': "****", 'mu': 112.5545, 'sigma': 0.6964, 'score': 110.4654, 'games': 582, 'win_rate': 0.4794, 'avg_rank': 0.7405}, '****': {'rank': 226, 'team': "****", 'mu': 112.4899, 'sigma': 0.6897, 'score': 110.4208, 'games': 190, 'win_rate': 0.3632, 'avg_rank': 0.9}, '****': {'rank': 227, 'team': "****", 'mu': 113.1575, 'sigma': 0.9141, 'score': 110.4152, 'games': 47, 'win_rate': 0.4681, 'avg_rank': 0.7447}, '****': {'rank': 228, 'team': "****", 'mu': 112.5899, 'sigma': 0.7283, 'score': 110.4051, 'games': 99, 'win_rate': 0.2828, 'avg_rank': 1.1313}, '****': {'rank': 229, 'team': "****", 'mu': 112.3959, 'sigma': 0.6881, 'score': 110.3316, 'games': 154, 'win_rate': 0.2597, 'avg_rank': 1.2532}, '****': {'rank': 230, 'team': "****", 'mu': 112.4322, 'sigma': 0.7013, 'score': 110.3284, 'games': 280, 'win_rate': 0.4464, 'avg_rank': 0.7571}, '****': {'rank': 231, 'team': "****", 'mu': 112.4587, 'sigma': 0.7167, 'score': 110.3084, 'games': 139, 'win_rate': 0.3885, 'avg_rank': 0.7914}, '****': {'rank': 232, 'team': "****", 'mu': 113.0219, 'sigma': 0.9045, 'score': 110.3083, 'games': 48, 'win_rate': 0.4583, 'avg_rank': 0.75}, '****': {'rank': 233, 'team': "****", 'mu': 112.3979, 'sigma': 0.7059, 'score': 110.2804, 'games': 188, 'win_rate': 0.4894, 'avg_rank': 0.7766}, '****': {'rank': 234, 'team': "****", 'mu': 112.3531, 'sigma': 0.6984, 'score': 110.2578, 'games': 169, 'win_rate': 0.4024, 'avg_rank': 0.8994}, '****': {'rank': 235, 'team': "****", 'mu': 112.4926, 'sigma': 0.746, 'score': 110.2544, 'games': 97, 'win_rate': 0.4021, 'avg_rank': 0.8144}, '****': {'rank': 236, 'team': "****", 'mu': 112.8586, 'sigma': 0.8832, 'score': 110.209, 'games': 50, 'win_rate': 0.4, 'avg_rank': 0.8}, '****': {'rank': 237, 'team': "****", 'mu': 112.2776, 'sigma': 0.6974, 'score': 110.1855, 'games': 263, 'win_rate': 0.4335, 'avg_rank': 0.73}, '****': {'rank': 238, 'team': "****", 'mu': 112.2517, 'sigma': 0.6959, 'score': 110.1638, 'games': 185, 'win_rate': 0.3892, 'avg_rank': 0.9027}, '****': {'rank': 239, 'team': "****", 'mu': 112.4165, 'sigma': 0.7591, 'score': 110.1391, 'games': 93, 'win_rate': 0.4839, 'avg_rank': 0.7957}, '****': {'rank': 240, 'team': "****", 'mu': 112.5841, 'sigma': 0.817, 'score': 110.1332, 'games': 69, 'win_rate': 0.5217, 'avg_rank': 0.6957}, '****': {'rank': 241, 'team': "****", 'mu': 112.3829, 'sigma': 0.7712, 'score': 110.0693, 'games': 85, 'win_rate': 0.4235, 'avg_rank': 0.8}, '****': {'rank': 242, 'team': "****", 'mu': 112.1241, 'sigma': 0.6952, 'score': 110.0384, 'games': 276, 'win_rate': 0.4565, 'avg_rank': 0.9022}, '****': {'rank': 243, 'team': "****", 'mu': 112.757, 'sigma': 0.9085, 'score': 110.0314, 'games': 55, 'win_rate': 0.5091, 'avg_rank': 0.8364}, '****': {'rank': 244, 'team': "****", 'mu': 112.2387, 'sigma': 0.7423, 'score': 110.0117, 'games': 84, 'win_rate': 0.1667, 'avg_rank': 1.5952}, '****': {'rank': 245, 'team': "****", 'mu': 112.0917, 'sigma': 0.7027, 'score': 109.9837, 'games': 1782, 'win_rate': 0.4585, 'avg_rank': 0.7789}, '****': {'rank': 246, 'team': "****", 'mu': 112.3349, 'sigma': 0.7931, 'score': 109.9556, 'games': 83, 'win_rate': 0.5422, 'avg_rank': 0.7711}, '****': {'rank': 247, 'team': "****", 'mu': 112.0693, 'sigma': 0.7049, 'score': 109.9547, 'games': 288, 'win_rate': 0.4271, 'avg_rank': 0.8819}, '****': {'rank': 248, 'team': "****", 'mu': 112.0625, 'sigma': 0.7062, 'score': 109.944, 'games': 1452, 'win_rate': 0.458, 'avg_rank': 0.7521}, '****': {'rank': 249, 'team': "****", 'mu': 112.2322, 'sigma': 0.7636, 'score': 109.9415, 'games': 84, 'win_rate': 0.381, 'avg_rank': 0.8452}, '****': {'rank': 250, 'team': "****", 'mu': 112.3681, 'sigma': 0.8257, 'score': 109.8911, 'games': 70, 'win_rate': 0.4714, 'avg_rank': 0.7571}, '****': {'rank': 251, 'team': "****", 'mu': 113.5543, 'sigma': 1.2231, 'score': 109.8849, 'games': 26, 'win_rate': 0.6538, 'avg_rank': 0.5769}, '****': {'rank': 252, 'team': "****", 'mu': 113.5421, 'sigma': 1.223, 'score': 109.8732, 'games': 25, 'win_rate': 0.6, 'avg_rank': 0.64}, '****': {'rank': 253, 'team': "****", 'mu': 111.9714, 'sigma': 0.7028, 'score': 109.863, 'games': 245, 'win_rate': 0.498, 'avg_rank': 0.8286}, '****': {'rank': 254, 'team': "****", 'mu': 113.6445, 'sigma': 1.2635, 'score': 109.854, 'games': 21, 'win_rate': 0.4762, 'avg_rank': 0.6667}, '****': {'rank': 255, 'team': "****", 'mu': 111.9936, 'sigma': 0.7176, 'score': 109.8408, 'games': 141, 'win_rate': 0.4468, 'avg_rank': 0.8723}, '****': {'rank': 256, 'team': "****", 'mu': 111.936, 'sigma': 0.7037, 'score': 109.825, 'games': 787, 'win_rate': 0.4562, 'avg_rank': 0.8018}, '****': {'rank': 257, 'team': "****", 'mu': 113.2472, 'sigma': 1.1497, 'score': 109.7981, 'games': 30, 'win_rate': 0.5667, 'avg_rank': 0.7}, '****': {'rank': 258, 'team': "****", 'mu': 113.7663, 'sigma': 1.324, 'score': 109.7945, 'games': 24, 'win_rate': 0.7917, 'avg_rank': 0.25}, '****': {'rank': 259, 'team': "****", 'mu': 111.8748, 'sigma': 0.7027, 'score': 109.7666, 'games': 247, 'win_rate': 0.4818, 'avg_rank': 0.834}, '****': {'rank': 260, 'team': "****", 'mu': 111.9497, 'sigma': 0.7374, 'score': 109.7377, 'games': 113, 'win_rate': 0.4602, 'avg_rank': 0.9115}, '****': {'rank': 261, 'team': "****", 'mu': 111.8178, 'sigma': 0.6959, 'score': 109.73, 'games': 441, 'win_rate': 0.4059, 'avg_rank': 0.9297}, '****': {'rank': 262, 'team': "****", 'mu': 111.7877, 'sigma': 0.689, 'score': 109.7206, 'games': 260, 'win_rate': 0.3577, 'avg_rank': 1.0731}, '****': {'rank': 263, 'team': "****", 'mu': 111.8268, 'sigma': 0.7022, 'score': 109.7202, 'games': 686, 'win_rate': 0.4563, 'avg_rank': 0.8571}, '****': {'rank': 264, 'team': "****", 'mu': 111.7962, 'sigma': 0.6958, 'score': 109.7087, 'games': 249, 'win_rate': 0.4498, 'avg_rank': 0.9398}, '****': {'rank': 265, 'team': "****", 'mu': 112.3164, 'sigma': 0.8709, 'score': 109.7037, 'games': 53, 'win_rate': 0.4151, 'avg_rank': 0.8302}, '****': {'rank': 266, 'team': "****", 'mu': 111.9744, 'sigma': 0.7627, 'score': 109.6863, 'games': 96, 'win_rate': 0.4792, 'avg_rank': 0.75}, '****': {'rank': 267, 'team': "****", 'mu': 111.7661, 'sigma': 0.6935, 'score': 109.6856, 'games': 1097, 'win_rate': 0.4877, 'avg_rank': 0.8122}, '****': {'rank': 268, 'team': "****", 'mu': 111.7695, 'sigma': 0.6975, 'score': 109.677, 'games': 364, 'win_rate': 0.4505, 'avg_rank': 0.9313}, '****': {'rank': 269, 'team': "****", 'mu': 111.7828, 'sigma': 0.7051, 'score': 109.6673, 'games': 163, 'win_rate': 0.4601, 'avg_rank': 0.9448}, '****': {'rank': 270, 'team': "****", 'mu': 111.7757, 'sigma': 0.7035, 'score': 109.6651, 'games': 169, 'win_rate': 0.497, 'avg_rank': 0.929}, '****': {'rank': 271, 'team': "****", 'mu': 111.7552, 'sigma': 0.702, 'score': 109.6492, 'games': 1866, 'win_rate': 0.5584, 'avg_rank': 0.672}, '****': {'rank': 272, 'team': "****", 'mu': 111.7862, 'sigma': 0.7127, 'score': 109.6481, 'games': 298, 'win_rate': 0.5067, 'avg_rank': 0.8423}, '****': {'rank': 273, 'team': "****", 'mu': 111.7561, 'sigma': 0.7054, 'score': 109.6398, 'games': 1432, 'win_rate': 0.4567, 'avg_rank': 0.7709}, '****': {'rank': 274, 'team': "****", 'mu': 111.7109, 'sigma': 0.6982, 'score': 109.6163, 'games': 2243, 'win_rate': 0.5181, 'avg_rank': 0.7325}, '****': {'rank': 275, 'team': "****", 'mu': 111.6955, 'sigma': 0.6962, 'score': 109.6068, 'games': 164, 'win_rate': 0.3293, 'avg_rank': 0.9756}, '****': {'rank': 276, 'team': "****", 'mu': 111.7053, 'sigma': 0.7037, 'score': 109.5943, 'games': 163, 'win_rate': 0.4663, 'avg_rank': 0.9509}, '****': {'rank': 277, 'team': "****", 'mu': 111.7175, 'sigma': 0.7094, 'score': 109.5894, 'games': 284, 'win_rate': 0.493, 'avg_rank': 0.8345}, '****': {'rank': 278, 'team': "****", 'mu': 111.6867, 'sigma': 0.6997, 'score': 109.5876, 'games': 2159, 'win_rate': 0.5359, 'avg_rank': 0.6952}, '****': {'rank': 279, 'team': "****", 'mu': 111.6468, 'sigma': 0.6872, 'score': 109.5852, 'games': 402, 'win_rate': 0.4204, 'avg_rank': 1.0547}, '****': {'rank': 280, 'team': "****", 'mu': 111.6552, 'sigma': 0.6936, 'score': 109.5744, 'games': 635, 'win_rate': 0.4205, 'avg_rank': 1.0929}, '****': {'rank': 281, 'team': "****", 'mu': 111.6759, 'sigma': 0.7019, 'score': 109.5702, 'games': 201, 'win_rate': 0.5075, 'avg_rank': 0.8856}, '****': {'rank': 282, 'team': "****", 'mu': 112.3601, 'sigma': 0.9325, 'score': 109.5626, 'games': 47, 'win_rate': 0.5745, 'avg_rank': 0.6809}, '****': {'rank': 283, 'team': "****", 'mu': 113.8939, 'sigma': 1.4444, 'score': 109.5607, 'games': 17, 'win_rate': 0.5882, 'avg_rank': 0.5882}, '****': {'rank': 284, 'team': "****", 'mu': 112.6599, 'sigma': 1.035, 'score': 109.555, 'games': 33, 'win_rate': 0.3939, 'avg_rank': 0.9697}, '****': {'rank': 285, 'team': "****", 'mu': 111.6494, 'sigma': 0.6985, 'score': 109.5538, 'games': 195, 'win_rate': 0.3333, 'avg_rank': 1.0615}, '****': {'rank': 286, 'team': "****", 'mu': 111.6731, 'sigma': 0.712, 'score': 109.537, 'games': 140, 'win_rate': 0.4357, 'avg_rank': 0.9357}, '****': {'rank': 287, 'team': "****", 'mu': 111.6464, 'sigma': 0.7053, 'score': 109.5306, 'games': 144, 'win_rate': 0.4097, 'avg_rank': 0.9792}, '****': {'rank': 288, 'team': "****", 'mu': 113.1899, 'sigma': 1.2239, 'score': 109.5182, 'games': 24, 'win_rate': 0.375, 'avg_rank': 1.0417}, '****': {'rank': 289, 'team': "****", 'mu': 111.5758, 'sigma': 0.688, 'score': 109.512, 'games': 733, 'win_rate': 0.3874, 'avg_rank': 1.146}, '****': {'rank': 290, 'team': "****", 'mu': 112.3306, 'sigma': 0.9397, 'score': 109.5115, 'games': 42, 'win_rate': 0.381, 'avg_rank': 0.7857}, '****': {'rank': 291, 'team': "****", 'mu': 111.6053, 'sigma': 0.7009, 'score': 109.5026, 'games': 196, 'win_rate': 0.4388, 'avg_rank': 0.8827}, '****': {'rank': 292, 'team': "****", 'mu': 111.5684, 'sigma': 0.6979, 'score': 109.4746, 'games': 322, 'win_rate': 0.4534, 'avg_rank': 0.7981}, '****': {'rank': 293, 'team': "****", 'mu': 112.707, 'sigma': 1.0782, 'score': 109.4725, 'games': 32, 'win_rate': 0.5625, 'avg_rank': 0.7812}, '****': {'rank': 294, 'team': "****", 'mu': 111.5714, 'sigma': 0.704, 'score': 109.4594, 'games': 125, 'win_rate': 0.36, 'avg_rank': 1.176}, '****': {'rank': 295, 'team': "****", 'mu': 111.5401, 'sigma': 0.7002, 'score': 109.4394, 'games': 238, 'win_rate': 0.416, 'avg_rank': 0.8782}, '****': {'rank': 296, 'team': "****", 'mu': 111.5351, 'sigma': 0.7033, 'score': 109.4253, 'games': 209, 'win_rate': 0.4737, 'avg_rank': 0.8038}, '****': {'rank': 297, 'team': "****", 'mu': 111.5104, 'sigma': 0.6969, 'score': 109.4197, 'games': 878, 'win_rate': 0.4875, 'avg_rank': 0.787}, '****': {'rank': 298, 'team': "****", 'mu': 111.9079, 'sigma': 0.8305, 'score': 109.4165, 'games': 61, 'win_rate': 0.4098, 'avg_rank': 0.7869}, '****': {'rank': 299, 'team': "****", 'mu': 111.7698, 'sigma': 0.7908, 'score': 109.3975, 'games': 66, 'win_rate': 0.2727, 'avg_rank': 1.2727}, '****': {'rank': 300, 'team': "****", 'mu': 111.498, 'sigma': 0.7074, 'score': 109.376, 'games': 1780, 'win_rate': 0.5185, 'avg_rank': 0.8073}, '****': {'rank': 301, 'team': "****", 'mu': 113.2434, 'sigma': 1.2954, 'score': 109.3572, 'games': 19, 'win_rate': 0.3684, 'avg_rank': 0.8947}, '****': {'rank': 302, 'team': "****", 'mu': 111.563, 'sigma': 0.7414, 'score': 109.3389, 'games': 89, 'win_rate': 0.3258, 'avg_rank': 1.0562}, '****': {'rank': 303, 'team': "****", 'mu': 111.3603, 'sigma': 0.6756, 'score': 109.3334, 'games': 419, 'win_rate': 0.1122, 'avg_rank': 1.2172}, '****': {'rank': 304, 'team': "****", 'mu': 111.4199, 'sigma': 0.7006, 'score': 109.318, 'games': 262, 'win_rate': 0.3969, 'avg_rank': 0.8855}, '****': {'rank': 305, 'team': "****", 'mu': 111.3126, 'sigma': 0.6781, 'score': 109.2784, 'games': 412, 'win_rate': 0.0777, 'avg_rank': 1.2597}, '****': {'rank': 306, 'team': "****", 'mu': 112.9982, 'sigma': 1.2562, 'score': 109.2298, 'games': 23, 'win_rate': 0.6522, 'avg_rank': 0.6087}, '****': {'rank': 307, 'team': "****", 'mu': 112.6164, 'sigma': 1.1304, 'score': 109.2252, 'games': 30, 'win_rate': 0.5333, 'avg_rank': 0.7}, '****': {'rank': 308, 'team': "****", 'mu': 111.3065, 'sigma': 0.6943, 'score': 109.2237, 'games': 531, 'win_rate': 0.3653, 'avg_rank': 0.9115}, '****': {'rank': 309, 'team': "****", 'mu': 113.9668, 'sigma': 1.5815, 'score': 109.2224, 'games': 12, 'win_rate': 0.4167, 'avg_rank': 0.5833}, '****': {'rank': 310, 'team': "****", 'mu': 111.3159, 'sigma': 0.7026, 'score': 109.2082, 'games': 654, 'win_rate': 0.5122, 'avg_rank': 0.7997}, '****': {'rank': 311, 'team': "****", 'mu': 111.3079, 'sigma': 0.702, 'score': 109.202, 'games': 148, 'win_rate': 0.4459, 'avg_rank': 0.8581}, '****': {'rank': 312, 'team': "****", 'mu': 111.2579, 'sigma': 0.6912, 'score': 109.1842, 'games': 447, 'win_rate': 0.3289, 'avg_rank': 0.953}, '****': {'rank': 313, 'team': "****", 'mu': 112.1461, 'sigma': 0.9913, 'score': 109.1722, 'games': 38, 'win_rate': 0.4211, 'avg_rank': 0.7368}, '****': {'rank': 314, 'team': "****", 'mu': 111.1576, 'sigma': 0.6941, 'score': 109.0755, 'games': 929, 'win_rate': 0.4758, 'avg_rank': 0.8095}, '****': {'rank': 315, 'team': "****", 'mu': 111.1306, 'sigma': 0.701, 'score': 109.0275, 'games': 682, 'win_rate': 0.4927, 'avg_rank': 0.8314}, '****': {'rank': 316, 'team': "****", 'mu': 111.1165, 'sigma': 0.6966, 'score': 109.0266, 'games': 213, 'win_rate': 0.4131, 'avg_rank': 1.0235}, '****': {'rank': 317, 'team': "****", 'mu': 111.0799, 'sigma': 0.6856, 'score': 109.0231, 'games': 232, 'win_rate': 0.2888, 'avg_rank': 1.1595}, '****': {'rank': 318, 'team': "****", 'mu': 111.2054, 'sigma': 0.7275, 'score': 109.0227, 'games': 105, 'win_rate': 0.3429, 'avg_rank': 1.1238}, '****': {'rank': 319, 'team': "****", 'mu': 111.2182, 'sigma': 0.7348, 'score': 109.0139, 'games': 89, 'win_rate': 0.0899, 'avg_rank': 1.4944}, '****': {'rank': 320, 'team': "****", 'mu': 111.1132, 'sigma': 0.7034, 'score': 109.0029, 'games': 862, 'win_rate': 0.4594, 'avg_rank': 0.8886}, '****': {'rank': 321, 'team': "****", 'mu': 111.1281, 'sigma': 0.7089, 'score': 109.0013, 'games': 219, 'win_rate': 0.411, 'avg_rank': 0.9635}, '****': {'rank': 322, 'team': "****", 'mu': 111.0924, 'sigma': 0.6976, 'score': 108.9995, 'games': 191, 'win_rate': 0.4241, 'avg_rank': 0.9162}, '****': {'rank': 323, 'team': "****", 'mu': 111.1574, 'sigma': 0.7194, 'score': 108.9993, 'games': 117, 'win_rate': 0.4274, 'avg_rank': 1.0}, '****': {'rank': 324, 'team': "****", 'mu': 111.0771, 'sigma': 0.6963, 'score': 108.9882, 'games': 230, 'win_rate': 0.3609, 'avg_rank': 1.0957}, '****': {'rank': 325, 'team': "****", 'mu': 111.0873, 'sigma': 0.7027, 'score': 108.9792, 'games': 238, 'win_rate': 0.4454, 'avg_rank': 0.9664}, '****': {'rank': 326, 'team': "****", 'mu': 111.6956, 'sigma': 0.9086, 'score': 108.9698, 'games': 46, 'win_rate': 0.5435, 'avg_rank': 0.7174}, '****': {'rank': 327, 'team': "****", 'mu': 111.6055, 'sigma': 0.8901, 'score': 108.9353, 'games': 48, 'win_rate': 0.2083, 'avg_rank': 1.3125}, '****': {'rank': 328, 'team': "****", 'mu': 111.0762, 'sigma': 0.7144, 'score': 108.9332, 'games': 115, 'win_rate': 0.3043, 'avg_rank': 1.2696}, '****': {'rank': 329, 'team': "****", 'mu': 110.9906, 'sigma': 0.6912, 'score': 108.9169, 'games': 893, 'win_rate': 0.4233, 'avg_rank': 0.9037}, '****': {'rank': 330, 'team': "****", 'mu': 111.0113, 'sigma': 0.7124, 'score': 108.8743, 'games': 182, 'win_rate': 0.467, 'avg_rank': 0.9176}, '****': {'rank': 331, 'team': "****", 'mu': 112.0516, 'sigma': 1.0611, 'score': 108.8684, 'games': 32, 'win_rate': 0.4375, 'avg_rank': 0.7812}, '****': {'rank': 332, 'team': "****", 'mu': 110.9056, 'sigma': 0.6951, 'score': 108.8202, 'games': 450, 'win_rate': 0.3956, 'avg_rank': 0.9578}, '****': {'rank': 333, 'team': "****", 'mu': 110.8727, 'sigma': 0.6915, 'score': 108.7982, 'games': 1869, 'win_rate': 0.3772, 'avg_rank': 0.8689}, '****': {'rank': 334, 'team': "****", 'mu': 110.9077, 'sigma': 0.7052, 'score': 108.7921, 'games': 235, 'win_rate': 0.417, 'avg_rank': 0.9447}, '****': {'rank': 335, 'team': "****", 'mu': 111.8633, 'sigma': 1.0312, 'score': 108.7696, 'games': 36, 'win_rate': 0.3889, 'avg_rank': 0.9722}, '****': {'rank': 336, 'team': "****", 'mu': 111.2645, 'sigma': 0.8336, 'score': 108.7638, 'games': 62, 'win_rate': 0.371, 'avg_rank': 1.0484}, '****': {'rank': 337, 'team': "****", 'mu': 111.3977, 'sigma': 0.899, 'score': 108.7007, 'games': 48, 'win_rate': 0.3333, 'avg_rank': 0.9583}, '****': {'rank': 338, 'team': "****", 'mu': 110.8212, 'sigma': 0.708, 'score': 108.6971, 'games': 122, 'win_rate': 0.3689, 'avg_rank': 1.0902}, '****': {'rank': 339, 'team': "****", 'mu': 111.6443, 'sigma': 0.9875, 'score': 108.6817, 'games': 40, 'win_rate': 0.475, 'avg_rank': 0.9}, '****': {'rank': 340, 'team': "****", 'mu': 110.7407, 'sigma': 0.6939, 'score': 108.6591, 'games': 333, 'win_rate': 0.3994, 'avg_rank': 0.9249}, '****': {'rank': 341, 'team': "****", 'mu': 110.676, 'sigma': 0.6925, 'score': 108.5985, 'games': 969, 'win_rate': 0.3942, 'avg_rank': 1.0114}, '****': {'rank': 342, 'team': "****", 'mu': 113.051, 'sigma': 1.4843, 'score': 108.598, 'games': 14, 'win_rate': 0.5, 'avg_rank': 0.7143}, '****': {'rank': 343, 'team': "****", 'mu': 110.6574, 'sigma': 0.6939, 'score': 108.5757, 'games': 488, 'win_rate': 0.4119, 'avg_rank': 0.9385}, '****': {'rank': 344, 'team': "****", 'mu': 110.6395, 'sigma': 0.6965, 'score': 108.55, 'games': 436, 'win_rate': 0.3555, 'avg_rank': 0.9289}, '****': {'rank': 345, 'team': "****", 'mu': 110.601, 'sigma': 0.689, 'score': 108.534, 'games': 246, 'win_rate': 0.3293, 'avg_rank': 1.2073}, '****': {'rank': 346, 'team': "****", 'mu': 110.6563, 'sigma': 0.7178, 'score': 108.5028, 'games': 116, 'win_rate': 0.3707, 'avg_rank': 1.0086}, '****': {'rank': 347, 'team': "****", 'mu': 110.5699, 'sigma': 0.6897, 'score': 108.5008, 'games': 469, 'win_rate': 0.4094, 'avg_rank': 0.9893}, '****': {'rank': 348, 'team': "****", 'mu': 110.5608, 'sigma': 0.692, 'score': 108.485, 'games': 256, 'win_rate': 0.4375, 'avg_rank': 1.0234}, '****': {'rank': 349, 'team': "****", 'mu': 110.5578, 'sigma': 0.6937, 'score': 108.4766, 'games': 256, 'win_rate': 0.3555, 'avg_rank': 1.0664}, '****': {'rank': 350, 'team': "****", 'mu': 110.5559, 'sigma': 0.694, 'score': 108.4739, 'games': 416, 'win_rate': 0.4111, 'avg_rank': 1.0409}, '****': {'rank': 351, 'team': "****", 'mu': 110.5489, 'sigma': 0.6936, 'score': 108.4681, 'games': 280, 'win_rate': 0.3893, 'avg_rank': 0.9643}, '****': {'rank': 352, 'team': "****", 'mu': 110.5424, 'sigma': 0.6929, 'score': 108.4636, 'games': 924, 'win_rate': 0.3929, 'avg_rank': 1.0119}, '****': {'rank': 353, 'team': "****", 'mu': 111.3998, 'sigma': 0.9793, 'score': 108.4618, 'games': 37, 'win_rate': 0.2162, 'avg_rank': 1.0811}, '****': {'rank': 354, 'team': "****", 'mu': 110.5306, 'sigma': 0.6951, 'score': 108.4452, 'games': 300, 'win_rate': 0.3367, 'avg_rank': 1.09}, '****': {'rank': 355, 'team': "****", 'mu': 110.4963, 'sigma': 0.6907, 'score': 108.4243, 'games': 467, 'win_rate': 0.3812, 'avg_rank': 0.9786}, '****': {'rank': 356, 'team': "****", 'mu': 110.4921, 'sigma': 0.6916, 'score': 108.4173, 'games': 2229, 'win_rate': 0.4065, 'avg_rank': 1.0099}, '****': {'rank': 357, 'team': "****", 'mu': 111.7064, 'sigma': 1.1059, 'score': 108.3886, 'games': 30, 'win_rate': 0.4, 'avg_rank': 1.0}, '****': {'rank': 358, 'team': "****", 'mu': 110.4223, 'sigma': 0.6916, 'score': 108.3476, 'games': 179, 'win_rate': 0.2793, 'avg_rank': 1.2179}, '****': {'rank': 359, 'team': "****", 'mu': 112.4471, 'sigma': 1.3685, 'score': 108.3415, 'games': 18, 'win_rate': 0.3889, 'avg_rank': 1.2222}, '****': {'rank': 360, 'team': "****", 'mu': 110.4111, 'sigma': 0.693, 'score': 108.3321, 'games': 1952, 'win_rate': 0.3519, 'avg_rank': 0.9534}, '****': {'rank': 361, 'team': "****", 'mu': 111.7207, 'sigma': 1.1409, 'score': 108.298, 'games': 28, 'win_rate': 0.5357, 'avg_rank': 0.8214}, '****': {'rank': 362, 'team': "****", 'mu': 110.4147, 'sigma': 0.7077, 'score': 108.2915, 'games': 140, 'win_rate': 0.4, 'avg_rank': 1.0571}, '****': {'rank': 363, 'team': "****", 'mu': 113.002, 'sigma': 1.5719, 'score': 108.2862, 'games': 12, 'win_rate': 0.25, 'avg_rank': 0.9167}, '****': {'rank': 364, 'team': "****", 'mu': 110.3515, 'sigma': 0.6908, 'score': 108.2791, 'games': 1236, 'win_rate': 0.4482, 'avg_rank': 0.8519}, '****': {'rank': 365, 'team': "****", 'mu': 110.4383, 'sigma': 0.7232, 'score': 108.2688, 'games': 102, 'win_rate': 0.2157, 'avg_rank': 1.3627}, '****': {'rank': 366, 'team': "****", 'mu': 110.3259, 'sigma': 0.6969, 'score': 108.2351, 'games': 973, 'win_rate': 0.3782, 'avg_rank': 0.9096}, '****': {'rank': 367, 'team': "****", 'mu': 110.4867, 'sigma': 0.7513, 'score': 108.2327, 'games': 90, 'win_rate': 0.3778, 'avg_rank': 1.1}, '****': {'rank': 368, 'team': "****", 'mu': 110.2985, 'sigma': 0.6928, 'score': 108.22, 'games': 1943, 'win_rate': 0.4246, 'avg_rank': 0.9233}, '****': {'rank': 369, 'team': "****", 'mu': 110.6207, 'sigma': 0.8015, 'score': 108.2164, 'games': 65, 'win_rate': 0.3692, 'avg_rank': 1.1692}, '****': {'rank': 370, 'team': "****", 'mu': 113.0283, 'sigma': 1.6122, 'score': 108.1917, 'games': 12, 'win_rate': 0.4167, 'avg_rank': 0.8333}, '****': {'rank': 371, 'team': "****", 'mu': 110.2521, 'sigma': 0.6919, 'score': 108.1764, 'games': 1153, 'win_rate': 0.4189, 'avg_rank': 0.9115}, '****': {'rank': 372, 'team': "****", 'mu': 110.258, 'sigma': 0.6951, 'score': 108.1728, 'games': 2093, 'win_rate': 0.3865, 'avg_rank': 0.9632}, '****': {'rank': 373, 'team': "****", 'mu': 110.2409, 'sigma': 0.6939, 'score': 108.1594, 'games': 440, 'win_rate': 0.3727, 'avg_rank': 0.9545}, '****': {'rank': 374, 'team': "****", 'mu': 110.2425, 'sigma': 0.6995, 'score': 108.1439, 'games': 908, 'win_rate': 0.3568, 'avg_rank': 1.0694}, '****': {'rank': 375, 'team': "****", 'mu': 110.2277, 'sigma': 0.6947, 'score': 108.1435, 'games': 134, 'win_rate': 0.2687, 'avg_rank': 1.3955}, '****': {'rank': 376, 'team': "****", 'mu': 110.1371, 'sigma': 0.6867, 'score': 108.0771, 'games': 178, 'win_rate': 0.3034, 'avg_rank': 1.1966}, '****': {'rank': 377, 'team': "****", 'mu': 110.755, 'sigma': 0.8935, 'score': 108.0744, 'games': 48, 'win_rate': 0.5, 'avg_rank': 0.875}, '****': {'rank': 378, 'team': "****", 'mu': 110.1521, 'sigma': 0.693, 'score': 108.0732, 'games': 213, 'win_rate': 0.3521, 'avg_rank': 1.1502}, '****': {'rank': 379, 'team': "****", 'mu': 110.1426, 'sigma': 0.6923, 'score': 108.0658, 'games': 236, 'win_rate': 0.3347, 'avg_rank': 1.0847}, '****': {'rank': 380, 'team': "****", 'mu': 110.1272, 'sigma': 0.6894, 'score': 108.0591, 'games': 370, 'win_rate': 0.3595, 'avg_rank': 1.0838}, '****': {'rank': 381, 'team': "****", 'mu': 110.3541, 'sigma': 0.7689, 'score': 108.0473, 'games': 88, 'win_rate': 0.4091, 'avg_rank': 1.0341}, '****': {'rank': 382, 'team': "****", 'mu': 110.1468, 'sigma': 0.7029, 'score': 108.038, 'games': 1730, 'win_rate': 0.448, 'avg_rank': 0.9624}, '****': {'rank': 383, 'team': "****", 'mu': 110.1062, 'sigma': 0.6898, 'score': 108.0369, 'games': 221, 'win_rate': 0.3077, 'avg_rank': 1.0633}, '****': {'rank': 384, 'team': "****", 'mu': 112.8182, 'sigma': 1.5997, 'score': 108.0192, 'games': 13, 'win_rate': 0.4615, 'avg_rank': 0.8462}, '****': {'rank': 385, 'team': "****", 'mu': 110.0841, 'sigma': 0.6935, 'score': 108.0035, 'games': 1899, 'win_rate': 0.3481, 'avg_rank': 1.0326}, '****': {'rank': 386, 'team': "****", 'mu': 110.2231, 'sigma': 0.744, 'score': 107.991, 'games': 86, 'win_rate': 0.2442, 'avg_rank': 1.1395}, '****': {'rank': 387, 'team': "****", 'mu': 110.5785, 'sigma': 0.8632, 'score': 107.9888, 'games': 57, 'win_rate': 0.4561, 'avg_rank': 0.9123}, '****': {'rank': 388, 'team': "****", 'mu': 110.3056, 'sigma': 0.7763, 'score': 107.9768, 'games': 71, 'win_rate': 0.0704, 'avg_rank': 1.5915}, '****': {'rank': 389, 'team': "****", 'mu': 110.0992, 'sigma': 0.7102, 'score': 107.9686, 'games': 140, 'win_rate': 0.4, 'avg_rank': 1.0286}, '****': {'rank': 390, 'team': "****", 'mu': 110.2082, 'sigma': 0.7498, 'score': 107.9587, 'games': 86, 'win_rate': 0.2907, 'avg_rank': 1.2326}, '****': {'rank': 391, 'team': "****", 'mu': 109.9965, 'sigma': 0.6848, 'score': 107.9422, 'games': 292, 'win_rate': 0.3048, 'avg_rank': 1.0548}, '****': {'rank': 392, 'team': "****", 'mu': 110.1795, 'sigma': 0.7539, 'score': 107.9176, 'games': 80, 'win_rate': 0.3, 'avg_rank': 1.2125}, '****': {'rank': 393, 'team': "****", 'mu': 110.0037, 'sigma': 0.6956, 'score': 107.9169, 'games': 150, 'win_rate': 0.08, 'avg_rank': 1.6267}, '****': {'rank': 394, 'team': "****", 'mu': 109.9792, 'sigma': 0.6887, 'score': 107.9132, 'games': 448, 'win_rate': 0.3415, 'avg_rank': 1.0714}, '****': {'rank': 395, 'team': "****", 'mu': 109.9822, 'sigma': 0.7006, 'score': 107.8803, 'games': 248, 'win_rate': 0.3387, 'avg_rank': 1.125}, '****': {'rank': 396, 'team': "****", 'mu': 109.9589, 'sigma': 0.6946, 'score': 107.8751, 'games': 288, 'win_rate': 0.3403, 'avg_rank': 1.1146}, '****': {'rank': 397, 'team': "****", 'mu': 110.0916, 'sigma': 0.7401, 'score': 107.8713, 'games': 93, 'win_rate': 0.3763, 'avg_rank': 1.172}, '****': {'rank': 398, 'team': "****", 'mu': 109.9071, 'sigma': 0.6873, 'score': 107.8453, 'games': 1124, 'win_rate': 0.2286, 'avg_rank': 1.1468}, '****': {'rank': 399, 'team': "****", 'mu': 109.8803, 'sigma': 0.6833, 'score': 107.8303, 'games': 242, 'win_rate': 0.2727, 'avg_rank': 1.4008}, '****': {'rank': 400, 'team': "****", 'mu': 109.8945, 'sigma': 0.6895, 'score': 107.826, 'games': 492, 'win_rate': 0.3598, 'avg_rank': 1.0813}, '****': {'rank': 401, 'team': "****", 'mu': 110.2269, 'sigma': 0.8087, 'score': 107.8009, 'games': 64, 'win_rate': 0.25, 'avg_rank': 1.2344}, '****': {'rank': 402, 'team': "****", 'mu': 109.8655, 'sigma': 0.6916, 'score': 107.7907, 'games': 227, 'win_rate': 0.3172, 'avg_rank': 1.1366}, '****': {'rank': 403, 'team': "****", 'mu': 109.8806, 'sigma': 0.7021, 'score': 107.7742, 'games': 151, 'win_rate': 0.2185, 'avg_rank': 1.3775}, '****': {'rank': 404, 'team': "****", 'mu': 109.8727, 'sigma': 0.7079, 'score': 107.749, 'games': 172, 'win_rate': 0.4244, 'avg_rank': 1.1105}, '****': {'rank': 405, 'team': "****", 'mu': 110.1903, 'sigma': 0.8194, 'score': 107.7322, 'games': 67, 'win_rate': 0.4179, 'avg_rank': 1.0299}, '****': {'rank': 406, 'team': "****", 'mu': 109.807, 'sigma': 0.6921, 'score': 107.7306, 'games': 393, 'win_rate': 0.4453, 'avg_rank': 1.0051}, '****': {'rank': 407, 'team': "****", 'mu': 109.7587, 'sigma': 0.6853, 'score': 107.7027, 'games': 2129, 'win_rate': 0.2865, 'avg_rank': 1.0474}, '****': {'rank': 408, 'team': "****", 'mu': 109.7613, 'sigma': 0.6897, 'score': 107.6922, 'games': 416, 'win_rate': 0.3702, 'avg_rank': 1.0601}, '****': {'rank': 409, 'team': "****", 'mu': 109.7861, 'sigma': 0.6996, 'score': 107.6872, 'games': 770, 'win_rate': 0.4247, 'avg_rank': 0.9286}, '****': {'rank': 410, 'team': "****", 'mu': 109.7596, 'sigma': 0.6927, 'score': 107.6816, 'games': 912, 'win_rate': 0.3388, 'avg_rank': 1.1941}, '****': {'rank': 411, 'team': "****", 'mu': 110.3717, 'sigma': 0.8978, 'score': 107.6783, 'games': 53, 'win_rate': 0.434, 'avg_rank': 1.0566}, '****': {'rank': 412, 'team': "****", 'mu': 109.6466, 'sigma': 0.6812, 'score': 107.603, 'games': 1208, 'win_rate': 0.0629, 'avg_rank': 0.8874}, '****': {'rank': 413, 'team': "****", 'mu': 110.1874, 'sigma': 0.8679, 'score': 107.5837, 'games': 53, 'win_rate': 0.0943, 'avg_rank': 1.6415}, '****': {'rank': 414, 'team': "****", 'mu': 109.6788, 'sigma': 0.6994, 'score': 107.5805, 'games': 153, 'win_rate': 0.2614, 'avg_rank': 1.3791}, '****': {'rank': 415, 'team': "****", 'mu': 109.9803, 'sigma': 0.8056, 'score': 107.5635, 'games': 63, 'win_rate': 0.3651, 'avg_rank': 1.254}, '****': {'rank': 416, 'team': "****", 'mu': 109.6283, 'sigma': 0.6937, 'score': 107.5472, 'games': 245, 'win_rate': 0.3592, 'avg_rank': 1.1102}, '****': {'rank': 417, 'team': "****", 'mu': 109.6949, 'sigma': 0.7185, 'score': 107.5393, 'games': 110, 'win_rate': 0.2818, 'avg_rank': 1.2727}, '****': {'rank': 418, 'team': "****", 'mu': 111.5667, 'sigma': 1.3441, 'score': 107.5344, 'games': 19, 'win_rate': 0.4211, 'avg_rank': 0.9474}, '****': {'rank': 419, 'team': "****", 'mu': 109.6518, 'sigma': 0.7085, 'score': 107.5264, 'games': 126, 'win_rate': 0.2381, 'avg_rank': 1.2937}, '****': {'rank': 420, 'team': "****", 'mu': 109.6057, 'sigma': 0.6968, 'score': 107.5154, 'games': 139, 'win_rate': 0.2518, 'avg_rank': 1.2086}, '****': {'rank': 421, 'team': "****", 'mu': 109.9467, 'sigma': 0.83, 'score': 107.4568, 'games': 63, 'win_rate': 0.3651, 'avg_rank': 1.1587}, '****': {'rank': 422, 'team': "****", 'mu': 109.5241, 'sigma': 0.6987, 'score': 107.428, 'games': 696, 'win_rate': 0.4138, 'avg_rank': 1.102}, '****': {'rank': 423, 'team': "****", 'mu': 109.4766, 'sigma': 0.6856, 'score': 107.4199, 'games': 271, 'win_rate': 0.2546, 'avg_rank': 1.4354}, '****': {'rank': 424, 'team': "****", 'mu': 109.5015, 'sigma': 0.6971, 'score': 107.4101, 'games': 280, 'win_rate': 0.3536, 'avg_rank': 1.1821}, '****': {'rank': 425, 'team': "****", 'mu': 109.4592, 'sigma': 0.6863, 'score': 107.4004, 'games': 636, 'win_rate': 0.0849, 'avg_rank': 1.1667}, '****': {'rank': 426, 'team': "****", 'mu': 109.4714, 'sigma': 0.6904, 'score': 107.4003, 'games': 294, 'win_rate': 0.2823, 'avg_rank': 1.2381}, '****': {'rank': 427, 'team': "****", 'mu': 109.8618, 'sigma': 0.8205, 'score': 107.4002, 'games': 73, 'win_rate': 0.0137, 'avg_rank': 1.411}, '****': {'rank': 428, 'team': "****", 'mu': 109.4596, 'sigma': 0.6871, 'score': 107.3982, 'games': 338, 'win_rate': 0.0592, 'avg_rank': 1.6036}, '****': {'rank': 429, 'team': "****", 'mu': 109.4621, 'sigma': 0.6996, 'score': 107.3632, 'games': 2158, 'win_rate': 0.3369, 'avg_rank': 1.2113}, '****': {'rank': 430, 'team': "****", 'mu': 109.4339, 'sigma': 0.6903, 'score': 107.363, 'games': 1069, 'win_rate': 0.3704, 'avg_rank': 1.0131}, '****': {'rank': 431, 'team': "****", 'mu': 109.4165, 'sigma': 0.6908, 'score': 107.3442, 'games': 1773, 'win_rate': 0.3457, 'avg_rank': 0.9836}, '****': {'rank': 432, 'team': "****", 'mu': 109.4169, 'sigma': 0.6933, 'score': 107.3372, 'games': 303, 'win_rate': 0.2937, 'avg_rank': 1.2904}, '****': {'rank': 433, 'team': "****", 'mu': 109.5158, 'sigma': 0.7313, 'score': 107.3218, 'games': 94, 'win_rate': 0.2447, 'avg_rank': 1.5319}, '****': {'rank': 434, 'team': "****", 'mu': 109.4236, 'sigma': 0.7008, 'score': 107.3213, 'games': 154, 'win_rate': 0.3896, 'avg_rank': 1.1688}, '****': {'rank': 435, 'team': "****", 'mu': 109.3786, 'sigma': 0.692, 'score': 107.3025, 'games': 218, 'win_rate': 0.1239, 'avg_rank': 1.8211}, '****': {'rank': 436, 'team': "****", 'mu': 109.3483, 'sigma': 0.6834, 'score': 107.2982, 'games': 444, 'win_rate': 0.2793, 'avg_rank': 1.1959}, '****': {'rank': 437, 'team': "****", 'mu': 109.5281, 'sigma': 0.7442, 'score': 107.2955, 'games': 88, 'win_rate': 0.25, 'avg_rank': 1.3068}, '****': {'rank': 438, 'team': "****", 'mu': 109.3416, 'sigma': 0.6824, 'score': 107.2943, 'games': 1288, 'win_rate': 0.111, 'avg_rank': 0.8261}, '****': {'rank': 439, 'team': "****", 'mu': 109.3183, 'sigma': 0.6834, 'score': 107.268, 'games': 1328, 'win_rate': 0.2816, 'avg_rank': 1.1544}, '****': {'rank': 440, 'team': "****", 'mu': 110.7281, 'sigma': 1.1581, 'score': 107.2537, 'games': 28, 'win_rate': 0.4643, 'avg_rank': 0.6786}, '****': {'rank': 441, 'team': "****", 'mu': 109.272, 'sigma': 0.6889, 'score': 107.2052, 'games': 1211, 'win_rate': 0.2791, 'avg_rank': 1.1214}, '****': {'rank': 442, 'team': "****", 'mu': 109.2996, 'sigma': 0.6985, 'score': 107.2042, 'games': 126, 'win_rate': 0.246, 'avg_rank': 1.5}, '****': {'rank': 443, 'team': "****", 'mu': 109.2667, 'sigma': 0.6915, 'score': 107.1923, 'games': 1230, 'win_rate': 0.3, 'avg_rank': 1.0049}, '****': {'rank': 444, 'team': "****", 'mu': 109.3358, 'sigma': 0.7309, 'score': 107.1431, 'games': 91, 'win_rate': 0.1209, 'avg_rank': 1.4835}, '****': {'rank': 445, 'team': "****", 'mu': 109.5705, 'sigma': 0.8102, 'score': 107.1399, 'games': 68, 'win_rate': 0.3088, 'avg_rank': 1.1471}, '****': {'rank': 446, 'team': "****", 'mu': 109.1959, 'sigma': 0.6888, 'score': 107.1294, 'games': 1656, 'win_rate': 0.3152, 'avg_rank': 1.1872}, '****': {'rank': 447, 'team': "****", 'mu': 109.1976, 'sigma': 0.6915, 'score': 107.123, 'games': 534, 'win_rate': 0.3876, 'avg_rank': 1.0262}, '****': {'rank': 448, 'team': "****", 'mu': 109.1684, 'sigma': 0.6893, 'score': 107.1006, 'games': 1875, 'win_rate': 0.3131, 'avg_rank': 1.1253}, '****': {'rank': 449, 'team': "****", 'mu': 109.1802, 'sigma': 0.6963, 'score': 107.0913, 'games': 152, 'win_rate': 0.1842, 'avg_rank': 1.5789}, '****': {'rank': 450, 'team': "****", 'mu': 109.1513, 'sigma': 0.6913, 'score': 107.0773, 'games': 1137, 'win_rate': 0.4028, 'avg_rank': 0.9938}, '****': {'rank': 451, 'team': "****", 'mu': 109.1443, 'sigma': 0.689, 'score': 107.0772, 'games': 2126, 'win_rate': 0.3523, 'avg_rank': 1.1279}, '****': {'rank': 452, 'team': "****", 'mu': 109.1851, 'sigma': 0.7101, 'score': 107.0548, 'games': 108, 'win_rate': 0.1667, 'avg_rank': 1.5926}, '****': {'rank': 453, 'team': "****", 'mu': 109.119, 'sigma': 0.6901, 'score': 107.0488, 'games': 1919, 'win_rate': 0.4325, 'avg_rank': 0.9192}, '****': {'rank': 454, 'team': "****", 'mu': 109.1505, 'sigma': 0.7107, 'score': 107.0183, 'games': 131, 'win_rate': 0.3511, 'avg_rank': 1.1985}, '****': {'rank': 455, 'team': "****", 'mu': 109.0734, 'sigma': 0.6948, 'score': 106.9891, 'games': 145, 'win_rate': 0.1034, 'avg_rank': 1.669}, '****': {'rank': 456, 'team': "****", 'mu': 109.0518, 'sigma': 0.6902, 'score': 106.9811, 'games': 1840, 'win_rate': 0.3217, 'avg_rank': 1.0848}, '****': {'rank': 457, 'team': "****", 'mu': 109.0153, 'sigma': 0.6878, 'score': 106.9518, 'games': 319, 'win_rate': 0.373, 'avg_rank': 1.1787}, '****': {'rank': 458, 'team': "****", 'mu': 109.9398, 'sigma': 1.0036, 'score': 106.929, 'games': 34, 'win_rate': 0.2941, 'avg_rank': 1.1176}, '****': {'rank': 459, 'team': "****", 'mu': 109.1522, 'sigma': 0.7447, 'score': 106.9183, 'games': 86, 'win_rate': 0.1395, 'avg_rank': 1.6395}, '****': {'rank': 460, 'team': "****", 'mu': 108.9694, 'sigma': 0.6878, 'score': 106.9058, 'games': 471, 'win_rate': 0.2739, 'avg_rank': 1.1805}, '****': {'rank': 461, 'team': "****", 'mu': 108.9779, 'sigma': 0.6921, 'score': 106.9015, 'games': 1060, 'win_rate': 0.2811, 'avg_rank': 1.1208}, '****': {'rank': 462, 'team': "****", 'mu': 108.9624, 'sigma': 0.689, 'score': 106.8954, 'games': 1775, 'win_rate': 0.2879, 'avg_rank': 1.147}, '****': {'rank': 463, 'team': "****", 'mu': 108.9564, 'sigma': 0.6871, 'score': 106.8951, 'games': 273, 'win_rate': 0.2711, 'avg_rank': 1.4103}, '****': {'rank': 464, 'team': "****", 'mu': 109.5621, 'sigma': 0.8899, 'score': 106.8924, 'games': 48, 'win_rate': 0.2083, 'avg_rank': 1.5833}, '****': {'rank': 465, 'team': "****", 'mu': 110.1666, 'sigma': 1.092, 'score': 106.8907, 'games': 34, 'win_rate': 0.3235, 'avg_rank': 1.2059}, '****': {'rank': 466, 'team': "****", 'mu': 108.9378, 'sigma': 0.6864, 'score': 106.8787, 'games': 385, 'win_rate': 0.2935, 'avg_rank': 1.3636}, '****': {'rank': 467, 'team': "****", 'mu': 109.0994, 'sigma': 0.7425, 'score': 106.872, 'games': 89, 'win_rate': 0.1124, 'avg_rank': 1.6067}, '****': {'rank': 468, 'team': "****", 'mu': 108.9307, 'sigma': 0.6888, 'score': 106.8644, 'games': 721, 'win_rate': 0.3107, 'avg_rank': 1.1151}, '****': {'rank': 469, 'team': "****", 'mu': 109.234, 'sigma': 0.7916, 'score': 106.8591, 'games': 78, 'win_rate': 0.1282, 'avg_rank': 1.7051}, '****': {'rank': 470, 'team': "****", 'mu': 108.9487, 'sigma': 0.6982, 'score': 106.8542, 'games': 167, 'win_rate': 0.3713, 'avg_rank': 1.0898}, '****': {'rank': 471, 'team': "****", 'mu': 108.8873, 'sigma': 0.6906, 'score': 106.8154, 'games': 1868, 'win_rate': 0.387, 'avg_rank': 1.1119}, '****': {'rank': 472, 'team': "****", 'mu': 108.8827, 'sigma': 0.6913, 'score': 106.809, 'games': 2865, 'win_rate': 0.3319, 'avg_rank': 1.185}, '****': {'rank': 473, 'team': "****", 'mu': 109.1964, 'sigma': 0.7968, 'score': 106.806, 'games': 70, 'win_rate': 0.3143, 'avg_rank': 1.2143}, '****': {'rank': 474, 'team': "****", 'mu': 108.8834, 'sigma': 0.6937, 'score': 106.8022, 'games': 143, 'win_rate': 0.1608, 'avg_rank': 1.4336}, '****': {'rank': 475, 'team': "****", 'mu': 108.8683, 'sigma': 0.6907, 'score': 106.7961, 'games': 1074, 'win_rate': 0.3557, 'avg_rank': 1.1927}, '****': {'rank': 476, 'team': "****", 'mu': 108.8241, 'sigma': 0.6882, 'score': 106.7595, 'games': 2220, 'win_rate': 0.3342, 'avg_rank': 1.1631}, '****': {'rank': 477, 'team': "****", 'mu': 108.8253, 'sigma': 0.6888, 'score': 106.759, 'games': 496, 'win_rate': 0.1008, 'avg_rank': 1.498}, '****': {'rank': 478, 'team': "****", 'mu': 108.8151, 'sigma': 0.6878, 'score': 106.7518, 'games': 161, 'win_rate': 0.1925, 'avg_rank': 1.1925}, '****': {'rank': 479, 'team': "****", 'mu': 108.8059, 'sigma': 0.6866, 'score': 106.7463, 'games': 737, 'win_rate': 0.3555, 'avg_rank': 1.0909}, '****': {'rank': 480, 'team': "****", 'mu': 108.8238, 'sigma': 0.693, 'score': 106.7447, 'games': 1690, 'win_rate': 0.4266, 'avg_rank': 0.9799}, '****': {'rank': 481, 'team': "****", 'mu': 110.3272, 'sigma': 1.1947, 'score': 106.743, 'games': 25, 'win_rate': 0.4, 'avg_rank': 0.96}, '****': {'rank': 482, 'team': "****", 'mu': 108.8094, 'sigma': 0.6892, 'score': 106.7418, 'games': 1127, 'win_rate': 0.2964, 'avg_rank': 1.0568}, '****': {'rank': 483, 'team': "****", 'mu': 108.7939, 'sigma': 0.6858, 'score': 106.7365, 'games': 1599, 'win_rate': 0.1726, 'avg_rank': 1.3827}, '****': {'rank': 484, 'team': "****", 'mu': 108.7722, 'sigma': 0.6866, 'score': 106.7125, 'games': 765, 'win_rate': 0.3216, 'avg_rank': 1.0248}, '****': {'rank': 485, 'team': "****", 'mu': 108.7637, 'sigma': 0.6921, 'score': 106.6873, 'games': 547, 'win_rate': 0.4278, 'avg_rank': 1.0841}, '****': {'rank': 486, 'team': "****", 'mu': 108.7551, 'sigma': 0.6933, 'score': 106.6751, 'games': 123, 'win_rate': 0.0325, 'avg_rank': 1.748}, '****': {'rank': 487, 'team': "****", 'mu': 108.6948, 'sigma': 0.6748, 'score': 106.6705, 'games': 811, 'win_rate': 0.1603, 'avg_rank': 0.561}, '****': {'rank': 488, 'team': "****", 'mu': 108.8252, 'sigma': 0.7217, 'score': 106.6601, 'games': 96, 'win_rate': 0.1771, 'avg_rank': 1.6146}, '****': {'rank': 489, 'team': "****", 'mu': 108.6828, 'sigma': 0.6774, 'score': 106.6505, 'games': 270, 'win_rate': 0.0778, 'avg_rank': 1.5889}, '****': {'rank': 490, 'team': "****", 'mu': 108.7001, 'sigma': 0.6839, 'score': 106.6483, 'games': 330, 'win_rate': 0.2939, 'avg_rank': 1.3364}, '****': {'rank': 491, 'team': "****", 'mu': 109.0147, 'sigma': 0.7935, 'score': 106.6343, 'games': 72, 'win_rate': 0.375, 'avg_rank': 1.1944}, '****': {'rank': 492, 'team': "****", 'mu': 108.6878, 'sigma': 0.6857, 'score': 106.6307, 'games': 874, 'win_rate': 0.3661, 'avg_rank': 1.1064}, '****': {'rank': 493, 'team': "****", 'mu': 108.7809, 'sigma': 0.7171, 'score': 106.6295, 'games': 108, 'win_rate': 0.0741, 'avg_rank': 1.7407}, '****': {'rank': 494, 'team': "****", 'mu': 108.6812, 'sigma': 0.6874, 'score': 106.6191, 'games': 310, 'win_rate': 0.3065, 'avg_rank': 1.2871}, '****': {'rank': 495, 'team': "****", 'mu': 108.6754, 'sigma': 0.6912, 'score': 106.6018, 'games': 488, 'win_rate': 0.334, 'avg_rank': 1.1742}, '****': {'rank': 496, 'team': "****", 'mu': 108.6441, 'sigma': 0.6834, 'score': 106.5938, 'games': 456, 'win_rate': 0.2588, 'avg_rank': 1.1996}, '****': {'rank': 497, 'team': "****", 'mu': 108.658, 'sigma': 0.6883, 'score': 106.5931, 'games': 1265, 'win_rate': 0.1992, 'avg_rank': 0.7565}, '****': {'rank': 498, 'team': "****", 'mu': 108.6746, 'sigma': 0.6939, 'score': 106.5929, 'games': 498, 'win_rate': 0.3293, 'avg_rank': 1.2088}, '****': {'rank': 499, 'team': "****", 'mu': 108.6492, 'sigma': 0.6857, 'score': 106.5921, 'games': 1177, 'win_rate': 0.2345, 'avg_rank': 1.2472}, '****': {'rank': 500, 'team': "****", 'mu': 108.6295, 'sigma': 0.6815, 'score': 106.5849, 'games': 439, 'win_rate': 0.1549, 'avg_rank': 1.3485}, '****': {'rank': 501, 'team': "****", 'mu': 108.6454, 'sigma': 0.6878, 'score': 106.582, 'games': 212, 'win_rate': 0.2736, 'avg_rank': 1.316}, '****': {'rank': 502, 'team': "****", 'mu': 108.6139, 'sigma': 0.6833, 'score': 106.564, 'games': 1676, 'win_rate': 0.2936, 'avg_rank': 1.1116}, '****': {'rank': 503, 'team': "****", 'mu': 108.6193, 'sigma': 0.6871, 'score': 106.5578, 'games': 490, 'win_rate': 0.3122, 'avg_rank': 1.2837}, '****': {'rank': 504, 'team': "****", 'mu': 109.157, 'sigma': 0.8697, 'score': 106.5478, 'games': 50, 'win_rate': 0.2, 'avg_rank': 1.14}, '****': {'rank': 505, 'team': "****", 'mu': 108.5873, 'sigma': 0.6811, 'score': 106.544, 'games': 701, 'win_rate': 0.1897, 'avg_rank': 1.2311}, '****': {'rank': 506, 'team': "****", 'mu': 108.5923, 'sigma': 0.691, 'score': 106.5192, 'games': 1527, 'win_rate': 0.2823, 'avg_rank': 1.1113}, '****': {'rank': 507, 'team': "****", 'mu': 108.5674, 'sigma': 0.6859, 'score': 106.5098, 'games': 721, 'win_rate': 0.2497, 'avg_rank': 1.2302}, '****': {'rank': 508, 'team': "****", 'mu': 108.5376, 'sigma': 0.6873, 'score': 106.4758, 'games': 2447, 'win_rate': 0.3907, 'avg_rank': 1.1488}, '****': {'rank': 509, 'team': "****", 'mu': 108.5127, 'sigma': 0.6816, 'score': 106.4679, 'games': 1106, 'win_rate': 0.2722, 'avg_rank': 1.2514}, '****': {'rank': 510, 'team': "****", 'mu': 108.5301, 'sigma': 0.6887, 'score': 106.464, 'games': 173, 'win_rate': 0.2486, 'avg_rank': 1.3526}, '****': {'rank': 511, 'team': "****", 'mu': 108.5333, 'sigma': 0.6916, 'score': 106.4584, 'games': 1859, 'win_rate': 0.3642, 'avg_rank': 1.1092}, '****': {'rank': 512, 'team': "****", 'mu': 109.2843, 'sigma': 0.9434, 'score': 106.454, 'games': 40, 'win_rate': 0.125, 'avg_rank': 1.3}, '****': {'rank': 513, 'team': "****", 'mu': 108.6791, 'sigma': 0.7487, 'score': 106.433, 'games': 106, 'win_rate': 0.066, 'avg_rank': 1.9906}, '****': {'rank': 514, 'team': "****", 'mu': 108.4588, 'sigma': 0.6816, 'score': 106.4141, 'games': 2562, 'win_rate': 0.3064, 'avg_rank': 1.123}, '****': {'rank': 515, 'team': "****", 'mu': 108.6817, 'sigma': 0.7633, 'score': 106.3918, 'games': 78, 'win_rate': 0.1795, 'avg_rank': 1.5128}, '****': {'rank': 516, 'team': "****", 'mu': 109.0916, 'sigma': 0.9022, 'score': 106.385, 'games': 48, 'win_rate': 0.3542, 'avg_rank': 1.1458}, '****': {'rank': 517, 'team': "****", 'mu': 108.6303, 'sigma': 0.7492, 'score': 106.3827, 'games': 89, 'win_rate': 0.1348, 'avg_rank': 1.4607}, '****': {'rank': 518, 'team': "****", 'mu': 108.4398, 'sigma': 0.6877, 'score': 106.3769, 'games': 2005, 'win_rate': 0.3277, 'avg_rank': 1.1486}, '****': {'rank': 519, 'team': "****", 'mu': 108.4326, 'sigma': 0.6858, 'score': 106.3751, 'games': 821, 'win_rate': 0.3216, 'avg_rank': 1.1413}, '****': {'rank': 520, 'team': "****", 'mu': 108.6228, 'sigma': 0.7581, 'score': 106.3484, 'games': 82, 'win_rate': 0.1707, 'avg_rank': 1.5854}, '****': {'rank': 521, 'team': "****", 'mu': 108.4199, 'sigma': 0.6919, 'score': 106.3442, 'games': 2243, 'win_rate': 0.3553, 'avg_rank': 1.1195}, '****': {'rank': 522, 'team': "****", 'mu': 108.3912, 'sigma': 0.6967, 'score': 106.301, 'games': 205, 'win_rate': 0.2927, 'avg_rank': 1.2293}, '****': {'rank': 523, 'team': "****", 'mu': 108.3773, 'sigma': 0.6958, 'score': 106.2898, 'games': 127, 'win_rate': 0.2283, 'avg_rank': 1.4016}, '****': {'rank': 524, 'team': "****", 'mu': 108.3401, 'sigma': 0.7003, 'score': 106.2392, 'games': 195, 'win_rate': 0.2923, 'avg_rank': 1.3077}, '****': {'rank': 525, 'team': "****", 'mu': 108.5919, 'sigma': 0.7898, 'score': 106.2226, 'games': 66, 'win_rate': 0.1818, 'avg_rank': 1.3939}, '****': {'rank': 526, 'team': "****", 'mu': 108.2665, 'sigma': 0.6842, 'score': 106.2138, 'games': 1111, 'win_rate': 0.2349, 'avg_rank': 1.5014}, '****': {'rank': 527, 'team': "****", 'mu': 108.2939, 'sigma': 0.6974, 'score': 106.2015, 'games': 166, 'win_rate': 0.2108, 'avg_rank': 1.3976}, '****': {'rank': 528, 'team': "****", 'mu': 108.2665, 'sigma': 0.6931, 'score': 106.1873, 'games': 284, 'win_rate': 0.1655, 'avg_rank': 1.7218}, '****': {'rank': 529, 'team': "****", 'mu': 108.2634, 'sigma': 0.6927, 'score': 106.1853, 'games': 2367, 'win_rate': 0.3185, 'avg_rank': 1.0832}, '****': {'rank': 530, 'team': "****", 'mu': 108.2544, 'sigma': 0.693, 'score': 106.1753, 'games': 158, 'win_rate': 0.2722, 'avg_rank': 1.4494}, '****': {'rank': 531, 'team': "****", 'mu': 108.4919, 'sigma': 0.7736, 'score': 106.1711, 'games': 77, 'win_rate': 0.1558, 'avg_rank': 1.6494}, '****': {'rank': 532, 'team': "****", 'mu': 111.0466, 'sigma': 1.6311, 'score': 106.1533, 'games': 12, 'win_rate': 0.5, 'avg_rank': 0.75}, '****': {'rank': 533, 'team': "****", 'mu': 108.2055, 'sigma': 0.6857, 'score': 106.1484, 'games': 1893, 'win_rate': 0.3629, 'avg_rank': 1.1241}, '****': {'rank': 534, 'team': "****", 'mu': 108.4017, 'sigma': 0.7602, 'score': 106.121, 'games': 76, 'win_rate': 0.0526, 'avg_rank': 1.7895}, '****': {'rank': 535, 'team': "****", 'mu': 108.1717, 'sigma': 0.6865, 'score': 106.1121, 'games': 2150, 'win_rate': 0.2219, 'avg_rank': 1.2572}, '****': {'rank': 536, 'team': "****", 'mu': 108.1496, 'sigma': 0.6856, 'score': 106.0929, 'games': 503, 'win_rate': 0.3101, 'avg_rank': 1.2008}, '****': {'rank': 537, 'team': "****", 'mu': 108.1742, 'sigma': 0.715, 'score': 106.0293, 'games': 102, 'win_rate': 0.0686, 'avg_rank': 1.6569}, '****': {'rank': 538, 'team': "****", 'mu': 108.0269, 'sigma': 0.6848, 'score': 105.9724, 'games': 1115, 'win_rate': 0.3166, 'avg_rank': 1.1632}, '****': {'rank': 539, 'team': "****", 'mu': 108.0082, 'sigma': 0.6871, 'score': 105.9469, 'games': 377, 'win_rate': 0.3316, 'avg_rank': 1.244}, '****': {'rank': 540, 'team': "****", 'mu': 108.0173, 'sigma': 0.6969, 'score': 105.9265, 'games': 140, 'win_rate': 0.05, 'avg_rank': 1.8143}, '****': {'rank': 541, 'team': "****", 'mu': 107.9667, 'sigma': 0.6897, 'score': 105.8976, 'games': 1151, 'win_rate': 0.3267, 'avg_rank': 1.2433}, '****': {'rank': 542, 'team': "****", 'mu': 107.9383, 'sigma': 0.6861, 'score': 105.8798, 'games': 549, 'win_rate': 0.3097, 'avg_rank': 1.286}, '****': {'rank': 543, 'team': "****", 'mu': 109.343, 'sigma': 1.16, 'score': 105.8631, 'games': 24, 'win_rate': 0.0417, 'avg_rank': 1.7917}, '****': {'rank': 544, 'team': "****", 'mu': 107.9041, 'sigma': 0.6821, 'score': 105.858, 'games': 198, 'win_rate': 0.096, 'avg_rank': 1.4747}, '****': {'rank': 545, 'team': "****", 'mu': 107.8834, 'sigma': 0.6843, 'score': 105.8307, 'games': 692, 'win_rate': 0.2818, 'avg_rank': 1.1821}, '****': {'rank': 546, 'team': "****", 'mu': 108.0837, 'sigma': 0.7524, 'score': 105.8267, 'games': 86, 'win_rate': 0.1744, 'avg_rank': 1.6163}, '****': {'rank': 547, 'team': "****", 'mu': 107.883, 'sigma': 0.6884, 'score': 105.8178, 'games': 434, 'win_rate': 0.2512, 'avg_rank': 1.3456}, '****': {'rank': 548, 'team': "****", 'mu': 107.868, 'sigma': 0.6841, 'score': 105.8159, 'games': 2279, 'win_rate': 0.3392, 'avg_rank': 1.2602}, '****': {'rank': 549, 'team': "****", 'mu': 107.9576, 'sigma': 0.7147, 'score': 105.8135, 'games': 109, 'win_rate': 0.1835, 'avg_rank': 1.3486}, '****': {'rank': 550, 'team': "****", 'mu': 107.8648, 'sigma': 0.6886, 'score': 105.7989, 'games': 199, 'win_rate': 0.0553, 'avg_rank': 1.6834}, '****': {'rank': 551, 'team': "****", 'mu': 108.8497, 'sigma': 1.0185, 'score': 105.7943, 'games': 34, 'win_rate': 0.1765, 'avg_rank': 1.5588}, '****': {'rank': 552, 'team': "****", 'mu': 107.8423, 'sigma': 0.6896, 'score': 105.7736, 'games': 974, 'win_rate': 0.3214, 'avg_rank': 1.194}, '****': {'rank': 553, 'team': "****", 'mu': 107.8316, 'sigma': 0.687, 'score': 105.7706, 'games': 599, 'win_rate': 0.2588, 'avg_rank': 1.2321}, '****': {'rank': 554, 'team': "****", 'mu': 107.8856, 'sigma': 0.7055, 'score': 105.7691, 'games': 122, 'win_rate': 0.1393, 'avg_rank': 1.7705}, '****': {'rank': 555, 'team': "****", 'mu': 107.8103, 'sigma': 0.6842, 'score': 105.7578, 'games': 2181, 'win_rate': 0.3338, 'avg_rank': 1.199}, '****': {'rank': 556, 'team': "****", 'mu': 108.0462, 'sigma': 0.77, 'score': 105.7362, 'games': 77, 'win_rate': 0.1558, 'avg_rank': 1.5974}, '****': {'rank': 557, 'team': "****", 'mu': 107.7799, 'sigma': 0.6859, 'score': 105.7221, 'games': 538, 'win_rate': 0.3123, 'avg_rank': 1.2955}, '****': {'rank': 558, 'team': "****", 'mu': 107.8031, 'sigma': 0.6941, 'score': 105.7208, 'games': 160, 'win_rate': 0.3, 'avg_rank': 1.2937}, '****': {'rank': 559, 'team': "****", 'mu': 107.7782, 'sigma': 0.6909, 'score': 105.7055, 'games': 2212, 'win_rate': 0.3458, 'avg_rank': 0.7676}, '****': {'rank': 560, 'team': "****", 'mu': 107.7217, 'sigma': 0.6803, 'score': 105.6808, 'games': 509, 'win_rate': 0.0806, 'avg_rank': 1.5992}, '****': {'rank': 561, 'team': "****", 'mu': 107.7449, 'sigma': 0.6902, 'score': 105.6744, 'games': 1443, 'win_rate': 0.2931, 'avg_rank': 0.7852}, '****': {'rank': 562, 'team': "****", 'mu': 107.8344, 'sigma': 0.7382, 'score': 105.6196, 'games': 111, 'win_rate': 0.0991, 'avg_rank': 1.9189}, '****': {'rank': 563, 'team': "****", 'mu': 107.7446, 'sigma': 0.7118, 'score': 105.6091, 'games': 108, 'win_rate': 0.037, 'avg_rank': 1.8426}, '****': {'rank': 564, 'team': "****", 'mu': 107.6475, 'sigma': 0.6829, 'score': 105.5987, 'games': 672, 'win_rate': 0.1637, 'avg_rank': 1.3929}, '****': {'rank': 565, 'team': "****", 'mu': 107.6372, 'sigma': 0.6809, 'score': 105.5944, 'games': 893, 'win_rate': 0.2195, 'avg_rank': 1.3135}, '****': {'rank': 566, 'team': "****", 'mu': 109.8226, 'sigma': 1.4211, 'score': 105.5593, 'games': 15, 'win_rate': 0.1333, 'avg_rank': 1.0667}, '****': {'rank': 567, 'team': "****", 'mu': 107.6124, 'sigma': 0.6984, 'score': 105.5171, 'games': 139, 'win_rate': 0.0288, 'avg_rank': 1.777}, '****': {'rank': 568, 'team': "****", 'mu': 107.508, 'sigma': 0.6788, 'score': 105.4717, 'games': 1838, 'win_rate': 0.161, 'avg_rank': 1.3814}, '****': {'rank': 569, 'team': "****", 'mu': 107.5055, 'sigma': 0.6823, 'score': 105.4586, 'games': 836, 'win_rate': 0.2871, 'avg_rank': 1.2919}, '****': {'rank': 570, 'team': "****", 'mu': 107.6334, 'sigma': 0.7284, 'score': 105.4482, 'games': 110, 'win_rate': 0.1818, 'avg_rank': 1.6909}, '****': {'rank': 571, 'team': "****", 'mu': 107.4668, 'sigma': 0.6925, 'score': 105.3892, 'games': 598, 'win_rate': 0.2124, 'avg_rank': 1.6589}, '****': {'rank': 572, 'team': "****", 'mu': 107.6912, 'sigma': 0.7674, 'score': 105.389, 'games': 79, 'win_rate': 0.0253, 'avg_rank': 1.9494}, '****': {'rank': 573, 'team': "****", 'mu': 107.4257, 'sigma': 0.6851, 'score': 105.3704, 'games': 876, 'win_rate': 0.3082, 'avg_rank': 1.3128}, '****': {'rank': 574, 'team': "****", 'mu': 109.5947, 'sigma': 1.4087, 'score': 105.3686, 'games': 15, 'win_rate': 0.2667, 'avg_rank': 1.2667}, '****': {'rank': 575, 'team': "****", 'mu': 107.4322, 'sigma': 0.6886, 'score': 105.3663, 'games': 193, 'win_rate': 0.057, 'avg_rank': 1.8912}, '****': {'rank': 576, 'team': "****", 'mu': 107.4535, 'sigma': 0.7055, 'score': 105.3369, 'games': 215, 'win_rate': 0.0372, 'avg_rank': 2.0465}, '****': {'rank': 577, 'team': "****", 'mu': 107.5575, 'sigma': 0.7441, 'score': 105.3252, 'games': 87, 'win_rate': 0.0575, 'avg_rank': 1.7701}, '****': {'rank': 578, 'team': "****", 'mu': 107.3786, 'sigma': 0.6856, 'score': 105.3218, 'games': 561, 'win_rate': 0.3012, 'avg_rank': 1.3565}, '****': {'rank': 579, 'team': "****", 'mu': 109.9859, 'sigma': 1.5605, 'score': 105.3045, 'games': 14, 'win_rate': 0.2143, 'avg_rank': 1.5714}, '****': {'rank': 580, 'team': "****", 'mu': 107.3847, 'sigma': 0.6939, 'score': 105.3029, 'games': 167, 'win_rate': 0.1677, 'avg_rank': 1.5269}, '****': {'rank': 581, 'team': "****", 'mu': 107.3328, 'sigma': 0.6842, 'score': 105.2802, 'games': 283, 'win_rate': 0.2226, 'avg_rank': 1.5901}, '****': {'rank': 582, 'team': "****", 'mu': 107.3389, 'sigma': 0.6874, 'score': 105.2768, 'games': 613, 'win_rate': 0.31, 'avg_rank': 1.3426}, '****': {'rank': 583, 'team': "****", 'mu': 107.3306, 'sigma': 0.6857, 'score': 105.2735, 'games': 588, 'win_rate': 0.2976, 'avg_rank': 1.398}, '****': {'rank': 584, 'team': "****", 'mu': 107.3226, 'sigma': 0.6942, 'score': 105.24, 'games': 216, 'win_rate': 0.2824, 'avg_rank': 1.4352}, '****': {'rank': 585, 'team': "****", 'mu': 107.2521, 'sigma': 0.6926, 'score': 105.1742, 'games': 176, 'win_rate': 0.2045, 'avg_rank': 1.5966}, '****': {'rank': 586, 'team': "****", 'mu': 107.1894, 'sigma': 0.6847, 'score': 105.1352, 'games': 2088, 'win_rate': 0.2768, 'avg_rank': 1.3142}, '****': {'rank': 587, 'team': "****", 'mu': 107.3013, 'sigma': 0.7225, 'score': 105.1338, 'games': 112, 'win_rate': 0.1071, 'avg_rank': 1.6786}, '****': {'rank': 588, 'team': "****", 'mu': 107.1837, 'sigma': 0.6857, 'score': 105.1264, 'games': 561, 'win_rate': 0.2585, 'avg_rank': 1.3886}, '****': {'rank': 589, 'team': "****", 'mu': 107.1755, 'sigma': 0.6863, 'score': 105.1168, 'games': 951, 'win_rate': 0.1693, 'avg_rank': 1.3796}, '****': {'rank': 590, 'team': "****", 'mu': 107.1722, 'sigma': 0.6867, 'score': 105.1121, 'games': 531, 'win_rate': 0.2712, 'avg_rank': 1.3879}, '****': {'rank': 591, 'team': "****", 'mu': 108.974, 'sigma': 1.2873, 'score': 105.1121, 'games': 24, 'win_rate': 0.2917, 'avg_rank': 1.5417}, '****': {'rank': 592, 'team': "****", 'mu': 107.2289, 'sigma': 0.7104, 'score': 105.0976, 'games': 143, 'win_rate': 0.2238, 'avg_rank': 1.5594}, '****': {'rank': 593, 'team': "****", 'mu': 107.1278, 'sigma': 0.681, 'score': 105.0847, 'games': 352, 'win_rate': 0.1392, 'avg_rank': 1.4943}, '****': {'rank': 594, 'team': "****", 'mu': 107.1095, 'sigma': 0.6817, 'score': 105.0642, 'games': 1376, 'win_rate': 0.2231, 'avg_rank': 1.3583}, '****': {'rank': 595, 'team': "****", 'mu': 109.0715, 'sigma': 1.3407, 'score': 105.0495, 'games': 18, 'win_rate': 0.2222, 'avg_rank': 1.6111}, '****': {'rank': 596, 'team': "****", 'mu': 108.225, 'sigma': 1.0685, 'score': 105.0196, 'games': 34, 'win_rate': 0.2059, 'avg_rank': 1.6471}, '****': {'rank': 597, 'team': "****", 'mu': 108.9322, 'sigma': 1.3136, 'score': 104.9914, 'games': 21, 'win_rate': 0.3333, 'avg_rank': 1.2857}, '****': {'rank': 598, 'team': "****", 'mu': 106.9949, 'sigma': 0.6778, 'score': 104.9615, 'games': 2630, 'win_rate': 0.1563, 'avg_rank': 1.3297}, '****': {'rank': 599, 'team': "****", 'mu': 107.0086, 'sigma': 0.6878, 'score': 104.9452, 'games': 789, 'win_rate': 0.2307, 'avg_rank': 1.4689}, '****': {'rank': 600, 'team': "****", 'mu': 107.0045, 'sigma': 0.6877, 'score': 104.9413, 'games': 1500, 'win_rate': 0.2627, 'avg_rank': 0.878}, '****': {'rank': 601, 'team': "****", 'mu': 106.9952, 'sigma': 0.6865, 'score': 104.9357, 'games': 2169, 'win_rate': 0.2923, 'avg_rank': 1.3974}, '****': {'rank': 602, 'team': "****", 'mu': 107.1297, 'sigma': 0.7352, 'score': 104.9242, 'games': 95, 'win_rate': 0.2421, 'avg_rank': 1.4526}, '****': {'rank': 603, 'team': "****", 'mu': 106.9683, 'sigma': 0.6878, 'score': 104.9048, 'games': 218, 'win_rate': 0.1972, 'avg_rank': 1.5917}, '****': {'rank': 604, 'team': "****", 'mu': 106.9205, 'sigma': 0.6863, 'score': 104.8617, 'games': 343, 'win_rate': 0.2187, 'avg_rank': 1.449}, '****': {'rank': 605, 'team': "****", 'mu': 106.9022, 'sigma': 0.6838, 'score': 104.8508, 'games': 892, 'win_rate': 0.1738, 'avg_rank': 1.4193}, '****': {'rank': 606, 'team': "****", 'mu': 106.9147, 'sigma': 0.6894, 'score': 104.8465, 'games': 1245, 'win_rate': 0.2313, 'avg_rank': 1.3751}, '****': {'rank': 607, 'team': "****", 'mu': 106.8852, 'sigma': 0.6858, 'score': 104.8277, 'games': 538, 'win_rate': 0.2249, 'avg_rank': 1.4926}, '****': {'rank': 608, 'team': "****", 'mu': 106.9099, 'sigma': 0.6988, 'score': 104.8136, 'games': 158, 'win_rate': 0.1899, 'avg_rank': 1.5949}, '****': {'rank': 609, 'team': "****", 'mu': 106.8631, 'sigma': 0.6843, 'score': 104.8103, 'games': 501, 'win_rate': 0.2695, 'avg_rank': 1.3932}, '****': {'rank': 610, 'team': "****", 'mu': 107.0109, 'sigma': 0.7364, 'score': 104.8017, 'games': 99, 'win_rate': 0.1111, 'avg_rank': 1.5859}, '****': {'rank': 611, 'team': "****", 'mu': 107.0391, 'sigma': 0.7465, 'score': 104.7997, 'games': 92, 'win_rate': 0.1087, 'avg_rank': 1.7826}, '****': {'rank': 612, 'team': "****", 'mu': 106.8563, 'sigma': 0.6865, 'score': 104.7968, 'games': 1228, 'win_rate': 0.3029, 'avg_rank': 1.3314}, '****': {'rank': 613, 'team': "****", 'mu': 106.8567, 'sigma': 0.6874, 'score': 104.7946, 'games': 654, 'win_rate': 0.2936, 'avg_rank': 1.3761}, '****': {'rank': 614, 'team': "****", 'mu': 106.8213, 'sigma': 0.6802, 'score': 104.7808, 'games': 475, 'win_rate': 0.2568, 'avg_rank': 1.4316}, '****': {'rank': 615, 'team': "****", 'mu': 106.8419, 'sigma': 0.6878, 'score': 104.7786, 'games': 222, 'win_rate': 0.1532, 'avg_rank': 1.7162}, '****': {'rank': 616, 'team': "****", 'mu': 106.9518, 'sigma': 0.7268, 'score': 104.7713, 'games': 108, 'win_rate': 0.0556, 'avg_rank': 1.8704}, '****': {'rank': 617, 'team': "****", 'mu': 106.7704, 'sigma': 0.6892, 'score': 104.7028, 'games': 783, 'win_rate': 0.1418, 'avg_rank': 1.5211}, '****': {'rank': 618, 'team': "****", 'mu': 106.7604, 'sigma': 0.6896, 'score': 104.6916, 'games': 609, 'win_rate': 0.0722, 'avg_rank': 1.8686}, '****': {'rank': 619, 'team': "****", 'mu': 106.7502, 'sigma': 0.6885, 'score': 104.6846, 'games': 940, 'win_rate': 0.183, 'avg_rank': 1.4138}, '****': {'rank': 620, 'team': "****", 'mu': 106.7378, 'sigma': 0.6868, 'score': 104.6772, 'games': 1927, 'win_rate': 0.1194, 'avg_rank': 1.5776}, '****': {'rank': 621, 'team': "****", 'mu': 106.7175, 'sigma': 0.6806, 'score': 104.6757, 'games': 497, 'win_rate': 0.159, 'avg_rank': 1.4708}, '****': {'rank': 622, 'team': "****", 'mu': 106.7581, 'sigma': 0.6948, 'score': 104.6738, 'games': 146, 'win_rate': 0.2192, 'avg_rank': 1.589}, '****': {'rank': 623, 'team': "****", 'mu': 107.0613, 'sigma': 0.7982, 'score': 104.6666, 'games': 73, 'win_rate': 0.1918, 'avg_rank': 1.6438}, '****': {'rank': 624, 'team': "****", 'mu': 106.7076, 'sigma': 0.6846, 'score': 104.6537, 'games': 2088, 'win_rate': 0.1082, 'avg_rank': 1.42}, '****': {'rank': 625, 'team': "****", 'mu': 107.0049, 'sigma': 0.7862, 'score': 104.6464, 'games': 77, 'win_rate': 0.1558, 'avg_rank': 1.8701}, '****': {'rank': 626, 'team': "****", 'mu': 106.8072, 'sigma': 0.7264, 'score': 104.6279, 'games': 105, 'win_rate': 0.1238, 'avg_rank': 1.7524}, '****': {'rank': 627, 'team': "****", 'mu': 106.6874, 'sigma': 0.6871, 'score': 104.626, 'games': 1079, 'win_rate': 0.2159, 'avg_rank': 1.3939}, '****': {'rank': 628, 'team': "****", 'mu': 106.7032, 'sigma': 0.6955, 'score': 104.6166, 'games': 526, 'win_rate': 0.0798, 'avg_rank': 1.8023}, '****': {'rank': 629, 'team': "****", 'mu': 106.6659, 'sigma': 0.6839, 'score': 104.6141, 'games': 301, 'win_rate': 0.0731, 'avg_rank': 1.5382}, '****': {'rank': 630, 'team': "****", 'mu': 106.6505, 'sigma': 0.6849, 'score': 104.5958, 'games': 2078, 'win_rate': 0.2113, 'avg_rank': 1.3985}, '****': {'rank': 631, 'team': "****", 'mu': 106.6484, 'sigma': 0.6874, 'score': 104.5863, 'games': 470, 'win_rate': 0.2149, 'avg_rank': 1.4787}, '****': {'rank': 632, 'team': "****", 'mu': 106.5998, 'sigma': 0.6876, 'score': 104.5371, 'games': 400, 'win_rate': 0.215, 'avg_rank': 1.5675}, '****': {'rank': 633, 'team': "****", 'mu': 106.616, 'sigma': 0.6933, 'score': 104.536, 'games': 408, 'win_rate': 0.2402, 'avg_rank': 1.5686}, '****': {'rank': 634, 'team': "****", 'mu': 106.876, 'sigma': 0.7827, 'score': 104.528, 'games': 83, 'win_rate': 0.0964, 'avg_rank': 2.012}, '****': {'rank': 635, 'team': "****", 'mu': 106.5469, 'sigma': 0.6776, 'score': 104.514, 'games': 1802, 'win_rate': 0.1853, 'avg_rank': 1.4689}, '****': {'rank': 636, 'team': "****", 'mu': 107.0028, 'sigma': 0.8299, 'score': 104.5129, 'games': 63, 'win_rate': 0.0794, 'avg_rank': 1.5556}, '****': {'rank': 637, 'team': "****", 'mu': 106.5527, 'sigma': 0.6829, 'score': 104.5041, 'games': 398, 'win_rate': 0.2563, 'avg_rank': 1.4497}, '****': {'rank': 638, 'team': "****", 'mu': 106.5446, 'sigma': 0.682, 'score': 104.4986, 'games': 2111, 'win_rate': 0.1222, 'avg_rank': 1.4263}, '****': {'rank': 639, 'team': "****", 'mu': 106.9972, 'sigma': 0.8336, 'score': 104.4964, 'games': 57, 'win_rate': 0.2281, 'avg_rank': 1.4561}, '****': {'rank': 640, 'team': "****", 'mu': 106.5077, 'sigma': 0.6894, 'score': 104.4395, 'games': 2176, 'win_rate': 0.2684, 'avg_rank': 1.4848}, '****': {'rank': 641, 'team': "****", 'mu': 106.5166, 'sigma': 0.6982, 'score': 104.4221, 'games': 131, 'win_rate': 0.1069, 'avg_rank': 1.855}, '****': {'rank': 642, 'team': "****", 'mu': 106.4724, 'sigma': 0.6843, 'score': 104.4196, 'games': 211, 'win_rate': 0.2227, 'avg_rank': 1.5308}, '****': {'rank': 643, 'team': "****", 'mu': 106.9002, 'sigma': 0.83, 'score': 104.4103, 'games': 57, 'win_rate': 0.193, 'avg_rank': 1.5088}, '****': {'rank': 644, 'team': "****", 'mu': 106.4633, 'sigma': 0.6854, 'score': 104.407, 'games': 461, 'win_rate': 0.1822, 'avg_rank': 1.5683}, '****': {'rank': 645, 'team': "****", 'mu': 107.0955, 'sigma': 0.9078, 'score': 104.3722, 'games': 45, 'win_rate': 0.1111, 'avg_rank': 1.8667}, '****': {'rank': 646, 'team': "****", 'mu': 106.4167, 'sigma': 0.6885, 'score': 104.3511, 'games': 1025, 'win_rate': 0.1932, 'avg_rank': 1.6234}, '****': {'rank': 647, 'team': "****", 'mu': 106.4068, 'sigma': 0.6887, 'score': 104.3405, 'games': 2778, 'win_rate': 0.2905, 'avg_rank': 1.3996}, '****': {'rank': 648, 'team': "****", 'mu': 106.4149, 'sigma': 0.6934, 'score': 104.3346, 'games': 247, 'win_rate': 0.2632, 'avg_rank': 1.4049}, '****': {'rank': 649, 'team': "****", 'mu': 106.445, 'sigma': 0.7053, 'score': 104.329, 'games': 134, 'win_rate': 0.2015, 'avg_rank': 1.6642}, '****': {'rank': 650, 'team': "****", 'mu': 106.7779, 'sigma': 0.8264, 'score': 104.2986, 'games': 58, 'win_rate': 0.1379, 'avg_rank': 1.6552}, '****': {'rank': 651, 'team': "****", 'mu': 106.5807, 'sigma': 0.7619, 'score': 104.2949, 'games': 79, 'win_rate': 0.1899, 'avg_rank': 1.5823}, '****': {'rank': 652, 'team': "****", 'mu': 106.9674, 'sigma': 0.8963, 'score': 104.2786, 'games': 52, 'win_rate': 0.0769, 'avg_rank': 2.1154}, '****': {'rank': 653, 'team': "****", 'mu': 106.3417, 'sigma': 0.6878, 'score': 104.2782, 'games': 865, 'win_rate': 0.2, 'avg_rank': 1.7595}, '****': {'rank': 654, 'team': "****", 'mu': 106.3864, 'sigma': 0.7044, 'score': 104.2733, 'games': 181, 'win_rate': 0.1436, 'avg_rank': 1.8453}, '****': {'rank': 655, 'team': "****", 'mu': 106.3196, 'sigma': 0.6832, 'score': 104.2699, 'games': 164, 'win_rate': 0.0732, 'avg_rank': 1.5854}, '****': {'rank': 656, 'team': "****", 'mu': 106.3304, 'sigma': 0.6883, 'score': 104.2654, 'games': 198, 'win_rate': 0.096, 'avg_rank': 1.6465}, '****': {'rank': 657, 'team': "****", 'mu': 107.9427, 'sigma': 1.2326, 'score': 104.2449, 'games': 22, 'win_rate': 0.1818, 'avg_rank': 1.4091}, '****': {'rank': 658, 'team': "****", 'mu': 106.3012, 'sigma': 0.6874, 'score': 104.239, 'games': 1367, 'win_rate': 0.2363, 'avg_rank': 1.4389}, '****': {'rank': 659, 'team': "****", 'mu': 106.295, 'sigma': 0.6863, 'score': 104.2361, 'games': 495, 'win_rate': 0.1737, 'avg_rank': 1.5152}, '****': {'rank': 660, 'team': "****", 'mu': 106.3364, 'sigma': 0.7005, 'score': 104.235, 'games': 134, 'win_rate': 0.1567, 'avg_rank': 1.5896}, '****': {'rank': 661, 'team': "****", 'mu': 106.2888, 'sigma': 0.686, 'score': 104.2308, 'games': 1390, 'win_rate': 0.054, 'avg_rank': 1.7165}, '****': {'rank': 662, 'team': "****", 'mu': 106.7576, 'sigma': 0.8521, 'score': 104.2013, 'games': 56, 'win_rate': 0.125, 'avg_rank': 1.8393}, '****': {'rank': 663, 'team': "****", 'mu': 106.2584, 'sigma': 0.6891, 'score': 104.1909, 'games': 1071, 'win_rate': 0.1867, 'avg_rank': 1.634}, '****': {'rank': 664, 'team': "****", 'mu': 106.2411, 'sigma': 0.6846, 'score': 104.1874, 'games': 1895, 'win_rate': 0.162, 'avg_rank': 1.4385}, '****': {'rank': 665, 'team': "****", 'mu': 106.5367, 'sigma': 0.7838, 'score': 104.1854, 'games': 71, 'win_rate': 0.1408, 'avg_rank': 1.507}, '****': {'rank': 666, 'team': "****", 'mu': 106.2112, 'sigma': 0.6855, 'score': 104.1546, 'games': 2176, 'win_rate': 0.0501, 'avg_rank': 1.9674}, '****': {'rank': 667, 'team': "****", 'mu': 106.2289, 'sigma': 0.6917, 'score': 104.1539, 'games': 1815, 'win_rate': 0.1967, 'avg_rank': 1.6182}, '****': {'rank': 668, 'team': "****", 'mu': 106.2038, 'sigma': 0.6868, 'score': 104.1435, 'games': 1028, 'win_rate': 0.2354, 'avg_rank': 1.5146}, '****': {'rank': 669, 'team': "****", 'mu': 106.2151, 'sigma': 0.6921, 'score': 104.1388, 'games': 346, 'win_rate': 0.2197, 'avg_rank': 1.3902}, '****': {'rank': 670, 'team': "****", 'mu': 106.148, 'sigma': 0.6854, 'score': 104.0917, 'games': 975, 'win_rate': 0.2277, 'avg_rank': 1.5426}, '****': {'rank': 671, 'team': "****", 'mu': 106.1116, 'sigma': 0.6813, 'score': 104.0676, 'games': 512, 'win_rate': 0.1504, 'avg_rank': 1.6035}, '****': {'rank': 672, 'team': "****", 'mu': 106.3331, 'sigma': 0.7621, 'score': 104.0467, 'games': 80, 'win_rate': 0.1375, 'avg_rank': 1.625}, '****': {'rank': 673, 'team': "****", 'mu': 106.7632, 'sigma': 0.9061, 'score': 104.0449, 'games': 51, 'win_rate': 0.1569, 'avg_rank': 1.5294}, '****': {'rank': 674, 'team': "****", 'mu': 106.1208, 'sigma': 0.6922, 'score': 104.044, 'games': 819, 'win_rate': 0.1612, 'avg_rank': 1.4835}, '****': {'rank': 675, 'team': "****", 'mu': 106.2189, 'sigma': 0.7251, 'score': 104.0437, 'games': 95, 'win_rate': 0.1368, 'avg_rank': 1.5684}, '****': {'rank': 676, 'team': "****", 'mu': 106.0918, 'sigma': 0.6833, 'score': 104.042, 'games': 542, 'win_rate': 0.1531, 'avg_rank': 1.607}, '****': {'rank': 677, 'team': "****", 'mu': 106.1425, 'sigma': 0.7073, 'score': 104.0207, 'games': 164, 'win_rate': 0.0122, 'avg_rank': 2.189}, '****': {'rank': 678, 'team': "****", 'mu': 106.0619, 'sigma': 0.6934, 'score': 103.9816, 'games': 542, 'win_rate': 0.024, 'avg_rank': 1.9594}, '****': {'rank': 679, 'team': "****", 'mu': 106.4092, 'sigma': 0.814, 'score': 103.9671, 'games': 61, 'win_rate': 0.0656, 'avg_rank': 1.5738}, '****': {'rank': 680, 'team': "****", 'mu': 106.0224, 'sigma': 0.6907, 'score': 103.9502, 'games': 229, 'win_rate': 0.0568, 'avg_rank': 1.7555}, '****': {'rank': 681, 'team': "****", 'mu': 105.9697, 'sigma': 0.6895, 'score': 103.9013, 'games': 1850, 'win_rate': 0.26, 'avg_rank': 1.3692}, '****': {'rank': 682, 'team': "****", 'mu': 105.9507, 'sigma': 0.6858, 'score': 103.8933, 'games': 2303, 'win_rate': 0.2262, 'avg_rank': 1.4594}, '****': {'rank': 683, 'team': "****", 'mu': 105.9385, 'sigma': 0.6871, 'score': 103.8772, 'games': 2059, 'win_rate': 0.1928, 'avg_rank': 1.6523}, '****': {'rank': 684, 'team': "****", 'mu': 105.9608, 'sigma': 0.7001, 'score': 103.8605, 'games': 307, 'win_rate': 0.0749, 'avg_rank': 2.0195}, '****': {'rank': 685, 'team': "****", 'mu': 105.9171, 'sigma': 0.6889, 'score': 103.8502, 'games': 617, 'win_rate': 0.128, 'avg_rank': 1.7034}, '****': {'rank': 686, 'team': "****", 'mu': 105.9266, 'sigma': 0.6929, 'score': 103.8478, 'games': 2286, 'win_rate': 0.2248, 'avg_rank': 1.4563}, '****': {'rank': 687, 'team': "****", 'mu': 105.9083, 'sigma': 0.6932, 'score': 103.8288, 'games': 932, 'win_rate': 0.2725, 'avg_rank': 1.4657}, '****': {'rank': 688, 'team': "****", 'mu': 105.8473, 'sigma': 0.6849, 'score': 103.7926, 'games': 1893, 'win_rate': 0.1992, 'avg_rank': 1.4982}, '****': {'rank': 689, 'team': "****", 'mu': 105.9258, 'sigma': 0.7203, 'score': 103.765, 'games': 109, 'win_rate': 0.1835, 'avg_rank': 1.6422}, '****': {'rank': 690, 'team': "****", 'mu': 105.7718, 'sigma': 0.6871, 'score': 103.7105, 'games': 941, 'win_rate': 0.2338, 'avg_rank': 1.4718}, '****': {'rank': 691, 'team': "****", 'mu': 105.7677, 'sigma': 0.6921, 'score': 103.6914, 'games': 2096, 'win_rate': 0.1951, 'avg_rank': 1.6288}, '****': {'rank': 692, 'team': "****", 'mu': 105.9682, 'sigma': 0.7603, 'score': 103.6874, 'games': 85, 'win_rate': 0.1294, 'avg_rank': 1.8}, '****': {'rank': 693, 'team': "****", 'mu': 105.7802, 'sigma': 0.7046, 'score': 103.6665, 'games': 128, 'win_rate': 0.1406, 'avg_rank': 1.6797}, '****': {'rank': 694, 'team': "****", 'mu': 105.719, 'sigma': 0.6862, 'score': 103.6604, 'games': 2657, 'win_rate': 0.2104, 'avg_rank': 1.5604}, '****': {'rank': 695, 'team': "****", 'mu': 105.7547, 'sigma': 0.7057, 'score': 103.6377, 'games': 129, 'win_rate': 0.0698, 'avg_rank': 1.7442}, '****': {'rank': 696, 'team': "****", 'mu': 106.0407, 'sigma': 0.807, 'score': 103.6197, 'games': 102, 'win_rate': 0.0588, 'avg_rank': 2.3725}, '****': {'rank': 697, 'team': "****", 'mu': 105.6555, 'sigma': 0.703, 'score': 103.5464, 'games': 339, 'win_rate': 0.0177, 'avg_rank': 1.9705}, '****': {'rank': 698, 'team': "****", 'mu': 105.6181, 'sigma': 0.6918, 'score': 103.5427, 'games': 1734, 'win_rate': 0.2018, 'avg_rank': 1.5196}, '****': {'rank': 699, 'team': "****", 'mu': 105.6137, 'sigma': 0.6931, 'score': 103.5345, 'games': 596, 'win_rate': 0.0319, 'avg_rank': 1.844}, '****': {'rank': 700, 'team': "****", 'mu': 105.6075, 'sigma': 0.6918, 'score': 103.5321, 'games': 861, 'win_rate': 0.2323, 'avg_rank': 1.4506}, '****': {'rank': 701, 'team': "****", 'mu': 105.5855, 'sigma': 0.6874, 'score': 103.5233, 'games': 818, 'win_rate': 0.2029, 'avg_rank': 1.6222}, '****': {'rank': 702, 'team': "****", 'mu': 105.5107, 'sigma': 0.6765, 'score': 103.4812, 'games': 947, 'win_rate': 0.0686, 'avg_rank': 1.7096}, '****': {'rank': 703, 'team': "****", 'mu': 105.5234, 'sigma': 0.6849, 'score': 103.4687, 'games': 2009, 'win_rate': 0.0667, 'avg_rank': 1.6063}, '****': {'rank': 704, 'team': "****", 'mu': 105.5124, 'sigma': 0.6872, 'score': 103.4509, 'games': 339, 'win_rate': 0.0855, 'avg_rank': 1.7375}, '****': {'rank': 705, 'team': "****", 'mu': 105.491, 'sigma': 0.6904, 'score': 103.4199, 'games': 595, 'win_rate': 0.1899, 'avg_rank': 1.6723}, '****': {'rank': 706, 'team': "****", 'mu': 105.48, 'sigma': 0.6876, 'score': 103.4172, 'games': 884, 'win_rate': 0.0317, 'avg_rank': 1.8609}, '****': {'rank': 707, 'team': "****", 'mu': 105.4963, 'sigma': 0.6944, 'score': 103.4132, 'games': 1802, 'win_rate': 0.0866, 'avg_rank': 1.7403}, '****': {'rank': 708, 'team': "****", 'mu': 105.5002, 'sigma': 0.702, 'score': 103.3943, 'games': 174, 'win_rate': 0.0977, 'avg_rank': 1.954}, '****': {'rank': 709, 'team': "****", 'mu': 105.4566, 'sigma': 0.6996, 'score': 103.3579, 'games': 365, 'win_rate': 0.1096, 'avg_rank': 2.0274}, '****': {'rank': 710, 'team': "****", 'mu': 106.1914, 'sigma': 0.9591, 'score': 103.3141, 'games': 44, 'win_rate': 0.1818, 'avg_rank': 1.7273}, '****': {'rank': 711, 'team': "****", 'mu': 105.3244, 'sigma': 0.6897, 'score': 103.2554, 'games': 1219, 'win_rate': 0.2158, 'avg_rank': 1.598}, '****': {'rank': 712, 'team': "****", 'mu': 105.3211, 'sigma': 0.6899, 'score': 103.2515, 'games': 584, 'win_rate': 0.1764, 'avg_rank': 1.6712}, '****': {'rank': 713, 'team': "****", 'mu': 105.3421, 'sigma': 0.6991, 'score': 103.2449, 'games': 211, 'win_rate': 0.0711, 'avg_rank': 2.0616}, '****': {'rank': 714, 'team': "****", 'mu': 105.299, 'sigma': 0.6875, 'score': 103.2364, 'games': 208, 'win_rate': 0.1202, 'avg_rank': 1.6827}, '****': {'rank': 715, 'team': "****", 'mu': 105.3657, 'sigma': 0.7125, 'score': 103.228, 'games': 123, 'win_rate': 0.1707, 'avg_rank': 1.748}, '****': {'rank': 716, 'team': "****", 'mu': 105.2816, 'sigma': 0.6872, 'score': 103.22, 'games': 2529, 'win_rate': 0.2009, 'avg_rank': 1.5987}, '****': {'rank': 717, 'team': "****", 'mu': 105.269, 'sigma': 0.6904, 'score': 103.1978, 'games': 1594, 'win_rate': 0.2039, 'avg_rank': 1.2698}, '****': {'rank': 718, 'team': "****", 'mu': 105.2693, 'sigma': 0.6924, 'score': 103.192, 'games': 729, 'win_rate': 0.192, 'avg_rank': 1.6077}, 'baseline_tactical_rule_agent_v1': {'rank': 719, 'team': "****", 'mu': 105.2553, 'sigma': 0.6879, 'score': 103.1915, 'games': 2542, 'win_rate': 0.3725, 'avg_rank': 1.0271}, '****': {'rank': 720, 'team': "****", 'mu': 105.3182, 'sigma': 0.709, 'score': 103.1911, 'games': 239, 'win_rate': 0.0251, 'avg_rank': 2.1046}, '****': {'rank': 721, 'team': "****", 'mu': 105.2416, 'sigma': 0.6937, 'score': 103.1605, 'games': 1448, 'win_rate': 0.1796, 'avg_rank': 1.2231}, '****': {'rank': 722, 'team': "****", 'mu': 105.2079, 'sigma': 0.6866, 'score': 103.148, 'games': 660, 'win_rate': 0.0379, 'avg_rank': 1.5045}, '****': {'rank': 723, 'team': "****", 'mu': 105.2081, 'sigma': 0.6919, 'score': 103.1323, 'games': 717, 'win_rate': 0.0544, 'avg_rank': 1.3808}, '****': {'rank': 724, 'team': "****", 'mu': 105.1818, 'sigma': 0.6869, 'score': 103.1212, 'games': 2317, 'win_rate': 0.2395, 'avg_rank': 1.4661}, '****': {'rank': 725, 'team': "****", 'mu': 105.1817, 'sigma': 0.6893, 'score': 103.1137, 'games': 241, 'win_rate': 0.112, 'avg_rank': 1.722}, '****': {'rank': 726, 'team': "****", 'mu': 105.1849, 'sigma': 0.6908, 'score': 103.1124, 'games': 316, 'win_rate': 0.0854, 'avg_rank': 1.7785}, '****': {'rank': 727, 'team': "****", 'mu': 105.2403, 'sigma': 0.7099, 'score': 103.1106, 'games': 184, 'win_rate': 0.087, 'avg_rank': 2.1033}, '****': {'rank': 728, 'team': "****", 'mu': 106.0086, 'sigma': 0.972, 'score': 103.0926, 'games': 46, 'win_rate': 0.0, 'avg_rank': 2.1087}, '****': {'rank': 729, 'team': "****", 'mu': 105.7518, 'sigma': 0.8878, 'score': 103.0882, 'games': 54, 'win_rate': 0.2778, 'avg_rank': 1.537}, '****': {'rank': 730, 'team': "****", 'mu': 105.3374, 'sigma': 0.7588, 'score': 103.0609, 'games': 80, 'win_rate': 0.0875, 'avg_rank': 1.7125}, '****': {'rank': 731, 'team': "****", 'mu': 105.1027, 'sigma': 0.6811, 'score': 103.0593, 'games': 2300, 'win_rate': 0.1317, 'avg_rank': 1.6252}, '****': {'rank': 732, 'team': "****", 'mu': 105.1226, 'sigma': 0.6881, 'score': 103.0584, 'games': 827, 'win_rate': 0.0339, 'avg_rank': 1.763}, '****': {'rank': 733, 'team': "****", 'mu': 105.1498, 'sigma': 0.7067, 'score': 103.0296, 'games': 306, 'win_rate': 0.0588, 'avg_rank': 2.098}, '****': {'rank': 734, 'team': "****", 'mu': 105.0604, 'sigma': 0.6842, 'score': 103.0079, 'games': 2000, 'win_rate': 0.025, 'avg_rank': 1.785}, '****': {'rank': 735, 'team': "****", 'mu': 105.0687, 'sigma': 0.6883, 'score': 103.0039, 'games': 2090, 'win_rate': 0.0526, 'avg_rank': 1.6325}, '****': {'rank': 736, 'team': "****", 'mu': 105.019, 'sigma': 0.6854, 'score': 102.9626, 'games': 2144, 'win_rate': 0.049, 'avg_rank': 1.7416}, '****': {'rank': 737, 'team': "****", 'mu': 105.588, 'sigma': 0.8772, 'score': 102.9565, 'games': 64, 'win_rate': 0.0, 'avg_rank': 2.1406}, '****': {'rank': 738, 'team': "****", 'mu': 105.1628, 'sigma': 0.737, 'score': 102.9517, 'games': 150, 'win_rate': 0.02, 'avg_rank': 2.2067}, '****': {'rank': 739, 'team': "****", 'mu': 104.9903, 'sigma': 0.6948, 'score': 102.906, 'games': 207, 'win_rate': 0.1353, 'avg_rank': 1.715}, '****': {'rank': 740, 'team': "****", 'mu': 104.9524, 'sigma': 0.6826, 'score': 102.9045, 'games': 475, 'win_rate': 0.0126, 'avg_rank': 1.8463}, '****': {'rank': 741, 'team': "****", 'mu': 104.9345, 'sigma': 0.6819, 'score': 102.8887, 'games': 1884, 'win_rate': 0.1269, 'avg_rank': 1.7192}, '****': {'rank': 742, 'team': "****", 'mu': 104.9798, 'sigma': 0.7, 'score': 102.8799, 'games': 142, 'win_rate': 0.0775, 'avg_rank': 1.6972}, '****': {'rank': 743, 'team': "****", 'mu': 104.8845, 'sigma': 0.6763, 'score': 102.8556, 'games': 1849, 'win_rate': 0.026, 'avg_rank': 1.7474}, '****': {'rank': 744, 'team': "****", 'mu': 104.9183, 'sigma': 0.6881, 'score': 102.8539, 'games': 828, 'win_rate': 0.0809, 'avg_rank': 1.7186}, '****': {'rank': 745, 'team': "****", 'mu': 104.9216, 'sigma': 0.6937, 'score': 102.8404, 'games': 748, 'win_rate': 0.0241, 'avg_rank': 1.8984}, '****': {'rank': 746, 'team': "****", 'mu': 105.1154, 'sigma': 0.7643, 'score': 102.8224, 'games': 106, 'win_rate': 0.0189, 'avg_rank': 2.0566}, '****': {'rank': 747, 'team': "****", 'mu': 105.0808, 'sigma': 0.76, 'score': 102.8007, 'games': 85, 'win_rate': 0.1059, 'avg_rank': 1.8}, '****': {'rank': 748, 'team': "****", 'mu': 104.9084, 'sigma': 0.7027, 'score': 102.8001, 'games': 124, 'win_rate': 0.0968, 'avg_rank': 1.7823}, '****': {'rank': 749, 'team': "****", 'mu': 104.859, 'sigma': 0.6922, 'score': 102.7823, 'games': 2312, 'win_rate': 0.1972, 'avg_rank': 1.6302}, '****': {'rank': 750, 'team': "****", 'mu': 104.8009, 'sigma': 0.6774, 'score': 102.7687, 'games': 1439, 'win_rate': 0.0278, 'avg_rank': 1.8068}, '****': {'rank': 751, 'team': "****", 'mu': 105.0206, 'sigma': 0.7509, 'score': 102.7678, 'games': 127, 'win_rate': 0.0236, 'avg_rank': 2.1575}, '****': {'rank': 752, 'team': "****", 'mu': 104.8172, 'sigma': 0.6927, 'score': 102.7391, 'games': 343, 'win_rate': 0.0583, 'avg_rank': 1.8426}, '****': {'rank': 753, 'team': "****", 'mu': 104.7804, 'sigma': 0.6846, 'score': 102.7265, 'games': 1231, 'win_rate': 0.2729, 'avg_rank': 1.5134}, '****': {'rank': 754, 'team': "****", 'mu': 104.8324, 'sigma': 0.707, 'score': 102.7113, 'games': 131, 'win_rate': 0.0382, 'avg_rank': 1.9542}, '****': {'rank': 755, 'team': "****", 'mu': 104.9347, 'sigma': 0.7457, 'score': 102.6975, 'games': 100, 'win_rate': 0.15, 'avg_rank': 1.88}, '****': {'rank': 756, 'team': "****", 'mu': 105.1622, 'sigma': 0.8263, 'score': 102.6833, 'games': 106, 'win_rate': 0.0094, 'avg_rank': 1.9151}, '****': {'rank': 757, 'team': "****", 'mu': 105.7795, 'sigma': 1.034, 'score': 102.6775, 'games': 39, 'win_rate': 0.0513, 'avg_rank': 2.0513}, '****': {'rank': 758, 'team': "****", 'mu': 104.9583, 'sigma': 0.7641, 'score': 102.666, 'games': 88, 'win_rate': 0.1364, 'avg_rank': 1.875}, '****': {'rank': 759, 'team': "****", 'mu': 105.6945, 'sigma': 1.0136, 'score': 102.6539, 'games': 43, 'win_rate': 0.1395, 'avg_rank': 2.1395}, '****': {'rank': 760, 'team': "****", 'mu': 104.6899, 'sigma': 0.6863, 'score': 102.6311, 'games': 1091, 'win_rate': 0.066, 'avg_rank': 1.8341}, '****': {'rank': 761, 'team': "****", 'mu': 105.8281, 'sigma': 1.068, 'score': 102.6241, 'games': 34, 'win_rate': 0.0294, 'avg_rank': 1.8529}, '****': {'rank': 762, 'team': "****", 'mu': 105.8255, 'sigma': 1.0676, 'score': 102.6226, 'games': 34, 'win_rate': 0.0, 'avg_rank': 1.7353}, '****': {'rank': 763, 'team': "****", 'mu': 104.7979, 'sigma': 0.7507, 'score': 102.5457, 'games': 187, 'win_rate': 0.0053, 'avg_rank': 2.3636}, '****': {'rank': 764, 'team': "****", 'mu': 104.601, 'sigma': 0.6861, 'score': 102.5427, 'games': 387, 'win_rate': 0.062, 'avg_rank': 1.832}, '****': {'rank': 765, 'team': "****", 'mu': 104.6771, 'sigma': 0.7145, 'score': 102.5336, 'games': 673, 'win_rate': 0.0713, 'avg_rank': 2.0817}, '****': {'rank': 766, 'team': "****", 'mu': 104.7826, 'sigma': 0.7499, 'score': 102.533, 'games': 86, 'win_rate': 0.0814, 'avg_rank': 1.8256}, '****': {'rank': 767, 'team': "****", 'mu': 104.9862, 'sigma': 0.8179, 'score': 102.5323, 'games': 65, 'win_rate': 0.0615, 'avg_rank': 1.8615}, '****': {'rank': 768, 'team': "****", 'mu': 104.9748, 'sigma': 0.8244, 'score': 102.5016, 'games': 65, 'win_rate': 0.0308, 'avg_rank': 1.8308}, '****': {'rank': 769, 'team': "****", 'mu': 105.4262, 'sigma': 0.9749, 'score': 102.5015, 'games': 44, 'win_rate': 0.1591, 'avg_rank': 1.7727}, '****': {'rank': 770, 'team': "****", 'mu': 104.545, 'sigma': 0.6875, 'score': 102.4824, 'games': 1758, 'win_rate': 0.0438, 'avg_rank': 1.7708}, '****': {'rank': 771, 'team': "****", 'mu': 104.5196, 'sigma': 0.6858, 'score': 102.4621, 'games': 861, 'win_rate': 0.0488, 'avg_rank': 1.7271}, '****': {'rank': 772, 'team': "****", 'mu': 104.5065, 'sigma': 0.6997, 'score': 102.4075, 'games': 224, 'win_rate': 0.0223, 'avg_rank': 1.9286}, '****': {'rank': 773, 'team': "****", 'mu': 104.7928, 'sigma': 0.8026, 'score': 102.385, 'games': 68, 'win_rate': 0.0441, 'avg_rank': 1.8676}, '****': {'rank': 774, 'team': "****", 'mu': 104.5375, 'sigma': 0.7215, 'score': 102.373, 'games': 146, 'win_rate': 0.0411, 'avg_rank': 2.1781}, '****': {'rank': 775, 'team': "****", 'mu': 104.5908, 'sigma': 0.7428, 'score': 102.3623, 'games': 163, 'win_rate': 0.0429, 'avg_rank': 2.3313}, '****': {'rank': 776, 'team': "****", 'mu': 104.4475, 'sigma': 0.7014, 'score': 102.3434, 'games': 482, 'win_rate': 0.0436, 'avg_rank': 2.0415}, '****': {'rank': 777, 'team': "****", 'mu': 104.5586, 'sigma': 0.741, 'score': 102.3357, 'games': 106, 'win_rate': 0.0755, 'avg_rank': 2.0472}, '****': {'rank': 778, 'team': "****", 'mu': 105.2241, 'sigma': 0.9632, 'score': 102.3345, 'games': 39, 'win_rate': 0.0, 'avg_rank': 1.9487}, '****': {'rank': 779, 'team': "****", 'mu': 104.4536, 'sigma': 0.7148, 'score': 102.3093, 'games': 2694, 'win_rate': 0.1295, 'avg_rank': 1.3976}, '****': {'rank': 780, 'team': "****", 'mu': 104.3022, 'sigma': 0.681, 'score': 102.2592, 'games': 679, 'win_rate': 0.0088, 'avg_rank': 1.8866}, '****': {'rank': 781, 'team': "****", 'mu': 104.2741, 'sigma': 0.6914, 'score': 102.1998, 'games': 469, 'win_rate': 0.0405, 'avg_rank': 1.8571}, '****': {'rank': 782, 'team': "****", 'mu': 104.2481, 'sigma': 0.6877, 'score': 102.1848, 'games': 882, 'win_rate': 0.0227, 'avg_rank': 1.7517}, '****': {'rank': 783, 'team': "****", 'mu': 105.6837, 'sigma': 1.1686, 'score': 102.178, 'games': 30, 'win_rate': 0.2, 'avg_rank': 1.8}, '****': {'rank': 784, 'team': "****", 'mu': 107.2109, 'sigma': 1.6789, 'score': 102.1743, 'games': 12, 'win_rate': 0.0, 'avg_rank': 1.9167}, '****': {'rank': 785, 'team': "****", 'mu': 104.3018, 'sigma': 0.7162, 'score': 102.1531, 'games': 133, 'win_rate': 0.0526, 'avg_rank': 2.0977}, '****': {'rank': 786, 'team': "****", 'mu': 104.2413, 'sigma': 0.7036, 'score': 102.1304, 'games': 974, 'win_rate': 0.039, 'avg_rank': 2.1027}, '****': {'rank': 787, 'team': "****", 'mu': 104.2437, 'sigma': 0.7099, 'score': 102.114, 'games': 591, 'win_rate': 0.0592, 'avg_rank': 2.0423}, '****': {'rank': 788, 'team': "****", 'mu': 104.1422, 'sigma': 0.6876, 'score': 102.0794, 'games': 1069, 'win_rate': 0.0281, 'avg_rank': 1.8372}, '****': {'rank': 789, 'team': "****", 'mu': 104.1157, 'sigma': 0.6851, 'score': 102.0603, 'games': 788, 'win_rate': 0.0114, 'avg_rank': 1.8211}, '****': {'rank': 790, 'team': "****", 'mu': 104.848, 'sigma': 0.9331, 'score': 102.0488, 'games': 45, 'win_rate': 0.1111, 'avg_rank': 1.7556}, '****': {'rank': 791, 'team': "****", 'mu': 104.0609, 'sigma': 0.6899, 'score': 101.9913, 'games': 827, 'win_rate': 0.1451, 'avg_rank': 1.5611}, '****': {'rank': 792, 'team': "****", 'mu': 104.035, 'sigma': 0.6885, 'score': 101.9696, 'games': 2564, 'win_rate': 0.181, 'avg_rank': 1.6568}, '****': {'rank': 793, 'team': "****", 'mu': 104.0105, 'sigma': 0.6851, 'score': 101.9551, 'games': 1702, 'win_rate': 0.0147, 'avg_rank': 1.7673}, '****': {'rank': 794, 'team': "****", 'mu': 104.0043, 'sigma': 0.6861, 'score': 101.9459, 'games': 2074, 'win_rate': 0.067, 'avg_rank': 1.7555}, '****': {'rank': 795, 'team': "****", 'mu': 106.6879, 'sigma': 1.5913, 'score': 101.9139, 'games': 14, 'win_rate': 0.1429, 'avg_rank': 1.6429}, '****': {'rank': 796, 'team': "****", 'mu': 103.99, 'sigma': 0.6932, 'score': 101.9103, 'games': 532, 'win_rate': 0.0432, 'avg_rank': 1.9023}, '****': {'rank': 797, 'team': "****", 'mu': 104.0779, 'sigma': 0.7244, 'score': 101.9048, 'games': 164, 'win_rate': 0.0305, 'avg_rank': 2.2683}, '****': {'rank': 798, 'team': "****", 'mu': 104.0967, 'sigma': 0.7307, 'score': 101.9046, 'games': 111, 'win_rate': 0.036, 'avg_rank': 1.982}, '****': {'rank': 799, 'team': "****", 'mu': 107.4042, 'sigma': 1.8344, 'score': 101.9011, 'games': 12, 'win_rate': 0.0, 'avg_rank': 2.0833}, '****': {'rank': 800, 'team': "****", 'mu': 105.3081, 'sigma': 1.1358, 'score': 101.9007, 'games': 26, 'win_rate': 0.0769, 'avg_rank': 1.9615}, '****': {'rank': 801, 'team': "****", 'mu': 103.9805, 'sigma': 0.6962, 'score': 101.8918, 'games': 756, 'win_rate': 0.0357, 'avg_rank': 1.8413}, '****': {'rank': 802, 'team': "****", 'mu': 104.6974, 'sigma': 0.9374, 'score': 101.8851, 'games': 48, 'win_rate': 0.0208, 'avg_rank': 1.9583}, '****': {'rank': 803, 'team': "****", 'mu': 105.0807, 'sigma': 1.0797, 'score': 101.8416, 'games': 32, 'win_rate': 0.125, 'avg_rank': 1.5938}, '****': {'rank': 804, 'team': "****", 'mu': 105.964, 'sigma': 1.3942, 'score': 101.7815, 'games': 20, 'win_rate': 0.1, 'avg_rank': 2.1}, '****': {'rank': 805, 'team': "****", 'mu': 103.851, 'sigma': 0.7156, 'score': 101.7041, 'games': 269, 'win_rate': 0.0112, 'avg_rank': 2.2342}, '****': {'rank': 806, 'team': "****", 'mu': 104.2113, 'sigma': 0.8481, 'score': 101.6669, 'games': 59, 'win_rate': 0.0847, 'avg_rank': 1.8475}, '****': {'rank': 807, 'team': "****", 'mu': 103.7276, 'sigma': 0.6927, 'score': 101.6495, 'games': 354, 'win_rate': 0.0734, 'avg_rank': 1.8785}, '****': {'rank': 808, 'team': "****", 'mu': 103.7418, 'sigma': 0.6998, 'score': 101.6424, 'games': 1187, 'win_rate': 0.0084, 'avg_rank': 2.064}, '****': {'rank': 809, 'team': "****", 'mu': 103.955, 'sigma': 0.7778, 'score': 101.6215, 'games': 81, 'win_rate': 0.0617, 'avg_rank': 1.9383}, '****': {'rank': 810, 'team': "****", 'mu': 104.1556, 'sigma': 0.8466, 'score': 101.6158, 'games': 67, 'win_rate': 0.0597, 'avg_rank': 1.9851}, '****': {'rank': 811, 'team': "****", 'mu': 103.6425, 'sigma': 0.6914, 'score': 101.5683, 'games': 390, 'win_rate': 0.1154, 'avg_rank': 1.8538}, '****': {'rank': 812, 'team': "****", 'mu': 103.6722, 'sigma': 0.7023, 'score': 101.5653, 'games': 858, 'win_rate': 0.1259, 'avg_rank': 1.8578}, '****': {'rank': 813, 'team': "****", 'mu': 103.5965, 'sigma': 0.6868, 'score': 101.5361, 'games': 892, 'win_rate': 0.0818, 'avg_rank': 1.7859}, '****': {'rank': 814, 'team': "****", 'mu': 103.6025, 'sigma': 0.6953, 'score': 101.5165, 'games': 216, 'win_rate': 0.0093, 'avg_rank': 2.0324}, '****': {'rank': 815, 'team': "****", 'mu': 103.5701, 'sigma': 0.6991, 'score': 101.4728, 'games': 555, 'win_rate': 0.0973, 'avg_rank': 1.8595}, '****': {'rank': 816, 'team': "****", 'mu': 103.5641, 'sigma': 0.7016, 'score': 101.4594, 'games': 823, 'win_rate': 0.051, 'avg_rank': 2.1154}, '****': {'rank': 817, 'team': "****", 'mu': 103.4728, 'sigma': 0.6889, 'score': 101.406, 'games': 535, 'win_rate': 0.0112, 'avg_rank': 2.0561}, '****': {'rank': 818, 'team': "****", 'mu': 103.5429, 'sigma': 0.719, 'score': 101.3858, 'games': 270, 'win_rate': 0.0593, 'avg_rank': 2.1407}, '****': {'rank': 819, 'team': "****", 'mu': 103.4599, 'sigma': 0.6933, 'score': 101.3799, 'games': 181, 'win_rate': 0.0166, 'avg_rank': 1.9669}, '****': {'rank': 820, 'team': "****", 'mu': 103.4178, 'sigma': 0.6923, 'score': 101.341, 'games': 741, 'win_rate': 0.0135, 'avg_rank': 1.9487}, '****': {'rank': 821, 'team': "****", 'mu': 103.3969, 'sigma': 0.6887, 'score': 101.3308, 'games': 512, 'win_rate': 0.0195, 'avg_rank': 1.8848}, '****': {'rank': 822, 'team': "****", 'mu': 105.7238, 'sigma': 1.4704, 'score': 101.3126, 'games': 16, 'win_rate': 0.3125, 'avg_rank': 1.0}, '****': {'rank': 823, 'team': "****", 'mu': 103.3422, 'sigma': 0.6883, 'score': 101.2773, 'games': 714, 'win_rate': 0.028, 'avg_rank': 1.9692}, '****': {'rank': 824, 'team': "****", 'mu': 103.3224, 'sigma': 0.6874, 'score': 101.2604, 'games': 2014, 'win_rate': 0.0199, 'avg_rank': 2.0233}, '****': {'rank': 825, 'team': "****", 'mu': 103.367, 'sigma': 0.7113, 'score': 101.2333, 'games': 387, 'win_rate': 0.0879, 'avg_rank': 2.0465}, '****': {'rank': 826, 'team': "****", 'mu': 103.608, 'sigma': 0.7997, 'score': 101.2089, 'games': 78, 'win_rate': 0.1154, 'avg_rank': 1.8077}, '****': {'rank': 827, 'team': "****", 'mu': 103.3026, 'sigma': 0.6992, 'score': 101.2049, 'games': 1778, 'win_rate': 0.0546, 'avg_rank': 2.009}, '****': {'rank': 828, 'team': "****", 'mu': 103.2697, 'sigma': 0.6938, 'score': 101.1884, 'games': 928, 'win_rate': 0.0668, 'avg_rank': 1.9483}, '****': {'rank': 829, 'team': "****", 'mu': 103.8243, 'sigma': 0.8867, 'score': 101.1642, 'games': 72, 'win_rate': 0.0278, 'avg_rank': 2.3056}, '****': {'rank': 830, 'team': "****", 'mu': 103.2119, 'sigma': 0.6977, 'score': 101.1188, 'games': 1089, 'win_rate': 0.1267, 'avg_rank': 1.7833}, '****': {'rank': 831, 'team': "****", 'mu': 103.1847, 'sigma': 0.6926, 'score': 101.1068, 'games': 853, 'win_rate': 0.027, 'avg_rank': 1.8792}, '****': {'rank': 832, 'team': "****", 'mu': 103.2812, 'sigma': 0.7267, 'score': 101.1011, 'games': 363, 'win_rate': 0.0138, 'avg_rank': 2.2837}, '****': {'rank': 833, 'team': "****", 'mu': 103.1799, 'sigma': 0.693, 'score': 101.1009, 'games': 929, 'win_rate': 0.0603, 'avg_rank': 2.0183}, '****': {'rank': 834, 'team': "****", 'mu': 103.1693, 'sigma': 0.6899, 'score': 101.0995, 'games': 347, 'win_rate': 0.0288, 'avg_rank': 1.9971}, '****': {'rank': 835, 'team': "****", 'mu': 103.2562, 'sigma': 0.7251, 'score': 101.0808, 'games': 226, 'win_rate': 0.0088, 'avg_rank': 2.3186}, '****': {'rank': 836, 'team': "****", 'mu': 103.2187, 'sigma': 0.7156, 'score': 101.072, 'games': 311, 'win_rate': 0.0129, 'avg_rank': 2.3055}, '****': {'rank': 837, 'team': "****", 'mu': 103.1422, 'sigma': 0.7032, 'score': 101.0328, 'games': 1155, 'win_rate': 0.0675, 'avg_rank': 2.0104}, '****': {'rank': 838, 'team': "****", 'mu': 103.2125, 'sigma': 0.7476, 'score': 100.9698, 'games': 112, 'win_rate': 0.0446, 'avg_rank': 2.0804}, '****': {'rank': 839, 'team': "****", 'mu': 104.9957, 'sigma': 1.3484, 'score': 100.9507, 'games': 23, 'win_rate': 0.1304, 'avg_rank': 1.8261}, '****': {'rank': 840, 'team': "****", 'mu': 103.1125, 'sigma': 0.7225, 'score': 100.945, 'games': 771, 'win_rate': 0.0597, 'avg_rank': 2.1453}, '****': {'rank': 841, 'team': "****", 'mu': 103.1138, 'sigma': 0.7249, 'score': 100.939, 'games': 206, 'win_rate': 0.0049, 'avg_rank': 2.3641}, '****': {'rank': 842, 'team': "****", 'mu': 103.5454, 'sigma': 0.8832, 'score': 100.8958, 'games': 67, 'win_rate': 0.0, 'avg_rank': 2.3582}, '****': {'rank': 843, 'team': "****", 'mu': 102.9187, 'sigma': 0.7011, 'score': 100.8154, 'games': 369, 'win_rate': 0.1003, 'avg_rank': 2.0407}, '****': {'rank': 844, 'team': "****", 'mu': 103.3156, 'sigma': 0.8442, 'score': 100.7831, 'games': 88, 'win_rate': 0.0114, 'avg_rank': 2.3864}, '****': {'rank': 845, 'team': "****", 'mu': 105.838, 'sigma': 1.6872, 'score': 100.7765, 'games': 19, 'win_rate': 0.0526, 'avg_rank': 2.2632}, '****': {'rank': 846, 'team': "****", 'mu': 102.9205, 'sigma': 0.7202, 'score': 100.7599, 'games': 152, 'win_rate': 0.0658, 'avg_rank': 2.125}, '****': {'rank': 847, 'team': "****", 'mu': 104.7986, 'sigma': 1.3467, 'score': 100.7586, 'games': 25, 'win_rate': 0.12, 'avg_rank': 2.08}, '****': {'rank': 848, 'team': "****", 'mu': 103.5125, 'sigma': 0.9217, 'score': 100.7474, 'games': 61, 'win_rate': 0.0492, 'avg_rank': 2.2623}, '****': {'rank': 849, 'team': "****", 'mu': 102.8091, 'sigma': 0.6989, 'score': 100.7123, 'games': 1925, 'win_rate': 0.0197, 'avg_rank': 1.9875}, '****': {'rank': 850, 'team': "****", 'mu': 102.8199, 'sigma': 0.7178, 'score': 100.6666, 'games': 169, 'win_rate': 0.0828, 'avg_rank': 2.0888}, '****': {'rank': 851, 'team': "****", 'mu': 103.0528, 'sigma': 0.8036, 'score': 100.6419, 'games': 88, 'win_rate': 0.0114, 'avg_rank': 2.3182}, '****': {'rank': 852, 'team': "****", 'mu': 102.8264, 'sigma': 0.7342, 'score': 100.6238, 'games': 332, 'win_rate': 0.0361, 'avg_rank': 2.3825}, '****': {'rank': 853, 'team': "****", 'mu': 103.3217, 'sigma': 0.9015, 'score': 100.6172, 'games': 87, 'win_rate': 0.0, 'avg_rank': 2.6207}, '****': {'rank': 854, 'team': "****", 'mu': 103.6224, 'sigma': 1.01, 'score': 100.5925, 'games': 45, 'win_rate': 0.0889, 'avg_rank': 2.0889}, '****': {'rank': 855, 'team': "****", 'mu': 102.8033, 'sigma': 0.7712, 'score': 100.4898, 'games': 80, 'win_rate': 0.0125, 'avg_rank': 2.075}, '****': {'rank': 856, 'team': "****", 'mu': 105.6507, 'sigma': 1.7394, 'score': 100.4325, 'games': 14, 'win_rate': 0.0714, 'avg_rank': 2.1429}, '****': {'rank': 857, 'team': "****", 'mu': 102.6798, 'sigma': 0.7774, 'score': 100.3475, 'games': 84, 'win_rate': 0.0833, 'avg_rank': 2.0357}, '****': {'rank': 858, 'team': "****", 'mu': 102.6676, 'sigma': 0.776, 'score': 100.3395, 'games': 88, 'win_rate': 0.0341, 'avg_rank': 2.1023}, '****': {'rank': 859, 'team': "****", 'mu': 102.4366, 'sigma': 0.7051, 'score': 100.3213, 'games': 530, 'win_rate': 0.0132, 'avg_rank': 1.9434}, '****': {'rank': 860, 'team': "****", 'mu': 102.6239, 'sigma': 0.7932, 'score': 100.2443, 'games': 83, 'win_rate': 0.0241, 'avg_rank': 2.0843}, '****': {'rank': 861, 'team': "****", 'mu': 102.4541, 'sigma': 0.7447, 'score': 100.22, 'games': 200, 'win_rate': 0.04, 'avg_rank': 2.42}, '****': {'rank': 862, 'team': "****", 'mu': 102.3559, 'sigma': 0.7204, 'score': 100.1946, 'games': 339, 'win_rate': 0.0678, 'avg_rank': 2.0413}, '****': {'rank': 863, 'team': "****", 'mu': 103.9533, 'sigma': 1.2677, 'score': 100.1501, 'games': 24, 'win_rate': 0.0417, 'avg_rank': 1.875}, '****': {'rank': 864, 'team': "****", 'mu': 102.3898, 'sigma': 0.7629, 'score': 100.1013, 'games': 90, 'win_rate': 0.0222, 'avg_rank': 2.1222}, '****': {'rank': 865, 'team': "****", 'mu': 102.8471, 'sigma': 0.9166, 'score': 100.0972, 'games': 64, 'win_rate': 0.0469, 'avg_rank': 2.2812}, '****': {'rank': 866, 'team': "****", 'mu': 102.1954, 'sigma': 0.7034, 'score': 100.0853, 'games': 915, 'win_rate': 0.0536, 'avg_rank': 2.0699}, '****': {'rank': 867, 'team': "****", 'mu': 105.9107, 'sigma': 1.9448, 'score': 100.0764, 'games': 13, 'win_rate': 0.0769, 'avg_rank': 2.0769}, '****': {'rank': 868, 'team': "****", 'mu': 102.2079, 'sigma': 0.7132, 'score': 100.0682, 'games': 144, 'win_rate': 0.0486, 'avg_rank': 2.0069}, '****': {'rank': 869, 'team': "****", 'mu': 105.0988, 'sigma': 1.6924, 'score': 100.0216, 'games': 16, 'win_rate': 0.1875, 'avg_rank': 2.125}, '****': {'rank': 870, 'team': "****", 'mu': 102.1568, 'sigma': 0.7148, 'score': 100.0125, 'games': 223, 'win_rate': 0.0359, 'avg_rank': 2.2691}, '****': {'rank': 871, 'team': "****", 'mu': 102.2397, 'sigma': 0.7606, 'score': 99.9578, 'games': 97, 'win_rate': 0.0206, 'avg_rank': 2.1443}, '****': {'rank': 872, 'team': "****", 'mu': 102.0763, 'sigma': 0.7067, 'score': 99.9563, 'games': 484, 'win_rate': 0.0579, 'avg_rank': 2.1343}, '****': {'rank': 873, 'team': "****", 'mu': 102.6826, 'sigma': 0.909, 'score': 99.9557, 'games': 53, 'win_rate': 0.0943, 'avg_rank': 1.9811}, '****': {'rank': 874, 'team': "****", 'mu': 102.0812, 'sigma': 0.7122, 'score': 99.9447, 'games': 474, 'win_rate': 0.0359, 'avg_rank': 2.2363}, '****': {'rank': 875, 'team': "****", 'mu': 102.0377, 'sigma': 0.7068, 'score': 99.9173, 'games': 1767, 'win_rate': 0.0209, 'avg_rank': 2.1302}, '****': {'rank': 876, 'team': "****", 'mu': 102.1615, 'sigma': 0.7599, 'score': 99.8816, 'games': 122, 'win_rate': 0.0164, 'avg_rank': 2.3033}, '****': {'rank': 877, 'team': "****", 'mu': 102.0431, 'sigma': 0.7255, 'score': 99.8667, 'games': 340, 'win_rate': 0.0853, 'avg_rank': 1.9882}, '****': {'rank': 878, 'team': "****", 'mu': 105.6937, 'sigma': 1.9642, 'score': 99.8012, 'games': 12, 'win_rate': 0.0, 'avg_rank': 2.1667}, '****': {'rank': 879, 'team': "****", 'mu': 101.9206, 'sigma': 0.7135, 'score': 99.78, 'games': 303, 'win_rate': 0.0066, 'avg_rank': 2.2079}, '****': {'rank': 880, 'team': "****", 'mu': 102.4183, 'sigma': 0.903, 'score': 99.7095, 'games': 57, 'win_rate': 0.0526, 'avg_rank': 2.1228}, '****': {'rank': 881, 'team': "****", 'mu': 101.8549, 'sigma': 0.7257, 'score': 99.6778, 'games': 368, 'win_rate': 0.0163, 'avg_rank': 2.2527}, '****': {'rank': 882, 'team': "****", 'mu': 102.4021, 'sigma': 0.9413, 'score': 99.5782, 'games': 70, 'win_rate': 0.0, 'avg_rank': 2.3857}, '****': {'rank': 883, 'team': "****", 'mu': 103.1998, 'sigma': 1.2091, 'score': 99.5726, 'games': 32, 'win_rate': 0.0312, 'avg_rank': 2.1562}, '****': {'rank': 884, 'team': "****", 'mu': 101.7128, 'sigma': 0.7295, 'score': 99.5245, 'games': 173, 'win_rate': 0.0173, 'avg_rank': 2.2197}, '****': {'rank': 885, 'team': "****", 'mu': 102.9039, 'sigma': 1.1276, 'score': 99.5213, 'games': 39, 'win_rate': 0.0513, 'avg_rank': 2.2564}, '****': {'rank': 886, 'team': "****", 'mu': 101.6344, 'sigma': 0.7275, 'score': 99.4518, 'games': 828, 'win_rate': 0.0048, 'avg_rank': 2.3768}, 'baseline_smarter_rule_agent_v1': {'rank': 887, 'team': "****", 'mu': 101.5249, 'sigma': 0.7125, 'score': 99.3875, 'games': 2501, 'win_rate': 0.1503, 'avg_rank': 1.5506}, '****': {'rank': 888, 'team': "****", 'mu': 101.5115, 'sigma': 0.7116, 'score': 99.3768, 'games': 1920, 'win_rate': 0.0578, 'avg_rank': 2.175}, '****': {'rank': 889, 'team': "****", 'mu': 101.5493, 'sigma': 0.7424, 'score': 99.322, 'games': 126, 'win_rate': 0.0079, 'avg_rank': 2.2143}, '****': {'rank': 890, 'team': "****", 'mu': 104.0156, 'sigma': 1.5832, 'score': 99.2659, 'games': 19, 'win_rate': 0.1053, 'avg_rank': 2.1579}, '****': {'rank': 891, 'team': "****", 'mu': 101.4046, 'sigma': 0.7239, 'score': 99.2329, 'games': 359, 'win_rate': 0.0084, 'avg_rank': 2.2953}, '****': {'rank': 892, 'team': "****", 'mu': 101.3375, 'sigma': 0.7043, 'score': 99.2245, 'games': 2745, 'win_rate': 0.0138, 'avg_rank': 2.1902}, '****': {'rank': 893, 'team': "****", 'mu': 101.3367, 'sigma': 0.7285, 'score': 99.1512, 'games': 481, 'win_rate': 0.0395, 'avg_rank': 2.343}, '****': {'rank': 894, 'team': "****", 'mu': 105.0573, 'sigma': 1.9935, 'score': 99.0768, 'games': 14, 'win_rate': 0.2143, 'avg_rank': 1.8571}, '****': {'rank': 895, 'team': "****", 'mu': 101.1612, 'sigma': 0.7091, 'score': 99.0339, 'games': 398, 'win_rate': 0.0176, 'avg_rank': 2.196}, '****': {'rank': 896, 'team': "****", 'mu': 101.2994, 'sigma': 0.7563, 'score': 99.0307, 'games': 145, 'win_rate': 0.0483, 'avg_rank': 2.3517}, '****': {'rank': 897, 'team': "****", 'mu': 101.174, 'sigma': 0.7165, 'score': 99.0246, 'games': 1243, 'win_rate': 0.004, 'avg_rank': 2.255}, '****': {'rank': 898, 'team': "****", 'mu': 102.3094, 'sigma': 1.0989, 'score': 99.0127, 'games': 36, 'win_rate': 0.0, 'avg_rank': 2.1667}, '****': {'rank': 899, 'team': "****", 'mu': 101.2272, 'sigma': 0.7413, 'score': 99.0033, 'games': 245, 'win_rate': 0.0082, 'avg_rank': 2.4245}, '****': {'rank': 900, 'team': "****", 'mu': 102.7322, 'sigma': 1.2464, 'score': 98.9929, 'games': 30, 'win_rate': 0.0, 'avg_rank': 2.2333}, '****': {'rank': 901, 'team': "****", 'mu': 101.0968, 'sigma': 0.7113, 'score': 98.963, 'games': 1829, 'win_rate': 0.0077, 'avg_rank': 2.1771}, '****': {'rank': 902, 'team': "****", 'mu': 101.0944, 'sigma': 0.7184, 'score': 98.9392, 'games': 376, 'win_rate': 0.0027, 'avg_rank': 2.3191}, '****': {'rank': 903, 'team': "****", 'mu': 103.4327, 'sigma': 1.5023, 'score': 98.9257, 'games': 22, 'win_rate': 0.0, 'avg_rank': 2.4091}, '****': {'rank': 904, 'team': "****", 'mu': 103.1576, 'sigma': 1.4108, 'score': 98.925, 'games': 20, 'win_rate': 0.25, 'avg_rank': 1.55}, '****': {'rank': 905, 'team': "****", 'mu': 101.0804, 'sigma': 0.7213, 'score': 98.9166, 'games': 769, 'win_rate': 0.0078, 'avg_rank': 2.225}, '****': {'rank': 906, 'team': "****", 'mu': 100.9615, 'sigma': 0.7151, 'score': 98.8163, 'games': 304, 'win_rate': 0.0296, 'avg_rank': 2.227}, '****': {'rank': 907, 'team': "****", 'mu': 101.0137, 'sigma': 0.7433, 'score': 98.7838, 'games': 368, 'win_rate': 0.0299, 'avg_rank': 2.4891}, '****': {'rank': 908, 'team': "****", 'mu': 100.9667, 'sigma': 0.7285, 'score': 98.7811, 'games': 333, 'win_rate': 0.009, 'avg_rank': 2.3213}, '****': {'rank': 909, 'team': "****", 'mu': 100.9585, 'sigma': 0.7293, 'score': 98.7707, 'games': 258, 'win_rate': 0.0116, 'avg_rank': 2.2519}, '****': {'rank': 910, 'team': "****", 'mu': 100.8878, 'sigma': 0.7088, 'score': 98.7615, 'games': 465, 'win_rate': 0.0366, 'avg_rank': 2.2796}, '****': {'rank': 911, 'team': "****", 'mu': 100.8462, 'sigma': 0.7078, 'score': 98.7226, 'games': 986, 'win_rate': 0.0193, 'avg_rank': 2.2201}, '****': {'rank': 912, 'team': "****", 'mu': 101.4704, 'sigma': 0.924, 'score': 98.6985, 'games': 82, 'win_rate': 0.0122, 'avg_rank': 2.5488}, 'baseline_genius_rule_agent_v1': {'rank': 913, 'team': "****", 'mu': 100.75, 'sigma': 0.7126, 'score': 98.6123, 'games': 2516, 'win_rate': 0.2289, 'avg_rank': 1.4583}, '****': {'rank': 914, 'team': "****", 'mu': 100.7065, 'sigma': 0.7188, 'score': 98.5502, 'games': 226, 'win_rate': 0.0044, 'avg_rank': 2.2699}, '****': {'rank': 915, 'team': "****", 'mu': 100.7118, 'sigma': 0.7323, 'score': 98.5149, 'games': 233, 'win_rate': 0.0, 'avg_rank': 2.309}, '****': {'rank': 916, 'team': "****", 'mu': 100.664, 'sigma': 0.7339, 'score': 98.4623, 'games': 576, 'win_rate': 0.0069, 'avg_rank': 2.4149}, '****': {'rank': 917, 'team': "****", 'mu': 100.5865, 'sigma': 0.7193, 'score': 98.4286, 'games': 930, 'win_rate': 0.0032, 'avg_rank': 2.3194}, '****': {'rank': 918, 'team': "****", 'mu': 102.3661, 'sigma': 1.3141, 'score': 98.4238, 'games': 22, 'win_rate': 0.0455, 'avg_rank': 1.9545}, '****': {'rank': 919, 'team': "****", 'mu': 101.8771, 'sigma': 1.1686, 'score': 98.3713, 'games': 33, 'win_rate': 0.1212, 'avg_rank': 2.0909}, '****': {'rank': 920, 'team': "****", 'mu': 100.5222, 'sigma': 0.7238, 'score': 98.3508, 'games': 198, 'win_rate': 0.0051, 'avg_rank': 2.2879}, '****': {'rank': 921, 'team': "****", 'mu': 100.872, 'sigma': 0.8781, 'score': 98.2378, 'games': 73, 'win_rate': 0.0, 'avg_rank': 2.3699}, '****': {'rank': 922, 'team': "****", 'mu': 100.3663, 'sigma': 0.7312, 'score': 98.1726, 'games': 222, 'win_rate': 0.009, 'avg_rank': 2.2928}, '****': {'rank': 923, 'team': "****", 'mu': 100.4279, 'sigma': 0.769, 'score': 98.1209, 'games': 252, 'win_rate': 0.0159, 'avg_rank': 2.6071}, 'baseline_box_farmer_agent_v1': {'rank': 924, 'team': "****", 'mu': 100.2128, 'sigma': 0.731, 'score': 98.0198, 'games': 2612, 'win_rate': 0.0758, 'avg_rank': 1.9698}, '****': {'rank': 925, 'team': "****", 'mu': 100.1765, 'sigma': 0.7216, 'score': 98.0118, 'games': 489, 'win_rate': 0.0041, 'avg_rank': 2.3149}, '****': {'rank': 926, 'team': "****", 'mu': 100.8498, 'sigma': 0.9537, 'score': 97.9885, 'games': 61, 'win_rate': 0.0, 'avg_rank': 2.3934}, '****': {'rank': 927, 'team': "****", 'mu': 100.1151, 'sigma': 0.7197, 'score': 97.956, 'games': 473, 'win_rate': 0.0021, 'avg_rank': 2.2939}, '****': {'rank': 928, 'team': "****", 'mu': 100.9877, 'sigma': 1.0118, 'score': 97.9523, 'games': 60, 'win_rate': 0.0, 'avg_rank': 2.5}, '****': {'rank': 929, 'team': "****", 'mu': 100.0322, 'sigma': 0.7228, 'score': 97.8638, 'games': 1173, 'win_rate': 0.0128, 'avg_rank': 2.2651}, '****': {'rank': 930, 'team': "****", 'mu': 102.0986, 'sigma': 1.4134, 'score': 97.8585, 'games': 26, 'win_rate': 0.0, 'avg_rank': 2.3462}, '****': {'rank': 931, 'team': "****", 'mu': 101.757, 'sigma': 1.3145, 'score': 97.8134, 'games': 22, 'win_rate': 0.2273, 'avg_rank': 1.5}, '****': {'rank': 932, 'team': "****", 'mu': 103.3835, 'sigma': 1.8601, 'score': 97.8032, 'games': 13, 'win_rate': 0.0, 'avg_rank': 2.3846}, '****': {'rank': 933, 'team': "****", 'mu': 100.07, 'sigma': 0.7727, 'score': 97.7519, 'games': 160, 'win_rate': 0.0, 'avg_rank': 2.4813}, '****': {'rank': 934, 'team': "****", 'mu': 99.9603, 'sigma': 0.7441, 'score': 97.7279, 'games': 196, 'win_rate': 0.0051, 'avg_rank': 2.5}, '****': {'rank': 935, 'team': "****", 'mu': 100.2542, 'sigma': 0.854, 'score': 97.6922, 'games': 141, 'win_rate': 0.0071, 'avg_rank': 2.5532}, '****': {'rank': 936, 'team': "****", 'mu': 99.8631, 'sigma': 0.7258, 'score': 97.6856, 'games': 2346, 'win_rate': 0.0038, 'avg_rank': 2.3269}, '****': {'rank': 937, 'team': "****", 'mu': 103.5771, 'sigma': 1.9815, 'score': 97.6327, 'games': 11, 'win_rate': 0.0, 'avg_rank': 2.0}, '****': {'rank': 938, 'team': "****", 'mu': 99.8674, 'sigma': 0.745, 'score': 97.6323, 'games': 199, 'win_rate': 0.0101, 'avg_rank': 2.4221}, '****': {'rank': 939, 'team': "****", 'mu': 104.316, 'sigma': 2.274, 'score': 97.494, 'games': 9, 'win_rate': 0.1111, 'avg_rank': 2.0}, '****': {'rank': 940, 'team': "****", 'mu': 101.4955, 'sigma': 1.3344, 'score': 97.4924, 'games': 26, 'win_rate': 0.0, 'avg_rank': 2.3462}, '****': {'rank': 941, 'team': "****", 'mu': 99.6742, 'sigma': 0.73, 'score': 97.4843, 'games': 2524, 'win_rate': 0.0075, 'avg_rank': 2.2532}, '****': {'rank': 942, 'team': "****", 'mu': 99.6718, 'sigma': 0.7558, 'score': 97.4043, 'games': 1267, 'win_rate': 0.0032, 'avg_rank': 2.4041}, '****': {'rank': 943, 'team': "****", 'mu': 102.6114, 'sigma': 1.7539, 'score': 97.3496, 'games': 16, 'win_rate': 0.0625, 'avg_rank': 2.25}, '****': {'rank': 944, 'team': "****", 'mu': 99.5914, 'sigma': 0.748, 'score': 97.3475, 'games': 605, 'win_rate': 0.0, 'avg_rank': 2.4017}, '****': {'rank': 945, 'team': "****", 'mu': 99.493, 'sigma': 0.73, 'score': 97.303, 'games': 921, 'win_rate': 0.0098, 'avg_rank': 2.316}, '****': {'rank': 946, 'team': "****", 'mu': 99.4767, 'sigma': 0.7283, 'score': 97.2919, 'games': 739, 'win_rate': 0.0149, 'avg_rank': 2.3018}, '****': {'rank': 947, 'team': "****", 'mu': 99.5147, 'sigma': 0.7609, 'score': 97.2319, 'games': 165, 'win_rate': 0.0, 'avg_rank': 2.5455}, '****': {'rank': 948, 'team': "****", 'mu': 99.4217, 'sigma': 0.7348, 'score': 97.2172, 'games': 1649, 'win_rate': 0.0042, 'avg_rank': 2.2826}, '****': {'rank': 949, 'team': "****", 'mu': 99.9107, 'sigma': 0.9052, 'score': 97.1951, 'games': 73, 'win_rate': 0.0274, 'avg_rank': 2.4247}, '****': {'rank': 950, 'team': "****", 'mu': 99.5293, 'sigma': 0.7783, 'score': 97.1944, 'games': 396, 'win_rate': 0.0177, 'avg_rank': 2.6313}, '****': {'rank': 951, 'team': "****", 'mu': 99.299, 'sigma': 0.7257, 'score': 97.1218, 'games': 919, 'win_rate': 0.0033, 'avg_rank': 2.4113}, '****': {'rank': 952, 'team': "****", 'mu': 102.3561, 'sigma': 1.7494, 'score': 97.108, 'games': 15, 'win_rate': 0.1333, 'avg_rank': 1.8667}, '****': {'rank': 953, 'team': "****", 'mu': 99.641, 'sigma': 0.8579, 'score': 97.0673, 'games': 84, 'win_rate': 0.0119, 'avg_rank': 2.4167}, '****': {'rank': 954, 'team': "****", 'mu': 100.1325, 'sigma': 1.0221, 'score': 97.0661, 'games': 57, 'win_rate': 0.0, 'avg_rank': 2.5614}, '****': {'rank': 955, 'team': "****", 'mu': 99.2794, 'sigma': 0.7382, 'score': 97.0647, 'games': 882, 'win_rate': 0.0159, 'avg_rank': 2.4478}, '****': {'rank': 956, 'team': "****", 'mu': 99.9623, 'sigma': 0.978, 'score': 97.0284, 'games': 51, 'win_rate': 0.0392, 'avg_rank': 2.4118}, '****': {'rank': 957, 'team': "****", 'mu': 99.39, 'sigma': 0.7922, 'score': 97.0133, 'games': 170, 'win_rate': 0.0, 'avg_rank': 2.6176}, '****': {'rank': 958, 'team': "****", 'mu': 100.7302, 'sigma': 1.2554, 'score': 96.964, 'games': 29, 'win_rate': 0.0345, 'avg_rank': 2.3103}, '****': {'rank': 959, 'team': "****", 'mu': 102.0745, 'sigma': 1.7117, 'score': 96.9395, 'games': 13, 'win_rate': 0.0, 'avg_rank': 2.3846}, '****': {'rank': 960, 'team': "****", 'mu': 103.6171, 'sigma': 2.2442, 'score': 96.8845, 'games': 14, 'win_rate': 0.0, 'avg_rank': 2.4286}, '****': {'rank': 961, 'team': "****", 'mu': 98.9478, 'sigma': 0.7281, 'score': 96.7636, 'games': 976, 'win_rate': 0.0174, 'avg_rank': 2.3371}, '****': {'rank': 962, 'team': "****", 'mu': 98.8887, 'sigma': 0.7447, 'score': 96.6547, 'games': 1988, 'win_rate': 0.003, 'avg_rank': 2.4582}, '****': {'rank': 963, 'team': "****", 'mu': 99.579, 'sigma': 0.9862, 'score': 96.6205, 'games': 76, 'win_rate': 0.0, 'avg_rank': 2.5526}, '****': {'rank': 964, 'team': "****", 'mu': 99.4089, 'sigma': 0.935, 'score': 96.6038, 'games': 85, 'win_rate': 0.0, 'avg_rank': 2.5882}, '****': {'rank': 965, 'team': "****", 'mu': 98.7967, 'sigma': 0.7321, 'score': 96.6004, 'games': 4409, 'win_rate': 0.0336, 'avg_rank': 2.169}, 'baseline_simple_rule_agent_v1': {'rank': 966, 'team': "****", 'mu': 98.8021, 'sigma': 0.7554, 'score': 96.5359, 'games': 2559, 'win_rate': 0.0539, 'avg_rank': 2.041}, '****': {'rank': 967, 'team': "****", 'mu': 98.7706, 'sigma': 0.7504, 'score': 96.5194, 'games': 339, 'win_rate': 0.0177, 'avg_rank': 2.5398}, '****': {'rank': 968, 'team': "****", 'mu': 98.9209, 'sigma': 0.8015, 'score': 96.5164, 'games': 121, 'win_rate': 0.0165, 'avg_rank': 2.5041}, '****': {'rank': 969, 'team': "****", 'mu': 99.0404, 'sigma': 0.9094, 'score': 96.312, 'games': 110, 'win_rate': 0.0273, 'avg_rank': 2.6091}, '****': {'rank': 970, 'team': "****", 'mu': 102.6973, 'sigma': 2.1292, 'score': 96.3097, 'games': 14, 'win_rate': 0.0, 'avg_rank': 2.5}, '****': {'rank': 971, 'team': "****", 'mu': 98.914, 'sigma': 0.8768, 'score': 96.2835, 'games': 107, 'win_rate': 0.0, 'avg_rank': 2.6262}, '****': {'rank': 972, 'team': "****", 'mu': 98.9332, 'sigma': 0.8987, 'score': 96.2371, 'games': 75, 'win_rate': 0.0133, 'avg_rank': 2.4267}, '****': {'rank': 973, 'team': "****", 'mu': 98.7257, 'sigma': 0.879, 'score': 96.0887, 'games': 81, 'win_rate': 0.0, 'avg_rank': 2.3951}, '****': {'rank': 974, 'team': "****", 'mu': 98.5046, 'sigma': 0.8192, 'score': 96.047, 'games': 107, 'win_rate': 0.0093, 'avg_rank': 2.514}, '****': {'rank': 975, 'team': "****", 'mu': 98.6128, 'sigma': 0.8632, 'score': 96.0231, 'games': 96, 'win_rate': 0.0, 'avg_rank': 2.5}, '****': {'rank': 976, 'team': "****", 'mu': 98.2369, 'sigma': 0.754, 'score': 95.9749, 'games': 1914, 'win_rate': 0.0063, 'avg_rank': 2.4624}, '****': {'rank': 977, 'team': "****", 'mu': 102.0705, 'sigma': 2.0389, 'score': 95.9538, 'games': 14, 'win_rate': 0.2143, 'avg_rank': 2.2143}, '****': {'rank': 978, 'team': "****", 'mu': 98.1885, 'sigma': 0.7738, 'score': 95.8671, 'games': 1367, 'win_rate': 0.0241, 'avg_rank': 2.5991}, '****': {'rank': 979, 'team': "****", 'mu': 101.7401, 'sigma': 1.9606, 'score': 95.8581, 'games': 19, 'win_rate': 0.0, 'avg_rank': 2.6842}, '****': {'rank': 980, 'team': "****", 'mu': 98.089, 'sigma': 0.7629, 'score': 95.8004, 'games': 457, 'win_rate': 0.0088, 'avg_rank': 2.5252}, '****': {'rank': 981, 'team': "****", 'mu': 102.1755, 'sigma': 2.1285, 'score': 95.79, 'games': 13, 'win_rate': 0.0, 'avg_rank': 2.3846}, '****': {'rank': 982, 'team': "****", 'mu': 98.7917, 'sigma': 1.0527, 'score': 95.6337, 'games': 95, 'win_rate': 0.0, 'avg_rank': 2.6842}, '****': {'rank': 983, 'team': "****", 'mu': 101.4467, 'sigma': 1.954, 'score': 95.5847, 'games': 12, 'win_rate': 0.0, 'avg_rank': 2.1667}, '****': {'rank': 984, 'team': "****", 'mu': 98.3529, 'sigma': 0.9339, 'score': 95.5512, 'games': 75, 'win_rate': 0.0133, 'avg_rank': 2.5067}, '****': {'rank': 985, 'team': "****", 'mu': 98.0258, 'sigma': 0.8527, 'score': 95.4677, 'games': 224, 'win_rate': 0.0089, 'avg_rank': 2.5893}, '****': {'rank': 986, 'team': "****", 'mu': 97.651, 'sigma': 0.7602, 'score': 95.3703, 'games': 813, 'win_rate': 0.0062, 'avg_rank': 2.4674}, '****': {'rank': 987, 'team': "****", 'mu': 97.6114, 'sigma': 0.7592, 'score': 95.3337, 'games': 2835, 'win_rate': 0.0071, 'avg_rank': 2.3933}, '****': {'rank': 988, 'team': "****", 'mu': 97.669, 'sigma': 0.7797, 'score': 95.33, 'games': 174, 'win_rate': 0.0057, 'avg_rank': 2.4885}, '****': {'rank': 989, 'team': "****", 'mu': 97.6127, 'sigma': 0.7623, 'score': 95.3257, 'games': 340, 'win_rate': 0.0088, 'avg_rank': 2.5941}, '****': {'rank': 990, 'team': "****", 'mu': 98.8015, 'sigma': 1.1877, 'score': 95.2383, 'games': 36, 'win_rate': 0.0556, 'avg_rank': 2.3056}, '****': {'rank': 991, 'team': "****", 'mu': 98.9301, 'sigma': 1.2401, 'score': 95.2098, 'games': 63, 'win_rate': 0.0, 'avg_rank': 2.6984}, '****': {'rank': 992, 'team': "****", 'mu': 97.6035, 'sigma': 0.8217, 'score': 95.1385, 'games': 150, 'win_rate': 0.0333, 'avg_rank': 2.5667}, '****': {'rank': 993, 'team': "****", 'mu': 97.5023, 'sigma': 0.788, 'score': 95.1383, 'games': 344, 'win_rate': 0.0029, 'avg_rank': 2.6628}, '****': {'rank': 994, 'team': "****", 'mu': 100.2838, 'sigma': 1.7207, 'score': 95.1218, 'games': 25, 'win_rate': 0.0, 'avg_rank': 2.64}, '****': {'rank': 995, 'team': "****", 'mu': 97.2935, 'sigma': 0.7677, 'score': 94.9905, 'games': 939, 'win_rate': 0.0192, 'avg_rank': 2.5911}, '****': {'rank': 996, 'team': "****", 'mu': 97.3238, 'sigma': 0.778, 'score': 94.9898, 'games': 1449, 'win_rate': 0.0186, 'avg_rank': 2.6059}, '****': {'rank': 997, 'team': "****", 'mu': 97.3052, 'sigma': 0.7829, 'score': 94.9565, 'games': 889, 'win_rate': 0.0169, 'avg_rank': 2.6198}, '****': {'rank': 998, 'team': "****", 'mu': 97.888, 'sigma': 0.9803, 'score': 94.947, 'games': 146, 'win_rate': 0.0, 'avg_rank': 2.6507}, '****': {'rank': 999, 'team': "****", 'mu': 97.0398, 'sigma': 0.7871, 'score': 94.6786, 'games': 2032, 'win_rate': 0.0192, 'avg_rank': 2.6097}, '****': {'rank': 1000, 'team': "****", 'mu': 97.1757, 'sigma': 0.9249, 'score': 94.4009, 'games': 131, 'win_rate': 0.0, 'avg_rank': 2.7023}, '****': {'rank': 1001, 'team': "****", 'mu': 96.7236, 'sigma': 0.7772, 'score': 94.3919, 'games': 1865, 'win_rate': 0.0097, 'avg_rank': 2.6155}, '****': {'rank': 1002, 'team': "****", 'mu': 96.7753, 'sigma': 0.7995, 'score': 94.3768, 'games': 428, 'win_rate': 0.0164, 'avg_rank': 2.6215}, '****': {'rank': 1003, 'team': "****", 'mu': 97.0582, 'sigma': 0.9097, 'score': 94.3292, 'games': 145, 'win_rate': 0.0, 'avg_rank': 2.7172}, '****': {'rank': 1004, 'team': "****", 'mu': 97.4411, 'sigma': 1.0993, 'score': 94.1431, 'games': 57, 'win_rate': 0.0, 'avg_rank': 2.4737}, '****': {'rank': 1005, 'team': "****", 'mu': 96.2809, 'sigma': 0.7876, 'score': 93.9183, 'games': 606, 'win_rate': 0.0066, 'avg_rank': 2.599}, '****': {'rank': 1006, 'team': "****", 'mu': 96.3367, 'sigma': 0.8077, 'score': 93.9135, 'games': 304, 'win_rate': 0.0, 'avg_rank': 2.6579}, '****': {'rank': 1007, 'team': "****", 'mu': 98.5822, 'sigma': 1.5589, 'score': 93.9057, 'games': 25, 'win_rate': 0.04, 'avg_rank': 2.52}, '****': {'rank': 1008, 'team': "****", 'mu': 97.0537, 'sigma': 1.0881, 'score': 93.7893, 'games': 63, 'win_rate': 0.0, 'avg_rank': 2.4762}, '****': {'rank': 1009, 'team': "****", 'mu': 99.8671, 'sigma': 2.0684, 'score': 93.6619, 'games': 17, 'win_rate': 0.0588, 'avg_rank': 2.5294}, '****': {'rank': 1010, 'team': "****", 'mu': 100.0479, 'sigma': 2.2342, 'score': 93.3453, 'games': 13, 'win_rate': 0.0, 'avg_rank': 2.4615}, '****': {'rank': 1011, 'team': "****", 'mu': 95.442, 'sigma': 0.7839, 'score': 93.0902, 'games': 902, 'win_rate': 0.0089, 'avg_rank': 2.6098}, '****': {'rank': 1012, 'team': "****", 'mu': 100.5608, 'sigma': 2.5221, 'score': 92.9945, 'games': 11, 'win_rate': 0.0, 'avg_rank': 2.5455}, '****': {'rank': 1013, 'team': "****", 'mu': 99.643, 'sigma': 2.2269, 'score': 92.9623, 'games': 12, 'win_rate': 0.0, 'avg_rank': 2.4167}, '****': {'rank': 1014, 'team': "****", 'mu': 95.2646, 'sigma': 0.7978, 'score': 92.871, 'games': 2329, 'win_rate': 0.0167, 'avg_rank': 2.596}, '****': {'rank': 1015, 'team': "****", 'mu': 97.5784, 'sigma': 1.5851, 'score': 92.8231, 'games': 27, 'win_rate': 0.037, 'avg_rank': 2.4074}, '****': {'rank': 1016, 'team': "****", 'mu': 98.6107, 'sigma': 1.933, 'score': 92.8117, 'games': 14, 'win_rate': 0.0, 'avg_rank': 2.4286}, '****': {'rank': 1017, 'team': "****", 'mu': 99.0781, 'sigma': 2.0896, 'score': 92.8093, 'games': 16, 'win_rate': 0.0, 'avg_rank': 2.6875}, '****': {'rank': 1018, 'team': "****", 'mu': 95.8775, 'sigma': 1.0754, 'score': 92.6513, 'games': 74, 'win_rate': 0.0, 'avg_rank': 2.6216}, '****': {'rank': 1019, 'team': "****", 'mu': 95.068, 'sigma': 0.8121, 'score': 92.6316, 'games': 532, 'win_rate': 0.0113, 'avg_rank': 2.656}, '****': {'rank': 1020, 'team': "****", 'mu': 98.6374, 'sigma': 2.046, 'score': 92.4994, 'games': 19, 'win_rate': 0.0, 'avg_rank': 2.5263}, '****': {'rank': 1021, 'team': "****", 'mu': 97.6251, 'sigma': 1.7171, 'score': 92.4737, 'games': 43, 'win_rate': 0.0, 'avg_rank': 2.7442}, '****': {'rank': 1022, 'team': "****", 'mu': 95.6473, 'sigma': 1.1261, 'score': 92.2692, 'games': 89, 'win_rate': 0.0, 'avg_rank': 2.7079}, '****': {'rank': 1023, 'team': "****", 'mu': 94.9178, 'sigma': 0.9659, 'score': 92.0202, 'games': 119, 'win_rate': 0.0, 'avg_rank': 2.6639}, '****': {'rank': 1024, 'team': "****", 'mu': 94.1755, 'sigma': 0.8223, 'score': 91.7085, 'games': 867, 'win_rate': 0.0046, 'avg_rank': 2.714}, '****': {'rank': 1025, 'team': "****", 'mu': 94.3105, 'sigma': 0.9806, 'score': 91.3687, 'games': 854, 'win_rate': 0.0, 'avg_rank': 2.8044}, '****': {'rank': 1026, 'team': "****", 'mu': 94.071, 'sigma': 0.9357, 'score': 91.264, 'games': 119, 'win_rate': 0.0, 'avg_rank': 2.6975}, '****': {'rank': 1027, 'team': "****", 'mu': 93.8279, 'sigma': 0.8633, 'score': 91.2381, 'games': 847, 'win_rate': 0.0047, 'avg_rank': 2.7403}, '****': {'rank': 1028, 'team': "****", 'mu': 97.3081, 'sigma': 2.071, 'score': 91.0952, 'games': 16, 'win_rate': 0.0, 'avg_rank': 2.5}, '****': {'rank': 1029, 'team': "****", 'mu': 94.6594, 'sigma': 1.2223, 'score': 90.9926, 'games': 67, 'win_rate': 0.0, 'avg_rank': 2.5522}, '****': {'rank': 1030, 'team': "****", 'mu': 94.1207, 'sigma': 1.0436, 'score': 90.99, 'games': 152, 'win_rate': 0.0, 'avg_rank': 2.7697}, '****': {'rank': 1031, 'team': "****", 'mu': 95.6516, 'sigma': 1.6978, 'score': 90.5583, 'games': 22, 'win_rate': 0.0, 'avg_rank': 2.6364}, '****': {'rank': 1032, 'team': "****", 'mu': 95.8759, 'sigma': 1.9068, 'score': 90.1554, 'games': 26, 'win_rate': 0.0, 'avg_rank': 2.8462}, '****': {'rank': 1033, 'team': "****", 'mu': 94.0329, 'sigma': 1.3384, 'score': 90.0177, 'games': 101, 'win_rate': 0.0, 'avg_rank': 2.6436}, '****': {'rank': 1034, 'team': "****", 'mu': 92.5542, 'sigma': 0.9477, 'score': 89.711, 'games': 133, 'win_rate': 0.0, 'avg_rank': 2.7068}, '****': {'rank': 1035, 'team': "****", 'mu': 93.5475, 'sigma': 1.4293, 'score': 89.2595, 'games': 78, 'win_rate': 0.0, 'avg_rank': 2.8333}, '****': {'rank': 1036, 'team': "****", 'mu': 91.2533, 'sigma': 0.9444, 'score': 88.4201, 'games': 993, 'win_rate': 0.0, 'avg_rank': 2.8489}, '****': {'rank': 1037, 'team': "****", 'mu': 90.7581, 'sigma': 1.0568, 'score': 87.5878, 'games': 444, 'win_rate': 0.0, 'avg_rank': 2.8514}, 'baseline_random_agent_v1': {'rank': 1038, 'team': "****", 'mu': 90.4548, 'sigma': 0.9674, 'score': 87.5525, 'games': 2609, 'win_rate': 0.0004, 'avg_rank': 2.775}, '****': {'rank': 1039, 'team': "****", 'mu': 94.6502, 'sigma': 2.5103, 'score': 87.1193, 'games': 18, 'win_rate': 0.0, 'avg_rank': 2.7222}, '****': {'rank': 1040, 'team': "****", 'mu': 89.9434, 'sigma': 1.0065, 'score': 86.9238, 'games': 2315, 'win_rate': 0.0, 'avg_rank': 2.8376}, '****': {'rank': 1041, 'team': "****", 'mu': 91.6528, 'sigma': 2.0321, 'score': 85.5566, 'games': 54, 'win_rate': 0.0, 'avg_rank': 2.7593}, '****': {'rank': 1042, 'team': "****", 'mu': 87.2714, 'sigma': 1.1356, 'score': 83.8648, 'games': 2917, 'win_rate': 0.0, 'avg_rank': 2.8512}, '****': {'rank': 1043, 'team': "****", 'mu': 93.3236, 'sigma': 3.2422, 'score': 83.5969, 'games': 15, 'win_rate': 0.0, 'avg_rank': 2.7333}, '****': {'rank': 1044, 'team': "****", 'mu': 93.8261, 'sigma': 3.7502, 'score': 82.5755, 'games': 14, 'win_rate': 0.0, 'avg_rank': 2.7857}, '****': {'rank': 1045, 'team': "****", 'mu': 89.3817, 'sigma': 3.8008, 'score': 77.9795, 'games': 21, 'win_rate': 0.0, 'avg_rank': 2.8095}, '****': {'rank': 1046, 'team': "****", 'mu': 81.708, 'sigma': 1.8961, 'score': 76.0198, 'games': 213, 'win_rate': 0.0, 'avg_rank': 2.8873}, '****': {'rank': 1047, 'team': "****", 'mu': 84.29, 'sigma': 2.8554, 'score': 75.7237, 'games': 52, 'win_rate': 0.0, 'avg_rank': 2.9038}, '****': {'rank': 1048, 'team': "****", 'mu': 83.5792, 'sigma': 3.3908, 'score': 73.4069, 'games': 16, 'win_rate': 0.0, 'avg_rank': 2.75}, '****': {'rank': 1049, 'team': "****", 'mu': 79.0416, 'sigma': 3.5463, 'score': 68.4026, 'games': 5, 'win_rate': 0.0, 'avg_rank': 2.0}, '****': {'rank': 1050, 'team': "****", 'mu': 25.0, 'sigma': 8.333, 'score': 0.001, 'games': 0, 'win_rate': 0.0, 'avg_rank': 0.0}, '****': {'rank': 1051, 'team': "****", 'mu': 25.0, 'sigma': 8.333, 'score': 0.001, 'games': 0, 'win_rate': 0.0, 'avg_rank': 0.0}}

SEQUENCE_LENGTH = 32
SEQUENCE_STRIDE = 16
SEQUENCE_BURN_IN = 8
SEQUENCE_BATCH_SIZE = 6 if V12_CPU_MODE else 8
SEQUENCE_BC_EPOCHS_V12 = 1 if V12_CPU_MODE else 3
ELITE_CLONE_COUNT = 3 if V12_CPU_MODE else 8
ELITE_CLONE_EPOCHS = 1 if V12_CPU_MODE else 2
MAX_SEQUENCES = 1600 if V12_CPU_MODE else 7000
MAX_ELITE_REPLAY_FRAMES = 32_000 if V12_CPU_MODE else 130_000
VALIDATE_EVERY = 4 if V12_CPU_MODE else 6
VALIDATION_MATCHES = 16 if V12_CPU_MODE else 40
FINAL_EVAL_MATCHES = 40 if V12_CPU_MODE else 112
TARGET_KL = 0.008
PLANNER_QUERY_PROB = 0.24 if V12_CPU_MODE else 0.30
PLANNER_DISTILL_COEF = 0.045
ELITE_BC_COEF = 0.035
ELITE_BC_BATCH = 128 if V12_CPU_MODE else 256

V12_CFG = TrainConfig(
    "v12_rankdrop_continuation_cpu" if V12_CPU_MODE else "v12_rankdrop_continuation_t4",
    SEED,
    num_envs=6 if V12_CPU_MODE else 18,
    episodes_per_update=8 if V12_CPU_MODE else 22,
    bc_teacher_steps=0,
    bc_chunk_size=4096,
    bc_epochs=1,
    max_steps=500,
    gamma=0.995,
    gae_lambda=0.95,
    learning_rate=1.6e-5 if V12_CPU_MODE else 2.8e-5,
    clip_coef=0.10,
    ent_coef=0.011,
    vf_coef=0.55,
    max_grad_norm=0.45,
    ppo_epochs=1 if V12_CPU_MODE else 2,
    minibatch_size=512 if V12_CPU_MODE else 1024,
    eval_matches=FINAL_EVAL_MATCHES,
    checkpoint_every=2 if V12_CPU_MODE else 4,
    league_snapshot_every=2 if V12_CPU_MODE else 4,
    league_max_size=6 if V12_CPU_MODE else 10,
)

V12_CURRICULUM = (
    [
        Stage("cpu_sequence_guard", 4, 1, 1, ("genius", "tactical"), "robust_v4", 0.72),
        Stage("cpu_rank_protect", 4, 1, 1, ("genius", "tactical"), "late_v4", 0.78),
    ]
    if V12_CPU_MODE
    else [
        Stage("sequence_guard", 8, 1, 1, ("smarter", "genius", "tactical"), "balanced_v4", 0.68),
        Stage("top8_pfsp", 14, 1, 1, ("genius", "tactical"), "competitive_v4", 0.82),
        Stage("longtail_rank_protect", 12, 1, 1, ("genius", "tactical"), "robust_v4", 0.88),
        Stage("late_tiebreak_guard", 8, 1, 1, ("genius", "tactical"), "late_v4", 0.86),
    ]
)
STAGE_TARGET_RANK.update({
    "cpu_sequence_guard": 0.58,
    "cpu_rank_protect": 0.56,
    "sequence_guard": 0.50,
    "top8_pfsp": 0.50,
    "longtail_rank_protect": 0.48,
    "late_tiebreak_guard": 0.48,
})
STAGE_EXTRA_CAP.update({
    "cpu_sequence_guard": 1,
    "cpu_rank_protect": 1,
    "sequence_guard": 2,
    "top8_pfsp": 3,
    "longtail_rank_protect": 3,
    "late_tiebreak_guard": 2,
})


def _v12_info(sid):
    return V12_LEADERBOARD_INFO.get(str(sid), {"rank": 999, "team": str(sid), "score": 0.0})


def _v12_read_inputs():
    payloads = {}
    manifest = None
    leaderboard_csv = None
    for path in Path("/kaggle/input").rglob("*"):
        if not path.is_file():
            continue
        lower = path.name.lower()
        if lower.endswith(".json"):
            try:
                raw = path.read_bytes()
                data = json.loads(raw)
                if "history" in data and "team_ids" in data:
                    payloads[str(path.stem)] = raw
                elif path.name == "rankdrop_top8_manifest.json":
                    manifest = data
            except Exception:
                pass
        elif lower == "current_leaderboard_live.csv":
            try:
                leaderboard_csv = path.read_text(encoding="utf-8")
            except Exception:
                pass
        elif lower.endswith(".zip"):
            try:
                with zipfile.ZipFile(path) as archive:
                    for name in archive.namelist():
                        lname = name.lower()
                        if lname.endswith(".json"):
                            raw = archive.read(name)
                            try:
                                data = json.loads(raw)
                            except Exception:
                                continue
                            if "history" in data and "team_ids" in data:
                                payloads[str(data.get("match_id", Path(name).stem))] = raw
                            elif name.endswith("rankdrop_top8_manifest.json"):
                                manifest = data
                        elif lname.endswith("current_leaderboard_live.csv"):
                            leaderboard_csv = archive.read(name).decode("utf-8", errors="replace")
            except Exception:
                pass
    return payloads, manifest, leaderboard_csv


def fetch_rankdrop_logs_v12(log_dir=V12_LOG_DIR):
    import pandas as pd
    import io

    log_dir = Path(log_dir)
    json_dir = log_dir / "json"
    json_dir.mkdir(parents=True, exist_ok=True)
    payloads, manifest, leaderboard_csv = _v12_read_inputs()
    if not payloads:
        raise RuntimeError("Không tìm thấy elite match JSON logs. Hãy attach elite-log training archive.")
    if leaderboard_csv:
        lb = pd.read_csv(io.StringIO(leaderboard_csv))
        info = {}
        for _, row in lb.iterrows():
            info[str(row["Submission ID"])] = {
                "rank": int(row["Rank"]),
                "team": str(row["Team"]),
                "score": float(row["Score"]),
            }
    else:
        info = V12_LEADERBOARD_INFO

    rows = []
    for match_id, raw in payloads.items():
        path = json_dir / f"{match_id}.json"
        if not path.exists():
            path.write_bytes(raw)
        data = json.loads(raw)
        ids = [str(x) for x in data["team_ids"]]
        ranks = [int(x) for x in data["ranks"]]
        has_target = any(sid in V12_TARGET_IDS for sid in ids)
        target_lost = any(sid in V12_TARGET_IDS and ranks[i] > 0 for i, sid in enumerate(ids))
        for slot, sid in enumerate(ids):
            if str(sid).startswith("baseline_"):
                continue
            reason = None
            train_rank = int(info.get(sid, {}).get("rank", 999))
            team = str(info.get(sid, {}).get("team", sid))
            weight_group = "other"
            if sid in V12_TOP8_IDS:
                reason = "top8_teacher"
                weight_group = "top8"
            elif has_target and target_lost and ranks[slot] == 0 and sid not in V12_TARGET_IDS:
                reason = "counter_winner"
                weight_group = "counter"
                train_rank = min(train_rank, 20)
            elif sid == "****" and ranks[slot] == 0:
                reason = "own_best_winner"
                weight_group = "own_best"
                train_rank = min(train_rank, 30)
            elif sid in V12_TARGET_IDS and ranks[slot] == 0 and sid != "****":
                reason = "own_winner"
                weight_group = "own"
                train_rank = min(train_rank, 45)
            if reason is None:
                continue
            rows.append({
                "rank": int(train_rank),
                "actual_rank": int(info.get(sid, {}).get("rank", 999)),
                "team": team,
                "agent_key": f"{reason}:{sid[:8]}",
                "submission_id": sid,
                "match_id": match_id,
                "path": str(path),
                "reason": reason,
                "weight_group": weight_group,
                "match_rank": int(ranks[slot]),
            })
    selected = pd.DataFrame(rows)
    if selected.empty:
        raise RuntimeError("V12 selected_top_logs.csv is empty")
    selected.to_csv(log_dir / "selected_top_logs.csv", index=False)
    counts = selected.groupby("reason").size().to_dict()
    print("Selected elite-log rows:", counts, "unique_matches=", selected["match_id"].nunique())
    return log_dir


# Trích recurrent sequences từ elite logs, gán weight theo rank, phase và chất lượng action.
def extract_elite_sequences(log_dir=TOP_LOG_DIR, max_sequences=MAX_SEQUENCES):
    import pandas as pd

    selected = pd.read_csv(Path(log_dir) / "selected_top_logs.csv")
    sequences = []
    seen = set()
    action_histogram = Counter()
    invalid_actions = 0
    for _, row in selected.iterrows():
        path = Path(row["path"])
        sid = str(row["submission_id"])
        match_id = str(row["match_id"])
        key = (match_id, sid)
        if key in seen or not path.exists():
            continue
        seen.add(key)
        data = json.loads(path.read_text(encoding="utf-8"))
        ids = [str(value) for value in data["team_ids"]]
        if sid not in ids:
            continue
        slot = ids.index(sid)
        match_rank = int(data["ranks"][slot])
        survival = int(data["survival_steps"][slot])
        actual_rank = int(row.get("actual_rank", row.get("rank", 999)))
        reason = str(row.get("reason", ""))
        team_name = str(row.get("team", ""))
        outcome_weight = (2.05, 1.18, 0.55, 0.16)[min(match_rank, 3)]
        if reason == "top8_teacher":
            meta_weight = 1.70 if actual_rank <= 3 else 1.45
        elif reason == "counter_winner":
            meta_weight = 1.35
        elif reason == "own_best_winner":
            meta_weight = 1.10
        else:
            meta_weight = 0.90
        if "****_TNTT" in team_name:
            meta_weight *= 1.14
        elif "****" in team_name:
            meta_weight *= 1.10
        elif "****" in team_name or "****" in team_name or "****" in team_name:
            meta_weight *= 1.06
        tracker = BombTracker()
        trajectory = []
        history = data.get("history", [])
        for step in range(1, len(history)):
            previous = history[step - 1]
            current = history[step]
            obs = obs_from_log_frame(previous)
            radius_lookup = tracker.update(obs)
            actions = current.get("actions")
            if actions is None or not bool(previous["alive"][slot]):
                trajectory = []
                continue
            action = int(actions[slot])
            if not 0 <= action <= 5:
                trajectory = []
                continue
            features, scalars = encode_obs(obs, slot, step, radius_lookup=radius_lookup)
            mask = legal_action_mask(obs, slot, radius_lookup=radius_lookup, strict_safety=True)
            if not bool(mask[action]):
                invalid_actions += 1
                trajectory = []
                continue
            phase_weight = 1.0
            if step <= 90:
                phase_weight *= 1.10
            elif step >= 300:
                phase_weight *= 1.18
            if action == 5:
                phase_weight *= 1.10 if step < 300 else 1.18
            if action == 0 and np.any(mask[1:5]):
                # Keep top ****/**** tactical waits, but suppress passive waiting
                # from our rank-drop targets.
                phase_weight *= 0.60 if reason == "top8_teacher" else 0.32
            if match_rank >= 2 and step >= max(1, survival - 12):
                phase_weight *= 0.20
            weight = float(outcome_weight * meta_weight * phase_weight)
            trajectory.append((features, scalars, mask, action, weight))
            action_histogram[action] += 1
            if len(trajectory) >= SEQUENCE_LENGTH:
                start = len(trajectory) - SEQUENCE_LENGTH
                if start % SEQUENCE_STRIDE == 0:
                    chunk = trajectory[-SEQUENCE_LENGTH:]
                    sequences.append({
                        "maps": np.stack([item[0] for item in chunk]),
                        "scalars": np.stack([item[1] for item in chunk]),
                        "masks": np.stack([item[2] for item in chunk]),
                        "actions": np.asarray([item[3] for item in chunk], dtype=np.int64),
                        "weights": np.asarray([item[4] for item in chunk], dtype=np.float32),
                        "sid": sid,
                        "team": team_name,
                        "leaderboard_rank": actual_rank,
                        "match_id": match_id,
                        "reason": reason,
                    })
    if len(sequences) > max_sequences:
        rng = np.random.default_rng(SEED + 1212)
        quality = np.asarray([
            float(sequence["weights"].mean())
            * (1.0 + 0.06 * np.count_nonzero(sequence["actions"] == 5))
            for sequence in sequences
        ], dtype=np.float64)
        quality /= quality.sum()
        keep = rng.choice(len(sequences), size=max_sequences, replace=False, p=quality)
        sequences = [sequences[int(index)] for index in keep]
    reason_counts = Counter(sequence.get("reason", "") for sequence in sequences)
    print(
        "Elite sequence dataset:",
        f"sequences={len(sequences)}",
        f"frames={len(sequences) * SEQUENCE_LENGTH}",
        f"matches={len(seen)}",
        f"invalid_or_unsafe={invalid_actions}",
        f"actions={dict(sorted(action_histogram.items()))}",
        f"reasons={dict(reason_counts)}",
    )
    if len(sequences) < (180 if V12_CPU_MODE else 800):
        raise RuntimeError("Too few V12 recurrent sequences were extracted")
    return sequences


def hard_validation_score_v12(metrics):
    return (
        22.0 * metrics["sole_win_rate"]
        + 7.5 * metrics["rank_0_rate"]
        - 15.0 * metrics["average_rank"]
        - 28.0 * metrics["rank_3_rate"]
        - 18.0 * metrics["draw_rate"]
        + 0.0015 * metrics["mean_steps"]
        - 4.0 * max(0.0, metrics.get("bomb_rate", 0.0) - 0.185)
        - 4.0 * max(0.0, 0.110 - metrics.get("bomb_rate", 0.0))
        - 2.5 * max(0.0, metrics.get("wait_rate", 0.0) - 0.105)
    )


hard_validation_score = hard_validation_score_v12


def configure_v12():
    global CFG, CURRICULUM, PFSP_POOL
    CFG = V12_CFG
    CURRICULUM = V12_CURRICULUM
    PFSP_POOL = PFSPPool(snapshot_limit=CFG.league_max_size)
    print("Continuation mode:", "CPU light" if V12_CPU_MODE else "T4 full")
    print("Continuation config:", CFG)
    print("Continuation curriculum:", [stage.name for stage in CURRICULUM])
    return PFSP_POOL


# Chọn checkpoint cuối bằng hard validation thay vì lấy checkpoint mới nhất.
def select_best_checkpoint_v12(model, pool, matches=64):
    preferred = (
        sorted(CHECKPOINT_DIR.glob("*v12_resume_raw.pth"))
        + sorted(CHECKPOINT_DIR.glob("*v12_after_rankdrop_sequence_bc.pth"))
        + sorted(CHECKPOINT_DIR.glob("*v4_best.pth"))
        + sorted(CHECKPOINT_DIR.glob("*v4_initial.pth"))
    )
    recent = sorted(CHECKPOINT_DIR.glob("ppo_*.pth"))[-8:]
    paths = []
    for path in preferred + recent:
        if path not in paths:
            paths.append(path)
    candidates = []
    for path in paths:
        trial = ActorCritic().to(DEVICE)
        state = torch.load(path, map_location=DEVICE)
        trial.load_state_dict(state["model_state_dict"])
        metrics = evaluate_hard_policy(trial, pool, matches=matches, seed=4_200_000)
        metrics["checkpoint"] = str(path)
        metrics["candidate_score"] = hard_validation_score_v12(metrics)
        if "resume_raw" in path.name:
            metrics["candidate_score"] += 0.25
        if "after_rankdrop_sequence_bc" in path.name:
            metrics["candidate_score"] += 0.15
        candidates.append((metrics["candidate_score"], path, metrics))
        del trial
        torch.cuda.empty_cache()
    current_metrics = evaluate_hard_policy(model, pool, matches=matches, seed=4_200_000)
    current_metrics["checkpoint"] = "current"
    current_metrics["candidate_score"] = hard_validation_score_v12(current_metrics)
    candidates.append((current_metrics["candidate_score"], None, current_metrics))
    candidates.sort(key=lambda item: item[0], reverse=True)
    _, best_path, best_metrics = candidates[0]
    if best_path is not None:
        state = torch.load(best_path, map_location=DEVICE)
        model.load_state_dict(state["model_state_dict"])
    report = {
        "selected": best_metrics,
        "candidates": [item[2] for item in candidates],
        "strategy": "V11 continuation + rank-drop/top8 sequence BC + conservative PFSP",
    }
    (ARTIFACT_DIR / "checkpoint_selection_v12.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
    print("Selected checkpoint:", json.dumps(best_metrics, indent=2))
    return best_metrics, best_path


print("V12 top8:", V12_TOP8_IDS)
print("V12 targets:", V12_TARGET_IDS)


## 9. Conservative Continuation

Đây là giai đoạn fine-tune từ checkpoint mạnh. Notebook không train quá
mạnh tay vì mục tiêu là cải thiện meta mà không làm mất độ ổn định.

Quy trình:

1. load resume policy;
2. lưu `resume_raw` để fallback;
3. sequence BC từ elite logs;
4. build elite clones và cache replay;
5. chạy PFSP PPO ngắn;
6. chọn checkpoint bằng hard validation.


In [ ]:
# Cell 9: Conservative continuation từ checkpoint mạnh.
# Mục tiêu:
# - Resume một policy đã ổn định.
# - Fine-tune bằng elite sequence BC + PFSP PPO ngắn.
# - Giữ learning rate nhỏ để tránh policy drift và regression.
# - Lưu resume_raw để có fallback nếu challenger yếu hơn.

# Cell 9: Conservative continuation từ checkpoint mạnh.
model = ActorCritic().to(DEVICE)
resume_update, resume_path, fresh_start = load_resume_or_fresh(model)
if fresh_start:
    raise RuntimeError(
        "V12 is a continuation run. Attach v11_continuation_bundle.zip, "
        "or a compatible submission.zip / policy.pt from the V11 run."
    )

pool = configure_v12()
optimizer = optim.Adam(model.parameters(), lr=CFG.learning_rate, eps=1e-5)
save_checkpoint(model, optimizer, int(resume_update), "v12_resume_raw")

rankdrop_log_dir = fetch_rankdrop_logs_v12()
rankdrop_sequences = extract_elite_sequences(rankdrop_log_dir)

train_sequence_bc(
    model,
    rankdrop_sequences,
    epochs=SEQUENCE_BC_EPOCHS_V12,
    learning_rate=1.8e-5 if V12_CPU_MODE else 2.4e-5,
    label="elite_log_continuation",
)
save_checkpoint(
    model,
    optim.Adam(model.parameters(), lr=CFG.learning_rate, eps=1e-5),
    int(resume_update),
    "v12_after_rankdrop_sequence_bc",
)

build_elite_clones(model, rankdrop_sequences, pool)
cache_elite_replay(model, rankdrop_sequences)

optimizer = optim.Adam(model.parameters(), lr=CFG.learning_rate, eps=1e-5)
history_v12, best_v12_path = train_curriculum_v4(
    model,
    optimizer,
    pool,
    start_update=int(resume_update),
)
print("Completed PFSP updates:", len(history_v12))
print("Best PFSP checkpoint:", best_v12_path)


## 10. Checkpoint Selection

Notebook đánh giá nhiều candidates:

- resume raw;
- sequence-BC checkpoint;
- PFSP best checkpoint;
- recent PPO checkpoints.

Score ưu tiên `sole_win_rate`, `rank_0_rate`, `average_rank` thấp, đồng
thời phạt `rank_3_rate`, draw, bomb rate quá lệch và wait rate cao.


In [ ]:
# Cell 10: Checkpoint selection và export final submission.
# Mục tiêu:
# - So sánh resume, sequence-BC checkpoint, best PFSP checkpoint và recent PPO checkpoints.
# - Chấm bằng hard validation score: sole win, rank0, avg rank, rank3, draw,
#   bomb rate và wait rate.
# - Export submission.zip từ checkpoint tốt nhất, không nhất thiết là checkpoint cuối.

# Cell 10: Checkpoint selection và export submission.zip.
selected_v12, selected_v12_path = select_best_checkpoint_v12(
    model,
    PFSP_POOL,
    matches=32 if V12_CPU_MODE else 72,
)
evaluation_v12 = evaluate_hard_policy(
    model,
    PFSP_POOL,
    matches=FINAL_EVAL_MATCHES,
    seed=4_400_000,
)
(ARTIFACT_DIR / "evaluation_v12.json").write_text(
    json.dumps(evaluation_v12, indent=2), encoding="utf-8"
)
submission_zip = export_submission(model)

continuation_path = ARTIFACT_DIR / "resume_v12_best.pth"
selected_update = int(resume_update)
checkpoint_name = str(selected_v12.get("checkpoint", ""))
if "ppo_" in checkpoint_name:
    try:
        selected_update = int(checkpoint_name.split("ppo_")[-1].split("_")[0])
    except Exception:
        selected_update = int(resume_update)
torch.save(
    {
        "model_state_dict": model.state_dict(),
        "global_update": selected_update,
        "stage_name": "v12_selected",
        "config": asdict(CFG),
    },
    continuation_path,
)
with zipfile.ZipFile(
    ARTIFACT_DIR / "v12_continuation_bundle.zip",
    "w",
    compression=zipfile.ZIP_DEFLATED,
) as archive:
    archive.write(continuation_path, arcname="resume_v12_best.pth")
    archive.write(DEPLOY_DIR / "policy.pt", arcname="policy.pt")
    archive.write(DEPLOY_DIR / "agent.py", arcname="agent.py")

print("\nMain submission:")
print(submission_zip)
print("Continuation input:")
print(ARTIFACT_DIR / "v12_continuation_bundle.zip")


## 11. Optional Runtime Smoke Test

Cell này dùng official runtime guard nếu package evaluation có sẵn.
Mục tiêu là kiểm tra `Agent.act(obs)` chạy dưới time budget và
submission package không thiếu file.


In [ ]:
# Cell 11: Optional runtime smoke test.
# Mục tiêu:
# - Dùng official runtime_guard nếu package evaluation có sẵn.
# - Kiểm tra Agent.act dưới time budget.
# - Chạy vài local matches để phát hiện lỗi packaging trước khi submit.

# Cell 11: Optional official runtime smoke test.
try:
    from competition.evaluation.runtime_guard import runtime_precheck
    from scripts.participant.run_local_match import run_match
except ModuleNotFoundError as exc:
    print("Optional smoke test skipped:", exc)
else:
    print("Official runtime precheck:")
    precheck_results = []
    for attempt in range(10):
        ok, note = runtime_precheck(
            str(DEPLOY_DIR / "agent.py"),
            timeout_s=0.1,
            startup_timeout_s=20.0,
        )
        precheck_results.append(ok)
        print(f"  {attempt + 1:02d}: {ok} | {note}")
    assert all(precheck_results), "Do not submit: official runtime precheck failed."
    run_match(
        [str(DEPLOY_DIR), "TacticalRuleAgent", "GeniusRuleAgent", "SmarterRuleAgent"],
        num_episodes=3,
        max_steps=500,
        seed=12345,
    )


## Kaggle Inputs Và Outputs

Input thường dùng:

- official participant kit hoặc competition repository;
- elite log archive cho imitation;
- resume bundle hoặc `submission.zip` từ một run mạnh trước đó.

Output chính:

```text
/kaggle/working/bomberland_artifacts/submission.zip
```

File zip nộp chỉ cần:

```text
agent.py
policy.pt
```
